# SECOM 반도체 공정 데이터 분석

## 0. 프로젝트 개요

이 노트북은 UCI SECOM 반도체 공정 센서 데이터를 사용해 제품의 정상(Pass)과 불량(Fail)을 판단하는 분석 흐름을 단계적으로 구현한다.

가장 먼저 할 일은 모델을 만드는 것이 아니라 데이터를 안정적으로 내려받고 읽는 것이다. 원본 센서 데이터, 원본 라벨, 시간 정보를 분리해서 보관해야 이후 결측값 분석, 클래스 불균형 분석, 전처리, 모델링에서 데이터 누수를 피할 수 있다.

이번 1차 구현에서 확인할 내용은 다음과 같다.

- SECOM 원본 데이터 자동 다운로드
- 센서 Feature 데이터와 라벨 데이터 불러오기
- 원본 라벨 `-1`, `1`을 분석용 라벨 `0`, `1`로 변환
- 날짜/시간 정보는 삭제하지 않고 별도 변수로 보관
- 데이터 Shape, 클래스 분포, 결측값 개수 확인


## 1. 라이브러리 Import

데이터를 읽고 기본 구조를 확인하기 위한 최소 라이브러리만 먼저 불러온다. 모델링 관련 라이브러리는 Train/Test 분리와 전처리 Pipeline을 만들 때 추가한다.

In [ ]:
from pathlib import Path
import random
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Seed와 경로 설정

반복 실행해도 같은 결과를 얻을 수 있도록 Seed를 고정한다. 원본 데이터는 `data/raw`에 저장하고, 분석 결과물은 이후 `outputs` 아래에 저장한다.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_PATH = RAW_DATA_DIR / "secom.data"
LABEL_PATH = RAW_DATA_DIR / "secom_labels.data"

FEATURE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/secom/secom.data"
LABEL_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/secom/secom_labels.data"

MISSING_THRESHOLD = 0.5

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")


## 3. SECOM 데이터 다운로드

Colab에서 새 런타임을 시작해도 자동으로 원본 데이터를 내려받을 수 있게 한다. 파일이 이미 존재하고 크기가 0보다 크면 다시 다운로드하지 않는다.

In [ ]:
def download_if_missing(url: str, destination: Path) -> None:
    """Download a file only when it is missing or empty."""
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Already exists: {destination.name} ({destination.stat().st_size:,} bytes)")
        return

    print(f"Downloading {destination.name}...")
    urllib.request.urlretrieve(url, destination)
    print(f"Saved: {destination} ({destination.stat().st_size:,} bytes)")


download_if_missing(FEATURE_URL, FEATURE_PATH)
download_if_missing(LABEL_URL, LABEL_PATH)


## 4. 데이터와 라벨 불러오기

`secom.data`는 센서 Feature이고, `secom_labels.data`는 원본 라벨과 측정 시간이다. 파일을 읽을 때는 `sep=r"\s+"`를 사용한다.

원본 라벨은 `-1 = 정상`, `1 = 불량`이다. 분석에서는 `0 = Pass`, `1 = Fail`로 변환한다. 날짜/시간은 초기 모델 입력에서는 제외하지만, 공정 드리프트 분석에 사용할 수 있도록 `timestamps`로 따로 보관한다.

In [ ]:
def load_secom_data(feature_path: Path, label_path: Path) -> tuple[pd.DataFrame, pd.Series, pd.Series, pd.DataFrame]:
    """Load SECOM features, converted labels, timestamps, and raw label data."""
    features = pd.read_csv(
        feature_path,
        sep=r"\s+",
        header=None,
        na_values="NaN",
    )
    features.columns = [f"feature_{idx:03d}" for idx in range(features.shape[1])]

    raw_labels = pd.read_csv(
        label_path,
        sep=r"\s+",
        header=None,
        engine="python",
    )

    raw_label = raw_labels.iloc[:, 0].astype(int)
    labels = raw_label.replace({-1: 0, 1: 1}).rename("target")
    timestamps = raw_labels.iloc[:, 1:].astype(str).agg(" ".join, axis=1).str.replace('"', '', regex=False)
    timestamps = pd.to_datetime(timestamps, errors="coerce", dayfirst=True).rename("timestamp")

    if len(features) != len(labels):
        raise ValueError(f"Feature rows ({len(features)}) and label rows ({len(labels)}) do not match.")

    return features, labels, timestamps, raw_labels


X, y, timestamps, raw_labels = load_secom_data(FEATURE_PATH, LABEL_PATH)

display(X.head())
display(pd.DataFrame({"target": y.head(), "timestamp": timestamps.head()}))


## 5. 데이터 구조 확인

모델링 전에 전체 행 수, Feature 개수, 라벨 개수, 날짜 범위, 중복 행, 무한대 값, 결측값을 먼저 확인한다. 이 단계에서 발견한 문제를 기준으로 다음 전처리 전략을 정한다.

In [ ]:
summary = {
    "n_samples": X.shape[0],
    "n_features": X.shape[1],
    "n_labels": y.shape[0],
    "date_min": timestamps.min(),
    "date_max": timestamps.max(),
    "duplicated_rows": int(X.duplicated().sum()),
    "infinite_values": int(np.isinf(X.to_numpy(dtype=float)).sum()),
    "missing_values": int(X.isna().sum().sum()),
}

pd.Series(summary, name="value")


## 6. 정상·불량 클래스 분포

SECOM 데이터는 정상(Pass)이 매우 많고 불량(Fail)이 적은 불균형 데이터다. Accuracy만으로 모델을 평가하면 불량 탐지 성능을 잘못 해석할 수 있으므로, 먼저 클래스 분포를 수치로 확인한다.

In [ ]:
class_distribution = (
    y.map({0: "Pass", 1: "Fail"})
    .value_counts()
    .rename_axis("class")
    .reset_index(name="count")
)
class_distribution["ratio"] = class_distribution["count"] / class_distribution["count"].sum()

display(class_distribution)
print("Raw label counts:")
display(raw_labels.iloc[:, 0].value_counts().sort_index())


## 7. 결측값 1차 확인

결측값은 전체 데이터에서 바로 채우지 않는다. 지금은 분포만 확인하고, 실제 대체 기준은 Train/Test 분리 후 Train 데이터에서만 학습한다.

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": X.isna().sum(),
    "missing_ratio": X.isna().mean(),
}).sort_values("missing_ratio", ascending=False)

high_missing_features = missing_summary[missing_summary["missing_ratio"] >= MISSING_THRESHOLD]

print(f"Total missing values: {int(X.isna().sum().sum()):,}")
print(f"Features with missing ratio >= {MISSING_THRESHOLD:.0%}: {len(high_missing_features)}")
display(missing_summary.head(20))


## 8. 기초 시각화

수치 요약만 보면 클래스 불균형과 결측값 패턴을 직관적으로 파악하기 어렵다. 여기서는 Pass/Fail 분포, 전체 Feature 결측률 분포, 결측률 상위 20개 Feature를 시각화한다.

이 그래프들은 모델 성능을 평가하기 전에 데이터 자체의 위험 요소를 확인하기 위한 것이다. 특히 Fail 샘플이 적기 때문에 이후 모델 평가는 Accuracy보다 Fail Recall, Fail Precision, Fail F1-score, PR-AUC를 중심으로 봐야 한다.


In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
class_plot_data = class_distribution.copy()
class_plot_data["ratio_pct"] = class_plot_data["ratio"] * 100

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(
    data=class_plot_data,
    x="class",
    y="count",
    hue="class",
    palette={"Pass": "#4C78A8", "Fail": "#E45756"},
    dodge=False,
    legend=False,
    ax=ax,
)

for idx, row in class_plot_data.reset_index(drop=True).iterrows():
    ax.text(
        idx,
        row["count"],
        f"{row['count']:,}\n({row['ratio_pct']:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=10,
    )

ax.set_title("SECOM Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Sample count")
ax.set_ylim(0, class_plot_data["count"].max() * 1.12)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(missing_summary["missing_ratio"] * 100, bins=30, color="#59A14F", ax=ax)
ax.axvline(MISSING_THRESHOLD * 100, color="#E45756", linestyle="--", label=f"Threshold: {MISSING_THRESHOLD:.0%}")
ax.set_title("Missing Ratio Distribution by Feature")
ax.set_xlabel("Missing ratio (%)")
ax.set_ylabel("Feature count")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "missing_ratio_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
top_missing = missing_summary.head(20).sort_values("missing_ratio")

fig, ax = plt.subplots(figsize=(8, 7))
sns.barplot(
    data=top_missing.reset_index(names="feature"),
    x="missing_ratio",
    y="feature",
    color="#F28E2B",
    ax=ax,
)
ax.axvline(MISSING_THRESHOLD, color="#E45756", linestyle="--", label=f"Threshold: {MISSING_THRESHOLD:.0%}")
ax.set_title("Top 20 Features by Missing Ratio")
ax.set_xlabel("Missing ratio")
ax.set_ylabel("Feature")
ax.xaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "top20_missing_features.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. 상수·저분산 Feature 분석

모델링 전에 모든 값이 같거나 거의 변하지 않는 Feature를 확인한다. 이런 Feature는 분류에 기여하지 못하거나 노이즈를 키울 수 있다.

여기서는 제거 후보만 정리한다. 실제 제거는 이후 Train/Test 분리 후 Pipeline 안에서 Train 데이터 기준으로 학습되도록 구성해야 데이터 누수를 피할 수 있다.


In [ ]:
from sklearn.feature_selection import VarianceThreshold

LOW_VARIANCE_THRESHOLD = 1e-8

n_unique = X.nunique(dropna=False)
constant_features = n_unique[n_unique <= 1].index.tolist()

X_for_variance = X.copy()
X_for_variance = X_for_variance.fillna(X_for_variance.median(numeric_only=True)).fillna(0)

zero_variance_selector = VarianceThreshold(threshold=0.0)
zero_variance_selector.fit(X_for_variance)
zero_variance_features = X.columns[~zero_variance_selector.get_support()].tolist()

feature_variance = X_for_variance.var().sort_values()
low_variance_features = feature_variance[feature_variance <= LOW_VARIANCE_THRESHOLD].index.tolist()

feature_screening_summary = pd.DataFrame(
    {
        "check": [
            "original_features",
            "constant_features",
            "zero_variance_features",
            "low_variance_features",
            "features_after_high_missing_filter",
        ],
        "count": [
            X.shape[1],
            len(constant_features),
            len(zero_variance_features),
            len(low_variance_features),
            X.shape[1] - len(high_missing_features),
        ],
    }
)

display(feature_screening_summary)
display(feature_variance.head(20).to_frame("variance"))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(np.log10(feature_variance + 1e-12), bins=40, color="#76B7B2", ax=ax)
ax.axvline(np.log10(LOW_VARIANCE_THRESHOLD + 1e-12), color="#E45756", linestyle="--", label="Low variance threshold")
ax.set_title("Feature Variance Distribution")
ax.set_xlabel("log10(variance + 1e-12)")
ax.set_ylabel("Feature count")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "feature_variance_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


## 10. 센서 Feature 분포 시각화

결측률이 낮고 분산이 있는 Feature를 골라 전체 분포를 먼저 확인한다. 센서 Feature는 익명화되어 있으므로 실제 공정 장비나 센서 이름을 임의로 붙이지 않는다.

히스토그램은 이상치, 치우침, 분포 폭을 빠르게 확인하기 위한 것이다.


In [ ]:
candidate_features = (
    missing_summary[missing_summary["missing_ratio"] < MISSING_THRESHOLD]
    .index.difference(constant_features)
)
selected_features = (
    feature_variance.loc[candidate_features]
    .sort_values(ascending=False)
    .head(6)
    .index.tolist()
)

print("Selected features for distribution plots:")
display(pd.Series(selected_features, name="feature"))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()

for ax, feature in zip(axes, selected_features):
    sns.histplot(X[feature], bins=40, kde=False, color="#4C78A8", ax=ax)
    ax.set_title(feature)
    ax.set_xlabel("Sensor value")
    ax.set_ylabel("Count")

for ax in axes[len(selected_features):]:
    ax.set_visible(False)

fig.suptitle("Selected Sensor Feature Distributions", y=1.02)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "selected_feature_histograms.png", dpi=150, bbox_inches="tight")
plt.show()


## 11. 정상·불량별 Feature 비교

Pass와 Fail의 샘플 수가 크게 다르기 때문에 중첩 히스토그램은 `density=True`로 비교한다. 개수 차이가 아니라 분포 모양 차이를 보는 것이 목적이다.

이 그래프는 모델의 판단 후보를 탐색하는 용도이며, 특정 Feature가 실제 불량 원인이라고 단정하지 않는다.


In [ ]:
plot_data = X[selected_features].copy()
plot_data["target_name"] = y.map({0: "Pass", 1: "Fail"})

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()

for ax, feature in zip(axes, selected_features):
    sns.histplot(
        data=plot_data,
        x=feature,
        hue="target_name",
        stat="density",
        common_norm=False,
        bins=40,
        element="step",
        palette={"Pass": "#4C78A8", "Fail": "#E45756"},
        ax=ax,
    )
    ax.set_title(feature)
    ax.set_xlabel("Sensor value")
    ax.set_ylabel("Density")

for ax in axes[len(selected_features):]:
    ax.set_visible(False)

fig.suptitle("Pass vs Fail Feature Distributions", y=1.02)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "pass_fail_feature_histograms.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()

for ax, feature in zip(axes, selected_features):
    sns.boxplot(
        data=plot_data,
        x="target_name",
        y=feature,
        hue="target_name",
        palette={"Pass": "#4C78A8", "Fail": "#E45756"},
        legend=False,
        showfliers=False,
        ax=ax,
    )
    ax.set_title(feature)
    ax.set_xlabel("Class")
    ax.set_ylabel("Sensor value")

for ax in axes[len(selected_features):]:
    ax.set_visible(False)

fig.suptitle("Pass vs Fail Feature Boxplots", y=1.02)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "pass_fail_feature_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()


## 12. Feature 간 상관관계 분석

590개 Feature 전체를 한 번에 히트맵으로 그리면 해석이 어렵다. 먼저 상관계수 절댓값이 높은 Feature 쌍을 표로 확인하고, 그중 일부 Feature만 선택해 히트맵으로 시각화한다.

상관관계가 높다는 것은 두 Feature가 함께 움직인다는 뜻이지, 하나가 다른 하나의 원인이라는 뜻은 아니다.


In [ ]:
corr_features = (
    missing_summary[missing_summary["missing_ratio"] < MISSING_THRESHOLD]
    .index.difference(constant_features)
)
corr_matrix = X[corr_features].corr().abs()
upper_mask = np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)
upper_corr = corr_matrix.where(upper_mask)

high_corr_pairs = (
    upper_corr.stack()
    .sort_values(ascending=False)
    .reset_index()
)
high_corr_pairs.columns = ["feature_1", "feature_2", "abs_correlation"]

display(high_corr_pairs.head(20))


In [ ]:
heatmap_features = pd.unique(high_corr_pairs[["feature_1", "feature_2"]].head(20).values.ravel()).tolist()
heatmap_features = heatmap_features[:20]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    X[heatmap_features].corr(),
    cmap="vlag",
    center=0,
    square=True,
    linewidths=0.3,
    cbar_kws={"label": "Correlation"},
    ax=ax,
)
ax.set_title("Correlation Heatmap of Selected Features")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "selected_feature_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


## 13. PCA 시각화

고차원 센서 데이터를 2차원으로 축소해 Pass와 Fail이 어느 정도 분리되는지 확인한다. PCA는 시각화용 탐색이며, 여기서 보이는 분리가 곧 최종 모델 성능을 의미하지는 않는다.

이 셀도 아직 최종 전처리 Pipeline이 아니다. 모델 학습 단계에서는 Train/Test 분리 후 Train 데이터 기준으로 결측값 대체와 표준화를 학습해야 한다.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

pca_features = corr_features.tolist()
pca_input = X[pca_features]

pca_imputer = SimpleImputer(strategy="median")
pca_scaler = StandardScaler()
pca_model = PCA(n_components=2, random_state=SEED)

pca_values = pca_model.fit_transform(
    pca_scaler.fit_transform(
        pca_imputer.fit_transform(pca_input)
    )
)

pca_df = pd.DataFrame(pca_values, columns=["PC1", "PC2"])
pca_df["target_name"] = y.map({0: "Pass", 1: "Fail"})

explained = pca_model.explained_variance_ratio_
print(f"PC1 explained variance ratio: {explained[0]:.4f}")
print(f"PC2 explained variance ratio: {explained[1]:.4f}")
print(f"Cumulative explained variance ratio: {explained.sum():.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="target_name",
    palette={"Pass": "#4C78A8", "Fail": "#E45756"},
    alpha=0.75,
    s=45,
    ax=ax,
)
ax.set_title("PCA Projection of SECOM Sensor Data")
ax.set_xlabel(f"PC1 ({explained[0]:.1%})")
ax.set_ylabel(f"PC2 ({explained[1]:.1%})")
ax.legend(title="Class")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "pca_projection.png", dpi=150, bbox_inches="tight")
plt.show()


## 14. Train/Test 분리

모델 평가를 위해 원본 Feature `X`와 라벨 `y`를 Train/Test로 먼저 나눈다. 이후 결측값 대체, 표준화, 저분산 Feature 제거는 Train 데이터에서만 학습하고 Test 데이터에는 `transform`만 적용한다.

SECOM 데이터는 Fail 샘플이 적으므로 `stratify=y`를 사용해 Train과 Test의 Pass/Fail 비율을 유지한다.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "dataset": ["train", "test"],
        "n_samples": [len(y_train), len(y_test)],
        "pass_count": [(y_train == 0).sum(), (y_test == 0).sum()],
        "fail_count": [(y_train == 1).sum(), (y_test == 1).sum()],
        "fail_ratio": [(y_train == 1).mean(), (y_test == 1).mean()],
    }
)

display(split_summary)


## 15. 전처리 Pipeline 구성

데이터 누수를 막기 위해 전처리 단계를 `sklearn.pipeline.Pipeline` 안에 넣는다. 이 구조에서는 Train 데이터로만 제거 기준, 결측값 중앙값, 스케일링 기준을 학습한다.

기본 전처리 순서는 다음과 같다.

1. 결측률이 높은 Feature 제거
2. 결측값 중앙값 대체
3. 상수·저분산 Feature 제거
4. 필요한 모델에 표준화 적용
5. 분류 모델 학습


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


class HighMissingFeatureDropper(BaseEstimator, TransformerMixin):
    """Drop columns whose missing ratio is greater than or equal to a threshold."""

    def __init__(self, threshold: float = 0.5):
        self.threshold = threshold

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X).copy()
        self.feature_names_in_ = X_df.columns.to_numpy()
        self.keep_columns_ = X_df.columns[X_df.isna().mean() < self.threshold].to_list()
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X, columns=self.feature_names_in_)
        return X_df.loc[:, self.keep_columns_]


def make_linear_pipeline(class_weight=None) -> Pipeline:
    """Create a leakage-safe preprocessing and Logistic Regression pipeline."""
    return Pipeline(
        steps=[
            ("drop_high_missing", HighMissingFeatureDropper(threshold=MISSING_THRESHOLD)),
            ("imputer", SimpleImputer(strategy="median")),
            ("variance", VarianceThreshold(threshold=LOW_VARIANCE_THRESHOLD)),
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    max_iter=3000,
                    class_weight=class_weight,
                    random_state=SEED,
                ),
            ),
        ]
    )


def make_tree_pipeline(class_weight=None) -> Pipeline:
    """Create a leakage-safe preprocessing and Random Forest pipeline."""
    return Pipeline(
        steps=[
            ("drop_high_missing", HighMissingFeatureDropper(threshold=MISSING_THRESHOLD)),
            ("imputer", SimpleImputer(strategy="median")),
            ("variance", VarianceThreshold(threshold=LOW_VARIANCE_THRESHOLD)),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=None,
                    min_samples_split=2,
                    min_samples_leaf=1,
                    class_weight=class_weight,
                    random_state=SEED,
                    n_jobs=-1,
                ),
            ),
        ]
    )


## 16. Dummy Classifier

Dummy Classifier는 다수 클래스만 예측하는 기준 모델이다. SECOM 데이터처럼 정상 데이터가 많은 경우 Accuracy가 높아 보여도 Fail을 거의 찾지 못할 수 있음을 확인하는 기준선으로 사용한다.


In [ ]:
dummy_pipeline = Pipeline(
    steps=[
        ("drop_high_missing", HighMissingFeatureDropper(threshold=MISSING_THRESHOLD)),
        ("imputer", SimpleImputer(strategy="median")),
        ("variance", VarianceThreshold(threshold=LOW_VARIANCE_THRESHOLD)),
        ("model", DummyClassifier(strategy="most_frequent", random_state=SEED)),
    ]
)


## 17. Logistic Regression

Logistic Regression은 선형 기준 모델이다. 표준화가 필요하므로 `StandardScaler`를 Pipeline에 포함한다.

기본 모델과 `class_weight="balanced"` 모델을 함께 비교해 클래스 불균형 보정이 Fail 탐지 성능에 어떤 영향을 주는지 확인한다.


In [ ]:
logistic_pipeline = make_linear_pipeline(class_weight=None)
balanced_logistic_pipeline = make_linear_pipeline(class_weight="balanced")


## 18. Random Forest

Random Forest는 비선형 관계와 Feature 상호작용을 포착할 수 있는 기준 모델이다. 표준화가 필수는 아니므로 Scaler 없이 별도 Pipeline을 사용한다.

기본 모델과 `class_weight="balanced"` 모델을 비교한다.


In [ ]:
random_forest_pipeline = make_tree_pipeline(class_weight=None)
balanced_random_forest_pipeline = make_tree_pipeline(class_weight="balanced")


## 19. 모델 성능 비교

모든 모델을 같은 Test 데이터에서 평가한다. 최종 모델은 Accuracy만으로 선택하지 않고 Fail Recall, Fail F1-score, PR-AUC, False Negative 수를 함께 확인한다.

이 단계에서는 기본 Threshold 0.5를 사용한다. Threshold 조정은 이후 Validation 또는 Cross Validation 예측 확률을 사용해 별도 단계에서 수행한다.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)


def get_positive_proba(model, X_data) -> np.ndarray:
    """Return predicted probability for the Fail class when available."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_data)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_data)
        score_range = scores.max() - scores.min()
        if score_range == 0:
            return np.full(shape=scores.shape, fill_value=0.5, dtype=float)
        return (scores - scores.min()) / score_range
    raise AttributeError("Model does not expose predict_proba or decision_function.")


def evaluate_classifier(name: str, imbalance_method: str, model, X_eval, y_eval, threshold: float = 0.5) -> dict:
    """Evaluate a fitted classifier with metrics focused on Fail detection."""
    y_proba = get_positive_proba(model, X_eval)
    y_pred = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    return {
        "model": name,
        "imbalance_method": imbalance_method,
        "threshold": threshold,
        "accuracy": accuracy_score(y_eval, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_eval, y_pred),
        "fail_precision": precision_score(y_eval, y_pred, pos_label=1, zero_division=0),
        "fail_recall": recall_score(y_eval, y_pred, pos_label=1, zero_division=0),
        "fail_f1": f1_score(y_eval, y_pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc_score(y_eval, y_proba),
        "pr_auc": average_precision_score(y_eval, y_proba),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


In [ ]:
models = {
    "Dummy Most Frequent": ("none", dummy_pipeline),
    "Logistic Regression": ("none", logistic_pipeline),
    "Logistic Regression Balanced": ("class_weight", balanced_logistic_pipeline),
    "Random Forest": ("none", random_forest_pipeline),
    "Random Forest Balanced": ("class_weight", balanced_random_forest_pipeline),
}

evaluation_rows = []
fitted_models = {}

for model_name, (imbalance_method, model) in models.items():
    print(f"Training: {model_name}")
    model.fit(X_train, y_train)
    fitted_models[model_name] = model
    evaluation_rows.append(
        evaluate_classifier(
            name=model_name,
            imbalance_method=imbalance_method,
            model=model,
            X_eval=X_test,
            y_eval=y_test,
            threshold=0.5,
        )
    )

results_df = (
    pd.DataFrame(evaluation_rows)
    .sort_values(["fail_recall", "fail_f1", "pr_auc"], ascending=False)
    .reset_index(drop=True)
)

display(results_df)


## 20. 혼동행렬

혼동행렬은 실제 Fail을 Pass로 놓친 False Negative 수를 직접 확인하기 위한 핵심 표다. 이 프로젝트에서는 실제 불량을 정상으로 놓치는 경우를 특히 중요하게 본다.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

best_model_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]
best_proba = get_positive_proba(best_model, X_test)
best_pred = (best_proba >= 0.5).astype(int)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_pred,
    display_labels=["Pass", "Fail"],
    cmap="Blues",
    values_format="d",
    ax=ax,
)
ax.set_title(f"Confusion Matrix: {best_model_name}")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "confusion_matrix_best_model.png", dpi=150, bbox_inches="tight")
plt.show()


## 21. ROC Curve와 Precision-Recall Curve

ROC Curve와 Precision-Recall Curve는 Threshold 변화에 따른 모델 성능을 확인하는 그래프다. 클래스 불균형이 큰 SECOM 데이터에서는 PR Curve와 PR-AUC가 특히 중요하다.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for model_name, model in fitted_models.items():
    y_proba = get_positive_proba(model, X_test)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f"{model_name} (AUC={roc_auc:.3f})")

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
ax.set_title("ROC Curve")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "roc_curve_models.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for model_name, model in fitted_models.items():
    y_proba = get_positive_proba(model, X_test)
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    ax.plot(recall, precision, label=f"{model_name} (AP={pr_auc:.3f})")

baseline = y_test.mean()
ax.axhline(baseline, linestyle="--", color="gray", label=f"Fail ratio={baseline:.3f}")
ax.set_title("Precision-Recall Curve")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend(loc="best", fontsize=8)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "precision_recall_curve_models.png", dpi=150, bbox_inches="tight")
plt.show()


## 22. Threshold 분석

기본 Threshold 0.5만 사용하면 불량 탐지 관점에서 적절하지 않을 수 있다. Threshold를 낮추면 Fail Recall이 증가할 수 있지만 False Positive도 늘어날 수 있다.

Threshold는 Test 데이터를 보고 고르지 않는다. 여기서는 Train 데이터 안에서 다시 Train/Validation을 나누고, Validation 성능으로 Threshold를 선택한 뒤 Test 데이터에서는 마지막 평가만 수행한다.


In [ ]:
from sklearn.base import clone

X_train_inner, X_valid, y_train_inner, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train,
)

threshold_candidate_name = "Logistic Regression Balanced"
threshold_candidate_model = clone(models[threshold_candidate_name][1])
threshold_candidate_model.fit(X_train_inner, y_train_inner)
valid_proba = get_positive_proba(threshold_candidate_model, X_valid)

threshold_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
threshold_rows = []

for threshold in threshold_values:
    valid_pred = (valid_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, valid_pred, labels=[0, 1]).ravel()
    threshold_rows.append(
        {
            "threshold": threshold,
            "fail_precision": precision_score(y_valid, valid_pred, pos_label=1, zero_division=0),
            "fail_recall": recall_score(y_valid, valid_pred, pos_label=1, zero_division=0),
            "fail_f1": f1_score(y_valid, valid_pred, pos_label=1, zero_division=0),
            "false_positive": fp,
            "false_negative": fn,
            "true_positive": tp,
            "true_negative": tn,
        }
    )

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)


In [ ]:
selected_threshold_row = (
    threshold_df.sort_values(
        ["fail_f1", "fail_recall", "false_negative"],
        ascending=[False, False, True],
    )
    .iloc[0]
)
selected_threshold = float(selected_threshold_row["threshold"])

print(f"Selected threshold from validation data: {selected_threshold:.2f}")
display(selected_threshold_row.to_frame("value"))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(threshold_df["threshold"], threshold_df["fail_precision"], marker="o", label="Fail Precision")
ax.plot(threshold_df["threshold"], threshold_df["fail_recall"], marker="o", label="Fail Recall")
ax.plot(threshold_df["threshold"], threshold_df["fail_f1"], marker="o", label="Fail F1-score")
ax.axvline(selected_threshold, color="#E45756", linestyle="--", label=f"Selected: {selected_threshold:.2f}")
ax.set_title("Validation Metrics by Decision Threshold")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "threshold_metrics_validation.png", dpi=150, bbox_inches="tight")
plt.show()


## 23. 최종 후보 모델 평가

Validation 데이터에서 선택한 Threshold를 사용해 Test 데이터에서 마지막 평가를 수행한다. 이 결과는 최종 보고서에서 가장 중요한 성능 근거가 된다.


In [ ]:
final_model_name = threshold_candidate_name
final_model = clone(models[final_model_name][1])
final_model.fit(X_train, y_train)

final_test_result = evaluate_classifier(
    name=final_model_name,
    imbalance_method=models[final_model_name][0],
    model=final_model,
    X_eval=X_test,
    y_eval=y_test,
    threshold=selected_threshold,
)

final_test_result_df = pd.DataFrame([final_test_result])
display(final_test_result_df)


## 24. Feature Importance

모델이 어떤 Feature를 중요하게 사용했는지 확인한다. SECOM Feature는 익명화되어 있으므로 실제 센서명이나 공정 원인을 임의로 붙이지 않는다.

Feature Importance는 모델 예측 기여도이지 인과관계가 아니다. 따라서 “Feature 59가 불량 예측에 높은 기여도를 보였다”처럼 표현하고, 실제 원인이라고 단정하지 않는다.


In [ ]:
from sklearn.inspection import permutation_importance


def get_selected_feature_names(fitted_pipeline: Pipeline, original_columns: pd.Index) -> list[str]:
    """Return feature names that remain after high-missing and variance filters."""
    dropper = fitted_pipeline.named_steps["drop_high_missing"]
    kept_after_missing = pd.Index(dropper.keep_columns_)
    variance_step = fitted_pipeline.named_steps["variance"]
    kept_after_variance = kept_after_missing[variance_step.get_support()]
    return kept_after_variance.tolist()


rf_importance_model_name = "Random Forest Balanced"
rf_importance_model = fitted_models[rf_importance_model_name]
rf_feature_names = get_selected_feature_names(rf_importance_model, X.columns)
rf_importance = pd.DataFrame(
    {
        "feature": rf_feature_names,
        "importance": rf_importance_model.named_steps["model"].feature_importances_,
    }
).sort_values("importance", ascending=False)

display(rf_importance.head(20))


In [ ]:
top_rf_importance = rf_importance.head(20).sort_values("importance")

fig, ax = plt.subplots(figsize=(8, 7))
sns.barplot(data=top_rf_importance, x="importance", y="feature", color="#59A14F", ax=ax)
ax.set_title("Random Forest Feature Importance")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "random_forest_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
logistic_feature_names = get_selected_feature_names(final_model, X.columns)
logistic_coefficients = final_model.named_steps["model"].coef_[0]
logistic_importance = pd.DataFrame(
    {
        "feature": logistic_feature_names,
        "coefficient": logistic_coefficients,
        "abs_coefficient": np.abs(logistic_coefficients),
    }
).sort_values("abs_coefficient", ascending=False)

display(logistic_importance.head(20))


In [ ]:
permutation_result = permutation_importance(
    final_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=SEED,
    n_jobs=-1,
)

permutation_importance_df = pd.DataFrame(
    {
        "feature": X_test.columns,
        "importance_mean": permutation_result.importances_mean,
        "importance_std": permutation_result.importances_std,
    }
).sort_values("importance_mean", ascending=False)

display(permutation_importance_df.head(20))


## 25. 새로운 데이터 예측

새로운 센서 데이터 한 행 또는 여러 행을 입력받아 Pass/Fail 예측과 Fail 확률을 반환하는 함수를 만든다. 결측값 대체와 표준화는 수동으로 다시 적용하지 않고 저장된 Pipeline 전체가 처리한다.


In [ ]:
def prepare_new_sample(sample) -> pd.DataFrame:
    """Convert a new sample into a DataFrame with the original SECOM feature columns."""
    if isinstance(sample, pd.Series):
        sample_df = sample.to_frame().T
    elif isinstance(sample, pd.DataFrame):
        sample_df = sample.copy()
    else:
        sample_array = np.asarray(sample)
        if sample_array.ndim == 1:
            sample_array = sample_array.reshape(1, -1)
        sample_df = pd.DataFrame(sample_array, columns=X.columns)

    missing_columns = [column for column in X.columns if column not in sample_df.columns]
    if missing_columns:
        raise ValueError(f"Missing required feature columns: {missing_columns[:5]} ...")

    return sample_df.loc[:, X.columns]


def predict_secom(sample, model=final_model, threshold: float = selected_threshold) -> pd.DataFrame:
    """Predict Pass/Fail and Fail probability for new SECOM sensor data."""
    sample_df = prepare_new_sample(sample)
    fail_probability = get_positive_proba(model, sample_df)
    prediction = np.where(fail_probability >= threshold, "Fail", "Pass")

    return pd.DataFrame(
        {
            "prediction": prediction,
            "fail_probability": fail_probability,
            "decision_threshold": threshold,
        },
        index=sample_df.index,
    )


example_prediction = predict_secom(X_test.head(5))
display(example_prediction)


## 26. 모델 저장

최종 Pipeline과 Threshold, Feature 목록, 성능 요약을 함께 저장한다. 저장된 Pipeline을 다시 불러오면 결측값 대체, Feature 제거, 표준화, 분류 모델을 같은 방식으로 적용할 수 있다.


In [ ]:
import joblib

MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_bundle = {
    "model": final_model,
    "threshold": selected_threshold,
    "feature_columns": X.columns.tolist(),
    "target_mapping": {0: "Pass", 1: "Fail"},
    "final_model_name": final_model_name,
    "test_metrics": final_test_result,
}

MODEL_PATH = MODEL_DIR / "secom_final_pipeline.joblib"
joblib.dump(model_bundle, MODEL_PATH)

loaded_bundle = joblib.load(MODEL_PATH)
print(f"Saved model bundle: {MODEL_PATH}")
print(f"Loaded model name: {loaded_bundle['final_model_name']}")


## 27. 결과 요약과 한계

이 노트북은 SECOM 센서 데이터를 읽고, 결측값과 클래스 불균형을 확인한 뒤, 데이터 누수를 피하는 Pipeline으로 기본 분류 모델을 비교한다. 또한 Validation 기반 Threshold 선택, Feature Importance, 새 데이터 예측 함수, 최종 Pipeline 저장까지 포함한다.

해석 시 다음 원칙을 지킨다.

- Accuracy가 높다고 좋은 모델이라고 단정하지 않는다.
- 실제 Fail을 몇 개 놓쳤는지 False Negative 수를 확인한다.
- Feature Importance는 예측 기여도이지 실제 공정 원인 또는 인과관계가 아니다.
- 익명 Feature에 임의의 센서명이나 장비명을 붙이지 않는다.
- 공개 SECOM 데이터 결과를 실제 Fab 양산 성능으로 일반화하지 않는다.

이후 확장할 작업은 웨이퍼 결함 패턴 이미지/맵 데이터가 확보되었을 때의 결함 패턴 분류, 설비 고장 예지용 시계열 Feature 구성, 실제 운영 데이터 기준의 시간 순서 검증이다.


## 28. 웨이퍼 검사·설비 이벤트 데이터 확장 인터페이스

현재 공개 SECOM 데이터는 센서 Feature와 정상·불량 라벨 중심이다. 웨이퍼 결함 패턴 분류와 설비 고장 예측을 하려면 별도의 웨이퍼 검사 데이터 또는 설비 이벤트 데이터가 필요하다.

아래 셀은 그런 데이터가 추가되었을 때 같은 프로젝트 구조에서 읽을 수 있게 만든 선택형 로더다. 파일이 없으면 실행을 멈추지 않고 안내 메시지만 출력한다.

권장 파일 위치와 예시는 다음과 같다.

- `data/raw/wafer_inspection.csv`: `wafer_id`, `x`, `y`, `defect_type`, `inspection_time`
- `data/raw/equipment_events.csv`: `equipment_id`, `timestamp`, `event_type`, `failure_label`

실제 컬럼명은 데이터 확보 후 조정한다.


In [ ]:
def read_optional_table(path: Path) -> pd.DataFrame | None:
    """Read an optional CSV/TSV/Excel table without failing when it is absent."""
    if not path.exists():
        print(f"Optional file not found: {path}")
        return None

    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="\t")
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    raise ValueError(f"Unsupported optional data format: {path.suffix}")


def first_existing_path(candidates: list[Path]) -> Path | None:
    """Return the first existing path from a list of candidates."""
    return next((path for path in candidates if path.exists()), None)


In [ ]:
wafer_candidates = [
    RAW_DATA_DIR / "wafer_inspection.csv",
    RAW_DATA_DIR / "wafer_inspection.tsv",
    RAW_DATA_DIR / "wafer_inspection.xlsx",
]
equipment_candidates = [
    RAW_DATA_DIR / "equipment_events.csv",
    RAW_DATA_DIR / "equipment_events.tsv",
    RAW_DATA_DIR / "equipment_events.xlsx",
]

wafer_path = first_existing_path(wafer_candidates)
equipment_path = first_existing_path(equipment_candidates)

wafer_inspection_df = read_optional_table(wafer_path) if wafer_path else None
equipment_events_df = read_optional_table(equipment_path) if equipment_path else None

optional_data_summary = pd.DataFrame(
    [
        {
            "dataset": "wafer_inspection",
            "loaded": wafer_inspection_df is not None,
            "path": str(wafer_path) if wafer_path else None,
            "rows": len(wafer_inspection_df) if wafer_inspection_df is not None else 0,
            "columns": wafer_inspection_df.shape[1] if wafer_inspection_df is not None else 0,
        },
        {
            "dataset": "equipment_events",
            "loaded": equipment_events_df is not None,
            "path": str(equipment_path) if equipment_path else None,
            "rows": len(equipment_events_df) if equipment_events_df is not None else 0,
            "columns": equipment_events_df.shape[1] if equipment_events_df is not None else 0,
        },
    ]
)

display(optional_data_summary)


## 29. 다음 확장 방향

센서 기반 Pass/Fail 예측이 현재 노트북의 중심이다. 웨이퍼 검사 데이터가 확보되면 결함 좌표 또는 웨이퍼 맵을 이용해 Center, Edge, Ring, Scratch 같은 결함 패턴 분류 모델을 추가할 수 있다.

설비 이벤트 데이터가 확보되면 센서 통계량과 이벤트 이력을 시간 기준으로 결합해 고장 가능성 예측 문제로 확장할 수 있다. 이때도 미래 정보를 학습에 섞지 않도록 시간 기준 Train/Test 분리와 누수 점검이 필요하다.


## 30. 클래스 불균형 처리 심화

SECOM 데이터는 Fail 샘플이 적기 때문에 기본 모델만으로는 실제 불량을 놓칠 수 있다. 여기서는 Class Weight 외에 Random Oversampling과 SMOTE를 비교한다.

Oversampling과 SMOTE는 반드시 Train 데이터에만 적용해야 한다. Test 데이터에는 샘플링을 절대 적용하지 않는다. Cross Validation에서도 각 Fold의 Train 부분에서만 샘플링이 일어나도록 `imblearn.pipeline.Pipeline`을 사용한다.

SMOTE는 고차원 센서 공간에서 합성 데이터를 만들기 때문에 실제 공정 상태를 충분히 표현하지 못할 수 있다는 한계가 있다.


In [ ]:
try:
    from imblearn.over_sampling import RandomOverSampler, SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    IMBLEARN_AVAILABLE = True
except ImportError:
    IMBLEARN_AVAILABLE = False
    print("imbalanced-learn is not installed. In Colab, run: !pip install imbalanced-learn")


def make_sampling_logistic_pipeline(sampler) -> "ImbPipeline":
    """Create a Logistic Regression pipeline with sampling applied only to training folds."""
    return ImbPipeline(
        steps=[
            ("drop_high_missing", HighMissingFeatureDropper(threshold=MISSING_THRESHOLD)),
            ("imputer", SimpleImputer(strategy="median")),
            ("variance", VarianceThreshold(threshold=LOW_VARIANCE_THRESHOLD)),
            ("scaler", StandardScaler()),
            ("sampler", sampler),
            (
                "model",
                LogisticRegression(
                    max_iter=3000,
                    random_state=SEED,
                ),
            ),
        ]
    )


In [ ]:
if IMBLEARN_AVAILABLE:
    sampling_models = {
        "Logistic Regression + Random Oversampling": (
            "random_oversampling",
            make_sampling_logistic_pipeline(RandomOverSampler(random_state=SEED)),
        ),
        "Logistic Regression + SMOTE": (
            "smote",
            make_sampling_logistic_pipeline(SMOTE(random_state=SEED, k_neighbors=5)),
        ),
    }

    sampling_rows = []

    for model_name, (imbalance_method, model) in sampling_models.items():
        print(f"Training: {model_name}")
        model.fit(X_train, y_train)
        fitted_models[model_name] = model
        sampling_rows.append(
            evaluate_classifier(
                name=model_name,
                imbalance_method=imbalance_method,
                model=model,
                X_eval=X_test,
                y_eval=y_test,
                threshold=0.5,
            )
        )

    sampling_results_df = pd.DataFrame(sampling_rows)
    all_results_df = (
        pd.concat([results_df, sampling_results_df], ignore_index=True)
        .sort_values(["fail_recall", "fail_f1", "pr_auc"], ascending=False)
        .reset_index(drop=True)
    )
else:
    sampling_results_df = pd.DataFrame()
    all_results_df = results_df.copy()

display(all_results_df)


## 31. Stratified Cross Validation

단일 Train/Test 분리 결과만 보면 모델 안정성을 판단하기 어렵다. Stratified Cross Validation을 사용하면 각 Fold에서 Pass/Fail 비율을 유지하면서 모델 성능의 평균과 변동성을 확인할 수 있다.

여기서도 전처리와 샘플링은 Fold 내부 Train 데이터에서만 학습되어야 한다.


In [ ]:
from sklearn.metrics import make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "fail_precision": make_scorer(precision_score, pos_label=1, zero_division=0),
    "fail_recall": make_scorer(recall_score, pos_label=1, zero_division=0),
    "fail_f1": make_scorer(f1_score, pos_label=1, zero_division=0),
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

cv_models = {
    "Logistic Regression Balanced": make_linear_pipeline(class_weight="balanced"),
    "Random Forest Balanced": make_tree_pipeline(class_weight="balanced"),
}

if IMBLEARN_AVAILABLE:
    cv_models["Logistic Regression + SMOTE"] = make_sampling_logistic_pipeline(
        SMOTE(random_state=SEED, k_neighbors=5)
    )

cv_rows = []

for model_name, model in cv_models.items():
    print(f"Cross-validating: {model_name}")
    cv_result = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    row = {"model": model_name}
    for metric_name in scoring:
        scores = cv_result[f"test_{metric_name}"]
        row[f"{metric_name}_mean"] = scores.mean()
        row[f"{metric_name}_std"] = scores.std()
    cv_rows.append(row)

cv_results_df = pd.DataFrame(cv_rows).sort_values(
    ["fail_recall_mean", "fail_f1_mean", "pr_auc_mean"],
    ascending=False,
)

display(cv_results_df)


## 32. MLP 신경망 비교

MLP는 머신러닝 모델과 비교하기 위한 심화 모델로 사용한다. 딥러닝이 항상 더 좋은 것은 아니므로, Fail Recall과 PR-AUC 관점에서 기본 모델들과 비교한다.

이 섹션은 TensorFlow가 설치된 환경에서만 실행된다. Colab에서는 일반적으로 TensorFlow가 기본 제공된다.


In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    TENSORFLOW_AVAILABLE = True
except ImportError:
    TENSORFLOW_AVAILABLE = False
    print("TensorFlow is not installed. In Colab, use a TensorFlow runtime or run: !pip install tensorflow")


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

if TENSORFLOW_AVAILABLE:
    tf.random.set_seed(SEED)

    X_train_nn, X_valid_nn, y_train_nn, y_valid_nn = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        random_state=SEED,
        stratify=y_train,
    )

    nn_preprocessor = Pipeline(
        steps=[
            ("drop_high_missing", HighMissingFeatureDropper(threshold=MISSING_THRESHOLD)),
            ("imputer", SimpleImputer(strategy="median")),
            ("variance", VarianceThreshold(threshold=LOW_VARIANCE_THRESHOLD)),
            ("scaler", StandardScaler()),
        ]
    )

    X_train_nn_processed = nn_preprocessor.fit_transform(X_train_nn)
    X_valid_nn_processed = nn_preprocessor.transform(X_valid_nn)
    X_test_nn_processed = nn_preprocessor.transform(X_test)

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=y_train_nn,
    )
    nn_class_weight = {0: class_weights[0], 1: class_weights[1]}

    mlp_model = keras.Sequential(
        [
            keras.layers.Input(shape=(X_train_nn_processed.shape[1],)),
            keras.layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4)),
            keras.layers.Dropout(0.3),
            keras.layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4)),
            keras.layers.Dropout(0.3),
            keras.layers.Dense(1, activation="sigmoid"),
        ]
    )

    mlp_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="roc_auc"),
            keras.metrics.AUC(name="pr_auc", curve="PR"),
        ],
    )

    mlp_callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_pr_auc",
            mode="max",
            patience=15,
            restore_best_weights=True,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_pr_auc",
            mode="max",
            patience=6,
            factor=0.5,
            min_lr=1e-5,
        ),
    ]

    mlp_history = mlp_model.fit(
        X_train_nn_processed,
        y_train_nn,
        validation_data=(X_valid_nn_processed, y_valid_nn),
        epochs=120,
        batch_size=32,
        class_weight=nn_class_weight,
        callbacks=mlp_callbacks,
        verbose=0,
    )
else:
    mlp_history = None
    mlp_model = None


In [ ]:
if TENSORFLOW_AVAILABLE:
    mlp_test_proba = mlp_model.predict(X_test_nn_processed, verbose=0).ravel()
    mlp_test_pred = (mlp_test_proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, mlp_test_pred, labels=[0, 1]).ravel()

    mlp_result = {
        "model": "MLP Neural Network",
        "imbalance_method": "class_weight",
        "threshold": 0.5,
        "accuracy": accuracy_score(y_test, mlp_test_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, mlp_test_pred),
        "fail_precision": precision_score(y_test, mlp_test_pred, pos_label=1, zero_division=0),
        "fail_recall": recall_score(y_test, mlp_test_pred, pos_label=1, zero_division=0),
        "fail_f1": f1_score(y_test, mlp_test_pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc_score(y_test, mlp_test_proba),
        "pr_auc": average_precision_score(y_test, mlp_test_proba),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }

    model_comparison_with_mlp = (
        pd.concat([all_results_df, pd.DataFrame([mlp_result])], ignore_index=True)
        .sort_values(["fail_recall", "fail_f1", "pr_auc"], ascending=False)
        .reset_index(drop=True)
    )
else:
    mlp_result = None
    model_comparison_with_mlp = all_results_df.copy()

display(model_comparison_with_mlp)


In [ ]:
if TENSORFLOW_AVAILABLE and mlp_history is not None:
    history_df = pd.DataFrame(mlp_history.history)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df["loss"], label="Train Loss")
    axes[0].plot(history_df["val_loss"], label="Validation Loss")
    axes[0].set_title("MLP Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Binary crossentropy")
    axes[0].legend()

    axes[1].plot(history_df["pr_auc"], label="Train PR-AUC")
    axes[1].plot(history_df["val_pr_auc"], label="Validation PR-AUC")
    axes[1].set_title("MLP PR-AUC")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("PR-AUC")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "mlp_training_history.png", dpi=150, bbox_inches="tight")
    plt.show()


## 33. Colab Project File Sync

Run the next cell to recreate the current local project files inside the Colab runtime. This writes src, scripts, tests, README, requirements, Makefile, and validation files.


In [ ]:
from pathlib import Path
import base64

PROJECT_FILES_B64 = {'.gitattributes': 'IyBBdXRvIGRldGVjdCB0ZXh0IGZpbGVzIGFuZCBwZXJmb3JtIExGIG5vcm1hbGl6YXRpb24KKiB0ZXh0PWF1dG8K', '.gitignore': 'IyBCeXRlLWNvbXBpbGVkIC8gb3B0aW1pemVkIC8gRExMIGZpbGVzDQpfX3B5Y2FjaGVfXy8NCioucHlbY29kXQ0KKiRweS5jbGFzcw0KDQojIEMgZXh0ZW5zaW9ucw0KKi5zbw0KDQojIERpc3RyaWJ1dGlvbiAvIHBhY2thZ2luZw0KLlB5dGhvbg0KYnVpbGQvDQpkZXZlbG9wLWVnZ3MvDQpkaXN0Lw0KZG93bmxvYWRzLw0KZGF0YS9yYXcvDQpvdXRwdXRzLw0KZWdncy8NCi5lZ2dzLw0KbGliLw0KbGliNjQvDQpwYXJ0cy8NCnNkaXN0Lw0KdmFyLw0Kd2hlZWxzLw0Kc2hhcmUvcHl0aG9uLXdoZWVscy8NCiouZWdnLWluZm8vDQouaW5zdGFsbGVkLmNmZw0KKi5lZ2cNCk1BTklGRVNUDQoNCiMgUHlJbnN0YWxsZXINCiMgIFVzdWFsbHkgdGhlc2UgZmlsZXMgYXJlIHdyaXR0ZW4gYnkgYSBweXRob24gc2NyaXB0IGZyb20gYSB0ZW1wbGF0ZQ0KIyAgYmVmb3JlIFB5SW5zdGFsbGVyIGJ1aWxkcyB0aGUgZXhlLCBzbyBhcyB0byBpbmplY3QgZGF0ZS9vdGhlciBpbmZvcyBpbnRvIGl0Lg0KKi5tYW5pZmVzdA0KKi5zcGVjDQoNCiMgSW5zdGFsbGVyIGxvZ3MNCnBpcC1sb2cudHh0DQpwaXAtZGVsZXRlLXRoaXMtZGlyZWN0b3J5LnR4dA0KDQojIFVuaXQgdGVzdCAvIGNvdmVyYWdlIHJlcG9ydHMNCmh0bWxjb3YvDQoudG94Lw0KLm5veC8NCi5jb3ZlcmFnZQ0KLmNvdmVyYWdlLioNCi5jYWNoZQ0Kbm9zZXRlc3RzLnhtbA0KY292ZXJhZ2UueG1sDQoqLmNvdmVyDQoqLnB5LGNvdmVyDQouaHlwb3RoZXNpcy8NCi5weXRlc3RfY2FjaGUvDQpjb3Zlci8NCg0KIyBUcmFuc2xhdGlvbnMNCioubW8NCioucG90DQoNCiMgRGphbmdvIHN0dWZmOg0KKi5sb2cNCmxvY2FsX3NldHRpbmdzLnB5DQpkYi5zcWxpdGUzDQpkYi5zcWxpdGUzLWpvdXJuYWwNCg0KIyBGbGFzayBzdHVmZjoNCmluc3RhbmNlLw0KLndlYmFzc2V0cy1jYWNoZQ0KDQojIFNjcmFweSBzdHVmZjoNCi5zY3JhcHkNCg0KIyBTcGhpbnggZG9jdW1lbnRhdGlvbg0KZG9jcy9fYnVpbGQvDQoNCiMgUHlCdWlsZGVyDQoucHlidWlsZGVyLw0KdGFyZ2V0Lw0KDQojIEp1cHl0ZXIgTm90ZWJvb2sNCi5pcHluYl9jaGVja3BvaW50cw0KDQojIElQeXRob24NCnByb2ZpbGVfZGVmYXVsdC8NCmlweXRob25fY29uZmlnLnB5DQoNCiMgcHllbnYNCiMgICBGb3IgYSBsaWJyYXJ5IG9yIHBhY2thZ2UsIHlvdSBtaWdodCB3YW50IHRvIGlnbm9yZSB0aGVzZSBmaWxlcyBzaW5jZSB0aGUgY29kZSBpcw0KIyAgIGludGVuZGVkIHRvIHJ1biBpbiBtdWx0aXBsZSBlbnZpcm9ubWVudHM7IG90aGVyd2lzZSwgY2hlY2sgdGhlbSBpbjoNCiMgLnB5dGhvbi12ZXJzaW9uDQoNCiMgcGlwZW52DQojICAgQWNjb3JkaW5nIHRvIHB5cGEvcGlwZW52IzU5OCwgaXQgaXMgcmVjb21tZW5kZWQgdG8gaW5jbHVkZSBQaXBmaWxlLmxvY2sgaW4gdmVyc2lvbiBjb250cm9sLg0KIyAgIEhvd2V2ZXIsIGluIGNhc2Ugb2YgY29sbGFib3JhdGlvbiwgaWYgaGF2aW5nIHBsYXRmb3JtLXNwZWNpZmljIGRlcGVuZGVuY2llcyBvciBkZXBlbmRlbmNpZXMNCiMgICBoYXZpbmcgbm8gY3Jvc3MtcGxhdGZvcm0gc3VwcG9ydCwgcGlwZW52IG1heSBpbnN0YWxsIGRlcGVuZGVuY2llcyB0aGF0IGRvbid0IHdvcmssIG9yIG5vdA0KIyAgIGluc3RhbGwgYWxsIG5lZWRlZCBkZXBlbmRlbmNpZXMuDQojUGlwZmlsZS5sb2NrDQoNCiMgVVYNCiMgICBTaW1pbGFyIHRvIFBpcGZpbGUubG9jaywgaXQgaXMgZ2VuZXJhbGx5IHJlY29tbWVuZGVkIHRvIGluY2x1ZGUgdXYubG9jayBpbiB2ZXJzaW9uIGNvbnRyb2wuDQojICAgVGhpcyBpcyBlc3BlY2lhbGx5IHJlY29tbWVuZGVkIGZvciBiaW5hcnkgcGFja2FnZXMgdG8gZW5zdXJlIHJlcHJvZHVjaWJpbGl0eSwgYW5kIGlzIG1vcmUNCiMgICBjb21tb25seSBpZ25vcmVkIGZvciBsaWJyYXJpZXMuDQojdXYubG9jaw0KDQojIHBvZXRyeQ0KIyAgIFNpbWlsYXIgdG8gUGlwZmlsZS5sb2NrLCBpdCBpcyBnZW5lcmFsbHkgcmVjb21tZW5kZWQgdG8gaW5jbHVkZSBwb2V0cnkubG9jayBpbiB2ZXJzaW9uIGNvbnRyb2wuDQojICAgVGhpcyBpcyBlc3BlY2lhbGx5IHJlY29tbWVuZGVkIGZvciBiaW5hcnkgcGFja2FnZXMgdG8gZW5zdXJlIHJlcHJvZHVjaWJpbGl0eSwgYW5kIGlzIG1vcmUNCiMgICBjb21tb25seSBpZ25vcmVkIGZvciBsaWJyYXJpZXMuDQojICAgaHR0cHM6Ly9weXRob24tcG9ldHJ5Lm9yZy9kb2NzL2Jhc2ljLXVzYWdlLyNjb21taXQteW91ci1wb2V0cnlsb2NrLWZpbGUtdG8tdmVyc2lvbi1jb250cm9sDQojcG9ldHJ5LmxvY2sNCg0KIyBwZG0NCiMgICBTaW1pbGFyIHRvIFBpcGZpbGUubG9jaywgaXQgaXMgZ2VuZXJhbGx5IHJlY29tbWVuZGVkIHRvIGluY2x1ZGUgcGRtLmxvY2sgaW4gdmVyc2lvbiBjb250cm9sLg0KI3BkbS5sb2NrDQojICAgcGRtIHN0b3JlcyBwcm9qZWN0LXdpZGUgY29uZmlndXJhdGlvbnMgaW4gLnBkbS50b21sLCBidXQgaXQgaXMgcmVjb21tZW5kZWQgdG8gbm90IGluY2x1ZGUgaXQNCiMgICBpbiB2ZXJzaW9uIGNvbnRyb2wuDQojICAgaHR0cHM6Ly9wZG0uZm1pbmcuZGV2L2xhdGVzdC91c2FnZS9wcm9qZWN0LyN3b3JraW5nLXdpdGgtdmVyc2lvbi1jb250cm9sDQoucGRtLnRvbWwNCi5wZG0tcHl0aG9uDQoucGRtLWJ1aWxkLw0KDQojIFBFUCA1ODI7IHVzZWQgYnkgZS5nLiBnaXRodWIuY29tL0RhdmlkLU9Db25ub3IvcHlmbG93IGFuZCBnaXRodWIuY29tL3BkbS1wcm9qZWN0L3BkbQ0KX19weXBhY2thZ2VzX18vDQoNCiMgQ2VsZXJ5IHN0dWZmDQpjZWxlcnliZWF0LXNjaGVkdWxlDQpjZWxlcnliZWF0LnBpZA0KDQojIFNhZ2VNYXRoIHBhcnNlZCBmaWxlcw0KKi5zYWdlLnB5DQoNCiMgRW52aXJvbm1lbnRzDQouZW52DQoudmVudg0KZW52Lw0KdmVudi8NCkVOVi8NCmVudi5iYWsvDQp2ZW52LmJhay8NCg0KIyBTcHlkZXIgcHJvamVjdCBzZXR0aW5ncw0KLnNweWRlcnByb2plY3QNCi5zcHlwcm9qZWN0DQoNCiMgUm9wZSBwcm9qZWN0IHNldHRpbmdzDQoucm9wZXByb2plY3QNCg0KIyBta2RvY3MgZG9jdW1lbnRhdGlvbg0KL3NpdGUNCg0KIyBteXB5DQoubXlweV9jYWNoZS8NCi5kbXlweS5qc29uDQpkbXlweS5qc29uDQoNCiMgUHlyZSB0eXBlIGNoZWNrZXINCi5weXJlLw0KDQojIHB5dHlwZSBzdGF0aWMgdHlwZSBhbmFseXplcg0KLnB5dHlwZS8NCg0KIyBDeXRob24gZGVidWcgc3ltYm9scw0KY3l0aG9uX2RlYnVnLw0KDQojIFB5Q2hhcm0NCiMgIEpldEJyYWlucyBzcGVjaWZpYyB0ZW1wbGF0ZSBpcyBtYWludGFpbmVkIGluIGEgc2VwYXJhdGUgSmV0QnJhaW5zLmdpdGlnbm9yZSB0aGF0IGNhbg0KIyAgYmUgZm91bmQgYXQgaHR0cHM6Ly9naXRodWIuY29tL2dpdGh1Yi9naXRpZ25vcmUvYmxvYi9tYWluL0dsb2JhbC9KZXRCcmFpbnMuZ2l0aWdub3JlDQojICBhbmQgY2FuIGJlIGFkZGVkIHRvIHRoZSBnbG9iYWwgZ2l0aWdub3JlIG9yIG1lcmdlZCBpbnRvIHRoaXMgZmlsZS4gIEZvciBhIG1vcmUgbnVjbGVhcg0KIyAgb3B0aW9uIChub3QgcmVjb21tZW5kZWQpIHlvdSBjYW4gdW5jb21tZW50IHRoZSBmb2xsb3dpbmcgdG8gaWdub3JlIHRoZSBlbnRpcmUgaWRlYSBmb2xkZXIuDQojLmlkZWEvDQoNCiMgUnVmZiBzdHVmZjoNCi5ydWZmX2NhY2hlLw0KDQojIFB5UEkgY29uZmlndXJhdGlvbiBmaWxlDQoucHlwaXJjDQoNCiMgQ3Vyc29yICANCiMgIEN1cnNvciBpcyBhbiBBSS1wb3dlcmVkIGNvZGUgZWRpdG9yLmAuY3Vyc29yaWdub3JlYCBzcGVjaWZpZXMgZmlsZXMvZGlyZWN0b3JpZXMgdG8gDQojICBleGNsdWRlIGZyb20gQUkgZmVhdHVyZXMgbGlrZSBhdXRvY29tcGxldGUgYW5kIGNvZGUgYW5hbHlzaXMuIFJlY29tbWVuZGVkIGZvciBzZW5zaXRpdmUgZGF0YQ0KIyAgcmVmZXIgdG8gaHR0cHM6Ly9kb2NzLmN1cnNvci5jb20vY29udGV4dC9pZ25vcmUtZmlsZXMNCi5jdXJzb3JpZ25vcmUNCi5jdXJzb3JpbmRleGluZ2lnbm9yZQ==', 'README.md': 'IyBTRUNPTSBTZW1pY29uZHVjdG9yIFByb2Nlc3MgRGF0YSBBbmFseXNpcw0KDQpVQ0kgU0VDT00gc2VtaWNvbmR1Y3RvciBwcm9jZXNzIHNlbnNvciBkYXRh66W8IOyCrOyaqe2VtCDsoJztkojsnZgg7KCV7IOBL+u2iOufieydhCDsmIjsuKHtlZjqs6AsIOydtO2bhCDsm6jsnbTtjbwg6rKA7IKsIOuNsOydtO2EsOyZgCDshKTruYQg7J2067Kk7Yq4IOuNsOydtO2EsOuhnCDtmZXsnqXtlaAg7IiYIOyeiOqyjCDrp4zrk6Ag67aE7ISdIOuFuO2KuOu2geyeheuLiOuLpC4NCg0KIyMgQ3VycmVudCBBcnRpZmFjdA0KDQotIGBTRUNPTS5pcHluYmA6IG1haW4gR29vZ2xlIENvbGFiIGFuYWx5c2lzIG5vdGVib29rDQotIGByZXF1aXJlbWVudHMudHh0YDogcGFja2FnZSBsaXN0IGZvciBsb2NhbCBleGVjdXRpb24NCi0gYE1ha2VmaWxlYDogc3RhbmRhcmQgcHJvamVjdCBjb21tYW5kcw0KLSBgc3JjL3NlY29tX2RhdGEucHlgOiByZXVzYWJsZSBkYXRhIGxvYWRpbmcgdXRpbGl0aWVzDQotIGBzcmMvZGF0YV9jb250cmFjdHMucHlgOiBzY2hlbWEgYW5kIHZhbHVlIGNvbnRyYWN0IGNoZWNrcyBmb3IgaW5wdXQvb3V0cHV0IHRhYmxlcw0KLSBgc3JjL3F1YWxpdHlfcmVwb3J0cy5weWA6IHN0YW5kYXJkIGRhdGEgcXVhbGl0eSByZXBvcnQgdXRpbGl0aWVzDQotIGBzcmMvbW9kZWxfcmVnaXN0cnkucHlgOiBtb2RlbCBidW5kbGUgc2F2ZS9sb2FkIGFuZCBwcmVkaWN0aW9uIHV0aWxpdGllcw0KLSBgc3JjL21vbml0b3JpbmcucHlgOiBwcm9jZXNzLXF1YWxpdHkgcHJlZGljdGlvbiBtb25pdG9yaW5nIHV0aWxpdGllcw0KLSBgc3JjL3JlcG9ydGluZy5weWA6IE1hcmtkb3duIHN1bW1hcnkgcmVwb3J0IHV0aWxpdGllcw0KLSBgc3JjL3NlY29tX21vZGVsaW5nLnB5YDogcmV1c2FibGUgcHJlcHJvY2Vzc2luZywgcHJlZGljdGlvbiwgYW5kIGV2YWx1YXRpb24gdXRpbGl0aWVzDQotIGBzcmMvc2Vjb21fdHJhaW5pbmcucHlgOiBtb2RlbCByYW5raW5nIGFuZCB0aHJlc2hvbGQgdHVuaW5nIGhlbHBlcnMNCi0gYHNyYy93YWZlcl9mZWF0dXJlcy5weWA6IHdhZmVyIGRlZmVjdCBzcGF0aWFsIGZlYXR1cmUgdXRpbGl0aWVzDQotIGBzcmMvZXF1aXBtZW50X2ZlYXR1cmVzLnB5YDogZXF1aXBtZW50IGV2ZW50IGZlYXR1cmUgdXRpbGl0aWVzDQotIGBzcmMvZmVhdHVyZV9zdG9yZS5weWA6IHNlbnNvciwgd2FmZXIsIGFuZCBlcXVpcG1lbnQgZmVhdHVyZSBhc3NlbWJseSB1dGlsaXRpZXMNCi0gYHNjcmlwdHMvcnVuX3F1YWxpdHlfcmVwb3J0LnB5YDogQ0xJIGZvciB3cml0aW5nIFNFQ09NIHF1YWxpdHkgcmVwb3J0IENTViBmaWxlcw0KLSBgc2NyaXB0cy90cmFpbl9zZWNvbV9tb2RlbC5weWA6IENMSSBmb3IgdHJhaW5pbmcsIHNlbGVjdGluZywgdHVuaW5nLCBhbmQgc2F2aW5nIFNFQ09NIG1vZGVscw0KLSBgc2NyaXB0cy9idWlsZF9hdXhpbGlhcnlfZmVhdHVyZXMucHlgOiBDTEkgZm9yIGJ1aWxkaW5nIHdhZmVyL2VxdWlwbWVudCBmZWF0dXJlIENTViBmaWxlcw0KLSBgc2NyaXB0cy9hc3NlbWJsZV9mZWF0dXJlX3RhYmxlLnB5YDogQ0xJIGZvciBhc3NlbWJsaW5nIG1vZGVsaW5nIGZlYXR1cmUgdGFibGVzDQotIGBzY3JpcHRzL3ByZWRpY3Rfd2l0aF9tb2RlbC5weWA6IENMSSBmb3IgYmF0Y2ggcHJlZGljdGlvbiB3aXRoIHNhdmVkIG1vZGVsIGJ1bmRsZXMNCi0gYHNjcmlwdHMvZ2VuZXJhdGVfbW9uaXRvcmluZ19yZXBvcnQucHlgOiBDTEkgZm9yIHByb2Nlc3MtcXVhbGl0eSBtb25pdG9yaW5nIHJlcG9ydHMNCi0gYHNjcmlwdHMvZ2VuZXJhdGVfc3VtbWFyeV9yZXBvcnQucHlgOiBDTEkgZm9yIE1hcmtkb3duIHN1bW1hcnkgcmVwb3J0cw0KLSBgc2NyaXB0cy9ydW5fcGlwZWxpbmUucHlgOiBDTEkgZm9yIG9yY2hlc3RyYXRpbmcgcXVhbGl0eSwgZmVhdHVyZSBhc3NlbWJseSwgcHJlZGljdGlvbiwgYW5kIG1vbml0b3Jpbmcgc3RlcHMNCi0gYHRlc3RzL2A6IHN5bnRoZXRpYy1kYXRhIHRlc3RzIGZvciBjb250cmFjdHMsIGxvYWRpbmcsIHF1YWxpdHksIHJlcG9ydGluZywgbW9kZWxpbmcsIG1vZGVsIHJlZ2lzdHJ5LCBtb25pdG9yaW5nLCB3YWZlciwgZXF1aXBtZW50LCBmZWF0dXJlLXN0b3JlLCBwaXBlbGluZSwgYW5kIENMSSB1dGlsaXRpZXMNCg0KIyMgSW1wbGVtZW50ZWQgU2NvcGUNCg0KLSBTRUNPTSDsm5Drs7gg642w7J207YSwIOyekOuPmSDri6TsmrTroZzrk5wNCi0g7IS87IScIEZlYXR1cmXsmYAg652867KoIOuhnOuUqQ0KLSDsm5Drs7gg652867KoIGAtMS8xYOydhCBgMD1QYXNzYCwgYDE9RmFpbGDroZwg67OA7ZmYDQotIOuCoOynnC/si5zqsIQg7KCV67O0IOuzhOuPhCDrs7TqtIANCi0g6rKw7Lih6rCSLCDtgbTrnpjsiqQg67aI6reg7ZiVLCDsg4HsiJgv7KCA67aE7IKwIEZlYXR1cmUg67aE7ISdDQotIOyjvOyalCBGZWF0dXJlIOu2hO2PrCwgUGFzcy9GYWlsIOu5hOq1kCwg7IOB6rSA6rSA6rOELCBQQ0Eg7Iuc6rCB7ZmUDQotIFRyYWluL1Rlc3Qg67aE66asDQotIOuNsOydtO2EsCDriITsiJgg67Cp7KeAIFBpcGVsaW5lDQotIER1bW15IENsYXNzaWZpZXIsIExvZ2lzdGljIFJlZ3Jlc3Npb24sIFJhbmRvbSBGb3Jlc3Qg67mE6rWQDQotIENsYXNzIFdlaWdodCwgUmFuZG9tIE92ZXJzYW1wbGluZywgU01PVEUNCi0gU3RyYXRpZmllZCBDcm9zcyBWYWxpZGF0aW9uDQotIFRocmVzaG9sZCDrtoTshJ0NCi0g7Zi864+Z7ZaJ66CsLCBST0MgQ3VydmUsIFByZWNpc2lvbi1SZWNhbGwgQ3VydmUNCi0gRmVhdHVyZSBJbXBvcnRhbmNlLCBQZXJtdXRhdGlvbiBJbXBvcnRhbmNlDQotIOyEoO2Dne2YlSBNTFAg7Iug6rK966edIOu5hOq1kA0KLSDstZzsooUgUGlwZWxpbmUg7KCA7J6l6rO8IOyDiCDshLzshJwg642w7J207YSwIOyYiOy4oSDtlajsiJgNCi0g7Juo7J207Y28IOqygOyCrC/shKTruYQg7J2067Kk7Yq4IOuNsOydtO2EsCDshKDtg53tmJUg66Gc642UDQoNCiMjIENvbGFiIFVzYWdlDQoNCjEuIGBTRUNPTS5pcHluYmDrpbwgR29vZ2xlIENvbGFi7JeQ7IScIOyXveuLiOuLpC4NCjIuIOuplOuJtOyXkOyEnCBgUnVudGltZSA+IFJ1biBhbGxg7J2EIOyLpO2Wie2VqeuLiOuLpC4NCjMuIGBpbWJhbGFuY2VkLWxlYXJuYCDrmJDripQgYHRlbnNvcmZsb3dg6rCAIOyXhuuKlCDrn7Dtg4DsnoTsnbTrqbQg7ZW064u5IOyEoO2Dne2YlSDshLnshZjsnYAg7JWI64K0IOuplOyLnOyngOulvCDstpzroKXtlanri4jri6QuDQo0LiDsi6Ttlokg6rKw6rO87JmAIOuqqOuNuCDtjIzsnbzsnYAg64W47Yq467aBIOq4sOykgCBgb3V0cHV0cy9gIOyVhOuemOyXkCDsg53shLHrkKnri4jri6QuDQoNCiMjIExvY2FsIFVzYWdlDQoNCmBgYGJhc2gNCnBpcCBpbnN0YWxsIC1yIHJlcXVpcmVtZW50cy50eHQNCmp1cHl0ZXIgbm90ZWJvb2sgU0VDT00uaXB5bmINCmBgYA0KDQojIyBTdGFuZGFyZCBDb21tYW5kcw0KDQpgYGBiYXNoDQptYWtlIGNoZWNrICAgICAgICAgICAgICAjIHZhbGlkYXRlIG5vdGVib29rIHN0cnVjdHVyZSBhbmQgcnVuIHRlc3RzDQptYWtlIHRyYWluLXNlY29tICAgICAgICAjIHRyYWluLCBldmFsdWF0ZSwgdHVuZSB0aHJlc2hvbGQsIGFuZCBzYXZlIGEgU0VDT00gbW9kZWwNCm1ha2UgcXVhbGl0eS1yZXBvcnQgICAgICMgZ2VuZXJhdGUgU0VDT00gcXVhbGl0eSBDU1YgcmVwb3J0cw0KbWFrZSBhdXhpbGlhcnktZmVhdHVyZXMgIyBidWlsZCB3YWZlci9lcXVpcG1lbnQgZmVhdHVyZSBDU1YgZmlsZXMNCm1ha2UgYXNzZW1ibGUtZmVhdHVyZXMgICMgYXNzZW1ibGUgb25lIG1vZGVsaW5nIGZlYXR1cmUgdGFibGUNCm1ha2UgcHJlZGljdCAgICAgICAgICAgICMgcnVuIGJhdGNoIHByZWRpY3Rpb24gd2l0aCBhIHNhdmVkIG1vZGVsIGJ1bmRsZQ0KbWFrZSBtb25pdG9yICAgICAgICAgICAgIyBnZW5lcmF0ZSBtb25pdG9yaW5nIENTViByZXBvcnRzDQptYWtlIHN1bW1hcnkgICAgICAgICAgICAjIGdlbmVyYXRlIE1hcmtkb3duIHN1bW1hcnkgcmVwb3J0DQptYWtlIHBpcGVsaW5lICAgICAgICAgICAjIHJ1biB0aGUgZnVsbCBDTEkgcGlwZWxpbmUNCmBgYA0KDQojIyBNb2RlbCBUcmFpbmluZyBDTEkNCg0KVHJhaW4gY2FuZGlkYXRlIG1vZGVscywgY2hvb3NlIHRoZSBiZXN0IHZhbGlkYXRpb24gbW9kZWwsIHR1bmUgdGhlIGRlY2lzaW9uIHRocmVzaG9sZCwgYW5kIHNhdmUgdGhlIGZpbmFsIG1vZGVsIGJ1bmRsZSBwbHVzIENTViByZXBvcnRzOg0KDQpgYGBiYXNoDQpweXRob24gc2NyaXB0cy90cmFpbl9zZWNvbV9tb2RlbC5weSAtLWRvd25sb2FkLXNlY29tDQpgYGANCg0KTWFpbiBvdXRwdXRzOg0KDQotIGBvdXRwdXRzL21vZGVscy9zZWNvbV9maW5hbF9waXBlbGluZS5qb2JsaWJgDQotIGBvdXRwdXRzL3JlcG9ydHMvbW9kZWxfbWV0cmljcy5jc3ZgDQotIGBvdXRwdXRzL3JlcG9ydHMvdGhyZXNob2xkX21ldHJpY3MuY3N2YA0KLSBgb3V0cHV0cy9wcmVkaWN0aW9ucy90ZXN0X3ByZWRpY3Rpb25zLmNzdmANCg0KIyMgUXVhbGl0eSBSZXBvcnQgQ0xJDQoNCkdlbmVyYXRlIHN0YW5kYXJkIFNFQ09NIHF1YWxpdHkgcmVwb3J0IENTViBmaWxlcyB3aXRoIGxvY2FsIHJhdyBmaWxlczoNCg0KYGBgYmFzaA0KcHl0aG9uIHNjcmlwdHMvcnVuX3F1YWxpdHlfcmVwb3J0LnB5IC0tcmF3LWRhdGEtZGlyIGRhdGEvcmF3IC0tb3V0cHV0LWRpciBvdXRwdXRzL3JlcG9ydHMNCmBgYA0KDQpEb3dubG9hZCB0aGUgVUNJIFNFQ09NIGZpbGVzIGZpcnN0LCB0aGVuIHdyaXRlIHJlcG9ydHM6DQoNCmBgYGJhc2gNCnB5dGhvbiBzY3JpcHRzL3J1bl9xdWFsaXR5X3JlcG9ydC5weSAtLWRvd25sb2FkDQpgYGANCg0KIyMgQXV4aWxpYXJ5IEZlYXR1cmUgQ0xJDQoNCkJ1aWxkIHdhZmVyIGFuZCBlcXVpcG1lbnQgZmVhdHVyZSBDU1YgZmlsZXMgZnJvbSByYXcgYXV4aWxpYXJ5IHRhYmxlczoNCg0KYGBgYmFzaA0KcHl0aG9uIHNjcmlwdHMvYnVpbGRfYXV4aWxpYXJ5X2ZlYXR1cmVzLnB5IC0td2FmZXItaW5wdXQgZGF0YS9yYXcvd2FmZXJfaW5zcGVjdGlvbi5jc3YgLS1lcXVpcG1lbnQtaW5wdXQgZGF0YS9yYXcvZXF1aXBtZW50X2V2ZW50cy5jc3YgLS13YWZlci1vdXRwdXQgZGF0YS9yYXcvd2FmZXJfZmVhdHVyZXMuY3N2IC0tZXF1aXBtZW50LW91dHB1dCBkYXRhL3Jhdy9lcXVpcG1lbnRfZmVhdHVyZXMuY3N2IC0tYWRkLXdhZmVyLXBhdHRlcm4tbGFiZWwNCmBgYA0KDQojIyBGZWF0dXJlIEFzc2VtYmx5IENMSQ0KDQpBc3NlbWJsZSBzZW5zb3IsIHdhZmVyLCBhbmQgZXF1aXBtZW50IGZlYXR1cmUgQ1NWIGZpbGVzIGludG8gb25lIG1vZGVsaW5nIHRhYmxlOg0KDQpgYGBiYXNoDQpweXRob24gc2NyaXB0cy9hc3NlbWJsZV9mZWF0dXJlX3RhYmxlLnB5IC0tc2Vuc29yLXBhdGggZGF0YS9yYXcvc2Vuc29yX2ZlYXR1cmVzLmNzdiAtLXdhZmVyLXBhdGggZGF0YS9yYXcvd2FmZXJfZmVhdHVyZXMuY3N2IC0tZXF1aXBtZW50LXBhdGggZGF0YS9yYXcvZXF1aXBtZW50X2ZlYXR1cmVzLmNzdiAtLW91dHB1dC1wYXRoIG91dHB1dHMvZmVhdHVyZXMvbW9kZWxpbmdfdGFibGUuY3N2DQpgYGANCg0KIyMgUHJlZGljdGlvbiBDTEkNCg0KUnVuIGJhdGNoIHByZWRpY3Rpb25zIHdpdGggYSBzYXZlZCBtb2RlbCBidW5kbGUgYW5kIGEgZmVhdHVyZSBDU1YgZmlsZToNCg0KYGBgYmFzaA0KcHl0aG9uIHNjcmlwdHMvcHJlZGljdF93aXRoX21vZGVsLnB5IC0tbW9kZWwtcGF0aCBvdXRwdXRzL21vZGVscy9zZWNvbV9maW5hbF9waXBlbGluZS5qb2JsaWIgLS1mZWF0dXJlcy1wYXRoIG91dHB1dHMvZmVhdHVyZXMvbW9kZWxpbmdfdGFibGUuY3N2IC0tb3V0cHV0LXBhdGggb3V0cHV0cy9wcmVkaWN0aW9ucy9wcmVkaWN0aW9ucy5jc3YgLS1pZC1jb2x1bW5zIHNhbXBsZV9pZCx3YWZlcl9pZA0KYGBgDQoNCiMjIE1vbml0b3JpbmcgQ0xJDQoNCkdlbmVyYXRlIHByb2Nlc3MtcXVhbGl0eSBtb25pdG9yaW5nIHJlcG9ydHMgZnJvbSBwcmVkaWN0aW9uIG91dHB1dHM6DQoNCmBgYGJhc2gNCnB5dGhvbiBzY3JpcHRzL2dlbmVyYXRlX21vbml0b3JpbmdfcmVwb3J0LnB5IC0tcHJlZGljdGlvbnMtcGF0aCBvdXRwdXRzL3ByZWRpY3Rpb25zL3ByZWRpY3Rpb25zLmNzdiAtLWdyb3VwLWNvbHVtbnMgd2FmZXJfaWQsZXF1aXBtZW50X2lkIC0tb3V0cHV0LWRpciBvdXRwdXRzL3JlcG9ydHMvbW9uaXRvcmluZw0KYGBgDQoNCiMjIEVuZC10by1FbmQgUGlwZWxpbmUgQ0xJDQoNClJ1biBxdWFsaXR5IHJlcG9ydGluZywgZmVhdHVyZSBhc3NlbWJseSwgcHJlZGljdGlvbiwgYW5kIG1vbml0b3JpbmcgaW4gb25lIGNvbW1hbmQ6DQoNCmBgYGJhc2gNCnB5dGhvbiBzY3JpcHRzL3J1bl9waXBlbGluZS5weSAtLXNlbnNvci1wYXRoIGRhdGEvcmF3L3NlbnNvcl9mZWF0dXJlcy5jc3YgLS13YWZlci1wYXRoIGRhdGEvcmF3L3dhZmVyX2ZlYXR1cmVzLmNzdiAtLWVxdWlwbWVudC1wYXRoIGRhdGEvcmF3L2VxdWlwbWVudF9mZWF0dXJlcy5jc3YgLS1tb2RlbC1wYXRoIG91dHB1dHMvbW9kZWxzL3NlY29tX2ZpbmFsX3BpcGVsaW5lLmpvYmxpYiAtLWlkLWNvbHVtbnMgc2FtcGxlX2lkLHdhZmVyX2lkIC0tbW9uaXRvcmluZy1ncm91cC1jb2x1bW5zIHdhZmVyX2lkLGVxdWlwbWVudF9pZA0KYGBgDQoNCiMjIFN1bW1hcnkgUmVwb3J0IENMSQ0KDQpHZW5lcmF0ZSBhIE1hcmtkb3duIHN1bW1hcnkgZnJvbSBhdmFpbGFibGUgQ1NWIHJlcG9ydHM6DQoNCmBgYGJhc2gNCnB5dGhvbiBzY3JpcHRzL2dlbmVyYXRlX3N1bW1hcnlfcmVwb3J0LnB5IC0tcmVwb3J0cy1kaXIgb3V0cHV0cy9yZXBvcnRzIC0tbW9uaXRvcmluZy1kaXIgb3V0cHV0cy9yZXBvcnRzL21vbml0b3JpbmcgLS1vdXRwdXQtcGF0aCBvdXRwdXRzL3JlcG9ydHMvc3VtbWFyeV9yZXBvcnQubWQNCmBgYA0KDQojIyBWYWxpZGF0aW9uDQoNCk5vdGVib29rIOq1rOyhsOyZgCDsvZTrk5wg7IWAIOusuOuyleydhCDruaDrpbTqsowg7ZmV7J247ZWY66Ck66m0IOuLpOydjCDrqoXroLnsnYQg7Iuk7ZaJ7ZWp64uI64ukLg0KDQpgYGBiYXNoDQpweXRob24gdmFsaWRhdGVfbm90ZWJvb2sucHkNCnB5dGhvbiAtbSB1bml0dGVzdCBkaXNjb3ZlciAtcyB0ZXN0cw0KYGBgDQoNCiMjIERhdGEgUG9saWN5DQoNCuybkOuzuCDrjbDsnbTthLDsmYAg7Iuk7ZaJIOyCsOy2nOusvOydgCBHaXTsl5Ag7Jis66as7KeAIOyViuyKteuLiOuLpC4NCg0KLSBgZGF0YS9yYXcvYA0KLSBgb3V0cHV0cy9gDQoNCiMjIE5leHQgRGF0YSBFeHRlbnNpb25zDQoNCuy2lOqwgCDrjbDsnbTthLDqsIAg7ZmV67O065CY66m0IOuLpOydjCDsnITsuZjsl5Ag64Sj6rOgIOuFuO2KuOu2geydmCDshKDtg53tmJUg66Gc642UIOyEueyFmOydhCDsi6Ttlontlanri4jri6QuDQoNCi0gYGRhdGEvcmF3L3dhZmVyX2luc3BlY3Rpb24uY3N2YA0KLSBgZGF0YS9yYXcvZXF1aXBtZW50X2V2ZW50cy5jc3ZgDQoNCuybqOydtO2NvCDqsrDtlagg7Yyo7YS0IOu2hOulmOyZgCDshKTruYQg6rOg7J6lIOyYiOy4oeydgCDsi6TsoJwg7Lus65+8IOq1rOyhsOqwgCDtmZXsnbjrkJwg65KkIOuzhOuPhCBGZWF0dXJlIEVuZ2luZWVyaW5n6rO8IOyLnOqwhCDquLDspIAg6rKA7Kad7J2EIOy2lOqwgO2VtOyVvCDtlanri4jri6QuDQo=', 'requirements.txt': 'bnVtcHkNCnBhbmRhcw0KbWF0cGxvdGxpYg0Kc2VhYm9ybg0Kc2Npa2l0LWxlYXJuDQppbWJhbGFuY2VkLWxlYXJuDQp0ZW5zb3JmbG93DQpqb2JsaWINCm9wZW5weXhsDQpub3RlYm9vaw0KDQo=', 'Makefile': 'UFlUSE9OID89IHB5dGhvbg0KDQouUEhPTlk6IHZhbGlkYXRlIHRlc3QgY2hlY2sgdHJhaW4tc2Vjb20gcXVhbGl0eS1yZXBvcnQgYXV4aWxpYXJ5LWZlYXR1cmVzIGFzc2VtYmxlLWZlYXR1cmVzIHByZWRpY3QgbW9uaXRvciBzdW1tYXJ5IHBpcGVsaW5lDQoNCnZhbGlkYXRlOg0KCSQoUFlUSE9OKSB2YWxpZGF0ZV9ub3RlYm9vay5weQ0KDQp0ZXN0Og0KCSQoUFlUSE9OKSAtbSB1bml0dGVzdCBkaXNjb3ZlciAtcyB0ZXN0cw0KDQpjaGVjazogdmFsaWRhdGUgdGVzdA0KDQp0cmFpbi1zZWNvbToNCgkkKFBZVEhPTikgc2NyaXB0cy90cmFpbl9zZWNvbV9tb2RlbC5weSAtLWRvd25sb2FkLXNlY29tIC0tcmF3LWRhdGEtZGlyIGRhdGEvcmF3IC0tbW9kZWwtb3V0cHV0IG91dHB1dHMvbW9kZWxzL3NlY29tX2ZpbmFsX3BpcGVsaW5lLmpvYmxpYiAtLW1ldHJpY3Mtb3V0cHV0IG91dHB1dHMvcmVwb3J0cy9tb2RlbF9tZXRyaWNzLmNzdiAtLXRocmVzaG9sZC1vdXRwdXQgb3V0cHV0cy9yZXBvcnRzL3RocmVzaG9sZF9tZXRyaWNzLmNzdiAtLXByZWRpY3Rpb25zLW91dHB1dCBvdXRwdXRzL3ByZWRpY3Rpb25zL3Rlc3RfcHJlZGljdGlvbnMuY3N2DQoNCnF1YWxpdHktcmVwb3J0Og0KCSQoUFlUSE9OKSBzY3JpcHRzL3J1bl9xdWFsaXR5X3JlcG9ydC5weSAtLXJhdy1kYXRhLWRpciBkYXRhL3JhdyAtLW91dHB1dC1kaXIgb3V0cHV0cy9yZXBvcnRzL3F1YWxpdHkNCg0KYXV4aWxpYXJ5LWZlYXR1cmVzOg0KCSQoUFlUSE9OKSBzY3JpcHRzL2J1aWxkX2F1eGlsaWFyeV9mZWF0dXJlcy5weSAtLXdhZmVyLWlucHV0IGRhdGEvcmF3L3dhZmVyX2luc3BlY3Rpb24uY3N2IC0tZXF1aXBtZW50LWlucHV0IGRhdGEvcmF3L2VxdWlwbWVudF9ldmVudHMuY3N2IC0td2FmZXItb3V0cHV0IGRhdGEvcmF3L3dhZmVyX2ZlYXR1cmVzLmNzdiAtLWVxdWlwbWVudC1vdXRwdXQgZGF0YS9yYXcvZXF1aXBtZW50X2ZlYXR1cmVzLmNzdiAtLWFkZC13YWZlci1wYXR0ZXJuLWxhYmVsDQoNCmFzc2VtYmxlLWZlYXR1cmVzOg0KCSQoUFlUSE9OKSBzY3JpcHRzL2Fzc2VtYmxlX2ZlYXR1cmVfdGFibGUucHkgLS1zZW5zb3ItcGF0aCBkYXRhL3Jhdy9zZW5zb3JfZmVhdHVyZXMuY3N2IC0td2FmZXItcGF0aCBkYXRhL3Jhdy93YWZlcl9mZWF0dXJlcy5jc3YgLS1lcXVpcG1lbnQtcGF0aCBkYXRhL3Jhdy9lcXVpcG1lbnRfZmVhdHVyZXMuY3N2IC0tb3V0cHV0LXBhdGggb3V0cHV0cy9mZWF0dXJlcy9tb2RlbGluZ190YWJsZS5jc3YNCg0KcHJlZGljdDoNCgkkKFBZVEhPTikgc2NyaXB0cy9wcmVkaWN0X3dpdGhfbW9kZWwucHkgLS1tb2RlbC1wYXRoIG91dHB1dHMvbW9kZWxzL3NlY29tX2ZpbmFsX3BpcGVsaW5lLmpvYmxpYiAtLWZlYXR1cmVzLXBhdGggb3V0cHV0cy9mZWF0dXJlcy9tb2RlbGluZ190YWJsZS5jc3YgLS1vdXRwdXQtcGF0aCBvdXRwdXRzL3ByZWRpY3Rpb25zL3ByZWRpY3Rpb25zLmNzdiAtLWlkLWNvbHVtbnMgc2FtcGxlX2lkLHdhZmVyX2lkDQoNCm1vbml0b3I6DQoJJChQWVRIT04pIHNjcmlwdHMvZ2VuZXJhdGVfbW9uaXRvcmluZ19yZXBvcnQucHkgLS1wcmVkaWN0aW9ucy1wYXRoIG91dHB1dHMvcHJlZGljdGlvbnMvcHJlZGljdGlvbnMuY3N2IC0tZ3JvdXAtY29sdW1ucyB3YWZlcl9pZCxlcXVpcG1lbnRfaWQgLS1vdXRwdXQtZGlyIG91dHB1dHMvcmVwb3J0cy9tb25pdG9yaW5nDQoNCnN1bW1hcnk6DQoJJChQWVRIT04pIHNjcmlwdHMvZ2VuZXJhdGVfc3VtbWFyeV9yZXBvcnQucHkgLS1yZXBvcnRzLWRpciBvdXRwdXRzL3JlcG9ydHMgLS1tb25pdG9yaW5nLWRpciBvdXRwdXRzL3JlcG9ydHMvbW9uaXRvcmluZyAtLW91dHB1dC1wYXRoIG91dHB1dHMvcmVwb3J0cy9zdW1tYXJ5X3JlcG9ydC5tZA0KDQpwaXBlbGluZToNCgkkKFBZVEhPTikgc2NyaXB0cy9ydW5fcGlwZWxpbmUucHkgLS1zZW5zb3ItcGF0aCBkYXRhL3Jhdy9zZW5zb3JfZmVhdHVyZXMuY3N2IC0td2FmZXItcGF0aCBkYXRhL3Jhdy93YWZlcl9mZWF0dXJlcy5jc3YgLS1lcXVpcG1lbnQtcGF0aCBkYXRhL3Jhdy9lcXVpcG1lbnRfZmVhdHVyZXMuY3N2IC0tbW9kZWwtcGF0aCBvdXRwdXRzL21vZGVscy9zZWNvbV9maW5hbF9waXBlbGluZS5qb2JsaWIgLS1pZC1jb2x1bW5zIHNhbXBsZV9pZCx3YWZlcl9pZCAtLW1vbml0b3JpbmctZ3JvdXAtY29sdW1ucyB3YWZlcl9pZCxlcXVpcG1lbnRfaWQNCg==', 'validate_notebook.py': 'IiIiVmFsaWRhdGUgdGhlIFNFQ09NIGFuYWx5c2lzIG5vdGVib29rIHN0cnVjdHVyZSBhbmQgUHl0aG9uIHN5bnRheC4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgYXN0DQppbXBvcnQganNvbg0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCg0KTk9URUJPT0tfUEFUSCA9IFBhdGgoIlNFQ09NLmlweW5iIikNCg0KUkVRVUlSRURfRklMRVMgPSBbDQogICAgUGF0aCgiUkVBRE1FLm1kIiksDQogICAgUGF0aCgicmVxdWlyZW1lbnRzLnR4dCIpLA0KICAgIFBhdGgoIk1ha2VmaWxlIiksDQogICAgUGF0aCgic3JjL3NlY29tX2RhdGEucHkiKSwNCiAgICBQYXRoKCJzcmMvZGF0YV9jb250cmFjdHMucHkiKSwNCiAgICBQYXRoKCJzcmMvc2Vjb21fbW9kZWxpbmcucHkiKSwNCiAgICBQYXRoKCJzcmMvd2FmZXJfZmVhdHVyZXMucHkiKSwNCiAgICBQYXRoKCJzcmMvZXF1aXBtZW50X2ZlYXR1cmVzLnB5IiksDQogICAgUGF0aCgic3JjL2ZlYXR1cmVfc3RvcmUucHkiKSwNCiAgICBQYXRoKCJzcmMvcXVhbGl0eV9yZXBvcnRzLnB5IiksDQogICAgUGF0aCgic3JjL21vZGVsX3JlZ2lzdHJ5LnB5IiksDQogICAgUGF0aCgic3JjL21vbml0b3JpbmcucHkiKSwNCiAgICBQYXRoKCJzcmMvcmVwb3J0aW5nLnB5IiksDQogICAgUGF0aCgic3JjL3NlY29tX3RyYWluaW5nLnB5IiksDQogICAgUGF0aCgic2NyaXB0cy9ydW5fcXVhbGl0eV9yZXBvcnQucHkiKSwNCiAgICBQYXRoKCJzY3JpcHRzL3RyYWluX3NlY29tX21vZGVsLnB5IiksDQogICAgUGF0aCgic2NyaXB0cy9hc3NlbWJsZV9mZWF0dXJlX3RhYmxlLnB5IiksDQogICAgUGF0aCgic2NyaXB0cy9idWlsZF9hdXhpbGlhcnlfZmVhdHVyZXMucHkiKSwNCiAgICBQYXRoKCJzY3JpcHRzL3ByZWRpY3Rfd2l0aF9tb2RlbC5weSIpLA0KICAgIFBhdGgoInNjcmlwdHMvZ2VuZXJhdGVfbW9uaXRvcmluZ19yZXBvcnQucHkiKSwNCiAgICBQYXRoKCJzY3JpcHRzL2dlbmVyYXRlX3N1bW1hcnlfcmVwb3J0LnB5IiksDQogICAgUGF0aCgic2NyaXB0cy9ydW5fcGlwZWxpbmUucHkiKSwNCiAgICBQYXRoKCJ0ZXN0cy90ZXN0X3NlY29tX2RhdGEucHkiKSwNCiAgICBQYXRoKCJ0ZXN0cy90ZXN0X2RhdGFfY29udHJhY3RzLnB5IiksDQogICAgUGF0aCgidGVzdHMvdGVzdF9zZWNvbV9tb2RlbGluZy5weSIpLA0KICAgIFBhdGgoInRlc3RzL3Rlc3Rfc2Vjb21fdHJhaW5pbmcucHkiKSwNCiAgICBQYXRoKCJ0ZXN0cy90ZXN0X3dhZmVyX2ZlYXR1cmVzLnB5IiksDQogICAgUGF0aCgidGVzdHMvdGVzdF9lcXVpcG1lbnRfZmVhdHVyZXMucHkiKSwNCiAgICBQYXRoKCJ0ZXN0cy90ZXN0X2ZlYXR1cmVfc3RvcmUucHkiKSwNCiAgICBQYXRoKCJ0ZXN0cy90ZXN0X3F1YWxpdHlfcmVwb3J0cy5weSIpLA0KICAgIFBhdGgoInRlc3RzL3Rlc3RfbW9kZWxfcmVnaXN0cnkucHkiKSwNCiAgICBQYXRoKCJ0ZXN0cy90ZXN0X21vbml0b3JpbmcucHkiKSwNCiAgICBQYXRoKCJ0ZXN0cy90ZXN0X3JlcG9ydGluZy5weSIpLA0KICAgIFBhdGgoInRlc3RzL3Rlc3RfcnVuX3F1YWxpdHlfcmVwb3J0LnB5IiksDQogICAgUGF0aCgidGVzdHMvdGVzdF9hc3NlbWJsZV9mZWF0dXJlX3RhYmxlLnB5IiksDQogICAgUGF0aCgidGVzdHMvdGVzdF9idWlsZF9hdXhpbGlhcnlfZmVhdHVyZXMucHkiKSwNCiAgICBQYXRoKCJ0ZXN0cy90ZXN0X3ByZWRpY3Rfd2l0aF9tb2RlbC5weSIpLA0KICAgIFBhdGgoInRlc3RzL3Rlc3RfZ2VuZXJhdGVfbW9uaXRvcmluZ19yZXBvcnQucHkiKSwNCiAgICBQYXRoKCJ0ZXN0cy90ZXN0X3J1bl9waXBlbGluZS5weSIpLA0KICAgIFBhdGgoInRlc3RzL3Rlc3RfdHJhaW5fc2Vjb21fbW9kZWwucHkiKSwNCl0NCg0KUkVRVUlSRURfU0VDVElPTlMgPSBbDQogICAgIiMjIDEuIOudvOydtOu4jOufrOumrCBJbXBvcnQiLA0KICAgICIjIyAyLiBTZWVk7JmAIOqyveuhnCDshKTsoJUiLA0KICAgICIjIyAzLiBTRUNPTSDrjbDsnbTthLAg64uk7Jq066Gc65OcIiwNCiAgICAiIyMgNC4g642w7J207YSw7JmAIOudvOuyqCDrtojrn6zsmKTquLAiLA0KICAgICIjIyA1LiDrjbDsnbTthLAg6rWs7KGwIO2ZleyduCIsDQogICAgIiMjIDYuIOygleyDgcK367aI65+JIO2BtOuemOyKpCDrtoTtj6wiLA0KICAgICIjIyA3LiDqsrDsuKHqsJIgMeywqCDtmZXsnbgiLA0KICAgICIjIyA4LiDquLDstIgg7Iuc6rCB7ZmUIiwNCiAgICAiIyMgOS4g7IOB7IiYwrfsoIDrtoTsgrAgRmVhdHVyZSDrtoTshJ0iLA0KICAgICIjIyAxMC4g7IS87IScIEZlYXR1cmUg67aE7Y+sIOyLnOqwge2ZlCIsDQogICAgIiMjIDExLiDsoJXsg4HCt+u2iOufieuzhCBGZWF0dXJlIOu5hOq1kCIsDQogICAgIiMjIDEyLiBGZWF0dXJlIOqwhCDsg4HqtIDqtIDqs4Qg67aE7ISdIiwNCiAgICAiIyMgMTMuIFBDQSDsi5zqsIHtmZQiLA0KICAgICIjIyAxNC4gVHJhaW4vVGVzdCDrtoTrpqwiLA0KICAgICIjIyAxNS4g7KCE7LKY66asIFBpcGVsaW5lIOq1rOyEsSIsDQogICAgIiMjIDE2LiBEdW1teSBDbGFzc2lmaWVyIiwNCiAgICAiIyMgMTcuIExvZ2lzdGljIFJlZ3Jlc3Npb24iLA0KICAgICIjIyAxOC4gUmFuZG9tIEZvcmVzdCIsDQogICAgIiMjIDE5LiDrqqjrjbgg7ISx64qlIOu5hOq1kCIsDQogICAgIiMjIDIwLiDtmLzrj5ntlonroKwiLA0KICAgICIjIyAyMS4gUk9DIEN1cnZl7JmAIFByZWNpc2lvbi1SZWNhbGwgQ3VydmUiLA0KICAgICIjIyAyMi4gVGhyZXNob2xkIOu2hOyEnSIsDQogICAgIiMjIDIzLiDstZzsooUg7ZuE67O0IOuqqOuNuCDtj4nqsIAiLA0KICAgICIjIyAyNC4gRmVhdHVyZSBJbXBvcnRhbmNlIiwNCiAgICAiIyMgMjUuIOyDiOuhnOyatCDrjbDsnbTthLAg7JiI7LihIiwNCiAgICAiIyMgMjYuIOuqqOuNuCDsoIDsnqUiLA0KICAgICIjIyAyNy4g6rKw6rO8IOyalOyVveqzvCDtlZzqs4QiLA0KICAgICIjIyAyOC4g7Juo7J207Y28IOqygOyCrMK37ISk67mEIOydtOuypO2KuCDrjbDsnbTthLAg7ZmV7J6lIOyduO2EsO2OmOydtOyKpCIsDQogICAgIiMjIDI5LiDri6TsnYwg7ZmV7J6lIOuwqe2WpSIsDQogICAgIiMjIDMwLiDtgbTrnpjsiqQg67aI6reg7ZiVIOyymOumrCDsi6ztmZQiLA0KICAgICIjIyAzMS4gU3RyYXRpZmllZCBDcm9zcyBWYWxpZGF0aW9uIiwNCiAgICAiIyMgMzIuIE1MUCDsi6Dqsr3rp50g67mE6rWQIiwNCl0NCg0KUkVRVUlSRURfVE9LRU5TID0gWw0KICAgICdzZXA9ciJcXHMrIicsDQogICAgInN0cmF0aWZ5PXkiLA0KICAgICJQaXBlbGluZSIsDQogICAgIkhpZ2hNaXNzaW5nRmVhdHVyZURyb3BwZXIiLA0KICAgICJEdW1teUNsYXNzaWZpZXIiLA0KICAgICJMb2dpc3RpY1JlZ3Jlc3Npb24iLA0KICAgICJSYW5kb21Gb3Jlc3RDbGFzc2lmaWVyIiwNCiAgICAiU01PVEUiLA0KICAgICJTdHJhdGlmaWVkS0ZvbGQiLA0KICAgICJwcmVkaWN0X3NlY29tIiwNCiAgICAiam9ibGliLmR1bXAiLA0KICAgICJ3YWZlcl9pbnNwZWN0aW9uIiwNCiAgICAiZXF1aXBtZW50X2V2ZW50cyIsDQpdDQoNCg0KZGVmIGxvYWRfbm90ZWJvb2socGF0aDogUGF0aCkgLT4gZGljdDoNCiAgICBpZiBub3QgcGF0aC5leGlzdHMoKToNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJOb3RlYm9vayBub3QgZm91bmQ6IHtwYXRofSIpDQogICAgcmV0dXJuIGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04LXNpZyIpKQ0KDQoNCmRlZiBjZWxsX3NvdXJjZShjZWxsOiBkaWN0KSAtPiBzdHI6DQogICAgcmV0dXJuICIiLmpvaW4oY2VsbC5nZXQoInNvdXJjZSIsIFtdKSkNCg0KDQpkZWYgdmFsaWRhdGVfc2VjdGlvbnMobWFya2Rvd25fdGV4dDogc3RyKSAtPiBsaXN0W3N0cl06DQogICAgcmV0dXJuIFtzZWN0aW9uIGZvciBzZWN0aW9uIGluIFJFUVVJUkVEX1NFQ1RJT05TIGlmIHNlY3Rpb24gbm90IGluIG1hcmtkb3duX3RleHRdDQoNCg0KZGVmIHZhbGlkYXRlX3Rva2Vucyhjb2RlX3RleHQ6IHN0cikgLT4gbGlzdFtzdHJdOg0KICAgIHJldHVybiBbdG9rZW4gZm9yIHRva2VuIGluIFJFUVVJUkVEX1RPS0VOUyBpZiB0b2tlbiBub3QgaW4gY29kZV90ZXh0XQ0KDQoNCmRlZiB2YWxpZGF0ZV9jb2RlX3N5bnRheChjb2RlX2NlbGxzOiBsaXN0W3N0cl0pIC0+IGxpc3Rbc3RyXToNCiAgICBlcnJvcnMgPSBbXQ0KICAgIGZvciBpZHgsIHNvdXJjZSBpbiBlbnVtZXJhdGUoY29kZV9jZWxscywgc3RhcnQ9MSk6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIGFzdC5wYXJzZShzb3VyY2UpDQogICAgICAgIGV4Y2VwdCBTeW50YXhFcnJvciBhcyBleGM6DQogICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYiQ29kZSBjZWxsIHtpZHh9OiB7ZXhjfSIpDQogICAgcmV0dXJuIGVycm9ycw0KDQoNCmRlZiB2YWxpZGF0ZV9yZXF1aXJlZF9maWxlcygpIC0+IGxpc3Rbc3RyXToNCiAgICByZXR1cm4gW3N0cihwYXRoKSBmb3IgcGF0aCBpbiBSRVFVSVJFRF9GSUxFUyBpZiBub3QgcGF0aC5leGlzdHMoKV0NCg0KDQpkZWYgbWFpbigpIC0+IGludDoNCiAgICBub3RlYm9vayA9IGxvYWRfbm90ZWJvb2soTk9URUJPT0tfUEFUSCkNCiAgICBjZWxscyA9IG5vdGVib29rLmdldCgiY2VsbHMiLCBbXSkNCiAgICBtYXJrZG93bl9jZWxscyA9IFtjZWxsX3NvdXJjZShjZWxsKSBmb3IgY2VsbCBpbiBjZWxscyBpZiBjZWxsLmdldCgiY2VsbF90eXBlIikgPT0gIm1hcmtkb3duIl0NCiAgICBjb2RlX2NlbGxzID0gW2NlbGxfc291cmNlKGNlbGwpIGZvciBjZWxsIGluIGNlbGxzIGlmIGNlbGwuZ2V0KCJjZWxsX3R5cGUiKSA9PSAiY29kZSJdDQoNCiAgICBtYXJrZG93bl90ZXh0ID0gIlxuIi5qb2luKG1hcmtkb3duX2NlbGxzKQ0KICAgIGNvZGVfdGV4dCA9ICJcblxuIi5qb2luKGNvZGVfY2VsbHMpDQoNCiAgICBtaXNzaW5nX3NlY3Rpb25zID0gdmFsaWRhdGVfc2VjdGlvbnMobWFya2Rvd25fdGV4dCkNCiAgICBtaXNzaW5nX3Rva2VucyA9IHZhbGlkYXRlX3Rva2Vucyhjb2RlX3RleHQpDQogICAgc3ludGF4X2Vycm9ycyA9IHZhbGlkYXRlX2NvZGVfc3ludGF4KGNvZGVfY2VsbHMpDQogICAgbWlzc2luZ19maWxlcyA9IHZhbGlkYXRlX3JlcXVpcmVkX2ZpbGVzKCkNCg0KICAgIHByaW50KGYiTm90ZWJvb2s6IHtOT1RFQk9PS19QQVRIfSIpDQogICAgcHJpbnQoZiJUb3RhbCBjZWxsczoge2xlbihjZWxscyl9IikNCiAgICBwcmludChmIk1hcmtkb3duIGNlbGxzOiB7bGVuKG1hcmtkb3duX2NlbGxzKX0iKQ0KICAgIHByaW50KGYiQ29kZSBjZWxsczoge2xlbihjb2RlX2NlbGxzKX0iKQ0KDQogICAgaWYgbWlzc2luZ19zZWN0aW9uczoNCiAgICAgICAgcHJpbnQoIlxuTWlzc2luZyBzZWN0aW9uczoiKQ0KICAgICAgICBmb3Igc2VjdGlvbiBpbiBtaXNzaW5nX3NlY3Rpb25zOg0KICAgICAgICAgICAgcHJpbnQoZiItIHtzZWN0aW9ufSIpDQoNCiAgICBpZiBtaXNzaW5nX3Rva2VuczoNCiAgICAgICAgcHJpbnQoIlxuTWlzc2luZyByZXF1aXJlZCBjb2RlIHRva2VuczoiKQ0KICAgICAgICBmb3IgdG9rZW4gaW4gbWlzc2luZ190b2tlbnM6DQogICAgICAgICAgICBwcmludChmIi0ge3Rva2VufSIpDQoNCiAgICBpZiBzeW50YXhfZXJyb3JzOg0KICAgICAgICBwcmludCgiXG5TeW50YXggZXJyb3JzOiIpDQogICAgICAgIGZvciBlcnJvciBpbiBzeW50YXhfZXJyb3JzOg0KICAgICAgICAgICAgcHJpbnQoZiItIHtlcnJvcn0iKQ0KDQogICAgaWYgbWlzc2luZ19maWxlczoNCiAgICAgICAgcHJpbnQoIlxuTWlzc2luZyBwcm9qZWN0IGZpbGVzOiIpDQogICAgICAgIGZvciBmaWxlX3BhdGggaW4gbWlzc2luZ19maWxlczoNCiAgICAgICAgICAgIHByaW50KGYiLSB7ZmlsZV9wYXRofSIpDQoNCiAgICBpZiBtaXNzaW5nX3NlY3Rpb25zIG9yIG1pc3NpbmdfdG9rZW5zIG9yIHN5bnRheF9lcnJvcnMgb3IgbWlzc2luZ19maWxlczoNCiAgICAgICAgcHJpbnQoIlxuVmFsaWRhdGlvbiBmYWlsZWQuIikNCiAgICAgICAgcmV0dXJuIDENCg0KICAgIHByaW50KCJcblZhbGlkYXRpb24gcGFzc2VkLiIpDQogICAgcmV0dXJuIDANCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQ0K', 'src/__init__.py': 'IiIiU0VDT00gYW5hbHlzaXMgc3VwcG9ydCBtb2R1bGVzLiIiIgo=', 'src/data_contracts.py': 'IiIiUmV1c2FibGUgZGF0YSBjb250cmFjdCBjaGVja3MgZm9yIG1hbnVmYWN0dXJpbmcgYW5hbHl0aWNzIHRhYmxlcy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHR5cGluZyBpbXBvcnQgSXRlcmFibGUKCmltcG9ydCBwYW5kYXMgYXMgcGQKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBDb250cmFjdFJlc3VsdDoKICAgICIiIlZhbGlkYXRpb24gcmVzdWx0IGZvciBhIGRhdGEgY29udHJhY3QuIiIiCgogICAgdGFibGVfbmFtZTogc3RyCiAgICBlcnJvcnM6IHR1cGxlW3N0ciwgLi4uXQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIG9rKHNlbGYpIC0+IGJvb2w6CiAgICAgICAgIiIiUmV0dXJuIFRydWUgd2hlbiBubyBjb250cmFjdCBlcnJvcnMgd2VyZSBmb3VuZC4iIiIKICAgICAgICByZXR1cm4gbm90IHNlbGYuZXJyb3JzCgogICAgZGVmIHJhaXNlX2lmX2ZhaWxlZChzZWxmKSAtPiBOb25lOgogICAgICAgICIiIlJhaXNlIFZhbHVlRXJyb3Igd2hlbiB2YWxpZGF0aW9uIGZhaWxlZC4iIiIKICAgICAgICBpZiBzZWxmLmVycm9yczoKICAgICAgICAgICAgam9pbmVkID0gIjsgIi5qb2luKHNlbGYuZXJyb3JzKQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYie3NlbGYudGFibGVfbmFtZX0gY29udHJhY3QgZmFpbGVkOiB7am9pbmVkfSIpCgoKZGVmIF9taXNzaW5nX2NvbHVtbnMoZGY6IHBkLkRhdGFGcmFtZSwgY29sdW1uczogSXRlcmFibGVbc3RyXSkgLT4gbGlzdFtzdHJdOgogICAgcmV0dXJuIFtjb2x1bW4gZm9yIGNvbHVtbiBpbiBjb2x1bW5zIGlmIGNvbHVtbiBub3QgaW4gZGYuY29sdW1uc10KCgpkZWYgcmVxdWlyZV9jb2x1bW5zKGRmOiBwZC5EYXRhRnJhbWUsIGNvbHVtbnM6IEl0ZXJhYmxlW3N0cl0sIHRhYmxlX25hbWU6IHN0cikgLT4gQ29udHJhY3RSZXN1bHQ6CiAgICAiIiJWYWxpZGF0ZSByZXF1aXJlZCBjb2x1bW5zLiIiIgogICAgbWlzc2luZyA9IF9taXNzaW5nX2NvbHVtbnMoZGYsIGNvbHVtbnMpCiAgICBlcnJvcnMgPSB0dXBsZShmIm1pc3NpbmcgY29sdW1uczoge21pc3Npbmd9IiBmb3IgXyBpbiBbMF0gaWYgbWlzc2luZykKICAgIHJldHVybiBDb250cmFjdFJlc3VsdCh0YWJsZV9uYW1lPXRhYmxlX25hbWUsIGVycm9ycz1lcnJvcnMpCgoKZGVmIHJlcXVpcmVfbnVtZXJpY19jb2x1bW5zKGRmOiBwZC5EYXRhRnJhbWUsIGNvbHVtbnM6IEl0ZXJhYmxlW3N0cl0sIHRhYmxlX25hbWU6IHN0cikgLT4gQ29udHJhY3RSZXN1bHQ6CiAgICAiIiJWYWxpZGF0ZSB0aGF0IGNvbHVtbnMgZXhpc3QgYW5kIGFyZSBudW1lcmljLiIiIgogICAgZXJyb3JzOiBsaXN0W3N0cl0gPSBbXQogICAgbWlzc2luZyA9IF9taXNzaW5nX2NvbHVtbnMoZGYsIGNvbHVtbnMpCiAgICBpZiBtaXNzaW5nOgogICAgICAgIGVycm9ycy5hcHBlbmQoZiJtaXNzaW5nIG51bWVyaWMgY29sdW1uczoge21pc3Npbmd9IikKCiAgICBmb3IgY29sdW1uIGluIGNvbHVtbnM6CiAgICAgICAgaWYgY29sdW1uIGluIGRmLmNvbHVtbnMgYW5kIG5vdCBwZC5hcGkudHlwZXMuaXNfbnVtZXJpY19kdHlwZShkZltjb2x1bW5dKToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIm5vbi1udW1lcmljIGNvbHVtbjoge2NvbHVtbn0iKQoKICAgIHJldHVybiBDb250cmFjdFJlc3VsdCh0YWJsZV9uYW1lPXRhYmxlX25hbWUsIGVycm9ycz10dXBsZShlcnJvcnMpKQoKCmRlZiByZXF1aXJlX3ZhbHVlX3NldCgKICAgIGRmOiBwZC5EYXRhRnJhbWUsCiAgICBjb2x1bW46IHN0ciwKICAgIGFsbG93ZWRfdmFsdWVzOiBzZXQsCiAgICB0YWJsZV9uYW1lOiBzdHIsCiAgICBhbGxvd19taXNzaW5nOiBib29sID0gRmFsc2UsCikgLT4gQ29udHJhY3RSZXN1bHQ6CiAgICAiIiJWYWxpZGF0ZSB0aGF0IGEgY29sdW1uIGNvbnRhaW5zIG9ubHkgYWxsb3dlZCB2YWx1ZXMuIiIiCiAgICBpZiBjb2x1bW4gbm90IGluIGRmLmNvbHVtbnM6CiAgICAgICAgcmV0dXJuIENvbnRyYWN0UmVzdWx0KHRhYmxlX25hbWU9dGFibGVfbmFtZSwgZXJyb3JzPShmIm1pc3NpbmcgY29sdW1uOiB7Y29sdW1ufSIsKSkKCiAgICB2YWx1ZXMgPSBkZltjb2x1bW5dCiAgICBpZiBhbGxvd19taXNzaW5nOgogICAgICAgIHZhbHVlcyA9IHZhbHVlcy5kcm9wbmEoKQoKICAgIGludmFsaWQgPSBzb3J0ZWQoc2V0KHZhbHVlcykuZGlmZmVyZW5jZShhbGxvd2VkX3ZhbHVlcykpCiAgICBlcnJvcnMgPSB0dXBsZShmImludmFsaWQgdmFsdWVzIGluIHtjb2x1bW59OiB7aW52YWxpZH0iIGZvciBfIGluIFswXSBpZiBpbnZhbGlkKQogICAgcmV0dXJuIENvbnRyYWN0UmVzdWx0KHRhYmxlX25hbWU9dGFibGVfbmFtZSwgZXJyb3JzPWVycm9ycykKCgpkZWYgcmVxdWlyZV9wcm9iYWJpbGl0eV9jb2x1bW4oZGY6IHBkLkRhdGFGcmFtZSwgY29sdW1uOiBzdHIsIHRhYmxlX25hbWU6IHN0cikgLT4gQ29udHJhY3RSZXN1bHQ6CiAgICAiIiJWYWxpZGF0ZSB0aGF0IGEgcHJvYmFiaWxpdHkgY29sdW1uIGV4aXN0cywgaXMgbnVtZXJpYywgYW5kIGxpZXMgaW4gWzAsIDFdLiIiIgogICAgbnVtZXJpY19yZXN1bHQgPSByZXF1aXJlX251bWVyaWNfY29sdW1ucyhkZiwgW2NvbHVtbl0sIHRhYmxlX25hbWUpCiAgICBlcnJvcnMgPSBsaXN0KG51bWVyaWNfcmVzdWx0LmVycm9ycykKICAgIGlmIGNvbHVtbiBpbiBkZi5jb2x1bW5zIGFuZCBwZC5hcGkudHlwZXMuaXNfbnVtZXJpY19kdHlwZShkZltjb2x1bW5dKToKICAgICAgICBpbnZhbGlkID0gZGZbY29sdW1uXS5kcm9wbmEoKS5sdCgwKS5hbnkoKSBvciBkZltjb2x1bW5dLmRyb3BuYSgpLmd0KDEpLmFueSgpCiAgICAgICAgaWYgaW52YWxpZDoKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmInByb2JhYmlsaXR5IG91dCBvZiByYW5nZSBbMCwgMV06IHtjb2x1bW59IikKICAgIHJldHVybiBDb250cmFjdFJlc3VsdCh0YWJsZV9uYW1lPXRhYmxlX25hbWUsIGVycm9ycz10dXBsZShlcnJvcnMpKQoKCmRlZiBtZXJnZV9jb250cmFjdF9yZXN1bHRzKHRhYmxlX25hbWU6IHN0ciwgcmVzdWx0czogSXRlcmFibGVbQ29udHJhY3RSZXN1bHRdKSAtPiBDb250cmFjdFJlc3VsdDoKICAgICIiIk1lcmdlIG11bHRpcGxlIGNvbnRyYWN0IHJlc3VsdHMgaW50byBvbmUuIiIiCiAgICBlcnJvcnM6IGxpc3Rbc3RyXSA9IFtdCiAgICBmb3IgcmVzdWx0IGluIHJlc3VsdHM6CiAgICAgICAgZXJyb3JzLmV4dGVuZChyZXN1bHQuZXJyb3JzKQogICAgcmV0dXJuIENvbnRyYWN0UmVzdWx0KHRhYmxlX25hbWU9dGFibGVfbmFtZSwgZXJyb3JzPXR1cGxlKGVycm9ycykpCgoKZGVmIHZhbGlkYXRlX3dhZmVyX2luc3BlY3Rpb25fY29udHJhY3QoZGY6IHBkLkRhdGFGcmFtZSkgLT4gQ29udHJhY3RSZXN1bHQ6CiAgICAiIiJWYWxpZGF0ZSB0aGUgZXhwZWN0ZWQgd2FmZXIgaW5zcGVjdGlvbiBjb29yZGluYXRlIHRhYmxlLiIiIgogICAgcmV0dXJuIG1lcmdlX2NvbnRyYWN0X3Jlc3VsdHMoCiAgICAgICAgIndhZmVyX2luc3BlY3Rpb24iLAogICAgICAgIFsKICAgICAgICAgICAgcmVxdWlyZV9jb2x1bW5zKGRmLCBbIndhZmVyX2lkIiwgIngiLCAieSJdLCAid2FmZXJfaW5zcGVjdGlvbiIpLAogICAgICAgICAgICByZXF1aXJlX251bWVyaWNfY29sdW1ucyhkZiwgWyJ4IiwgInkiXSwgIndhZmVyX2luc3BlY3Rpb24iKSwKICAgICAgICBdLAogICAgKQoKCmRlZiB2YWxpZGF0ZV9lcXVpcG1lbnRfZXZlbnRzX2NvbnRyYWN0KGRmOiBwZC5EYXRhRnJhbWUpIC0+IENvbnRyYWN0UmVzdWx0OgogICAgIiIiVmFsaWRhdGUgdGhlIGV4cGVjdGVkIGVxdWlwbWVudCBldmVudCB0YWJsZS4iIiIKICAgIHJldHVybiByZXF1aXJlX2NvbHVtbnMoCiAgICAgICAgZGYsCiAgICAgICAgWyJlcXVpcG1lbnRfaWQiLCAidGltZXN0YW1wIiwgImV2ZW50X3R5cGUiXSwKICAgICAgICAiZXF1aXBtZW50X2V2ZW50cyIsCiAgICApCgoKZGVmIHZhbGlkYXRlX3ByZWRpY3Rpb25zX2NvbnRyYWN0KGRmOiBwZC5EYXRhRnJhbWUpIC0+IENvbnRyYWN0UmVzdWx0OgogICAgIiIiVmFsaWRhdGUgcHJlZGljdGlvbiBvdXRwdXQgdXNlZCBieSBtb25pdG9yaW5nIHJlcG9ydHMuIiIiCiAgICByZXR1cm4gbWVyZ2VfY29udHJhY3RfcmVzdWx0cygKICAgICAgICAicHJlZGljdGlvbnMiLAogICAgICAgIFsKICAgICAgICAgICAgcmVxdWlyZV9jb2x1bW5zKGRmLCBbInByZWRpY3Rpb24iLCAiZmFpbF9wcm9iYWJpbGl0eSJdLCAicHJlZGljdGlvbnMiKSwKICAgICAgICAgICAgcmVxdWlyZV92YWx1ZV9zZXQoZGYsICJwcmVkaWN0aW9uIiwgeyJQYXNzIiwgIkZhaWwifSwgInByZWRpY3Rpb25zIiksCiAgICAgICAgICAgIHJlcXVpcmVfcHJvYmFiaWxpdHlfY29sdW1uKGRmLCAiZmFpbF9wcm9iYWJpbGl0eSIsICJwcmVkaWN0aW9ucyIpLAogICAgICAgIF0sCiAgICApCgoKZGVmIHZhbGlkYXRlX21vZGVsaW5nX3RhYmxlX2NvbnRyYWN0KGRmOiBwZC5EYXRhRnJhbWUsIGZlYXR1cmVfY29sdW1uczogbGlzdFtzdHJdKSAtPiBDb250cmFjdFJlc3VsdDoKICAgICIiIlZhbGlkYXRlIHRoYXQgYSBtb2RlbGluZyB0YWJsZSBjb250YWlucyByZXF1aXJlZCBtb2RlbCBmZWF0dXJlIGNvbHVtbnMuIiIiCiAgICByZXR1cm4gcmVxdWlyZV9jb2x1bW5zKGRmLCBmZWF0dXJlX2NvbHVtbnMsICJtb2RlbGluZ190YWJsZSIpCg==', 'src/equipment_features.py': 'IiIiRmVhdHVyZSBlbmdpbmVlcmluZyB1dGlsaXRpZXMgZm9yIGVxdWlwbWVudCBldmVudCBhbmQgZmFpbHVyZSBkYXRhLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHBhbmRhcyBhcyBwZAoKCmRlZiB2YWxpZGF0ZV9lcXVpcG1lbnRfY29sdW1ucygKICAgIGV2ZW50c19kZjogcGQuRGF0YUZyYW1lLAogICAgZXF1aXBtZW50X2lkX2NvbDogc3RyID0gImVxdWlwbWVudF9pZCIsCiAgICB0aW1lc3RhbXBfY29sOiBzdHIgPSAidGltZXN0YW1wIiwKICAgIGV2ZW50X3R5cGVfY29sOiBzdHIgPSAiZXZlbnRfdHlwZSIsCikgLT4gTm9uZToKICAgICIiIlZhbGlkYXRlIHRoYXQgYW4gZXF1aXBtZW50IGV2ZW50IHRhYmxlIGhhcyByZXF1aXJlZCBjb2x1bW5zLiIiIgogICAgcmVxdWlyZWQgPSB7ZXF1aXBtZW50X2lkX2NvbCwgdGltZXN0YW1wX2NvbCwgZXZlbnRfdHlwZV9jb2x9CiAgICBtaXNzaW5nID0gc29ydGVkKHJlcXVpcmVkLmRpZmZlcmVuY2UoZXZlbnRzX2RmLmNvbHVtbnMpKQogICAgaWYgbWlzc2luZzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTWlzc2luZyBlcXVpcG1lbnQgZXZlbnQgY29sdW1uczoge21pc3Npbmd9IikKCgpkZWYgcHJlcGFyZV9lcXVpcG1lbnRfZXZlbnRzKAogICAgZXZlbnRzX2RmOiBwZC5EYXRhRnJhbWUsCiAgICBlcXVpcG1lbnRfaWRfY29sOiBzdHIgPSAiZXF1aXBtZW50X2lkIiwKICAgIHRpbWVzdGFtcF9jb2w6IHN0ciA9ICJ0aW1lc3RhbXAiLAogICAgZXZlbnRfdHlwZV9jb2w6IHN0ciA9ICJldmVudF90eXBlIiwKKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJSZXR1cm4gZXZlbnRzIHNvcnRlZCBieSBlcXVpcG1lbnQgYW5kIHRpbWVzdGFtcCB3aXRoIHBhcnNlZCBkYXRldGltZXMuIiIiCiAgICB2YWxpZGF0ZV9lcXVpcG1lbnRfY29sdW1ucygKICAgICAgICBldmVudHNfZGYsCiAgICAgICAgZXF1aXBtZW50X2lkX2NvbD1lcXVpcG1lbnRfaWRfY29sLAogICAgICAgIHRpbWVzdGFtcF9jb2w9dGltZXN0YW1wX2NvbCwKICAgICAgICBldmVudF90eXBlX2NvbD1ldmVudF90eXBlX2NvbCwKICAgICkKICAgIHJlc3VsdCA9IGV2ZW50c19kZi5jb3B5KCkKICAgIHJlc3VsdFt0aW1lc3RhbXBfY29sXSA9IHBkLnRvX2RhdGV0aW1lKHJlc3VsdFt0aW1lc3RhbXBfY29sXSwgZXJyb3JzPSJjb2VyY2UiKQogICAgaWYgcmVzdWx0W3RpbWVzdGFtcF9jb2xdLmlzbmEoKS5hbnkoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFcXVpcG1lbnQgZXZlbnQgdGltZXN0YW1wcyBjb250YWluIHVucGFyc2FibGUgdmFsdWVzLiIpCiAgICByZXR1cm4gcmVzdWx0LnNvcnRfdmFsdWVzKFtlcXVpcG1lbnRfaWRfY29sLCB0aW1lc3RhbXBfY29sXSkucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQoKCmRlZiBlcXVpcG1lbnRfZXZlbnRfZmVhdHVyZXMoCiAgICBldmVudHNfZGY6IHBkLkRhdGFGcmFtZSwKICAgIGVxdWlwbWVudF9pZF9jb2w6IHN0ciA9ICJlcXVpcG1lbnRfaWQiLAogICAgdGltZXN0YW1wX2NvbDogc3RyID0gInRpbWVzdGFtcCIsCiAgICBldmVudF90eXBlX2NvbDogc3RyID0gImV2ZW50X3R5cGUiLAogICAgZmFpbHVyZV9sYWJlbF9jb2w6IHN0ciB8IE5vbmUgPSAiZmFpbHVyZV9sYWJlbCIsCikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiQWdncmVnYXRlIGVxdWlwbWVudCBldmVudCBsb2dzIGludG8gZXF1aXBtZW50LWxldmVsIHByZWRpY3RpdmUgZmVhdHVyZXMuIiIiCiAgICBwcmVwYXJlZCA9IHByZXBhcmVfZXF1aXBtZW50X2V2ZW50cygKICAgICAgICBldmVudHNfZGYsCiAgICAgICAgZXF1aXBtZW50X2lkX2NvbD1lcXVpcG1lbnRfaWRfY29sLAogICAgICAgIHRpbWVzdGFtcF9jb2w9dGltZXN0YW1wX2NvbCwKICAgICAgICBldmVudF90eXBlX2NvbD1ldmVudF90eXBlX2NvbCwKICAgICkKCiAgICBncm91cGVkID0gcHJlcGFyZWQuZ3JvdXBieShlcXVpcG1lbnRfaWRfY29sLCBkcm9wbmE9RmFsc2UpCiAgICBmZWF0dXJlcyA9IGdyb3VwZWQuYWdnKAogICAgICAgIGV2ZW50X2NvdW50PShldmVudF90eXBlX2NvbCwgInNpemUiKSwKICAgICAgICBmaXJzdF9ldmVudF90aW1lPSh0aW1lc3RhbXBfY29sLCAibWluIiksCiAgICAgICAgbGFzdF9ldmVudF90aW1lPSh0aW1lc3RhbXBfY29sLCAibWF4IiksCiAgICAgICAgdW5pcXVlX2V2ZW50X3R5cGVzPShldmVudF90eXBlX2NvbCwgIm51bmlxdWUiKSwKICAgICkKICAgIGZlYXR1cmVzWyJvYnNlcnZhdGlvbl9ob3VycyJdID0gKAogICAgICAgIGZlYXR1cmVzWyJsYXN0X2V2ZW50X3RpbWUiXSAtIGZlYXR1cmVzWyJmaXJzdF9ldmVudF90aW1lIl0KICAgICkuZHQudG90YWxfc2Vjb25kcygpIC8gMzYwMC4wCiAgICBmZWF0dXJlc1siZXZlbnRfcmF0ZV9wZXJfaG91ciJdID0gZmVhdHVyZXNbImV2ZW50X2NvdW50Il0gLyBmZWF0dXJlc1sib2JzZXJ2YXRpb25faG91cnMiXS5jbGlwKGxvd2VyPTEpCgogICAgZXZlbnRfdHlwZV9jb3VudHMgPSBwZC5jcm9zc3RhYihwcmVwYXJlZFtlcXVpcG1lbnRfaWRfY29sXSwgcHJlcGFyZWRbZXZlbnRfdHlwZV9jb2xdKQogICAgZXZlbnRfdHlwZV9jb3VudHMgPSBldmVudF90eXBlX2NvdW50cy5hZGRfcHJlZml4KCJldmVudF90eXBlX2NvdW50XyIpCiAgICBvdXRwdXQgPSBmZWF0dXJlcy5qb2luKGV2ZW50X3R5cGVfY291bnRzKS5maWxsbmEoMCkKCiAgICBpZiBmYWlsdXJlX2xhYmVsX2NvbCBhbmQgZmFpbHVyZV9sYWJlbF9jb2wgaW4gcHJlcGFyZWQuY29sdW1uczoKICAgICAgICBmYWlsdXJlID0gcHJlcGFyZWQuZ3JvdXBieShlcXVpcG1lbnRfaWRfY29sKVtmYWlsdXJlX2xhYmVsX2NvbF0ubWF4KCkucmVuYW1lKCJmYWlsdXJlX2xhYmVsIikKICAgICAgICBvdXRwdXQgPSBvdXRwdXQuam9pbihmYWlsdXJlKQoKICAgIHJldHVybiBvdXRwdXQucmVzZXRfaW5kZXgoKS5yZW5hbWUoY29sdW1ucz17ZXF1aXBtZW50X2lkX2NvbDogImVxdWlwbWVudF9pZCJ9KQoKCmRlZiBhZGRfdGltZV9zaW5jZV9wcmV2aW91c19ldmVudCgKICAgIGV2ZW50c19kZjogcGQuRGF0YUZyYW1lLAogICAgZXF1aXBtZW50X2lkX2NvbDogc3RyID0gImVxdWlwbWVudF9pZCIsCiAgICB0aW1lc3RhbXBfY29sOiBzdHIgPSAidGltZXN0YW1wIiwKICAgIGV2ZW50X3R5cGVfY29sOiBzdHIgPSAiZXZlbnRfdHlwZSIsCikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiQWRkIGVsYXBzZWQgaG91cnMgc2luY2UgdGhlIHByZXZpb3VzIGV2ZW50IGZvciBlYWNoIGVxdWlwbWVudCB1bml0LiIiIgogICAgcHJlcGFyZWQgPSBwcmVwYXJlX2VxdWlwbWVudF9ldmVudHMoCiAgICAgICAgZXZlbnRzX2RmLAogICAgICAgIGVxdWlwbWVudF9pZF9jb2w9ZXF1aXBtZW50X2lkX2NvbCwKICAgICAgICB0aW1lc3RhbXBfY29sPXRpbWVzdGFtcF9jb2wsCiAgICAgICAgZXZlbnRfdHlwZV9jb2w9ZXZlbnRfdHlwZV9jb2wsCiAgICApCiAgICBlbGFwc2VkID0gcHJlcGFyZWQuZ3JvdXBieShlcXVpcG1lbnRfaWRfY29sKVt0aW1lc3RhbXBfY29sXS5kaWZmKCkuZHQudG90YWxfc2Vjb25kcygpIC8gMzYwMC4wCiAgICBwcmVwYXJlZFsiaG91cnNfc2luY2VfcHJldmlvdXNfZXZlbnQiXSA9IGVsYXBzZWQuZmlsbG5hKDApCiAgICByZXR1cm4gcHJlcGFyZWQK', 'src/feature_store.py': 'IiIiVXRpbGl0aWVzIGZvciBhc3NlbWJsaW5nIHNlbnNvciwgd2FmZXIsIGFuZCBlcXVpcG1lbnQgZmVhdHVyZSB0YWJsZXMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKCmltcG9ydCBwYW5kYXMgYXMgcGQKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBKb2luUmVwb3J0OgogICAgIiIiU3VtbWFyeSBvZiBhIGZlYXR1cmUtdGFibGUgam9pbiBvcGVyYXRpb24uIiIiCgogICAgdGFibGVfbmFtZTogc3RyCiAgICBqb2luX2tleTogc3RyCiAgICBsZWZ0X3Jvd3M6IGludAogICAgcmlnaHRfcm93czogaW50CiAgICBvdXRwdXRfcm93czogaW50CiAgICB1bm1hdGNoZWRfcm93czogaW50CgogICAgZGVmIHRvX2RpY3Qoc2VsZikgLT4gZGljdFtzdHIsIGludCB8IHN0cl06CiAgICAgICAgIiIiUmV0dXJuIGEgZGljdGlvbmFyeSByZXByZXNlbnRhdGlvbiBmb3IgcmVwb3J0aW5nLiIiIgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJ0YWJsZV9uYW1lIjogc2VsZi50YWJsZV9uYW1lLAogICAgICAgICAgICAiam9pbl9rZXkiOiBzZWxmLmpvaW5fa2V5LAogICAgICAgICAgICAibGVmdF9yb3dzIjogc2VsZi5sZWZ0X3Jvd3MsCiAgICAgICAgICAgICJyaWdodF9yb3dzIjogc2VsZi5yaWdodF9yb3dzLAogICAgICAgICAgICAib3V0cHV0X3Jvd3MiOiBzZWxmLm91dHB1dF9yb3dzLAogICAgICAgICAgICAidW5tYXRjaGVkX3Jvd3MiOiBzZWxmLnVubWF0Y2hlZF9yb3dzLAogICAgICAgIH0KCgpkZWYgdmFsaWRhdGVfa2V5X2NvbHVtbnMoZGY6IHBkLkRhdGFGcmFtZSwga2V5czogbGlzdFtzdHJdLCB0YWJsZV9uYW1lOiBzdHIpIC0+IE5vbmU6CiAgICAiIiJWYWxpZGF0ZSB0aGF0IGFsbCBrZXkgY29sdW1ucyBleGlzdCBpbiBhIHRhYmxlLiIiIgogICAgbWlzc2luZyA9IFtrZXkgZm9yIGtleSBpbiBrZXlzIGlmIGtleSBub3QgaW4gZGYuY29sdW1uc10KICAgIGlmIG1pc3Npbmc6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInt0YWJsZV9uYW1lfSBpcyBtaXNzaW5nIGtleSBjb2x1bW5zOiB7bWlzc2luZ30iKQoKCmRlZiBwcmVmaXhfZmVhdHVyZV9jb2x1bW5zKGRmOiBwZC5EYXRhRnJhbWUsIGtleXM6IGxpc3Rbc3RyXSwgcHJlZml4OiBzdHIpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIlByZWZpeCBub24ta2V5IGZlYXR1cmUgY29sdW1ucyB0byBhdm9pZCBjb2xsaXNpb25zIGFmdGVyIGpvaW5zLiIiIgogICAgcmVuYW1lX21hcCA9IHtjb2x1bW46IGYie3ByZWZpeH17Y29sdW1ufSIgZm9yIGNvbHVtbiBpbiBkZi5jb2x1bW5zIGlmIGNvbHVtbiBub3QgaW4ga2V5c30KICAgIHJldHVybiBkZi5yZW5hbWUoY29sdW1ucz1yZW5hbWVfbWFwKQoKCmRlZiBsZWZ0X2pvaW5fZmVhdHVyZXMoCiAgICBiYXNlX2RmOiBwZC5EYXRhRnJhbWUsCiAgICBmZWF0dXJlX2RmOiBwZC5EYXRhRnJhbWUsCiAgICBrZXlzOiBsaXN0W3N0cl0sCiAgICB0YWJsZV9uYW1lOiBzdHIsCiAgICBmZWF0dXJlX3ByZWZpeDogc3RyLAopIC0+IHR1cGxlW3BkLkRhdGFGcmFtZSwgSm9pblJlcG9ydF06CiAgICAiIiJMZWZ0IGpvaW4gb25lIGZlYXR1cmUgdGFibGUgb250byBhIGJhc2UgdGFibGUgYW5kIHJldHVybiBhIGpvaW4gcmVwb3J0LiIiIgogICAgdmFsaWRhdGVfa2V5X2NvbHVtbnMoYmFzZV9kZiwga2V5cywgImJhc2VfZGYiKQogICAgdmFsaWRhdGVfa2V5X2NvbHVtbnMoZmVhdHVyZV9kZiwga2V5cywgdGFibGVfbmFtZSkKCiAgICByaWdodCA9IHByZWZpeF9mZWF0dXJlX2NvbHVtbnMoZmVhdHVyZV9kZiwga2V5cz1rZXlzLCBwcmVmaXg9ZmVhdHVyZV9wcmVmaXgpCiAgICBiZWZvcmVfY29scyA9IHNldChiYXNlX2RmLmNvbHVtbnMpCiAgICBvdXRwdXQgPSBiYXNlX2RmLm1lcmdlKHJpZ2h0LCBvbj1rZXlzLCBob3c9ImxlZnQiLCBpbmRpY2F0b3I9ZiJfe3RhYmxlX25hbWV9X21lcmdlIikKICAgIHVubWF0Y2hlZCA9IGludCgob3V0cHV0W2YiX3t0YWJsZV9uYW1lfV9tZXJnZSJdID09ICJsZWZ0X29ubHkiKS5zdW0oKSkKICAgIG91dHB1dCA9IG91dHB1dC5kcm9wKGNvbHVtbnM9W2YiX3t0YWJsZV9uYW1lfV9tZXJnZSJdKQoKICAgIHJlcG9ydCA9IEpvaW5SZXBvcnQoCiAgICAgICAgdGFibGVfbmFtZT10YWJsZV9uYW1lLAogICAgICAgIGpvaW5fa2V5PSIsIi5qb2luKGtleXMpLAogICAgICAgIGxlZnRfcm93cz1sZW4oYmFzZV9kZiksCiAgICAgICAgcmlnaHRfcm93cz1sZW4oZmVhdHVyZV9kZiksCiAgICAgICAgb3V0cHV0X3Jvd3M9bGVuKG91dHB1dCksCiAgICAgICAgdW5tYXRjaGVkX3Jvd3M9dW5tYXRjaGVkLAogICAgKQoKICAgIGR1cGxpY2F0ZWRfY29sdW1ucyA9IFtjb2x1bW4gZm9yIGNvbHVtbiBpbiBvdXRwdXQuY29sdW1ucyBpZiBjb2x1bW4gaW4gYmVmb3JlX2NvbHMgYW5kIGNvbHVtbiBub3QgaW4gYmFzZV9kZi5jb2x1bW5zXQogICAgaWYgZHVwbGljYXRlZF9jb2x1bW5zOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbmV4cGVjdGVkIGR1cGxpY2F0ZWQgY29sdW1ucyBhZnRlciBqb2luOiB7ZHVwbGljYXRlZF9jb2x1bW5zfSIpCgogICAgcmV0dXJuIG91dHB1dCwgcmVwb3J0CgoKZGVmIGFzc2VtYmxlX2ZlYXR1cmVfdGFibGUoCiAgICBzZW5zb3JfZGY6IHBkLkRhdGFGcmFtZSwKICAgIHdhZmVyX2ZlYXR1cmVzOiBwZC5EYXRhRnJhbWUgfCBOb25lID0gTm9uZSwKICAgIGVxdWlwbWVudF9mZWF0dXJlczogcGQuRGF0YUZyYW1lIHwgTm9uZSA9IE5vbmUsCiAgICBzZW5zb3Jfd2FmZXJfa2V5czogbGlzdFtzdHJdIHwgTm9uZSA9IE5vbmUsCiAgICBzZW5zb3JfZXF1aXBtZW50X2tleXM6IGxpc3Rbc3RyXSB8IE5vbmUgPSBOb25lLAopIC0+IHR1cGxlW3BkLkRhdGFGcmFtZSwgcGQuRGF0YUZyYW1lXToKICAgICIiIkFzc2VtYmxlIHNlbnNvciwgd2FmZXIsIGFuZCBlcXVpcG1lbnQgZmVhdHVyZXMgaW50byBvbmUgbW9kZWxpbmcgdGFibGUuIiIiCiAgICBvdXRwdXQgPSBzZW5zb3JfZGYuY29weSgpCiAgICByZXBvcnRzOiBsaXN0W0pvaW5SZXBvcnRdID0gW10KCiAgICBpZiB3YWZlcl9mZWF0dXJlcyBpcyBub3QgTm9uZToKICAgICAgICBrZXlzID0gc2Vuc29yX3dhZmVyX2tleXMgb3IgWyJ3YWZlcl9pZCJdCiAgICAgICAgb3V0cHV0LCByZXBvcnQgPSBsZWZ0X2pvaW5fZmVhdHVyZXMoCiAgICAgICAgICAgIG91dHB1dCwKICAgICAgICAgICAgd2FmZXJfZmVhdHVyZXMsCiAgICAgICAgICAgIGtleXM9a2V5cywKICAgICAgICAgICAgdGFibGVfbmFtZT0id2FmZXJfZmVhdHVyZXMiLAogICAgICAgICAgICBmZWF0dXJlX3ByZWZpeD0id2FmZXJfIiwKICAgICAgICApCiAgICAgICAgcmVwb3J0cy5hcHBlbmQocmVwb3J0KQoKICAgIGlmIGVxdWlwbWVudF9mZWF0dXJlcyBpcyBub3QgTm9uZToKICAgICAgICBrZXlzID0gc2Vuc29yX2VxdWlwbWVudF9rZXlzIG9yIFsiZXF1aXBtZW50X2lkIl0KICAgICAgICBvdXRwdXQsIHJlcG9ydCA9IGxlZnRfam9pbl9mZWF0dXJlcygKICAgICAgICAgICAgb3V0cHV0LAogICAgICAgICAgICBlcXVpcG1lbnRfZmVhdHVyZXMsCiAgICAgICAgICAgIGtleXM9a2V5cywKICAgICAgICAgICAgdGFibGVfbmFtZT0iZXF1aXBtZW50X2ZlYXR1cmVzIiwKICAgICAgICAgICAgZmVhdHVyZV9wcmVmaXg9ImVxdWlwbWVudF8iLAogICAgICAgICkKICAgICAgICByZXBvcnRzLmFwcGVuZChyZXBvcnQpCgogICAgcmV0dXJuIG91dHB1dCwgcGQuRGF0YUZyYW1lKFtyZXBvcnQudG9fZGljdCgpIGZvciByZXBvcnQgaW4gcmVwb3J0c10pCgoKZGVmIGZlYXR1cmVfbWlzc2luZ25lc3NfcmVwb3J0KGZlYXR1cmVfdGFibGU6IHBkLkRhdGFGcmFtZSkgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiUmV0dXJuIG1pc3NpbmctdmFsdWUgY291bnRzIGFuZCByYXRpb3MgZm9yIGEgbW9kZWxpbmcgZmVhdHVyZSB0YWJsZS4iIiIKICAgIHJlcG9ydCA9IHBkLkRhdGFGcmFtZSgKICAgICAgICB7CiAgICAgICAgICAgICJtaXNzaW5nX2NvdW50IjogZmVhdHVyZV90YWJsZS5pc25hKCkuc3VtKCksCiAgICAgICAgICAgICJtaXNzaW5nX3JhdGlvIjogZmVhdHVyZV90YWJsZS5pc25hKCkubWVhbigpLAogICAgICAgIH0KICAgICkKICAgIHJldHVybiByZXBvcnQuc29ydF92YWx1ZXMoWyJtaXNzaW5nX3JhdGlvIiwgIm1pc3NpbmdfY291bnQiXSwgYXNjZW5kaW5nPUZhbHNlKQo=', 'src/model_registry.py': 'IiIiTW9kZWwgYnVuZGxlIHNhdmUvbG9hZCB1dGlsaXRpZXMgZm9yIFNFQ09NIHByZWRpY3Rpb24gd29ya2Zsb3dzLiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcw0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQpmcm9tIHR5cGluZyBpbXBvcnQgQW55DQoNCmltcG9ydCBwYW5kYXMgYXMgcGQNCg0KZnJvbSBzcmMuc2Vjb21fbW9kZWxpbmcgaW1wb3J0IHByZWRpY3Rfd2l0aF90aHJlc2hvbGQNCg0KDQpAZGF0YWNsYXNzDQpjbGFzcyBNb2RlbEJ1bmRsZToNCiAgICAiIiJTZXJpYWxpemFibGUgbW9kZWwgYnVuZGxlIHdpdGggcHJlZGljdGlvbiBtZXRhZGF0YS4iIiINCg0KICAgIG1vZGVsOiBBbnkNCiAgICB0aHJlc2hvbGQ6IGZsb2F0DQogICAgZmVhdHVyZV9jb2x1bW5zOiBsaXN0W3N0cl0NCiAgICB0YXJnZXRfbWFwcGluZzogZGljdFtpbnQsIHN0cl0NCiAgICBtb2RlbF9uYW1lOiBzdHINCiAgICBtZXRyaWNzOiBkaWN0W3N0ciwgQW55XSB8IE5vbmUgPSBOb25lDQoNCiAgICBkZWYgdG9fZGljdChzZWxmKSAtPiBkaWN0W3N0ciwgQW55XToNCiAgICAgICAgIiIiUmV0dXJuIGEgam9ibGliLXNlcmlhbGl6YWJsZSBkaWN0aW9uYXJ5LiIiIg0KICAgICAgICByZXR1cm4gew0KICAgICAgICAgICAgIm1vZGVsIjogc2VsZi5tb2RlbCwNCiAgICAgICAgICAgICJ0aHJlc2hvbGQiOiBzZWxmLnRocmVzaG9sZCwNCiAgICAgICAgICAgICJmZWF0dXJlX2NvbHVtbnMiOiBzZWxmLmZlYXR1cmVfY29sdW1ucywNCiAgICAgICAgICAgICJ0YXJnZXRfbWFwcGluZyI6IHNlbGYudGFyZ2V0X21hcHBpbmcsDQogICAgICAgICAgICAibW9kZWxfbmFtZSI6IHNlbGYubW9kZWxfbmFtZSwNCiAgICAgICAgICAgICJtZXRyaWNzIjogc2VsZi5tZXRyaWNzIG9yIHt9LA0KICAgICAgICB9DQoNCiAgICBAY2xhc3NtZXRob2QNCiAgICBkZWYgZnJvbV9kaWN0KGNscywgZGF0YTogZGljdFtzdHIsIEFueV0pIC0+ICJNb2RlbEJ1bmRsZSI6DQogICAgICAgICIiIkNyZWF0ZSBhIE1vZGVsQnVuZGxlIGZyb20gYSBkaWN0aW9uYXJ5LiIiIg0KICAgICAgICByZXF1aXJlZCA9IHsibW9kZWwiLCAidGhyZXNob2xkIiwgImZlYXR1cmVfY29sdW1ucyIsICJ0YXJnZXRfbWFwcGluZyIsICJtb2RlbF9uYW1lIn0NCiAgICAgICAgbWlzc2luZyA9IHNvcnRlZChyZXF1aXJlZC5kaWZmZXJlbmNlKGRhdGEpKQ0KICAgICAgICBpZiBtaXNzaW5nOg0KICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk1vZGVsIGJ1bmRsZSBpcyBtaXNzaW5nIGtleXM6IHttaXNzaW5nfSIpDQoNCiAgICAgICAgcmV0dXJuIGNscygNCiAgICAgICAgICAgIG1vZGVsPWRhdGFbIm1vZGVsIl0sDQogICAgICAgICAgICB0aHJlc2hvbGQ9ZmxvYXQoZGF0YVsidGhyZXNob2xkIl0pLA0KICAgICAgICAgICAgZmVhdHVyZV9jb2x1bW5zPWxpc3QoZGF0YVsiZmVhdHVyZV9jb2x1bW5zIl0pLA0KICAgICAgICAgICAgdGFyZ2V0X21hcHBpbmc9ZGljdChkYXRhWyJ0YXJnZXRfbWFwcGluZyJdKSwNCiAgICAgICAgICAgIG1vZGVsX25hbWU9c3RyKGRhdGFbIm1vZGVsX25hbWUiXSksDQogICAgICAgICAgICBtZXRyaWNzPWRpY3QoZGF0YS5nZXQoIm1ldHJpY3MiKSBvciB7fSksDQogICAgICAgICkNCg0KDQpkZWYgc2F2ZV9tb2RlbF9idW5kbGUoYnVuZGxlOiBNb2RlbEJ1bmRsZSwgcGF0aDogUGF0aCkgLT4gUGF0aDoNCiAgICAiIiJTYXZlIGEgbW9kZWwgYnVuZGxlIHRvIGRpc2sgd2l0aCBqb2JsaWIuIiIiDQogICAgaW1wb3J0IGpvYmxpYg0KDQogICAgcGF0aCA9IFBhdGgocGF0aCkNCiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgam9ibGliLmR1bXAoYnVuZGxlLnRvX2RpY3QoKSwgcGF0aCkNCiAgICByZXR1cm4gcGF0aA0KDQoNCmRlZiBsb2FkX21vZGVsX2J1bmRsZShwYXRoOiBQYXRoKSAtPiBNb2RlbEJ1bmRsZToNCiAgICAiIiJMb2FkIGEgbW9kZWwgYnVuZGxlIGZyb20gZGlzay4iIiINCiAgICBpbXBvcnQgam9ibGliDQoNCiAgICBwYXRoID0gUGF0aChwYXRoKQ0KICAgIGRhdGEgPSBqb2JsaWIubG9hZChwYXRoKQ0KICAgIGlmIGlzaW5zdGFuY2UoZGF0YSwgTW9kZWxCdW5kbGUpOg0KICAgICAgICByZXR1cm4gZGF0YQ0KICAgIGlmIG5vdCBpc2luc3RhbmNlKGRhdGEsIGRpY3QpOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJNb2RlbCBidW5kbGUgZmlsZSBtdXN0IGNvbnRhaW4gYSBkaWN0IG9yIE1vZGVsQnVuZGxlLiIpDQogICAgcmV0dXJuIE1vZGVsQnVuZGxlLmZyb21fZGljdChkYXRhKQ0KDQoNCmRlZiBwcmVwYXJlX2ZlYXR1cmVzX2Zvcl9idW5kbGUoZmVhdHVyZV90YWJsZTogcGQuRGF0YUZyYW1lLCBidW5kbGU6IE1vZGVsQnVuZGxlKSAtPiBwZC5EYXRhRnJhbWU6DQogICAgIiIiUmV0dXJuIGZlYXR1cmUgY29sdW1ucyBpbiB0aGUgZXhhY3Qgb3JkZXIgZXhwZWN0ZWQgYnkgYSBtb2RlbCBidW5kbGUuIiIiDQogICAgbWlzc2luZyA9IFtjb2x1bW4gZm9yIGNvbHVtbiBpbiBidW5kbGUuZmVhdHVyZV9jb2x1bW5zIGlmIGNvbHVtbiBub3QgaW4gZmVhdHVyZV90YWJsZS5jb2x1bW5zXQ0KICAgIGlmIG1pc3Npbmc6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJNaXNzaW5nIHJlcXVpcmVkIG1vZGVsIGZlYXR1cmUgY29sdW1uczoge21pc3Npbmd9IikNCiAgICByZXR1cm4gZmVhdHVyZV90YWJsZS5sb2NbOiwgYnVuZGxlLmZlYXR1cmVfY29sdW1uc10NCg0KDQpkZWYgcHJlZGljdF9mcm9tX2J1bmRsZShmZWF0dXJlX3RhYmxlOiBwZC5EYXRhRnJhbWUsIGJ1bmRsZTogTW9kZWxCdW5kbGUpIC0+IHBkLkRhdGFGcmFtZToNCiAgICAiIiJQcmVkaWN0IFBhc3MvRmFpbCBsYWJlbHMgZnJvbSBhIHNhdmVkIG1vZGVsIGJ1bmRsZS4iIiINCiAgICBmZWF0dXJlcyA9IHByZXBhcmVfZmVhdHVyZXNfZm9yX2J1bmRsZShmZWF0dXJlX3RhYmxlLCBidW5kbGUpDQogICAgcmV0dXJuIHByZWRpY3Rfd2l0aF90aHJlc2hvbGQoYnVuZGxlLm1vZGVsLCBmZWF0dXJlcywgdGhyZXNob2xkPWJ1bmRsZS50aHJlc2hvbGQpLnRvX2ZyYW1lKA0KICAgICAgICBpbmRleD1mZWF0dXJlX3RhYmxlLmluZGV4DQogICAgKQ0K', 'src/monitoring.py': 'IiIiTW9uaXRvcmluZyByZXBvcnRzIGZvciBwcmVkaWN0aW9uIG91dHB1dHMgYW5kIHByb2Nlc3MgcXVhbGl0eSByaXNrLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHBhbmRhcyBhcyBwZAoKClJFUVVJUkVEX1BSRURJQ1RJT05fQ09MVU1OUyA9IHsicHJlZGljdGlvbiIsICJmYWlsX3Byb2JhYmlsaXR5In0KCgpkZWYgdmFsaWRhdGVfcHJlZGljdGlvbl9jb2x1bW5zKHByZWRpY3Rpb25zOiBwZC5EYXRhRnJhbWUpIC0+IE5vbmU6CiAgICAiIiJWYWxpZGF0ZSByZXF1aXJlZCBwcmVkaWN0aW9uIG91dHB1dCBjb2x1bW5zLiIiIgogICAgbWlzc2luZyA9IHNvcnRlZChSRVFVSVJFRF9QUkVESUNUSU9OX0NPTFVNTlMuZGlmZmVyZW5jZShwcmVkaWN0aW9ucy5jb2x1bW5zKSkKICAgIGlmIG1pc3Npbmc6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk1pc3NpbmcgcHJlZGljdGlvbiBjb2x1bW5zOiB7bWlzc2luZ30iKQoKCmRlZiBvdmVyYWxsX3Jpc2tfc3VtbWFyeSgKICAgIHByZWRpY3Rpb25zOiBwZC5EYXRhRnJhbWUsCiAgICBwcm9iYWJpbGl0eV9jb2w6IHN0ciA9ICJmYWlsX3Byb2JhYmlsaXR5IiwKICAgIHByZWRpY3Rpb25fY29sOiBzdHIgPSAicHJlZGljdGlvbiIsCiAgICBoaWdoX3Jpc2tfdGhyZXNob2xkOiBmbG9hdCA9IDAuNSwKKSAtPiBwZC5TZXJpZXM6CiAgICAiIiJSZXR1cm4gYW4gb3ZlcmFsbCBwcm9jZXNzLXF1YWxpdHkgcmlzayBzdW1tYXJ5IGZyb20gcHJlZGljdGlvbiByb3dzLiIiIgogICAgdmFsaWRhdGVfcHJlZGljdGlvbl9jb2x1bW5zKHByZWRpY3Rpb25zKQogICAgZmFpbF9wcm9iYWJpbGl0eSA9IHByZWRpY3Rpb25zW3Byb2JhYmlsaXR5X2NvbF0KICAgIGZhaWxfcHJlZGljdGlvbnMgPSBwcmVkaWN0aW9uc1twcmVkaWN0aW9uX2NvbF0uYXN0eXBlKHN0cikuc3RyLmxvd2VyKCkuZXEoImZhaWwiKQogICAgaGlnaF9yaXNrID0gZmFpbF9wcm9iYWJpbGl0eSA+PSBoaWdoX3Jpc2tfdGhyZXNob2xkCgogICAgcmV0dXJuIHBkLlNlcmllcygKICAgICAgICB7CiAgICAgICAgICAgICJuX3ByZWRpY3Rpb25zIjogbGVuKHByZWRpY3Rpb25zKSwKICAgICAgICAgICAgIm1lYW5fZmFpbF9wcm9iYWJpbGl0eSI6IGZsb2F0KGZhaWxfcHJvYmFiaWxpdHkubWVhbigpKSwKICAgICAgICAgICAgIm1heF9mYWlsX3Byb2JhYmlsaXR5IjogZmxvYXQoZmFpbF9wcm9iYWJpbGl0eS5tYXgoKSksCiAgICAgICAgICAgICJwcmVkaWN0ZWRfZmFpbF9jb3VudCI6IGludChmYWlsX3ByZWRpY3Rpb25zLnN1bSgpKSwKICAgICAgICAgICAgInByZWRpY3RlZF9mYWlsX3JhdGlvIjogZmxvYXQoZmFpbF9wcmVkaWN0aW9ucy5tZWFuKCkpLAogICAgICAgICAgICAiaGlnaF9yaXNrX2NvdW50IjogaW50KGhpZ2hfcmlzay5zdW0oKSksCiAgICAgICAgICAgICJoaWdoX3Jpc2tfcmF0aW8iOiBmbG9hdChoaWdoX3Jpc2subWVhbigpKSwKICAgICAgICAgICAgImhpZ2hfcmlza190aHJlc2hvbGQiOiBoaWdoX3Jpc2tfdGhyZXNob2xkLAogICAgICAgIH0sCiAgICAgICAgbmFtZT0idmFsdWUiLAogICAgKQoKCmRlZiBncm91cF9yaXNrX3N1bW1hcnkoCiAgICBwcmVkaWN0aW9uczogcGQuRGF0YUZyYW1lLAogICAgZ3JvdXBfY29sczogbGlzdFtzdHJdLAogICAgcHJvYmFiaWxpdHlfY29sOiBzdHIgPSAiZmFpbF9wcm9iYWJpbGl0eSIsCiAgICBwcmVkaWN0aW9uX2NvbDogc3RyID0gInByZWRpY3Rpb24iLAogICAgaGlnaF9yaXNrX3RocmVzaG9sZDogZmxvYXQgPSAwLjUsCiAgICBhbGVydF9yYXRpb190aHJlc2hvbGQ6IGZsb2F0ID0gMC4yNSwKKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJBZ2dyZWdhdGUgcHJlZGljdGlvbiByaXNrIGJ5IHdhZmVyLCBlcXVpcG1lbnQsIGxvdCwgb3Igb3RoZXIgZ3JvdXAgY29sdW1ucy4iIiIKICAgIHZhbGlkYXRlX3ByZWRpY3Rpb25fY29sdW1ucyhwcmVkaWN0aW9ucykKICAgIG1pc3NpbmdfZ3JvdXBzID0gW2NvbHVtbiBmb3IgY29sdW1uIGluIGdyb3VwX2NvbHMgaWYgY29sdW1uIG5vdCBpbiBwcmVkaWN0aW9ucy5jb2x1bW5zXQogICAgaWYgbWlzc2luZ19ncm91cHM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk1pc3NpbmcgZ3JvdXAgY29sdW1uczoge21pc3NpbmdfZ3JvdXBzfSIpCgogICAgZGF0YSA9IHByZWRpY3Rpb25zLmNvcHkoKQogICAgZGF0YVsiX3ByZWRpY3RlZF9mYWlsIl0gPSBkYXRhW3ByZWRpY3Rpb25fY29sXS5hc3R5cGUoc3RyKS5zdHIubG93ZXIoKS5lcSgiZmFpbCIpCiAgICBkYXRhWyJfaGlnaF9yaXNrIl0gPSBkYXRhW3Byb2JhYmlsaXR5X2NvbF0gPj0gaGlnaF9yaXNrX3RocmVzaG9sZAoKICAgIHN1bW1hcnkgPSAoCiAgICAgICAgZGF0YS5ncm91cGJ5KGdyb3VwX2NvbHMsIGRyb3BuYT1GYWxzZSkKICAgICAgICAuYWdnKAogICAgICAgICAgICBuX3ByZWRpY3Rpb25zPShwcm9iYWJpbGl0eV9jb2wsICJzaXplIiksCiAgICAgICAgICAgIG1lYW5fZmFpbF9wcm9iYWJpbGl0eT0ocHJvYmFiaWxpdHlfY29sLCAibWVhbiIpLAogICAgICAgICAgICBtYXhfZmFpbF9wcm9iYWJpbGl0eT0ocHJvYmFiaWxpdHlfY29sLCAibWF4IiksCiAgICAgICAgICAgIHByZWRpY3RlZF9mYWlsX2NvdW50PSgiX3ByZWRpY3RlZF9mYWlsIiwgInN1bSIpLAogICAgICAgICAgICBoaWdoX3Jpc2tfY291bnQ9KCJfaGlnaF9yaXNrIiwgInN1bSIpLAogICAgICAgICkKICAgICAgICAucmVzZXRfaW5kZXgoKQogICAgKQogICAgc3VtbWFyeVsicHJlZGljdGVkX2ZhaWxfcmF0aW8iXSA9IHN1bW1hcnlbInByZWRpY3RlZF9mYWlsX2NvdW50Il0gLyBzdW1tYXJ5WyJuX3ByZWRpY3Rpb25zIl0KICAgIHN1bW1hcnlbImhpZ2hfcmlza19yYXRpbyJdID0gc3VtbWFyeVsiaGlnaF9yaXNrX2NvdW50Il0gLyBzdW1tYXJ5WyJuX3ByZWRpY3Rpb25zIl0KICAgIHN1bW1hcnlbImFsZXJ0X2ZsYWciXSA9IHN1bW1hcnlbImhpZ2hfcmlza19yYXRpbyJdID49IGFsZXJ0X3JhdGlvX3RocmVzaG9sZAoKICAgIHJldHVybiBzdW1tYXJ5LnNvcnRfdmFsdWVzKAogICAgICAgIFsiYWxlcnRfZmxhZyIsICJoaWdoX3Jpc2tfcmF0aW8iLCAibWVhbl9mYWlsX3Byb2JhYmlsaXR5Il0sCiAgICAgICAgYXNjZW5kaW5nPVtGYWxzZSwgRmFsc2UsIEZhbHNlXSwKICAgICkucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQoKCmRlZiB0b3Bfcmlza19wcmVkaWN0aW9ucygKICAgIHByZWRpY3Rpb25zOiBwZC5EYXRhRnJhbWUsCiAgICB0b3BfbjogaW50ID0gMjAsCiAgICBwcm9iYWJpbGl0eV9jb2w6IHN0ciA9ICJmYWlsX3Byb2JhYmlsaXR5IiwKKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJSZXR1cm4gdGhlIGhpZ2hlc3QtcmlzayBwcmVkaWN0aW9uIHJvd3MuIiIiCiAgICB2YWxpZGF0ZV9wcmVkaWN0aW9uX2NvbHVtbnMocHJlZGljdGlvbnMpCiAgICByZXR1cm4gcHJlZGljdGlvbnMuc29ydF92YWx1ZXMocHJvYmFiaWxpdHlfY29sLCBhc2NlbmRpbmc9RmFsc2UpLmhlYWQodG9wX24pLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkK', 'src/quality_reports.py': 'IiIiUXVhbGl0eSByZXBvcnQgdXRpbGl0aWVzIGZvciBTRUNPTSBhbmQgYXNzZW1ibGVkIG1hbnVmYWN0dXJpbmcgZGF0YXNldHMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSBzcmMuc2Vjb21fZGF0YSBpbXBvcnQgU2Vjb21EYXRhc2V0CgoKZGVmIGRhdGFzZXRfb3ZlcnZpZXcoZGF0YXNldDogU2Vjb21EYXRhc2V0KSAtPiBwZC5TZXJpZXM6CiAgICAiIiJSZXR1cm4gYSBjb21wYWN0IG92ZXJ2aWV3IG9mIGEgbG9hZGVkIFNFQ09NIGRhdGFzZXQuIiIiCiAgICBmZWF0dXJlcyA9IGRhdGFzZXQuZmVhdHVyZXMKICAgIGxhYmVscyA9IGRhdGFzZXQubGFiZWxzCiAgICB0aW1lc3RhbXBzID0gZGF0YXNldC50aW1lc3RhbXBzCgogICAgcmV0dXJuIHBkLlNlcmllcygKICAgICAgICB7CiAgICAgICAgICAgICJuX3NhbXBsZXMiOiBmZWF0dXJlcy5zaGFwZVswXSwKICAgICAgICAgICAgIm5fZmVhdHVyZXMiOiBmZWF0dXJlcy5zaGFwZVsxXSwKICAgICAgICAgICAgIm5fbGFiZWxzIjogbGFiZWxzLnNoYXBlWzBdLAogICAgICAgICAgICAicGFzc19jb3VudCI6IGludCgobGFiZWxzID09IDApLnN1bSgpKSwKICAgICAgICAgICAgImZhaWxfY291bnQiOiBpbnQoKGxhYmVscyA9PSAxKS5zdW0oKSksCiAgICAgICAgICAgICJmYWlsX3JhdGlvIjogZmxvYXQoKGxhYmVscyA9PSAxKS5tZWFuKCkpLAogICAgICAgICAgICAiZGF0ZV9taW4iOiB0aW1lc3RhbXBzLm1pbigpLAogICAgICAgICAgICAiZGF0ZV9tYXgiOiB0aW1lc3RhbXBzLm1heCgpLAogICAgICAgICAgICAidGltZXN0YW1wX21pc3NpbmciOiBpbnQodGltZXN0YW1wcy5pc25hKCkuc3VtKCkpLAogICAgICAgICAgICAiZHVwbGljYXRlZF9yb3dzIjogaW50KGZlYXR1cmVzLmR1cGxpY2F0ZWQoKS5zdW0oKSksCiAgICAgICAgICAgICJpbmZpbml0ZV92YWx1ZXMiOiBpbnQobnAuaXNpbmYoZmVhdHVyZXMudG9fbnVtcHkoZHR5cGU9ZmxvYXQpKS5zdW0oKSksCiAgICAgICAgICAgICJtaXNzaW5nX3ZhbHVlcyI6IGludChmZWF0dXJlcy5pc25hKCkuc3VtKCkuc3VtKCkpLAogICAgICAgIH0sCiAgICAgICAgbmFtZT0idmFsdWUiLAogICAgKQoKCmRlZiBjbGFzc19kaXN0cmlidXRpb24obGFiZWxzOiBwZC5TZXJpZXMpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIlJldHVybiBQYXNzL0ZhaWwgY2xhc3MgY291bnRzIGFuZCByYXRpb3MuIiIiCiAgICBkaXN0cmlidXRpb24gPSAoCiAgICAgICAgbGFiZWxzLm1hcCh7MDogIlBhc3MiLCAxOiAiRmFpbCJ9KQogICAgICAgIC52YWx1ZV9jb3VudHMoKQogICAgICAgIC5yZW5hbWVfYXhpcygiY2xhc3MiKQogICAgICAgIC5yZXNldF9pbmRleChuYW1lPSJjb3VudCIpCiAgICApCiAgICBkaXN0cmlidXRpb25bInJhdGlvIl0gPSBkaXN0cmlidXRpb25bImNvdW50Il0gLyBkaXN0cmlidXRpb25bImNvdW50Il0uc3VtKCkKICAgIHJldHVybiBkaXN0cmlidXRpb24KCgpkZWYgbWlzc2luZ25lc3NfcmVwb3J0KGZlYXR1cmVzOiBwZC5EYXRhRnJhbWUpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIlJldHVybiBmZWF0dXJlLWxldmVsIG1pc3NpbmcgY291bnRzIGFuZCByYXRpb3MuIiIiCiAgICByZXBvcnQgPSBwZC5EYXRhRnJhbWUoCiAgICAgICAgewogICAgICAgICAgICAibWlzc2luZ19jb3VudCI6IGZlYXR1cmVzLmlzbmEoKS5zdW0oKSwKICAgICAgICAgICAgIm1pc3NpbmdfcmF0aW8iOiBmZWF0dXJlcy5pc25hKCkubWVhbigpLAogICAgICAgIH0KICAgICkKICAgIHJldHVybiByZXBvcnQuc29ydF92YWx1ZXMoWyJtaXNzaW5nX3JhdGlvIiwgIm1pc3NpbmdfY291bnQiXSwgYXNjZW5kaW5nPUZhbHNlKQoKCmRlZiBoaWdoX21pc3NpbmdfZmVhdHVyZXMoZmVhdHVyZXM6IHBkLkRhdGFGcmFtZSwgdGhyZXNob2xkOiBmbG9hdCA9IDAuNSkgLT4gbGlzdFtzdHJdOgogICAgIiIiUmV0dXJuIGZlYXR1cmUgbmFtZXMgd2hvc2UgbWlzc2luZyByYXRpbyBpcyBncmVhdGVyIHRoYW4gb3IgZXF1YWwgdG8gdGhyZXNob2xkLiIiIgogICAgcmVwb3J0ID0gbWlzc2luZ25lc3NfcmVwb3J0KGZlYXR1cmVzKQogICAgcmV0dXJuIHJlcG9ydFtyZXBvcnRbIm1pc3NpbmdfcmF0aW8iXSA+PSB0aHJlc2hvbGRdLmluZGV4LnRvbGlzdCgpCgoKZGVmIGNvbnN0YW50X2ZlYXR1cmVzKGZlYXR1cmVzOiBwZC5EYXRhRnJhbWUpIC0+IGxpc3Rbc3RyXToKICAgICIiIlJldHVybiBmZWF0dXJlIG5hbWVzIHdpdGggYSBzaW5nbGUgdW5pcXVlIHZhbHVlLCBjb3VudGluZyBOYU4gYXMgYSB2YWx1ZS4iIiIKICAgIG5fdW5pcXVlID0gZmVhdHVyZXMubnVuaXF1ZShkcm9wbmE9RmFsc2UpCiAgICByZXR1cm4gbl91bmlxdWVbbl91bmlxdWUgPD0gMV0uaW5kZXgudG9saXN0KCkKCgpkZWYgcXVhbGl0eV9yZXBvcnRfYnVuZGxlKGRhdGFzZXQ6IFNlY29tRGF0YXNldCwgbWlzc2luZ190aHJlc2hvbGQ6IGZsb2F0ID0gMC41KSAtPiBkaWN0W3N0ciwgcGQuRGF0YUZyYW1lIHwgcGQuU2VyaWVzIHwgbGlzdFtzdHJdXToKICAgICIiIlJldHVybiB0aGUgc3RhbmRhcmQgc2V0IG9mIGRhdGEgcXVhbGl0eSByZXBvcnRzIGZvciBhIFNFQ09NIGRhdGFzZXQuIiIiCiAgICByZXR1cm4gewogICAgICAgICJvdmVydmlldyI6IGRhdGFzZXRfb3ZlcnZpZXcoZGF0YXNldCksCiAgICAgICAgImNsYXNzX2Rpc3RyaWJ1dGlvbiI6IGNsYXNzX2Rpc3RyaWJ1dGlvbihkYXRhc2V0LmxhYmVscyksCiAgICAgICAgIm1pc3NpbmduZXNzIjogbWlzc2luZ25lc3NfcmVwb3J0KGRhdGFzZXQuZmVhdHVyZXMpLAogICAgICAgICJoaWdoX21pc3NpbmdfZmVhdHVyZXMiOiBoaWdoX21pc3NpbmdfZmVhdHVyZXMoZGF0YXNldC5mZWF0dXJlcywgdGhyZXNob2xkPW1pc3NpbmdfdGhyZXNob2xkKSwKICAgICAgICAiY29uc3RhbnRfZmVhdHVyZXMiOiBjb25zdGFudF9mZWF0dXJlcyhkYXRhc2V0LmZlYXR1cmVzKSwKICAgIH0K', 'src/reporting.py': 'IiIiTWFya2Rvd24gcmVwb3J0aW5nIHV0aWxpdGllcyBmb3IgU0VDT00gYW5hbHl0aWNzIG91dHB1dHMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwYW5kYXMgYXMgcGQKCgpkZWYgZGF0YWZyYW1lX3RvX21hcmtkb3duKGRmOiBwZC5EYXRhRnJhbWUsIG1heF9yb3dzOiBpbnQgPSAyMCkgLT4gc3RyOgogICAgIiIiUmVuZGVyIGEgRGF0YUZyYW1lIGFzIGEgc2ltcGxlIEdpdEh1Yi1jb21wYXRpYmxlIE1hcmtkb3duIHRhYmxlLiIiIgogICAgaWYgZGYuZW1wdHk6CiAgICAgICAgcmV0dXJuICJfTm8gcm93cy5fIgoKICAgIGRpc3BsYXlfZGYgPSBkZi5oZWFkKG1heF9yb3dzKS5jb3B5KCkKICAgIGhlYWRlcnMgPSBbc3RyKGNvbHVtbikgZm9yIGNvbHVtbiBpbiBkaXNwbGF5X2RmLmNvbHVtbnNdCiAgICByb3dzID0gW1tzdHIodmFsdWUpIGZvciB2YWx1ZSBpbiByb3ddIGZvciByb3cgaW4gZGlzcGxheV9kZi50b19udW1weSgpXQoKICAgIGxpbmVzID0gWwogICAgICAgICJ8ICIgKyAiIHwgIi5qb2luKGhlYWRlcnMpICsgIiB8IiwKICAgICAgICAifCAiICsgIiB8ICIuam9pbihbIi0tLSJdICogbGVuKGhlYWRlcnMpKSArICIgfCIsCiAgICBdCiAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgbGluZXMuYXBwZW5kKCJ8ICIgKyAiIHwgIi5qb2luKHJvdykgKyAiIHwiKQogICAgcmV0dXJuICJcbiIuam9pbihsaW5lcykKCgpkZWYgcmVhZF9jc3ZfaWZfZXhpc3RzKHBhdGg6IFBhdGgpIC0+IHBkLkRhdGFGcmFtZSB8IE5vbmU6CiAgICAiIiJSZWFkIGEgQ1NWIGZpbGUgb25seSB3aGVuIGl0IGV4aXN0cy4iIiIKICAgIHBhdGggPSBQYXRoKHBhdGgpCiAgICBpZiBub3QgcGF0aC5leGlzdHMoKToKICAgICAgICByZXR1cm4gTm9uZQogICAgcmV0dXJuIHBkLnJlYWRfY3N2KHBhdGgpCgoKZGVmIGFkZF90YWJsZV9zZWN0aW9uKAogICAgc2VjdGlvbnM6IGxpc3Rbc3RyXSwKICAgIHRpdGxlOiBzdHIsCiAgICBjc3ZfcGF0aDogUGF0aCwKICAgIG1heF9yb3dzOiBpbnQgPSAyMCwKKSAtPiBOb25lOgogICAgIiIiQXBwZW5kIGEgTWFya2Rvd24gc2VjdGlvbiBmb3IgYSBDU1YgdGFibGUgd2hlbiB0aGUgZmlsZSBleGlzdHMuIiIiCiAgICBkZiA9IHJlYWRfY3N2X2lmX2V4aXN0cyhjc3ZfcGF0aCkKICAgIGlmIGRmIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuCiAgICBzZWN0aW9ucy5hcHBlbmQoZiIjIyB7dGl0bGV9XG5cblNvdXJjZTogYHtjc3ZfcGF0aH1gXG5cbntkYXRhZnJhbWVfdG9fbWFya2Rvd24oZGYsIG1heF9yb3dzPW1heF9yb3dzKX0iKQoKCmRlZiBidWlsZF9zdW1tYXJ5X3JlcG9ydCgKICAgIHJlcG9ydHNfZGlyOiBQYXRoID0gUGF0aCgib3V0cHV0cy9yZXBvcnRzIiksCiAgICBtb25pdG9yaW5nX2RpcjogUGF0aCA9IFBhdGgoIm91dHB1dHMvcmVwb3J0cy9tb25pdG9yaW5nIiksCiAgICBvdXRwdXRfcGF0aDogUGF0aCA9IFBhdGgoIm91dHB1dHMvcmVwb3J0cy9zdW1tYXJ5X3JlcG9ydC5tZCIpLAopIC0+IHN0cjoKICAgICIiIkJ1aWxkIGEgTWFya2Rvd24gc3VtbWFyeSByZXBvcnQgZnJvbSBnZW5lcmF0ZWQgQ1NWIHJlcG9ydCBmaWxlcy4iIiIKICAgIHJlcG9ydHNfZGlyID0gUGF0aChyZXBvcnRzX2RpcikKICAgIG1vbml0b3JpbmdfZGlyID0gUGF0aChtb25pdG9yaW5nX2RpcikKCiAgICBzZWN0aW9ucyA9IFsKICAgICAgICAiIyBTRUNPTSBNYW51ZmFjdHVyaW5nIEFuYWx5dGljcyBTdW1tYXJ5IiwKICAgICAgICAoCiAgICAgICAgICAgICJUaGlzIHJlcG9ydCBzdW1tYXJpemVzIGF2YWlsYWJsZSBkYXRhIHF1YWxpdHksIGZlYXR1cmUgYXNzZW1ibHksICIKICAgICAgICAgICAgInByZWRpY3Rpb24gbW9uaXRvcmluZywgYW5kIHJpc2sgb3V0cHV0cyBnZW5lcmF0ZWQgYnkgdGhlIHBpcGVsaW5lLiIKICAgICAgICApLAogICAgXQoKICAgIGFkZF90YWJsZV9zZWN0aW9uKHNlY3Rpb25zLCAiRGF0YSBPdmVydmlldyIsIHJlcG9ydHNfZGlyIC8gInF1YWxpdHkiIC8gIm92ZXJ2aWV3LmNzdiIpCiAgICBhZGRfdGFibGVfc2VjdGlvbihzZWN0aW9ucywgIkNsYXNzIERpc3RyaWJ1dGlvbiIsIHJlcG9ydHNfZGlyIC8gInF1YWxpdHkiIC8gImNsYXNzX2Rpc3RyaWJ1dGlvbi5jc3YiKQogICAgYWRkX3RhYmxlX3NlY3Rpb24oc2VjdGlvbnMsICJUb3AgTWlzc2luZyBGZWF0dXJlcyIsIHJlcG9ydHNfZGlyIC8gInF1YWxpdHkiIC8gIm1pc3NpbmduZXNzLmNzdiIsIG1heF9yb3dzPTIwKQogICAgYWRkX3RhYmxlX3NlY3Rpb24oc2VjdGlvbnMsICJGZWF0dXJlIEpvaW4gUmVwb3J0IiwgcmVwb3J0c19kaXIgLyAiZmVhdHVyZV9qb2luX3JlcG9ydC5jc3YiKQogICAgYWRkX3RhYmxlX3NlY3Rpb24oc2VjdGlvbnMsICJGZWF0dXJlIE1pc3NpbmduZXNzIFJlcG9ydCIsIHJlcG9ydHNfZGlyIC8gImZlYXR1cmVfbWlzc2luZ25lc3NfcmVwb3J0LmNzdiIsIG1heF9yb3dzPTIwKQogICAgYWRkX3RhYmxlX3NlY3Rpb24oc2VjdGlvbnMsICJPdmVyYWxsIFJpc2sgU3VtbWFyeSIsIG1vbml0b3JpbmdfZGlyIC8gIm92ZXJhbGxfcmlza19zdW1tYXJ5LmNzdiIpCiAgICBhZGRfdGFibGVfc2VjdGlvbihzZWN0aW9ucywgIkdyb3VwIFJpc2sgU3VtbWFyeSIsIG1vbml0b3JpbmdfZGlyIC8gImdyb3VwX3Jpc2tfc3VtbWFyeS5jc3YiLCBtYXhfcm93cz0yMCkKICAgIGFkZF90YWJsZV9zZWN0aW9uKHNlY3Rpb25zLCAiVG9wIFJpc2sgUHJlZGljdGlvbnMiLCBtb25pdG9yaW5nX2RpciAvICJ0b3Bfcmlza19wcmVkaWN0aW9ucy5jc3YiLCBtYXhfcm93cz0yMCkKCiAgICByZXBvcnQgPSAiXG5cbiIuam9pbihzZWN0aW9ucykgKyAiXG4iCiAgICBvdXRwdXRfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgb3V0cHV0X3BhdGgud3JpdGVfdGV4dChyZXBvcnQsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICByZXR1cm4gcmVwb3J0Cg==', 'src/secom_data.py': 'IiIiRGF0YSBsb2FkaW5nIHV0aWxpdGllcyBmb3IgdGhlIFNFQ09NIGFuYWx5c2lzIHByb2plY3QuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdXJsbGliLnJlcXVlc3QgaW1wb3J0IHVybHJldHJpZXZlCgppbXBvcnQgcGFuZGFzIGFzIHBkCgoKRkVBVFVSRV9VUkwgPSAiaHR0cHM6Ly9hcmNoaXZlLmljcy51Y2kuZWR1L21sL21hY2hpbmUtbGVhcm5pbmctZGF0YWJhc2VzL3NlY29tL3NlY29tLmRhdGEiCkxBQkVMX1VSTCA9ICJodHRwczovL2FyY2hpdmUuaWNzLnVjaS5lZHUvbWwvbWFjaGluZS1sZWFybmluZy1kYXRhYmFzZXMvc2Vjb20vc2Vjb21fbGFiZWxzLmRhdGEiCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgU2Vjb21EYXRhc2V0OgogICAgIiIiQ29udGFpbmVyIGZvciBsb2FkZWQgU0VDT00gZmVhdHVyZSwgbGFiZWwsIGFuZCB0aW1lc3RhbXAgZGF0YS4iIiIKCiAgICBmZWF0dXJlczogcGQuRGF0YUZyYW1lCiAgICBsYWJlbHM6IHBkLlNlcmllcwogICAgdGltZXN0YW1wczogcGQuU2VyaWVzCiAgICByYXdfbGFiZWxzOiBwZC5EYXRhRnJhbWUKCgpkZWYgZG93bmxvYWRfaWZfbWlzc2luZyh1cmw6IHN0ciwgZGVzdGluYXRpb246IFBhdGgpIC0+IFBhdGg6CiAgICAiIiJEb3dubG9hZCBhIGZpbGUgb25seSB3aGVuIGl0IGlzIG1pc3Npbmcgb3IgZW1wdHkuIiIiCiAgICBkZXN0aW5hdGlvbiA9IFBhdGgoZGVzdGluYXRpb24pCiAgICBkZXN0aW5hdGlvbi5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIGlmIGRlc3RpbmF0aW9uLmV4aXN0cygpIGFuZCBkZXN0aW5hdGlvbi5zdGF0KCkuc3Rfc2l6ZSA+IDA6CiAgICAgICAgcmV0dXJuIGRlc3RpbmF0aW9uCgogICAgdXJscmV0cmlldmUodXJsLCBkZXN0aW5hdGlvbikKICAgIHJldHVybiBkZXN0aW5hdGlvbgoKCmRlZiBkZWZhdWx0X3NlY29tX3BhdGhzKHJhd19kYXRhX2RpcjogUGF0aCkgLT4gdHVwbGVbUGF0aCwgUGF0aF06CiAgICAiIiJSZXR1cm4gdGhlIGRlZmF1bHQgU0VDT00gZmVhdHVyZSBhbmQgbGFiZWwgZmlsZSBwYXRocy4iIiIKICAgIHJhd19kYXRhX2RpciA9IFBhdGgocmF3X2RhdGFfZGlyKQogICAgcmV0dXJuIHJhd19kYXRhX2RpciAvICJzZWNvbS5kYXRhIiwgcmF3X2RhdGFfZGlyIC8gInNlY29tX2xhYmVscy5kYXRhIgoKCmRlZiBkb3dubG9hZF9zZWNvbV9kYXRhc2V0KHJhd19kYXRhX2RpcjogUGF0aCkgLT4gdHVwbGVbUGF0aCwgUGF0aF06CiAgICAiIiJEb3dubG9hZCB0aGUgVUNJIFNFQ09NIGZlYXR1cmUgYW5kIGxhYmVsIGZpbGVzIGlmIG5lZWRlZC4iIiIKICAgIGZlYXR1cmVfcGF0aCwgbGFiZWxfcGF0aCA9IGRlZmF1bHRfc2Vjb21fcGF0aHMocmF3X2RhdGFfZGlyKQogICAgZG93bmxvYWRfaWZfbWlzc2luZyhGRUFUVVJFX1VSTCwgZmVhdHVyZV9wYXRoKQogICAgZG93bmxvYWRfaWZfbWlzc2luZyhMQUJFTF9VUkwsIGxhYmVsX3BhdGgpCiAgICByZXR1cm4gZmVhdHVyZV9wYXRoLCBsYWJlbF9wYXRoCgoKZGVmIGxvYWRfc2Vjb21fZGF0YShmZWF0dXJlX3BhdGg6IFBhdGgsIGxhYmVsX3BhdGg6IFBhdGgpIC0+IFNlY29tRGF0YXNldDoKICAgICIiIkxvYWQgU0VDT00gZmVhdHVyZXMsIGNvbnZlcnRlZCBsYWJlbHMsIHRpbWVzdGFtcHMsIGFuZCByYXcgbGFiZWxzLgoKICAgIE9yaWdpbmFsIGxhYmVscyBhcmUgbWFwcGVkIGFzIGZvbGxvd3M6CiAgICAtIC0xIC0+IDAgKFBhc3MpCiAgICAtIDEgLT4gMSAoRmFpbCkKICAgICIiIgogICAgZmVhdHVyZV9wYXRoID0gUGF0aChmZWF0dXJlX3BhdGgpCiAgICBsYWJlbF9wYXRoID0gUGF0aChsYWJlbF9wYXRoKQoKICAgIGZlYXR1cmVzID0gcGQucmVhZF9jc3YoCiAgICAgICAgZmVhdHVyZV9wYXRoLAogICAgICAgIHNlcD1yIlxzKyIsCiAgICAgICAgaGVhZGVyPU5vbmUsCiAgICAgICAgbmFfdmFsdWVzPSJOYU4iLAogICAgKQogICAgZmVhdHVyZXMuY29sdW1ucyA9IFtmImZlYXR1cmVfe2lkeDowM2R9IiBmb3IgaWR4IGluIHJhbmdlKGZlYXR1cmVzLnNoYXBlWzFdKV0KCiAgICByYXdfbGFiZWxzID0gcGQucmVhZF9jc3YoCiAgICAgICAgbGFiZWxfcGF0aCwKICAgICAgICBzZXA9ciJccysiLAogICAgICAgIGhlYWRlcj1Ob25lLAogICAgICAgIGVuZ2luZT0icHl0aG9uIiwKICAgICkKCiAgICByYXdfbGFiZWwgPSByYXdfbGFiZWxzLmlsb2NbOiwgMF0uYXN0eXBlKGludCkKICAgIGxhYmVscyA9IHJhd19sYWJlbC5yZXBsYWNlKHstMTogMCwgMTogMX0pLnJlbmFtZSgidGFyZ2V0IikKCiAgICB0aW1lc3RhbXBzID0gcmF3X2xhYmVscy5pbG9jWzosIDE6XS5hc3R5cGUoc3RyKS5hZ2coIiAiLmpvaW4sIGF4aXM9MSkKICAgIHRpbWVzdGFtcHMgPSB0aW1lc3RhbXBzLnN0ci5yZXBsYWNlKCciJywgIiIsIHJlZ2V4PUZhbHNlKQogICAgdGltZXN0YW1wcyA9IHBkLnRvX2RhdGV0aW1lKHRpbWVzdGFtcHMsIGVycm9ycz0iY29lcmNlIiwgZGF5Zmlyc3Q9VHJ1ZSkucmVuYW1lKCJ0aW1lc3RhbXAiKQoKICAgIGlmIGxlbihmZWF0dXJlcykgIT0gbGVuKGxhYmVscyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgZiJGZWF0dXJlIHJvd3MgKHtsZW4oZmVhdHVyZXMpfSkgYW5kIGxhYmVsIHJvd3MgKHtsZW4obGFiZWxzKX0pIGRvIG5vdCBtYXRjaC4iCiAgICAgICAgKQoKICAgIHJldHVybiBTZWNvbURhdGFzZXQoCiAgICAgICAgZmVhdHVyZXM9ZmVhdHVyZXMsCiAgICAgICAgbGFiZWxzPWxhYmVscywKICAgICAgICB0aW1lc3RhbXBzPXRpbWVzdGFtcHMsCiAgICAgICAgcmF3X2xhYmVscz1yYXdfbGFiZWxzLAogICAgKQoKCmRlZiByZWFkX29wdGlvbmFsX3RhYmxlKHBhdGg6IFBhdGggfCBOb25lKSAtPiBwZC5EYXRhRnJhbWUgfCBOb25lOgogICAgIiIiUmVhZCBhbiBvcHRpb25hbCBDU1YvVFNWL0V4Y2VsIHRhYmxlIHdpdGhvdXQgZmFpbGluZyB3aGVuIGl0IGlzIGFic2VudC4iIiIKICAgIGlmIHBhdGggaXMgTm9uZToKICAgICAgICByZXR1cm4gTm9uZQoKICAgIHBhdGggPSBQYXRoKHBhdGgpCiAgICBpZiBub3QgcGF0aC5leGlzdHMoKToKICAgICAgICByZXR1cm4gTm9uZQoKICAgIHN1ZmZpeCA9IHBhdGguc3VmZml4Lmxvd2VyKCkKICAgIGlmIHN1ZmZpeCA9PSAiLmNzdiI6CiAgICAgICAgcmV0dXJuIHBkLnJlYWRfY3N2KHBhdGgpCiAgICBpZiBzdWZmaXggaW4geyIudHN2IiwgIi50eHQifToKICAgICAgICByZXR1cm4gcGQucmVhZF9jc3YocGF0aCwgc2VwPSJcdCIpCiAgICBpZiBzdWZmaXggaW4geyIueGxzeCIsICIueGxzIn06CiAgICAgICAgcmV0dXJuIHBkLnJlYWRfZXhjZWwocGF0aCkKCiAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5zdXBwb3J0ZWQgb3B0aW9uYWwgZGF0YSBmb3JtYXQ6IHtwYXRoLnN1ZmZpeH0iKQoKCmRlZiBmaXJzdF9leGlzdGluZ19wYXRoKGNhbmRpZGF0ZXM6IGxpc3RbUGF0aF0pIC0+IFBhdGggfCBOb25lOgogICAgIiIiUmV0dXJuIHRoZSBmaXJzdCBleGlzdGluZyBwYXRoIGZyb20gYSBsaXN0IG9mIGNhbmRpZGF0ZXMuIiIiCiAgICByZXR1cm4gbmV4dCgoUGF0aChwYXRoKSBmb3IgcGF0aCBpbiBjYW5kaWRhdGVzIGlmIFBhdGgocGF0aCkuZXhpc3RzKCkpLCBOb25lKQo=', 'src/secom_modeling.py': 'IiIiTW9kZWxpbmcgdXRpbGl0aWVzIGZvciBsZWFrYWdlLXNhZmUgU0VDT00gY2xhc3NpZmljYXRpb24gd29ya2Zsb3dzLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgoKQGRhdGFjbGFzcwpjbGFzcyBQcmVkaWN0aW9uUmVzdWx0OgogICAgIiIiQ29udGFpbmVyIGZvciB0aHJlc2hvbGQtYmFzZWQgY2xhc3NpZmljYXRpb24gb3V0cHV0LiIiIgoKICAgIHByZWRpY3Rpb246IG5wLm5kYXJyYXkKICAgIGZhaWxfcHJvYmFiaWxpdHk6IG5wLm5kYXJyYXkKICAgIGRlY2lzaW9uX3RocmVzaG9sZDogZmxvYXQKCiAgICBkZWYgdG9fZnJhbWUoc2VsZiwgaW5kZXg6IHBkLkluZGV4IHwgTm9uZSA9IE5vbmUpIC0+IHBkLkRhdGFGcmFtZToKICAgICAgICAiIiJSZXR1cm4gcHJlZGljdGlvbnMgYXMgYSB0YWJ1bGFyIHJlc3VsdC4iIiIKICAgICAgICByZXR1cm4gcGQuRGF0YUZyYW1lKAogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAicHJlZGljdGlvbiI6IHNlbGYucHJlZGljdGlvbiwKICAgICAgICAgICAgICAgICJmYWlsX3Byb2JhYmlsaXR5Ijogc2VsZi5mYWlsX3Byb2JhYmlsaXR5LAogICAgICAgICAgICAgICAgImRlY2lzaW9uX3RocmVzaG9sZCI6IHNlbGYuZGVjaXNpb25fdGhyZXNob2xkLAogICAgICAgICAgICB9LAogICAgICAgICAgICBpbmRleD1pbmRleCwKICAgICAgICApCgoKY2xhc3MgSGlnaE1pc3NpbmdGZWF0dXJlRHJvcHBlcjoKICAgICIiIkRyb3AgY29sdW1ucyB3aG9zZSBtaXNzaW5nIHJhdGlvIGlzIGdyZWF0ZXIgdGhhbiBvciBlcXVhbCB0byBhIHRocmVzaG9sZC4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgdGhyZXNob2xkOiBmbG9hdCA9IDAuNSk6CiAgICAgICAgc2VsZi50aHJlc2hvbGQgPSB0aHJlc2hvbGQKCiAgICBkZWYgZml0KHNlbGYsIFg6IHBkLkRhdGFGcmFtZSwgeTogQW55ID0gTm9uZSkgLT4gIkhpZ2hNaXNzaW5nRmVhdHVyZURyb3BwZXIiOgogICAgICAgICIiIkxlYXJuIHdoaWNoIGNvbHVtbnMgc2hvdWxkIGJlIHJldGFpbmVkLiIiIgogICAgICAgIFhfZGYgPSBwZC5EYXRhRnJhbWUoWCkuY29weSgpCiAgICAgICAgc2VsZi5mZWF0dXJlX25hbWVzX2luXyA9IFhfZGYuY29sdW1ucy50b19udW1weSgpCiAgICAgICAgc2VsZi5rZWVwX2NvbHVtbnNfID0gWF9kZi5jb2x1bW5zW1hfZGYuaXNuYSgpLm1lYW4oKSA8IHNlbGYudGhyZXNob2xkXS50b19saXN0KCkKICAgICAgICByZXR1cm4gc2VsZgoKICAgIGRlZiB0cmFuc2Zvcm0oc2VsZiwgWDogcGQuRGF0YUZyYW1lKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAgICAgIiIiUmV0dXJuIHRoZSByZXRhaW5lZCBjb2x1bW5zLiIiIgogICAgICAgIHNlbGYuX2NoZWNrX2lzX2ZpdHRlZCgpCiAgICAgICAgWF9kZiA9IHBkLkRhdGFGcmFtZShYLCBjb2x1bW5zPXNlbGYuZmVhdHVyZV9uYW1lc19pbl8pCiAgICAgICAgcmV0dXJuIFhfZGYubG9jWzosIHNlbGYua2VlcF9jb2x1bW5zX10KCiAgICBkZWYgZml0X3RyYW5zZm9ybShzZWxmLCBYOiBwZC5EYXRhRnJhbWUsIHk6IEFueSA9IE5vbmUpIC0+IHBkLkRhdGFGcmFtZToKICAgICAgICAiIiJGaXQgdGhlIHRyYW5zZm9ybWVyIGFuZCByZXR1cm4gdHJhbnNmb3JtZWQgZGF0YS4iIiIKICAgICAgICByZXR1cm4gc2VsZi5maXQoWCwgeSkudHJhbnNmb3JtKFgpCgogICAgZGVmIGdldF9wYXJhbXMoc2VsZiwgZGVlcDogYm9vbCA9IFRydWUpIC0+IGRpY3Rbc3RyLCBmbG9hdF06CiAgICAgICAgIiIiUmV0dXJuIHBhcmFtZXRlcnMgZm9yIHNrbGVhcm4gY29tcGF0aWJpbGl0eS4iIiIKICAgICAgICByZXR1cm4geyJ0aHJlc2hvbGQiOiBzZWxmLnRocmVzaG9sZH0KCiAgICBkZWYgc2V0X3BhcmFtcyhzZWxmLCAqKnBhcmFtczogQW55KSAtPiAiSGlnaE1pc3NpbmdGZWF0dXJlRHJvcHBlciI6CiAgICAgICAgIiIiU2V0IHBhcmFtZXRlcnMgZm9yIHNrbGVhcm4gY29tcGF0aWJpbGl0eS4iIiIKICAgICAgICBmb3Iga2V5LCB2YWx1ZSBpbiBwYXJhbXMuaXRlbXMoKToKICAgICAgICAgICAgc2V0YXR0cihzZWxmLCBrZXksIHZhbHVlKQogICAgICAgIHJldHVybiBzZWxmCgogICAgZGVmIF9jaGVja19pc19maXR0ZWQoc2VsZikgLT4gTm9uZToKICAgICAgICBpZiBub3QgaGFzYXR0cihzZWxmLCAia2VlcF9jb2x1bW5zXyIpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkhpZ2hNaXNzaW5nRmVhdHVyZURyb3BwZXIgbXVzdCBiZSBmaXR0ZWQgYmVmb3JlIHRyYW5zZm9ybS4iKQoKCmRlZiBtYWtlX2xpbmVhcl9waXBlbGluZSgKICAgIG1pc3NpbmdfdGhyZXNob2xkOiBmbG9hdCA9IDAuNSwKICAgIGxvd192YXJpYW5jZV90aHJlc2hvbGQ6IGZsb2F0ID0gMWUtOCwKICAgIGNsYXNzX3dlaWdodDogc3RyIHwgZGljdFtpbnQsIGZsb2F0XSB8IE5vbmUgPSBOb25lLAogICAgcmFuZG9tX3N0YXRlOiBpbnQgPSA0MiwKKToKICAgICIiIkNyZWF0ZSBhIGxlYWthZ2Utc2FmZSBwcmVwcm9jZXNzaW5nIGFuZCBMb2dpc3RpYyBSZWdyZXNzaW9uIHBpcGVsaW5lLiIiIgogICAgZnJvbSBza2xlYXJuLmZlYXR1cmVfc2VsZWN0aW9uIGltcG9ydCBWYXJpYW5jZVRocmVzaG9sZAogICAgZnJvbSBza2xlYXJuLmltcHV0ZSBpbXBvcnQgU2ltcGxlSW1wdXRlcgogICAgZnJvbSBza2xlYXJuLmxpbmVhcl9tb2RlbCBpbXBvcnQgTG9naXN0aWNSZWdyZXNzaW9uCiAgICBmcm9tIHNrbGVhcm4ucGlwZWxpbmUgaW1wb3J0IFBpcGVsaW5lCiAgICBmcm9tIHNrbGVhcm4ucHJlcHJvY2Vzc2luZyBpbXBvcnQgU3RhbmRhcmRTY2FsZXIKCiAgICByZXR1cm4gUGlwZWxpbmUoCiAgICAgICAgc3RlcHM9WwogICAgICAgICAgICAoImRyb3BfaGlnaF9taXNzaW5nIiwgSGlnaE1pc3NpbmdGZWF0dXJlRHJvcHBlcih0aHJlc2hvbGQ9bWlzc2luZ190aHJlc2hvbGQpKSwKICAgICAgICAgICAgKCJpbXB1dGVyIiwgU2ltcGxlSW1wdXRlcihzdHJhdGVneT0ibWVkaWFuIikpLAogICAgICAgICAgICAoInZhcmlhbmNlIiwgVmFyaWFuY2VUaHJlc2hvbGQodGhyZXNob2xkPWxvd192YXJpYW5jZV90aHJlc2hvbGQpKSwKICAgICAgICAgICAgKCJzY2FsZXIiLCBTdGFuZGFyZFNjYWxlcigpKSwKICAgICAgICAgICAgKAogICAgICAgICAgICAgICAgIm1vZGVsIiwKICAgICAgICAgICAgICAgIExvZ2lzdGljUmVncmVzc2lvbigKICAgICAgICAgICAgICAgICAgICBtYXhfaXRlcj0zMDAwLAogICAgICAgICAgICAgICAgICAgIGNsYXNzX3dlaWdodD1jbGFzc193ZWlnaHQsCiAgICAgICAgICAgICAgICAgICAgcmFuZG9tX3N0YXRlPXJhbmRvbV9zdGF0ZSwKICAgICAgICAgICAgICAgICksCiAgICAgICAgICAgICksCiAgICAgICAgXQogICAgKQoKCmRlZiBtYWtlX3RyZWVfcGlwZWxpbmUoCiAgICBtaXNzaW5nX3RocmVzaG9sZDogZmxvYXQgPSAwLjUsCiAgICBsb3dfdmFyaWFuY2VfdGhyZXNob2xkOiBmbG9hdCA9IDFlLTgsCiAgICBjbGFzc193ZWlnaHQ6IHN0ciB8IGRpY3RbaW50LCBmbG9hdF0gfCBOb25lID0gTm9uZSwKICAgIHJhbmRvbV9zdGF0ZTogaW50ID0gNDIsCiAgICBuX2VzdGltYXRvcnM6IGludCA9IDMwMCwKKToKICAgICIiIkNyZWF0ZSBhIGxlYWthZ2Utc2FmZSBwcmVwcm9jZXNzaW5nIGFuZCBSYW5kb20gRm9yZXN0IHBpcGVsaW5lLiIiIgogICAgZnJvbSBza2xlYXJuLmVuc2VtYmxlIGltcG9ydCBSYW5kb21Gb3Jlc3RDbGFzc2lmaWVyCiAgICBmcm9tIHNrbGVhcm4uZmVhdHVyZV9zZWxlY3Rpb24gaW1wb3J0IFZhcmlhbmNlVGhyZXNob2xkCiAgICBmcm9tIHNrbGVhcm4uaW1wdXRlIGltcG9ydCBTaW1wbGVJbXB1dGVyCiAgICBmcm9tIHNrbGVhcm4ucGlwZWxpbmUgaW1wb3J0IFBpcGVsaW5lCgogICAgcmV0dXJuIFBpcGVsaW5lKAogICAgICAgIHN0ZXBzPVsKICAgICAgICAgICAgKCJkcm9wX2hpZ2hfbWlzc2luZyIsIEhpZ2hNaXNzaW5nRmVhdHVyZURyb3BwZXIodGhyZXNob2xkPW1pc3NpbmdfdGhyZXNob2xkKSksCiAgICAgICAgICAgICgiaW1wdXRlciIsIFNpbXBsZUltcHV0ZXIoc3RyYXRlZ3k9Im1lZGlhbiIpKSwKICAgICAgICAgICAgKCJ2YXJpYW5jZSIsIFZhcmlhbmNlVGhyZXNob2xkKHRocmVzaG9sZD1sb3dfdmFyaWFuY2VfdGhyZXNob2xkKSksCiAgICAgICAgICAgICgKICAgICAgICAgICAgICAgICJtb2RlbCIsCiAgICAgICAgICAgICAgICBSYW5kb21Gb3Jlc3RDbGFzc2lmaWVyKAogICAgICAgICAgICAgICAgICAgIG5fZXN0aW1hdG9ycz1uX2VzdGltYXRvcnMsCiAgICAgICAgICAgICAgICAgICAgY2xhc3Nfd2VpZ2h0PWNsYXNzX3dlaWdodCwKICAgICAgICAgICAgICAgICAgICByYW5kb21fc3RhdGU9cmFuZG9tX3N0YXRlLAogICAgICAgICAgICAgICAgICAgIG5fam9icz0tMSwKICAgICAgICAgICAgICAgICksCiAgICAgICAgICAgICksCiAgICAgICAgXQogICAgKQoKCmRlZiBnZXRfcG9zaXRpdmVfcHJvYmEobW9kZWw6IEFueSwgWDogcGQuRGF0YUZyYW1lKSAtPiBucC5uZGFycmF5OgogICAgIiIiUmV0dXJuIHByZWRpY3RlZCBwcm9iYWJpbGl0eSBmb3IgdGhlIEZhaWwgY2xhc3Mgd2hlbiBhdmFpbGFibGUuIiIiCiAgICBpZiBoYXNhdHRyKG1vZGVsLCAicHJlZGljdF9wcm9iYSIpOgogICAgICAgIHByb2JhID0gbnAuYXNhcnJheShtb2RlbC5wcmVkaWN0X3Byb2JhKFgpKQogICAgICAgIGlmIHByb2JhLm5kaW0gIT0gMiBvciBwcm9iYS5zaGFwZVsxXSA8IDI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInByZWRpY3RfcHJvYmEgbXVzdCByZXR1cm4gYSAyRCBhcnJheSB3aXRoIGF0IGxlYXN0IHR3byBjb2x1bW5zLiIpCiAgICAgICAgcmV0dXJuIHByb2JhWzosIDFdCgogICAgaWYgaGFzYXR0cihtb2RlbCwgImRlY2lzaW9uX2Z1bmN0aW9uIik6CiAgICAgICAgc2NvcmVzID0gbnAuYXNhcnJheShtb2RlbC5kZWNpc2lvbl9mdW5jdGlvbihYKSwgZHR5cGU9ZmxvYXQpCiAgICAgICAgc2NvcmVfcmFuZ2UgPSBzY29yZXMubWF4KCkgLSBzY29yZXMubWluKCkKICAgICAgICBpZiBzY29yZV9yYW5nZSA9PSAwOgogICAgICAgICAgICByZXR1cm4gbnAuZnVsbChzaGFwZT1zY29yZXMuc2hhcGUsIGZpbGxfdmFsdWU9MC41LCBkdHlwZT1mbG9hdCkKICAgICAgICByZXR1cm4gKHNjb3JlcyAtIHNjb3Jlcy5taW4oKSkgLyBzY29yZV9yYW5nZQoKICAgIHJhaXNlIEF0dHJpYnV0ZUVycm9yKCJNb2RlbCBkb2VzIG5vdCBleHBvc2UgcHJlZGljdF9wcm9iYSBvciBkZWNpc2lvbl9mdW5jdGlvbi4iKQoKCmRlZiBwcmVkaWN0X3dpdGhfdGhyZXNob2xkKG1vZGVsOiBBbnksIFg6IHBkLkRhdGFGcmFtZSwgdGhyZXNob2xkOiBmbG9hdCA9IDAuNSkgLT4gUHJlZGljdGlvblJlc3VsdDoKICAgICIiIlByZWRpY3QgUGFzcy9GYWlsIGxhYmVscyBhbmQgRmFpbCBwcm9iYWJpbGl0aWVzIHVzaW5nIGEgZGVjaXNpb24gdGhyZXNob2xkLiIiIgogICAgZmFpbF9wcm9iYWJpbGl0eSA9IGdldF9wb3NpdGl2ZV9wcm9iYShtb2RlbCwgWCkKICAgIHByZWRpY3Rpb24gPSBucC53aGVyZShmYWlsX3Byb2JhYmlsaXR5ID49IHRocmVzaG9sZCwgIkZhaWwiLCAiUGFzcyIpCiAgICByZXR1cm4gUHJlZGljdGlvblJlc3VsdCgKICAgICAgICBwcmVkaWN0aW9uPXByZWRpY3Rpb24sCiAgICAgICAgZmFpbF9wcm9iYWJpbGl0eT1mYWlsX3Byb2JhYmlsaXR5LAogICAgICAgIGRlY2lzaW9uX3RocmVzaG9sZD10aHJlc2hvbGQsCiAgICApCgoKZGVmIGV2YWx1YXRlX2NsYXNzaWZpZXIoCiAgICBuYW1lOiBzdHIsCiAgICBpbWJhbGFuY2VfbWV0aG9kOiBzdHIsCiAgICBtb2RlbDogQW55LAogICAgWF9ldmFsOiBwZC5EYXRhRnJhbWUsCiAgICB5X2V2YWw6IHBkLlNlcmllcyB8IG5wLm5kYXJyYXksCiAgICB0aHJlc2hvbGQ6IGZsb2F0ID0gMC41LAopIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgIiIiRXZhbHVhdGUgYSBmaXR0ZWQgY2xhc3NpZmllciB3aXRoIG1ldHJpY3MgZm9jdXNlZCBvbiBGYWlsIGRldGVjdGlvbi4iIiIKICAgIGZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCAoCiAgICAgICAgYWNjdXJhY3lfc2NvcmUsCiAgICAgICAgYXZlcmFnZV9wcmVjaXNpb25fc2NvcmUsCiAgICAgICAgYmFsYW5jZWRfYWNjdXJhY3lfc2NvcmUsCiAgICAgICAgY29uZnVzaW9uX21hdHJpeCwKICAgICAgICBmMV9zY29yZSwKICAgICAgICBwcmVjaXNpb25fc2NvcmUsCiAgICAgICAgcmVjYWxsX3Njb3JlLAogICAgICAgIHJvY19hdWNfc2NvcmUsCiAgICApCgogICAgeV9wcm9iYSA9IGdldF9wb3NpdGl2ZV9wcm9iYShtb2RlbCwgWF9ldmFsKQogICAgeV9wcmVkID0gKHlfcHJvYmEgPj0gdGhyZXNob2xkKS5hc3R5cGUoaW50KQogICAgdG4sIGZwLCBmbiwgdHAgPSBjb25mdXNpb25fbWF0cml4KHlfZXZhbCwgeV9wcmVkLCBsYWJlbHM9WzAsIDFdKS5yYXZlbCgpCgogICAgcmV0dXJuIHsKICAgICAgICAibW9kZWwiOiBuYW1lLAogICAgICAgICJpbWJhbGFuY2VfbWV0aG9kIjogaW1iYWxhbmNlX21ldGhvZCwKICAgICAgICAidGhyZXNob2xkIjogdGhyZXNob2xkLAogICAgICAgICJhY2N1cmFjeSI6IGFjY3VyYWN5X3Njb3JlKHlfZXZhbCwgeV9wcmVkKSwKICAgICAgICAiYmFsYW5jZWRfYWNjdXJhY3kiOiBiYWxhbmNlZF9hY2N1cmFjeV9zY29yZSh5X2V2YWwsIHlfcHJlZCksCiAgICAgICAgImZhaWxfcHJlY2lzaW9uIjogcHJlY2lzaW9uX3Njb3JlKHlfZXZhbCwgeV9wcmVkLCBwb3NfbGFiZWw9MSwgemVyb19kaXZpc2lvbj0wKSwKICAgICAgICAiZmFpbF9yZWNhbGwiOiByZWNhbGxfc2NvcmUoeV9ldmFsLCB5X3ByZWQsIHBvc19sYWJlbD0xLCB6ZXJvX2RpdmlzaW9uPTApLAogICAgICAgICJmYWlsX2YxIjogZjFfc2NvcmUoeV9ldmFsLCB5X3ByZWQsIHBvc19sYWJlbD0xLCB6ZXJvX2RpdmlzaW9uPTApLAogICAgICAgICJyb2NfYXVjIjogcm9jX2F1Y19zY29yZSh5X2V2YWwsIHlfcHJvYmEpLAogICAgICAgICJwcl9hdWMiOiBhdmVyYWdlX3ByZWNpc2lvbl9zY29yZSh5X2V2YWwsIHlfcHJvYmEpLAogICAgICAgICJ0biI6IHRuLAogICAgICAgICJmcCI6IGZwLAogICAgICAgICJmbiI6IGZuLAogICAgICAgICJ0cCI6IHRwLAogICAgfQo=', 'src/secom_training.py': 'IiIiVHJhaW5pbmcgaGVscGVycyBmb3IgU0VDT00gbW9kZWwgc2VsZWN0aW9uIGFuZCB0aHJlc2hvbGQgdHVuaW5nLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucy5hYmMgaW1wb3J0IEl0ZXJhYmxlLCBTZXF1ZW5jZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCgpERUZBVUxUX01PREVMX1JBTktJTkcgPSAoImZhaWxfcmVjYWxsIiwgImZhaWxfZjEiLCAicHJfYXVjIikKREVGQVVMVF9USFJFU0hPTERfUkFOS0lORyA9ICgiZmFpbF9mMSIsICJmYWlsX3JlY2FsbCIsICJwcl9hdWMiKQoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIFNlbGVjdGVkUmVzdWx0OgogICAgIiIiU2VsZWN0ZWQgcm93IG1ldGFkYXRhIGZyb20gYSByYW5rZWQgbW9kZWwgb3IgdGhyZXNob2xkIHRhYmxlLiIiIgoKICAgIHJvdzogZGljdFtzdHIsIEFueV0KICAgIHJhbmtpbmdfY29sdW1uczogdHVwbGVbc3RyLCAuLi5dCgogICAgQHByb3BlcnR5CiAgICBkZWYgbmFtZShzZWxmKSAtPiBzdHIgfCBOb25lOgogICAgICAgICIiIlJldHVybiB0aGUgc2VsZWN0ZWQgbW9kZWwgbmFtZSB3aGVuIHByZXNlbnQuIiIiCiAgICAgICAgdmFsdWUgPSBzZWxmLnJvdy5nZXQoIm1vZGVsIikKICAgICAgICByZXR1cm4gc3RyKHZhbHVlKSBpZiB2YWx1ZSBpcyBub3QgTm9uZSBlbHNlIE5vbmUKCgpkZWYgcGFyc2VfdGhyZXNob2xkcyh2YWx1ZTogc3RyIHwgSXRlcmFibGVbZmxvYXRdIHwgTm9uZSkgLT4gbGlzdFtmbG9hdF06CiAgICAiIiJQYXJzZSBjb21tYS1zZXBhcmF0ZWQgdGhyZXNob2xkcyBhbmQgdmFsaWRhdGUgdGhleSBhcmUgcHJvYmFiaWxpdGllcy4iIiIKICAgIGlmIHZhbHVlIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIFswLjEsIDAuMiwgMC4zLCAwLjQsIDAuNSwgMC42LCAwLjddCgogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgc3RyKToKICAgICAgICByYXdfdmFsdWVzID0gW2l0ZW0uc3RyaXAoKSBmb3IgaXRlbSBpbiB2YWx1ZS5zcGxpdCgiLCIpIGlmIGl0ZW0uc3RyaXAoKV0KICAgICAgICB0aHJlc2hvbGRzID0gW2Zsb2F0KGl0ZW0pIGZvciBpdGVtIGluIHJhd192YWx1ZXNdCiAgICBlbHNlOgogICAgICAgIHRocmVzaG9sZHMgPSBbZmxvYXQoaXRlbSkgZm9yIGl0ZW0gaW4gdmFsdWVdCgogICAgaW52YWxpZCA9IFt0aHJlc2hvbGQgZm9yIHRocmVzaG9sZCBpbiB0aHJlc2hvbGRzIGlmIHRocmVzaG9sZCA8IDAgb3IgdGhyZXNob2xkID4gMV0KICAgIGlmIGludmFsaWQ6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlRocmVzaG9sZHMgbXVzdCBiZSBiZXR3ZWVuIDAgYW5kIDE6IHtpbnZhbGlkfSIpCiAgICBpZiBub3QgdGhyZXNob2xkczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJBdCBsZWFzdCBvbmUgdGhyZXNob2xkIGlzIHJlcXVpcmVkLiIpCiAgICByZXR1cm4gdGhyZXNob2xkcwoKCmRlZiByYW5rX3Jlc3VsdHMoCiAgICByZXN1bHRzOiBwZC5EYXRhRnJhbWUsCiAgICByYW5raW5nX2NvbHVtbnM6IFNlcXVlbmNlW3N0cl0gPSBERUZBVUxUX01PREVMX1JBTktJTkcsCikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiU29ydCBhIHJlc3VsdCB0YWJsZSBmcm9tIGJlc3QgdG8gd29yc3QgdXNpbmcgZGVzY2VuZGluZyBtZXRyaWMgY29sdW1ucy4iIiIKICAgIHJhbmtpbmdfY29sdW1ucyA9IHR1cGxlKHJhbmtpbmdfY29sdW1ucykKICAgIG1pc3NpbmcgPSBbY29sdW1uIGZvciBjb2x1bW4gaW4gcmFua2luZ19jb2x1bW5zIGlmIGNvbHVtbiBub3QgaW4gcmVzdWx0cy5jb2x1bW5zXQogICAgaWYgbWlzc2luZzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTWlzc2luZyByYW5raW5nIGNvbHVtbnM6IHttaXNzaW5nfSIpCiAgICByZXR1cm4gcmVzdWx0cy5zb3J0X3ZhbHVlcyhsaXN0KHJhbmtpbmdfY29sdW1ucyksIGFzY2VuZGluZz1GYWxzZSkucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQoKCmRlZiBzZWxlY3RfdG9wX3Jlc3VsdCgKICAgIHJlc3VsdHM6IHBkLkRhdGFGcmFtZSwKICAgIHJhbmtpbmdfY29sdW1uczogU2VxdWVuY2Vbc3RyXSA9IERFRkFVTFRfTU9ERUxfUkFOS0lORywKKSAtPiBTZWxlY3RlZFJlc3VsdDoKICAgICIiIlJldHVybiB0aGUgaGlnaGVzdCByYW5rZWQgcm93IGZyb20gYSBtb2RlbCBvciB0aHJlc2hvbGQgcmVzdWx0IHRhYmxlLiIiIgogICAgaWYgcmVzdWx0cy5lbXB0eToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDYW5ub3Qgc2VsZWN0IGZyb20gYW4gZW1wdHkgcmVzdWx0IHRhYmxlLiIpCgogICAgcmFua2luZ19jb2x1bW5zID0gdHVwbGUocmFua2luZ19jb2x1bW5zKQogICAgcmFua2VkID0gcmFua19yZXN1bHRzKHJlc3VsdHMsIHJhbmtpbmdfY29sdW1ucz1yYW5raW5nX2NvbHVtbnMpCiAgICByZXR1cm4gU2VsZWN0ZWRSZXN1bHQocm93PXJhbmtlZC5pbG9jWzBdLnRvX2RpY3QoKSwgcmFua2luZ19jb2x1bW5zPXJhbmtpbmdfY29sdW1ucykKCgpkZWYgYnVpbGRfdGhyZXNob2xkX21ldHJpY3MoCiAgICB5X3RydWU6IHBkLlNlcmllcyB8IG5wLm5kYXJyYXksCiAgICBmYWlsX3Byb2JhYmlsaXR5OiBucC5uZGFycmF5LAogICAgdGhyZXNob2xkczogSXRlcmFibGVbZmxvYXRdLAopIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIkV2YWx1YXRlIHByZWNpc2lvbiwgcmVjYWxsLCBGMSwgYW5kIGNvbmZ1c2lvbiBjb3VudHMgZm9yIHRocmVzaG9sZHMuIiIiCiAgICBmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgKAogICAgICAgIGF2ZXJhZ2VfcHJlY2lzaW9uX3Njb3JlLAogICAgICAgIGNvbmZ1c2lvbl9tYXRyaXgsCiAgICAgICAgZjFfc2NvcmUsCiAgICAgICAgcHJlY2lzaW9uX3Njb3JlLAogICAgICAgIHJlY2FsbF9zY29yZSwKICAgICAgICByb2NfYXVjX3Njb3JlLAogICAgKQoKICAgIHJvd3MgPSBbXQogICAgeV90cnVlX2FycmF5ID0gbnAuYXNhcnJheSh5X3RydWUpCiAgICBmYWlsX3Byb2JhYmlsaXR5ID0gbnAuYXNhcnJheShmYWlsX3Byb2JhYmlsaXR5LCBkdHlwZT1mbG9hdCkKICAgIGZvciB0aHJlc2hvbGQgaW4gcGFyc2VfdGhyZXNob2xkcyh0aHJlc2hvbGRzKToKICAgICAgICB5X3ByZWQgPSAoZmFpbF9wcm9iYWJpbGl0eSA+PSB0aHJlc2hvbGQpLmFzdHlwZShpbnQpCiAgICAgICAgdG4sIGZwLCBmbiwgdHAgPSBjb25mdXNpb25fbWF0cml4KHlfdHJ1ZV9hcnJheSwgeV9wcmVkLCBsYWJlbHM9WzAsIDFdKS5yYXZlbCgpCiAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJ0aHJlc2hvbGQiOiB0aHJlc2hvbGQsCiAgICAgICAgICAgICAgICAiZmFpbF9wcmVjaXNpb24iOiBwcmVjaXNpb25fc2NvcmUoCiAgICAgICAgICAgICAgICAgICAgeV90cnVlX2FycmF5LCB5X3ByZWQsIHBvc19sYWJlbD0xLCB6ZXJvX2RpdmlzaW9uPTAKICAgICAgICAgICAgICAgICksCiAgICAgICAgICAgICAgICAiZmFpbF9yZWNhbGwiOiByZWNhbGxfc2NvcmUoeV90cnVlX2FycmF5LCB5X3ByZWQsIHBvc19sYWJlbD0xLCB6ZXJvX2RpdmlzaW9uPTApLAogICAgICAgICAgICAgICAgImZhaWxfZjEiOiBmMV9zY29yZSh5X3RydWVfYXJyYXksIHlfcHJlZCwgcG9zX2xhYmVsPTEsIHplcm9fZGl2aXNpb249MCksCiAgICAgICAgICAgICAgICAicm9jX2F1YyI6IHJvY19hdWNfc2NvcmUoeV90cnVlX2FycmF5LCBmYWlsX3Byb2JhYmlsaXR5KSwKICAgICAgICAgICAgICAgICJwcl9hdWMiOiBhdmVyYWdlX3ByZWNpc2lvbl9zY29yZSh5X3RydWVfYXJyYXksIGZhaWxfcHJvYmFiaWxpdHkpLAogICAgICAgICAgICAgICAgInRuIjogdG4sCiAgICAgICAgICAgICAgICAiZnAiOiBmcCwKICAgICAgICAgICAgICAgICJmbiI6IGZuLAogICAgICAgICAgICAgICAgInRwIjogdHAsCiAgICAgICAgICAgIH0KICAgICAgICApCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MpCg==', 'src/wafer_features.py': 'IiIiRmVhdHVyZSBlbmdpbmVlcmluZyB1dGlsaXRpZXMgZm9yIHdhZmVyIGRlZmVjdCBpbnNwZWN0aW9uIGRhdGEuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKClJFUVVJUkVEX1dBRkVSX0NPTFVNTlMgPSB7IndhZmVyX2lkIiwgIngiLCAieSJ9CgoKZGVmIHZhbGlkYXRlX3dhZmVyX2NvbHVtbnMoCiAgICB3YWZlcl9kZjogcGQuRGF0YUZyYW1lLAogICAgd2FmZXJfaWRfY29sOiBzdHIgPSAid2FmZXJfaWQiLAogICAgeF9jb2w6IHN0ciA9ICJ4IiwKICAgIHlfY29sOiBzdHIgPSAieSIsCikgLT4gTm9uZToKICAgICIiIlZhbGlkYXRlIHRoYXQgYSB3YWZlciBpbnNwZWN0aW9uIHRhYmxlIGhhcyByZXF1aXJlZCBjb29yZGluYXRlIGNvbHVtbnMuIiIiCiAgICByZXF1aXJlZCA9IHt3YWZlcl9pZF9jb2wsIHhfY29sLCB5X2NvbH0KICAgIG1pc3NpbmcgPSBzb3J0ZWQocmVxdWlyZWQuZGlmZmVyZW5jZSh3YWZlcl9kZi5jb2x1bW5zKSkKICAgIGlmIG1pc3Npbmc6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk1pc3Npbmcgd2FmZXIgaW5zcGVjdGlvbiBjb2x1bW5zOiB7bWlzc2luZ30iKQoKCmRlZiBhZGRfcmFkaWFsX3Bvc2l0aW9uKAogICAgd2FmZXJfZGY6IHBkLkRhdGFGcmFtZSwKICAgIHhfY29sOiBzdHIgPSAieCIsCiAgICB5X2NvbDogc3RyID0gInkiLAogICAgb3V0cHV0X2NvbDogc3RyID0gInJhZGl1c19ub3JtIiwKKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJBZGQgbm9ybWFsaXplZCByYWRpYWwgZGlzdGFuY2UgZnJvbSB0aGUgd2FmZXIgY2VudGVyLgoKICAgIENvb3JkaW5hdGVzIGFyZSBub3JtYWxpemVkIHVzaW5nIHRoZSBtYXhpbXVtIG9ic2VydmVkIGFic29sdXRlIHgveSBkaXN0YW5jZS4KICAgIFRoaXMga2VlcHMgdGhlIHV0aWxpdHkgdXNhYmxlIGZvciBzeW50aGV0aWMsIHBpeGVsLCBvciBkaWUtZ3JpZCBjb29yZGluYXRlcy4KICAgICIiIgogICAgcmVzdWx0ID0gd2FmZXJfZGYuY29weSgpCiAgICB4X2NlbnRlcmVkID0gcmVzdWx0W3hfY29sXSAtIHJlc3VsdFt4X2NvbF0ubWVhbigpCiAgICB5X2NlbnRlcmVkID0gcmVzdWx0W3lfY29sXSAtIHJlc3VsdFt5X2NvbF0ubWVhbigpCiAgICByYWRpdXMgPSBucC5zcXJ0KHhfY2VudGVyZWQqKjIgKyB5X2NlbnRlcmVkKioyKQogICAgbWF4X3JhZGl1cyA9IHJhZGl1cy5tYXgoKQogICAgcmVzdWx0W291dHB1dF9jb2xdID0gMC4wIGlmIG1heF9yYWRpdXMgPT0gMCBlbHNlIHJhZGl1cyAvIG1heF9yYWRpdXMKICAgIHJldHVybiByZXN1bHQKCgpkZWYgY2xhc3NpZnlfcmFkaWFsX3pvbmUocmFkaXVzX25vcm06IHBkLlNlcmllcywgY2VudGVyX2N1dG9mZjogZmxvYXQgPSAwLjMzLCBlZGdlX2N1dG9mZjogZmxvYXQgPSAwLjc1KSAtPiBwZC5TZXJpZXM6CiAgICAiIiJDbGFzc2lmeSBub3JtYWxpemVkIHJhZGl1cyBpbnRvIGNlbnRlciwgbWlkZGxlLCBhbmQgZWRnZSB6b25lcy4iIiIKICAgIHJldHVybiBwZC5TZXJpZXMoCiAgICAgICAgbnAuc2VsZWN0KAogICAgICAgICAgICBbcmFkaXVzX25vcm0gPD0gY2VudGVyX2N1dG9mZiwgcmFkaXVzX25vcm0gPj0gZWRnZV9jdXRvZmZdLAogICAgICAgICAgICBbImNlbnRlciIsICJlZGdlIl0sCiAgICAgICAgICAgIGRlZmF1bHQ9Im1pZGRsZSIsCiAgICAgICAgKSwKICAgICAgICBpbmRleD1yYWRpdXNfbm9ybS5pbmRleCwKICAgICAgICBuYW1lPSJyYWRpYWxfem9uZSIsCiAgICApCgoKZGVmIGFkZF9zcGF0aWFsX2JpbnMoCiAgICB3YWZlcl9kZjogcGQuRGF0YUZyYW1lLAogICAgeF9jb2w6IHN0ciA9ICJ4IiwKICAgIHlfY29sOiBzdHIgPSAieSIsCikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiQWRkIHJhZGlhbCB6b25lIGFuZCBxdWFkcmFudCBjb2x1bW5zIGZvciB3YWZlciBkZWZlY3QgY29vcmRpbmF0ZXMuIiIiCiAgICByZXN1bHQgPSBhZGRfcmFkaWFsX3Bvc2l0aW9uKHdhZmVyX2RmLCB4X2NvbD14X2NvbCwgeV9jb2w9eV9jb2wpCiAgICByZXN1bHRbInJhZGlhbF96b25lIl0gPSBjbGFzc2lmeV9yYWRpYWxfem9uZShyZXN1bHRbInJhZGl1c19ub3JtIl0pCiAgICByZXN1bHRbInF1YWRyYW50Il0gPSBucC5zZWxlY3QoCiAgICAgICAgWwogICAgICAgICAgICAocmVzdWx0W3hfY29sXSA+PSByZXN1bHRbeF9jb2xdLm1lYW4oKSkgJiAocmVzdWx0W3lfY29sXSA+PSByZXN1bHRbeV9jb2xdLm1lYW4oKSksCiAgICAgICAgICAgIChyZXN1bHRbeF9jb2xdIDwgcmVzdWx0W3hfY29sXS5tZWFuKCkpICYgKHJlc3VsdFt5X2NvbF0gPj0gcmVzdWx0W3lfY29sXS5tZWFuKCkpLAogICAgICAgICAgICAocmVzdWx0W3hfY29sXSA8IHJlc3VsdFt4X2NvbF0ubWVhbigpKSAmIChyZXN1bHRbeV9jb2xdIDwgcmVzdWx0W3lfY29sXS5tZWFuKCkpLAogICAgICAgICAgICAocmVzdWx0W3hfY29sXSA+PSByZXN1bHRbeF9jb2xdLm1lYW4oKSkgJiAocmVzdWx0W3lfY29sXSA8IHJlc3VsdFt5X2NvbF0ubWVhbigpKSwKICAgICAgICBdLAogICAgICAgIFsiUTEiLCAiUTIiLCAiUTMiLCAiUTQiXSwKICAgICAgICBkZWZhdWx0PSJ1bmtub3duIiwKICAgICkKICAgIHJldHVybiByZXN1bHQKCgpkZWYgd2FmZXJfZGVmZWN0X2ZlYXR1cmVzKAogICAgd2FmZXJfZGY6IHBkLkRhdGFGcmFtZSwKICAgIHdhZmVyX2lkX2NvbDogc3RyID0gIndhZmVyX2lkIiwKICAgIHhfY29sOiBzdHIgPSAieCIsCiAgICB5X2NvbDogc3RyID0gInkiLAogICAgZGVmZWN0X3R5cGVfY29sOiBzdHIgfCBOb25lID0gImRlZmVjdF90eXBlIiwKKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJBZ2dyZWdhdGUgY29vcmRpbmF0ZS1sZXZlbCB3YWZlciBkZWZlY3RzIGludG8gd2FmZXItbGV2ZWwgZmVhdHVyZXMuIiIiCiAgICB2YWxpZGF0ZV93YWZlcl9jb2x1bW5zKHdhZmVyX2RmLCB3YWZlcl9pZF9jb2w9d2FmZXJfaWRfY29sLCB4X2NvbD14X2NvbCwgeV9jb2w9eV9jb2wpCiAgICBlbnJpY2hlZCA9IGFkZF9zcGF0aWFsX2JpbnMod2FmZXJfZGYsIHhfY29sPXhfY29sLCB5X2NvbD15X2NvbCkKCiAgICBncm91cGVkID0gZW5yaWNoZWQuZ3JvdXBieSh3YWZlcl9pZF9jb2wsIGRyb3BuYT1GYWxzZSkKICAgIGZlYXR1cmVzID0gZ3JvdXBlZC5hZ2coCiAgICAgICAgZGVmZWN0X2NvdW50PSh4X2NvbCwgInNpemUiKSwKICAgICAgICB4X21lYW49KHhfY29sLCAibWVhbiIpLAogICAgICAgIHhfc3RkPSh4X2NvbCwgInN0ZCIpLAogICAgICAgIHlfbWVhbj0oeV9jb2wsICJtZWFuIiksCiAgICAgICAgeV9zdGQ9KHlfY29sLCAic3RkIiksCiAgICAgICAgcmFkaXVzX21lYW49KCJyYWRpdXNfbm9ybSIsICJtZWFuIiksCiAgICAgICAgcmFkaXVzX3N0ZD0oInJhZGl1c19ub3JtIiwgInN0ZCIpLAogICAgICAgIHJhZGl1c19tYXg9KCJyYWRpdXNfbm9ybSIsICJtYXgiKSwKICAgICkKICAgIGZlYXR1cmVzID0gZmVhdHVyZXMuZmlsbG5hKDApCgogICAgem9uZV9jb3VudHMgPSBwZC5jcm9zc3RhYihlbnJpY2hlZFt3YWZlcl9pZF9jb2xdLCBlbnJpY2hlZFsicmFkaWFsX3pvbmUiXSkKICAgIHF1YWRyYW50X2NvdW50cyA9IHBkLmNyb3NzdGFiKGVucmljaGVkW3dhZmVyX2lkX2NvbF0sIGVucmljaGVkWyJxdWFkcmFudCJdKQoKICAgIGZvciB6b25lIGluIFsiY2VudGVyIiwgIm1pZGRsZSIsICJlZGdlIl06CiAgICAgICAgaWYgem9uZSBub3QgaW4gem9uZV9jb3VudHM6CiAgICAgICAgICAgIHpvbmVfY291bnRzW3pvbmVdID0gMAogICAgZm9yIHF1YWRyYW50IGluIFsiUTEiLCAiUTIiLCAiUTMiLCAiUTQiXToKICAgICAgICBpZiBxdWFkcmFudCBub3QgaW4gcXVhZHJhbnRfY291bnRzOgogICAgICAgICAgICBxdWFkcmFudF9jb3VudHNbcXVhZHJhbnRdID0gMAoKICAgIHpvbmVfcmF0aW9zID0gem9uZV9jb3VudHNbWyJjZW50ZXIiLCAibWlkZGxlIiwgImVkZ2UiXV0uZGl2KHpvbmVfY291bnRzLnN1bShheGlzPTEpLCBheGlzPTApCiAgICB6b25lX3JhdGlvcyA9IHpvbmVfcmF0aW9zLmFkZF9wcmVmaXgoInpvbmVfcmF0aW9fIikKCiAgICBxdWFkcmFudF9yYXRpb3MgPSBxdWFkcmFudF9jb3VudHNbWyJRMSIsICJRMiIsICJRMyIsICJRNCJdXS5kaXYocXVhZHJhbnRfY291bnRzLnN1bShheGlzPTEpLCBheGlzPTApCiAgICBxdWFkcmFudF9yYXRpb3MgPSBxdWFkcmFudF9yYXRpb3MuYWRkX3ByZWZpeCgicXVhZHJhbnRfcmF0aW9fIikKICAgIHF1YWRyYW50X2ltYmFsYW5jZSA9IHF1YWRyYW50X3JhdGlvcy5tYXgoYXhpcz0xKSAtIHF1YWRyYW50X3JhdGlvcy5taW4oYXhpcz0xKQogICAgcXVhZHJhbnRfaW1iYWxhbmNlLm5hbWUgPSAicXVhZHJhbnRfaW1iYWxhbmNlIgoKICAgIG91dHB1dCA9IGZlYXR1cmVzLmpvaW4oem9uZV9yYXRpb3MpLmpvaW4ocXVhZHJhbnRfcmF0aW9zKS5qb2luKHF1YWRyYW50X2ltYmFsYW5jZSkKCiAgICBpZiBkZWZlY3RfdHlwZV9jb2wgYW5kIGRlZmVjdF90eXBlX2NvbCBpbiBlbnJpY2hlZC5jb2x1bW5zOgogICAgICAgIHR5cGVfY291bnRzID0gcGQuY3Jvc3N0YWIoZW5yaWNoZWRbd2FmZXJfaWRfY29sXSwgZW5yaWNoZWRbZGVmZWN0X3R5cGVfY29sXSkKICAgICAgICB0eXBlX3JhdGlvcyA9IHR5cGVfY291bnRzLmRpdih0eXBlX2NvdW50cy5zdW0oYXhpcz0xKSwgYXhpcz0wKS5hZGRfcHJlZml4KCJkZWZlY3RfdHlwZV9yYXRpb18iKQogICAgICAgIG91dHB1dCA9IG91dHB1dC5qb2luKHR5cGVfcmF0aW9zKQoKICAgIHJldHVybiBvdXRwdXQucmVzZXRfaW5kZXgoKS5yZW5hbWUoY29sdW1ucz17d2FmZXJfaWRfY29sOiAid2FmZXJfaWQifSkKCgpkZWYgaGV1cmlzdGljX3dhZmVyX3BhdHRlcm5fbGFiZWwoZmVhdHVyZXM6IHBkLkRhdGFGcmFtZSkgLT4gcGQuU2VyaWVzOgogICAgIiIiQXNzaWduIGEgc2ltcGxlIGhldXJpc3RpYyB3YWZlciBkZWZlY3QgcGF0dGVybiBsYWJlbCBmcm9tIGVuZ2luZWVyZWQgZmVhdHVyZXMuIiIiCiAgICByZXF1aXJlZCA9IHsiem9uZV9yYXRpb19jZW50ZXIiLCAiem9uZV9yYXRpb19lZGdlIiwgInF1YWRyYW50X2ltYmFsYW5jZSJ9CiAgICBtaXNzaW5nID0gc29ydGVkKHJlcXVpcmVkLmRpZmZlcmVuY2UoZmVhdHVyZXMuY29sdW1ucykpCiAgICBpZiBtaXNzaW5nOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJNaXNzaW5nIHdhZmVyIGZlYXR1cmUgY29sdW1uczoge21pc3Npbmd9IikKCiAgICBsYWJlbHMgPSBucC5zZWxlY3QoCiAgICAgICAgWwogICAgICAgICAgICBmZWF0dXJlc1siem9uZV9yYXRpb19lZGdlIl0gPj0gMC42MCwKICAgICAgICAgICAgZmVhdHVyZXNbInpvbmVfcmF0aW9fY2VudGVyIl0gPj0gMC42MCwKICAgICAgICAgICAgZmVhdHVyZXNbInF1YWRyYW50X2ltYmFsYW5jZSJdID49IDAuNTAsCiAgICAgICAgXSwKICAgICAgICBbImVkZ2UiLCAiY2VudGVyIiwgImxvY2FsaXplZCJdLAogICAgICAgIGRlZmF1bHQ9Im1peGVkIiwKICAgICkKICAgIHJldHVybiBwZC5TZXJpZXMobGFiZWxzLCBpbmRleD1mZWF0dXJlcy5pbmRleCwgbmFtZT0icGF0dGVybl9sYWJlbCIpCg==', 'scripts/assemble_feature_table.py': 'IiIiQXNzZW1ibGUgc2Vuc29yLCB3YWZlciwgYW5kIGVxdWlwbWVudCBmZWF0dXJlcyBpbnRvIGEgbW9kZWxpbmcgdGFibGUuIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IGFyZ3BhcnNlDQppbXBvcnQgc3lzDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KUFJPSkVDVF9ST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0NCmlmIHN0cihQUk9KRUNUX1JPT1QpIG5vdCBpbiBzeXMucGF0aDoNCiAgICBzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBST0pFQ1RfUk9PVCkpDQoNCmltcG9ydCBwYW5kYXMgYXMgcGQNCg0KZnJvbSBzcmMuZmVhdHVyZV9zdG9yZSBpbXBvcnQgYXNzZW1ibGVfZmVhdHVyZV90YWJsZSwgZmVhdHVyZV9taXNzaW5nbmVzc19yZXBvcnQNCg0KDQpkZWYgcGFyc2Vfa2V5X2xpc3QodmFsdWU6IHN0cikgLT4gbGlzdFtzdHJdOg0KICAgICIiIlBhcnNlIGEgY29tbWEtc2VwYXJhdGVkIGtleSBsaXN0LiIiIg0KICAgIGtleXMgPSBbaXRlbS5zdHJpcCgpIGZvciBpdGVtIGluIHZhbHVlLnNwbGl0KCIsIikgaWYgaXRlbS5zdHJpcCgpXQ0KICAgIGlmIG5vdCBrZXlzOg0KICAgICAgICByYWlzZSBhcmdwYXJzZS5Bcmd1bWVudFR5cGVFcnJvcigiQXQgbGVhc3Qgb25lIGtleSBjb2x1bW4gaXMgcmVxdWlyZWQuIikNCiAgICByZXR1cm4ga2V5cw0KDQoNCmRlZiBwYXJzZV9hcmdzKCkgLT4gYXJncGFyc2UuTmFtZXNwYWNlOg0KICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJBc3NlbWJsZSBtYW51ZmFjdHVyaW5nIGZlYXR1cmUgdGFibGVzLiIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zZW5zb3ItcGF0aCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSwgaGVscD0iQ1NWIGZpbGUgY29udGFpbmluZyBzZW5zb3IgZmVhdHVyZXMuIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXdhZmVyLXBhdGgiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9Tm9uZSwgaGVscD0iT3B0aW9uYWwgd2FmZXIgZmVhdHVyZSBDU1YgZmlsZS4iKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZXF1aXBtZW50LXBhdGgiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9Tm9uZSwgaGVscD0iT3B0aW9uYWwgZXF1aXBtZW50IGZlYXR1cmUgQ1NWIGZpbGUuIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KA0KICAgICAgICAiLS1zZW5zb3Itd2FmZXIta2V5cyIsDQogICAgICAgIHR5cGU9cGFyc2Vfa2V5X2xpc3QsDQogICAgICAgIGRlZmF1bHQ9WyJ3YWZlcl9pZCJdLA0KICAgICAgICBoZWxwPSJDb21tYS1zZXBhcmF0ZWQgam9pbiBrZXlzIGJldHdlZW4gc2Vuc29yIGFuZCB3YWZlciBmZWF0dXJlcy4iLA0KICAgICkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KA0KICAgICAgICAiLS1zZW5zb3ItZXF1aXBtZW50LWtleXMiLA0KICAgICAgICB0eXBlPXBhcnNlX2tleV9saXN0LA0KICAgICAgICBkZWZhdWx0PVsiZXF1aXBtZW50X2lkIl0sDQogICAgICAgIGhlbHA9IkNvbW1hLXNlcGFyYXRlZCBqb2luIGtleXMgYmV0d2VlbiBzZW5zb3IgYW5kIGVxdWlwbWVudCBmZWF0dXJlcy4iLA0KICAgICkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KA0KICAgICAgICAiLS1vdXRwdXQtcGF0aCIsDQogICAgICAgIHR5cGU9UGF0aCwNCiAgICAgICAgZGVmYXVsdD1QYXRoKCJvdXRwdXRzL2ZlYXR1cmVzL21vZGVsaW5nX3RhYmxlLmNzdiIpLA0KICAgICAgICBoZWxwPSJPdXRwdXQgQ1NWIHBhdGggZm9yIGFzc2VtYmxlZCBtb2RlbGluZyB0YWJsZS4iLA0KICAgICkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KA0KICAgICAgICAiLS1yZXBvcnQtZGlyIiwNCiAgICAgICAgdHlwZT1QYXRoLA0KICAgICAgICBkZWZhdWx0PVBhdGgoIm91dHB1dHMvcmVwb3J0cyIpLA0KICAgICAgICBoZWxwPSJEaXJlY3Rvcnkgd2hlcmUgam9pbiBhbmQgbWlzc2luZ25lc3MgcmVwb3J0cyB3aWxsIGJlIHdyaXR0ZW4uIiwNCiAgICApDQogICAgcmV0dXJuIHBhcnNlci5wYXJzZV9hcmdzKCkNCg0KDQpkZWYgcmVhZF9vcHRpb25hbF9jc3YocGF0aDogUGF0aCB8IE5vbmUpIC0+IHBkLkRhdGFGcmFtZSB8IE5vbmU6DQogICAgIiIiUmVhZCBhbiBvcHRpb25hbCBDU1YgZmlsZS4iIiINCiAgICBpZiBwYXRoIGlzIE5vbmU6DQogICAgICAgIHJldHVybiBOb25lDQogICAgaWYgbm90IHBhdGguZXhpc3RzKCk6DQogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiT3B0aW9uYWwgZmVhdHVyZSBmaWxlIG5vdCBmb3VuZDoge3BhdGh9IikNCiAgICByZXR1cm4gcGQucmVhZF9jc3YocGF0aCkNCg0KDQpkZWYgYXNzZW1ibGVfZnJvbV9wYXRocygNCiAgICBzZW5zb3JfcGF0aDogUGF0aCwNCiAgICBvdXRwdXRfcGF0aDogUGF0aCwNCiAgICByZXBvcnRfZGlyOiBQYXRoLA0KICAgIHdhZmVyX3BhdGg6IFBhdGggfCBOb25lID0gTm9uZSwNCiAgICBlcXVpcG1lbnRfcGF0aDogUGF0aCB8IE5vbmUgPSBOb25lLA0KICAgIHNlbnNvcl93YWZlcl9rZXlzOiBsaXN0W3N0cl0gfCBOb25lID0gTm9uZSwNCiAgICBzZW5zb3JfZXF1aXBtZW50X2tleXM6IGxpc3Rbc3RyXSB8IE5vbmUgPSBOb25lLA0KKSAtPiB0dXBsZVtQYXRoLCBQYXRoLCBQYXRoXToNCiAgICAiIiJBc3NlbWJsZSBmZWF0dXJlIHRhYmxlcyBmcm9tIENTViBwYXRocyBhbmQgd3JpdGUgb3V0cHV0IGZpbGVzLiIiIg0KICAgIGlmIG5vdCBzZW5zb3JfcGF0aC5leGlzdHMoKToNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJTZW5zb3IgZmVhdHVyZSBmaWxlIG5vdCBmb3VuZDoge3NlbnNvcl9wYXRofSIpDQoNCiAgICBzZW5zb3JfZGYgPSBwZC5yZWFkX2NzdihzZW5zb3JfcGF0aCkNCiAgICB3YWZlcl9kZiA9IHJlYWRfb3B0aW9uYWxfY3N2KHdhZmVyX3BhdGgpDQogICAgZXF1aXBtZW50X2RmID0gcmVhZF9vcHRpb25hbF9jc3YoZXF1aXBtZW50X3BhdGgpDQoNCiAgICBmZWF0dXJlX3RhYmxlLCBqb2luX3JlcG9ydCA9IGFzc2VtYmxlX2ZlYXR1cmVfdGFibGUoDQogICAgICAgIHNlbnNvcl9kZj1zZW5zb3JfZGYsDQogICAgICAgIHdhZmVyX2ZlYXR1cmVzPXdhZmVyX2RmLA0KICAgICAgICBlcXVpcG1lbnRfZmVhdHVyZXM9ZXF1aXBtZW50X2RmLA0KICAgICAgICBzZW5zb3Jfd2FmZXJfa2V5cz1zZW5zb3Jfd2FmZXJfa2V5cywNCiAgICAgICAgc2Vuc29yX2VxdWlwbWVudF9rZXlzPXNlbnNvcl9lcXVpcG1lbnRfa2V5cywNCiAgICApDQogICAgbWlzc2luZ25lc3MgPSBmZWF0dXJlX21pc3NpbmduZXNzX3JlcG9ydChmZWF0dXJlX3RhYmxlKQ0KDQogICAgb3V0cHV0X3BhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICByZXBvcnRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCg0KICAgIGpvaW5fcmVwb3J0X3BhdGggPSByZXBvcnRfZGlyIC8gImZlYXR1cmVfam9pbl9yZXBvcnQuY3N2Ig0KICAgIG1pc3NpbmduZXNzX3BhdGggPSByZXBvcnRfZGlyIC8gImZlYXR1cmVfbWlzc2luZ25lc3NfcmVwb3J0LmNzdiINCg0KICAgIGZlYXR1cmVfdGFibGUudG9fY3N2KG91dHB1dF9wYXRoLCBpbmRleD1GYWxzZSkNCiAgICBqb2luX3JlcG9ydC50b19jc3Yoam9pbl9yZXBvcnRfcGF0aCwgaW5kZXg9RmFsc2UpDQogICAgbWlzc2luZ25lc3MudG9fY3N2KG1pc3NpbmduZXNzX3BhdGgsIGluZGV4X2xhYmVsPSJmZWF0dXJlIikNCg0KICAgIHJldHVybiBvdXRwdXRfcGF0aCwgam9pbl9yZXBvcnRfcGF0aCwgbWlzc2luZ25lc3NfcGF0aA0KDQoNCmRlZiBtYWluKCkgLT4gaW50Og0KICAgIGFyZ3MgPSBwYXJzZV9hcmdzKCkNCiAgICBvdXRwdXRfcGF0aCwgam9pbl9yZXBvcnRfcGF0aCwgbWlzc2luZ25lc3NfcGF0aCA9IGFzc2VtYmxlX2Zyb21fcGF0aHMoDQogICAgICAgIHNlbnNvcl9wYXRoPWFyZ3Muc2Vuc29yX3BhdGgsDQogICAgICAgIHdhZmVyX3BhdGg9YXJncy53YWZlcl9wYXRoLA0KICAgICAgICBlcXVpcG1lbnRfcGF0aD1hcmdzLmVxdWlwbWVudF9wYXRoLA0KICAgICAgICBzZW5zb3Jfd2FmZXJfa2V5cz1hcmdzLnNlbnNvcl93YWZlcl9rZXlzLA0KICAgICAgICBzZW5zb3JfZXF1aXBtZW50X2tleXM9YXJncy5zZW5zb3JfZXF1aXBtZW50X2tleXMsDQogICAgICAgIG91dHB1dF9wYXRoPWFyZ3Mub3V0cHV0X3BhdGgsDQogICAgICAgIHJlcG9ydF9kaXI9YXJncy5yZXBvcnRfZGlyLA0KICAgICkNCiAgICBwcmludChmIk1vZGVsaW5nIHRhYmxlIHdyaXR0ZW4gdG86IHtvdXRwdXRfcGF0aH0iKQ0KICAgIHByaW50KGYiSm9pbiByZXBvcnQgd3JpdHRlbiB0bzoge2pvaW5fcmVwb3J0X3BhdGh9IikNCiAgICBwcmludChmIk1pc3NpbmduZXNzIHJlcG9ydCB3cml0dGVuIHRvOiB7bWlzc2luZ25lc3NfcGF0aH0iKQ0KICAgIHJldHVybiAwDQoNCg0KaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoNCiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkNCg==', 'scripts/build_auxiliary_features.py': 'IiIiQnVpbGQgd2FmZXIgYW5kIGVxdWlwbWVudCBmZWF0dXJlIENTViBmaWxlcyBmcm9tIHJhdyBhdXhpbGlhcnkgZGF0YS4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgYXJncGFyc2UNCmltcG9ydCBzeXMNCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KDQpQUk9KRUNUX1JPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXQ0KaWYgc3RyKFBST0pFQ1RfUk9PVCkgbm90IGluIHN5cy5wYXRoOg0KICAgIHN5cy5wYXRoLmluc2VydCgwLCBzdHIoUFJPSkVDVF9ST09UKSkNCg0KaW1wb3J0IHBhbmRhcyBhcyBwZA0KDQpmcm9tIHNyYy5kYXRhX2NvbnRyYWN0cyBpbXBvcnQgdmFsaWRhdGVfZXF1aXBtZW50X2V2ZW50c19jb250cmFjdCwgdmFsaWRhdGVfd2FmZXJfaW5zcGVjdGlvbl9jb250cmFjdA0KZnJvbSBzcmMuZXF1aXBtZW50X2ZlYXR1cmVzIGltcG9ydCBlcXVpcG1lbnRfZXZlbnRfZmVhdHVyZXMNCmZyb20gc3JjLndhZmVyX2ZlYXR1cmVzIGltcG9ydCBoZXVyaXN0aWNfd2FmZXJfcGF0dGVybl9sYWJlbCwgd2FmZXJfZGVmZWN0X2ZlYXR1cmVzDQoNCg0KZGVmIHBhcnNlX29wdGlvbmFsX3BhdGgodmFsdWU6IHN0ciB8IE5vbmUpIC0+IFBhdGggfCBOb25lOg0KICAgICIiIlBhcnNlIG9wdGlvbmFsIHBhdGggYXJndW1lbnRzLiIiIg0KICAgIGlmIHZhbHVlIGlzIE5vbmUgb3Igbm90IHN0cih2YWx1ZSkuc3RyaXAoKToNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICByZXR1cm4gUGF0aCh2YWx1ZSkNCg0KDQpkZWYgcGFyc2VfYXJncygpIC0+IGFyZ3BhcnNlLk5hbWVzcGFjZToNCiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iQnVpbGQgd2FmZXIgYW5kIGVxdWlwbWVudCBmZWF0dXJlIENTViBmaWxlcy4iKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0td2FmZXItaW5wdXQiLCB0eXBlPXBhcnNlX29wdGlvbmFsX3BhdGgsIGRlZmF1bHQ9Tm9uZSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWVxdWlwbWVudC1pbnB1dCIsIHR5cGU9cGFyc2Vfb3B0aW9uYWxfcGF0aCwgZGVmYXVsdD1Ob25lKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoDQogICAgICAgICItLXdhZmVyLW91dHB1dCIsDQogICAgICAgIHR5cGU9UGF0aCwNCiAgICAgICAgZGVmYXVsdD1QYXRoKCJkYXRhL3Jhdy93YWZlcl9mZWF0dXJlcy5jc3YiKSwNCiAgICAgICAgaGVscD0iT3V0cHV0IENTViBwYXRoIGZvciB3YWZlci1sZXZlbCBmZWF0dXJlcy4iLA0KICAgICkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KA0KICAgICAgICAiLS1lcXVpcG1lbnQtb3V0cHV0IiwNCiAgICAgICAgdHlwZT1QYXRoLA0KICAgICAgICBkZWZhdWx0PVBhdGgoImRhdGEvcmF3L2VxdWlwbWVudF9mZWF0dXJlcy5jc3YiKSwNCiAgICAgICAgaGVscD0iT3V0cHV0IENTViBwYXRoIGZvciBlcXVpcG1lbnQtbGV2ZWwgZmVhdHVyZXMuIiwNCiAgICApDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgNCiAgICAgICAgIi0tYWRkLXdhZmVyLXBhdHRlcm4tbGFiZWwiLA0KICAgICAgICBhY3Rpb249InN0b3JlX3RydWUiLA0KICAgICAgICBoZWxwPSJBZGQgaGV1cmlzdGljIHdhZmVyIHBhdHRlcm4gbGFiZWxzIHRvIHdhZmVyIGZlYXR1cmUgb3V0cHV0LiIsDQogICAgKQ0KICAgIHJldHVybiBwYXJzZXIucGFyc2VfYXJncygpDQoNCg0KZGVmIGJ1aWxkX3dhZmVyX2ZlYXR1cmVzX2ZpbGUoDQogICAgd2FmZXJfaW5wdXQ6IFBhdGgsDQogICAgd2FmZXJfb3V0cHV0OiBQYXRoLA0KICAgIGFkZF9wYXR0ZXJuX2xhYmVsOiBib29sID0gRmFsc2UsDQopIC0+IFBhdGg6DQogICAgIiIiQnVpbGQgYSB3YWZlci1sZXZlbCBmZWF0dXJlIENTViBmcm9tIGNvb3JkaW5hdGUtbGV2ZWwgaW5zcGVjdGlvbiBkYXRhLiIiIg0KICAgIGlmIG5vdCB3YWZlcl9pbnB1dC5leGlzdHMoKToNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJXYWZlciBpbnB1dCBub3QgZm91bmQ6IHt3YWZlcl9pbnB1dH0iKQ0KDQogICAgd2FmZXJfZGYgPSBwZC5yZWFkX2Nzdih3YWZlcl9pbnB1dCkNCiAgICB2YWxpZGF0ZV93YWZlcl9pbnNwZWN0aW9uX2NvbnRyYWN0KHdhZmVyX2RmKS5yYWlzZV9pZl9mYWlsZWQoKQ0KICAgIGZlYXR1cmVzID0gd2FmZXJfZGVmZWN0X2ZlYXR1cmVzKHdhZmVyX2RmKQ0KICAgIGlmIGFkZF9wYXR0ZXJuX2xhYmVsOg0KICAgICAgICBmZWF0dXJlc1sicGF0dGVybl9sYWJlbCJdID0gaGV1cmlzdGljX3dhZmVyX3BhdHRlcm5fbGFiZWwoZmVhdHVyZXMpDQoNCiAgICB3YWZlcl9vdXRwdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICBmZWF0dXJlcy50b19jc3Yod2FmZXJfb3V0cHV0LCBpbmRleD1GYWxzZSkNCiAgICByZXR1cm4gd2FmZXJfb3V0cHV0DQoNCg0KZGVmIGJ1aWxkX2VxdWlwbWVudF9mZWF0dXJlc19maWxlKGVxdWlwbWVudF9pbnB1dDogUGF0aCwgZXF1aXBtZW50X291dHB1dDogUGF0aCkgLT4gUGF0aDoNCiAgICAiIiJCdWlsZCBhbiBlcXVpcG1lbnQtbGV2ZWwgZmVhdHVyZSBDU1YgZnJvbSBldmVudCBsb2cgZGF0YS4iIiINCiAgICBpZiBub3QgZXF1aXBtZW50X2lucHV0LmV4aXN0cygpOg0KICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkVxdWlwbWVudCBpbnB1dCBub3QgZm91bmQ6IHtlcXVpcG1lbnRfaW5wdXR9IikNCg0KICAgIGVxdWlwbWVudF9kZiA9IHBkLnJlYWRfY3N2KGVxdWlwbWVudF9pbnB1dCkNCiAgICB2YWxpZGF0ZV9lcXVpcG1lbnRfZXZlbnRzX2NvbnRyYWN0KGVxdWlwbWVudF9kZikucmFpc2VfaWZfZmFpbGVkKCkNCiAgICBmZWF0dXJlcyA9IGVxdWlwbWVudF9ldmVudF9mZWF0dXJlcyhlcXVpcG1lbnRfZGYpDQoNCiAgICBlcXVpcG1lbnRfb3V0cHV0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgZmVhdHVyZXMudG9fY3N2KGVxdWlwbWVudF9vdXRwdXQsIGluZGV4PUZhbHNlKQ0KICAgIHJldHVybiBlcXVpcG1lbnRfb3V0cHV0DQoNCg0KZGVmIGJ1aWxkX2F1eGlsaWFyeV9mZWF0dXJlcygNCiAgICB3YWZlcl9pbnB1dDogUGF0aCB8IE5vbmUgPSBOb25lLA0KICAgIGVxdWlwbWVudF9pbnB1dDogUGF0aCB8IE5vbmUgPSBOb25lLA0KICAgIHdhZmVyX291dHB1dDogUGF0aCA9IFBhdGgoImRhdGEvcmF3L3dhZmVyX2ZlYXR1cmVzLmNzdiIpLA0KICAgIGVxdWlwbWVudF9vdXRwdXQ6IFBhdGggPSBQYXRoKCJkYXRhL3Jhdy9lcXVpcG1lbnRfZmVhdHVyZXMuY3N2IiksDQogICAgYWRkX3dhZmVyX3BhdHRlcm5fbGFiZWw6IGJvb2wgPSBGYWxzZSwNCikgLT4gZGljdFtzdHIsIFBhdGhdOg0KICAgICIiIkJ1aWxkIGF2YWlsYWJsZSBhdXhpbGlhcnkgZmVhdHVyZSBmaWxlcyBhbmQgcmV0dXJuIG91dHB1dCBwYXRocy4iIiINCiAgICBvdXRwdXRzOiBkaWN0W3N0ciwgUGF0aF0gPSB7fQ0KDQogICAgaWYgd2FmZXJfaW5wdXQgaXMgbm90IE5vbmU6DQogICAgICAgIG91dHB1dHNbIndhZmVyIl0gPSBidWlsZF93YWZlcl9mZWF0dXJlc19maWxlKA0KICAgICAgICAgICAgd2FmZXJfaW5wdXQ9d2FmZXJfaW5wdXQsDQogICAgICAgICAgICB3YWZlcl9vdXRwdXQ9d2FmZXJfb3V0cHV0LA0KICAgICAgICAgICAgYWRkX3BhdHRlcm5fbGFiZWw9YWRkX3dhZmVyX3BhdHRlcm5fbGFiZWwsDQogICAgICAgICkNCg0KICAgIGlmIGVxdWlwbWVudF9pbnB1dCBpcyBub3QgTm9uZToNCiAgICAgICAgb3V0cHV0c1siZXF1aXBtZW50Il0gPSBidWlsZF9lcXVpcG1lbnRfZmVhdHVyZXNfZmlsZSgNCiAgICAgICAgICAgIGVxdWlwbWVudF9pbnB1dD1lcXVpcG1lbnRfaW5wdXQsDQogICAgICAgICAgICBlcXVpcG1lbnRfb3V0cHV0PWVxdWlwbWVudF9vdXRwdXQsDQogICAgICAgICkNCg0KICAgIGlmIG5vdCBvdXRwdXRzOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJBdCBsZWFzdCBvbmUgb2YgLS13YWZlci1pbnB1dCBvciAtLWVxdWlwbWVudC1pbnB1dCBpcyByZXF1aXJlZC4iKQ0KDQogICAgcmV0dXJuIG91dHB1dHMNCg0KDQpkZWYgbWFpbigpIC0+IGludDoNCiAgICBhcmdzID0gcGFyc2VfYXJncygpDQogICAgb3V0cHV0cyA9IGJ1aWxkX2F1eGlsaWFyeV9mZWF0dXJlcygNCiAgICAgICAgd2FmZXJfaW5wdXQ9YXJncy53YWZlcl9pbnB1dCwNCiAgICAgICAgZXF1aXBtZW50X2lucHV0PWFyZ3MuZXF1aXBtZW50X2lucHV0LA0KICAgICAgICB3YWZlcl9vdXRwdXQ9YXJncy53YWZlcl9vdXRwdXQsDQogICAgICAgIGVxdWlwbWVudF9vdXRwdXQ9YXJncy5lcXVpcG1lbnRfb3V0cHV0LA0KICAgICAgICBhZGRfd2FmZXJfcGF0dGVybl9sYWJlbD1hcmdzLmFkZF93YWZlcl9wYXR0ZXJuX2xhYmVsLA0KICAgICkNCiAgICBmb3IgbmFtZSwgcGF0aCBpbiBvdXRwdXRzLml0ZW1zKCk6DQogICAgICAgIHByaW50KGYie25hbWV9IGZlYXR1cmVzIHdyaXR0ZW4gdG86IHtwYXRofSIpDQogICAgcmV0dXJuIDANCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQ0K', 'scripts/generate_monitoring_report.py': 'IiIiR2VuZXJhdGUgcHJvY2Vzcy1xdWFsaXR5IG1vbml0b3JpbmcgcmVwb3J0cyBmcm9tIHByZWRpY3Rpb24gQ1NWIGZpbGVzLiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBhcmdwYXJzZQ0KaW1wb3J0IHN5cw0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNClBST0pFQ1RfUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdDQppZiBzdHIoUFJPSkVDVF9ST09UKSBub3QgaW4gc3lzLnBhdGg6DQogICAgc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQUk9KRUNUX1JPT1QpKQ0KDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCmZyb20gc3JjLm1vbml0b3JpbmcgaW1wb3J0IGdyb3VwX3Jpc2tfc3VtbWFyeSwgb3ZlcmFsbF9yaXNrX3N1bW1hcnksIHRvcF9yaXNrX3ByZWRpY3Rpb25zDQoNCg0KZGVmIHBhcnNlX2dyb3VwX2NvbHVtbnModmFsdWU6IHN0ciB8IE5vbmUpIC0+IGxpc3Rbc3RyXToNCiAgICAiIiJQYXJzZSBvcHRpb25hbCBjb21tYS1zZXBhcmF0ZWQgZ3JvdXAgY29sdW1ucy4iIiINCiAgICBpZiB2YWx1ZSBpcyBOb25lIG9yIG5vdCB2YWx1ZS5zdHJpcCgpOg0KICAgICAgICByZXR1cm4gW10NCiAgICByZXR1cm4gW2l0ZW0uc3RyaXAoKSBmb3IgaXRlbSBpbiB2YWx1ZS5zcGxpdCgiLCIpIGlmIGl0ZW0uc3RyaXAoKV0NCg0KDQpkZWYgcGFyc2VfYXJncygpIC0+IGFyZ3BhcnNlLk5hbWVzcGFjZToNCiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iR2VuZXJhdGUgbW9uaXRvcmluZyByZXBvcnRzIGZyb20gcHJlZGljdGlvbiBvdXRwdXRzLiIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgNCiAgICAgICAgIi0tcHJlZGljdGlvbnMtcGF0aCIsDQogICAgICAgIHR5cGU9UGF0aCwNCiAgICAgICAgcmVxdWlyZWQ9VHJ1ZSwNCiAgICAgICAgaGVscD0iUHJlZGljdGlvbiBDU1YgZmlsZSBnZW5lcmF0ZWQgYnkgcHJlZGljdF93aXRoX21vZGVsLnB5LiIsDQogICAgKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoDQogICAgICAgICItLW91dHB1dC1kaXIiLA0KICAgICAgICB0eXBlPVBhdGgsDQogICAgICAgIGRlZmF1bHQ9UGF0aCgib3V0cHV0cy9yZXBvcnRzL21vbml0b3JpbmciKSwNCiAgICAgICAgaGVscD0iRGlyZWN0b3J5IHdoZXJlIG1vbml0b3JpbmcgcmVwb3J0cyB3aWxsIGJlIHdyaXR0ZW4uIiwNCiAgICApDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgNCiAgICAgICAgIi0tZ3JvdXAtY29sdW1ucyIsDQogICAgICAgIHR5cGU9cGFyc2VfZ3JvdXBfY29sdW1ucywNCiAgICAgICAgZGVmYXVsdD1bXSwNCiAgICAgICAgaGVscD0iT3B0aW9uYWwgY29tbWEtc2VwYXJhdGVkIGNvbHVtbnMgZm9yIGdyb3VwZWQgcmlzayByZXBvcnRzLiIsDQogICAgKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoDQogICAgICAgICItLWhpZ2gtcmlzay10aHJlc2hvbGQiLA0KICAgICAgICB0eXBlPWZsb2F0LA0KICAgICAgICBkZWZhdWx0PTAuNSwNCiAgICAgICAgaGVscD0iRmFpbCBwcm9iYWJpbGl0eSB0aHJlc2hvbGQgZm9yIGhpZ2gtcmlzayByb3dzLiIsDQogICAgKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoDQogICAgICAgICItLWFsZXJ0LXJhdGlvLXRocmVzaG9sZCIsDQogICAgICAgIHR5cGU9ZmxvYXQsDQogICAgICAgIGRlZmF1bHQ9MC4yNSwNCiAgICAgICAgaGVscD0iR3JvdXAgaGlnaC1yaXNrIHJhdGlvIHRocmVzaG9sZCBmb3IgYWxlcnQgZmxhZ3MuIiwNCiAgICApDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10b3AtbiIsIHR5cGU9aW50LCBkZWZhdWx0PTIwLCBoZWxwPSJOdW1iZXIgb2YgdG9wLXJpc2sgcm93cyB0byBleHBvcnQuIikNCiAgICByZXR1cm4gcGFyc2VyLnBhcnNlX2FyZ3MoKQ0KDQoNCmRlZiBnZW5lcmF0ZV9tb25pdG9yaW5nX3JlcG9ydHMoDQogICAgcHJlZGljdGlvbnNfcGF0aDogUGF0aCwNCiAgICBvdXRwdXRfZGlyOiBQYXRoLA0KICAgIGdyb3VwX2NvbHVtbnM6IGxpc3Rbc3RyXSB8IE5vbmUgPSBOb25lLA0KICAgIGhpZ2hfcmlza190aHJlc2hvbGQ6IGZsb2F0ID0gMC41LA0KICAgIGFsZXJ0X3JhdGlvX3RocmVzaG9sZDogZmxvYXQgPSAwLjI1LA0KICAgIHRvcF9uOiBpbnQgPSAyMCwNCikgLT4gZGljdFtzdHIsIFBhdGhdOg0KICAgICIiIkdlbmVyYXRlIG1vbml0b3JpbmcgcmVwb3J0IENTViBmaWxlcyBhbmQgcmV0dXJuIHRoZWlyIHBhdGhzLiIiIg0KICAgIGlmIG5vdCBwcmVkaWN0aW9uc19wYXRoLmV4aXN0cygpOg0KICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIlByZWRpY3Rpb24gQ1NWIG5vdCBmb3VuZDoge3ByZWRpY3Rpb25zX3BhdGh9IikNCg0KICAgIG91dHB1dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KICAgIHByZWRpY3Rpb25zID0gcGQucmVhZF9jc3YocHJlZGljdGlvbnNfcGF0aCkNCg0KICAgIG92ZXJhbGxfcGF0aCA9IG91dHB1dF9kaXIgLyAib3ZlcmFsbF9yaXNrX3N1bW1hcnkuY3N2Ig0KICAgIHRvcF9yaXNrX3BhdGggPSBvdXRwdXRfZGlyIC8gInRvcF9yaXNrX3ByZWRpY3Rpb25zLmNzdiINCiAgICBncm91cF9wYXRoID0gb3V0cHV0X2RpciAvICJncm91cF9yaXNrX3N1bW1hcnkuY3N2Ig0KDQogICAgb3ZlcmFsbCA9IG92ZXJhbGxfcmlza19zdW1tYXJ5KHByZWRpY3Rpb25zLCBoaWdoX3Jpc2tfdGhyZXNob2xkPWhpZ2hfcmlza190aHJlc2hvbGQpDQogICAgb3ZlcmFsbC50b19mcmFtZSgpLnJlc2V0X2luZGV4KCkucmVuYW1lKGNvbHVtbnM9eyJpbmRleCI6ICJtZXRyaWMifSkudG9fY3N2KG92ZXJhbGxfcGF0aCwgaW5kZXg9RmFsc2UpDQoNCiAgICB0b3Bfcmlza19wcmVkaWN0aW9ucyhwcmVkaWN0aW9ucywgdG9wX249dG9wX24pLnRvX2Nzdih0b3Bfcmlza19wYXRoLCBpbmRleD1GYWxzZSkNCg0KICAgIHBhdGhzID0gew0KICAgICAgICAib3ZlcmFsbCI6IG92ZXJhbGxfcGF0aCwNCiAgICAgICAgInRvcF9yaXNrIjogdG9wX3Jpc2tfcGF0aCwNCiAgICB9DQoNCiAgICBncm91cF9jb2x1bW5zID0gZ3JvdXBfY29sdW1ucyBvciBbXQ0KICAgIGlmIGdyb3VwX2NvbHVtbnM6DQogICAgICAgIGdyb3VwX3Jpc2tfc3VtbWFyeSgNCiAgICAgICAgICAgIHByZWRpY3Rpb25zLA0KICAgICAgICAgICAgZ3JvdXBfY29scz1ncm91cF9jb2x1bW5zLA0KICAgICAgICAgICAgaGlnaF9yaXNrX3RocmVzaG9sZD1oaWdoX3Jpc2tfdGhyZXNob2xkLA0KICAgICAgICAgICAgYWxlcnRfcmF0aW9fdGhyZXNob2xkPWFsZXJ0X3JhdGlvX3RocmVzaG9sZCwNCiAgICAgICAgKS50b19jc3YoZ3JvdXBfcGF0aCwgaW5kZXg9RmFsc2UpDQogICAgICAgIHBhdGhzWyJncm91cCJdID0gZ3JvdXBfcGF0aA0KDQogICAgcmV0dXJuIHBhdGhzDQoNCg0KZGVmIG1haW4oKSAtPiBpbnQ6DQogICAgYXJncyA9IHBhcnNlX2FyZ3MoKQ0KICAgIHBhdGhzID0gZ2VuZXJhdGVfbW9uaXRvcmluZ19yZXBvcnRzKA0KICAgICAgICBwcmVkaWN0aW9uc19wYXRoPWFyZ3MucHJlZGljdGlvbnNfcGF0aCwNCiAgICAgICAgb3V0cHV0X2Rpcj1hcmdzLm91dHB1dF9kaXIsDQogICAgICAgIGdyb3VwX2NvbHVtbnM9YXJncy5ncm91cF9jb2x1bW5zLA0KICAgICAgICBoaWdoX3Jpc2tfdGhyZXNob2xkPWFyZ3MuaGlnaF9yaXNrX3RocmVzaG9sZCwNCiAgICAgICAgYWxlcnRfcmF0aW9fdGhyZXNob2xkPWFyZ3MuYWxlcnRfcmF0aW9fdGhyZXNob2xkLA0KICAgICAgICB0b3Bfbj1hcmdzLnRvcF9uLA0KICAgICkNCiAgICBmb3IgbmFtZSwgcGF0aCBpbiBwYXRocy5pdGVtcygpOg0KICAgICAgICBwcmludChmIntuYW1lfSByZXBvcnQgd3JpdHRlbiB0bzoge3BhdGh9IikNCiAgICByZXR1cm4gMA0KDQoNCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6DQogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpDQo=', 'scripts/generate_summary_report.py': 'IiIiR2VuZXJhdGUgYSBNYXJrZG93biBzdW1tYXJ5IHJlcG9ydCBmcm9tIHBpcGVsaW5lIENTViBvdXRwdXRzLiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBhcmdwYXJzZQ0KaW1wb3J0IHN5cw0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNClBST0pFQ1RfUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdDQppZiBzdHIoUFJPSkVDVF9ST09UKSBub3QgaW4gc3lzLnBhdGg6DQogICAgc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQUk9KRUNUX1JPT1QpKQ0KDQpmcm9tIHNyYy5yZXBvcnRpbmcgaW1wb3J0IGJ1aWxkX3N1bW1hcnlfcmVwb3J0DQoNCg0KZGVmIHBhcnNlX2FyZ3MoKSAtPiBhcmdwYXJzZS5OYW1lc3BhY2U6DQogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IkdlbmVyYXRlIGEgTWFya2Rvd24gc3VtbWFyeSByZXBvcnQuIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJlcG9ydHMtZGlyIiwgdHlwZT1QYXRoLCBkZWZhdWx0PVBhdGgoIm91dHB1dHMvcmVwb3J0cyIpKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbW9uaXRvcmluZy1kaXIiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9UGF0aCgib3V0cHV0cy9yZXBvcnRzL21vbml0b3JpbmciKSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dC1wYXRoIiwgdHlwZT1QYXRoLCBkZWZhdWx0PVBhdGgoIm91dHB1dHMvcmVwb3J0cy9zdW1tYXJ5X3JlcG9ydC5tZCIpKQ0KICAgIHJldHVybiBwYXJzZXIucGFyc2VfYXJncygpDQoNCg0KZGVmIG1haW4oKSAtPiBpbnQ6DQogICAgYXJncyA9IHBhcnNlX2FyZ3MoKQ0KICAgIGJ1aWxkX3N1bW1hcnlfcmVwb3J0KA0KICAgICAgICByZXBvcnRzX2Rpcj1hcmdzLnJlcG9ydHNfZGlyLA0KICAgICAgICBtb25pdG9yaW5nX2Rpcj1hcmdzLm1vbml0b3JpbmdfZGlyLA0KICAgICAgICBvdXRwdXRfcGF0aD1hcmdzLm91dHB1dF9wYXRoLA0KICAgICkNCiAgICBwcmludChmIlN1bW1hcnkgcmVwb3J0IHdyaXR0ZW4gdG86IHthcmdzLm91dHB1dF9wYXRofSIpDQogICAgcmV0dXJuIDANCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQ0K', 'scripts/predict_with_model.py': 'IiIiUnVuIHByZWRpY3Rpb25zIHdpdGggYSBzYXZlZCBTRUNPTSBtb2RlbCBidW5kbGUgYW5kIGZlYXR1cmUgQ1NWLiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBhcmdwYXJzZQ0KaW1wb3J0IHN5cw0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNClBST0pFQ1RfUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdDQppZiBzdHIoUFJPSkVDVF9ST09UKSBub3QgaW4gc3lzLnBhdGg6DQogICAgc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQUk9KRUNUX1JPT1QpKQ0KDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCmZyb20gc3JjLm1vZGVsX3JlZ2lzdHJ5IGltcG9ydCBsb2FkX21vZGVsX2J1bmRsZSwgcHJlZGljdF9mcm9tX2J1bmRsZQ0KDQoNCmRlZiBwYXJzZV9pZF9jb2x1bW5zKHZhbHVlOiBzdHIgfCBOb25lKSAtPiBsaXN0W3N0cl06DQogICAgIiIiUGFyc2Ugb3B0aW9uYWwgY29tbWEtc2VwYXJhdGVkIGlkZW50aWZpZXIgY29sdW1ucy4iIiINCiAgICBpZiB2YWx1ZSBpcyBOb25lIG9yIG5vdCB2YWx1ZS5zdHJpcCgpOg0KICAgICAgICByZXR1cm4gW10NCiAgICByZXR1cm4gW2l0ZW0uc3RyaXAoKSBmb3IgaXRlbSBpbiB2YWx1ZS5zcGxpdCgiLCIpIGlmIGl0ZW0uc3RyaXAoKV0NCg0KDQpkZWYgcGFyc2VfYXJncygpIC0+IGFyZ3BhcnNlLk5hbWVzcGFjZToNCiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iUHJlZGljdCBQYXNzL0ZhaWwgdXNpbmcgYSBzYXZlZCBtb2RlbCBidW5kbGUuIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1vZGVsLXBhdGgiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUsIGhlbHA9IlBhdGggdG8gYSBzYXZlZCBtb2RlbCBidW5kbGUuIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWZlYXR1cmVzLXBhdGgiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUsIGhlbHA9IkNTViBmaWxlIGNvbnRhaW5pbmcgZmVhdHVyZSByb3dzLiIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgNCiAgICAgICAgIi0tb3V0cHV0LXBhdGgiLA0KICAgICAgICB0eXBlPVBhdGgsDQogICAgICAgIGRlZmF1bHQ9UGF0aCgib3V0cHV0cy9wcmVkaWN0aW9ucy9wcmVkaWN0aW9ucy5jc3YiKSwNCiAgICAgICAgaGVscD0iT3V0cHV0IENTViBwYXRoIGZvciBwcmVkaWN0aW9ucy4iLA0KICAgICkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KA0KICAgICAgICAiLS1pZC1jb2x1bW5zIiwNCiAgICAgICAgdHlwZT1wYXJzZV9pZF9jb2x1bW5zLA0KICAgICAgICBkZWZhdWx0PVtdLA0KICAgICAgICBoZWxwPSJPcHRpb25hbCBjb21tYS1zZXBhcmF0ZWQgaWRlbnRpZmllciBjb2x1bW5zIHRvIGluY2x1ZGUgaW4gcHJlZGljdGlvbiBvdXRwdXQuIiwNCiAgICApDQogICAgcmV0dXJuIHBhcnNlci5wYXJzZV9hcmdzKCkNCg0KDQpkZWYgcnVuX3ByZWRpY3Rpb24oDQogICAgbW9kZWxfcGF0aDogUGF0aCwNCiAgICBmZWF0dXJlc19wYXRoOiBQYXRoLA0KICAgIG91dHB1dF9wYXRoOiBQYXRoLA0KICAgIGlkX2NvbHVtbnM6IGxpc3Rbc3RyXSB8IE5vbmUgPSBOb25lLA0KKSAtPiBQYXRoOg0KICAgICIiIkxvYWQgYSBtb2RlbCBidW5kbGUgYW5kIGZlYXR1cmUgdGFibGUsIHRoZW4gd3JpdGUgcHJlZGljdGlvbiByZXN1bHRzLiIiIg0KICAgIGlmIG5vdCBmZWF0dXJlc19wYXRoLmV4aXN0cygpOg0KICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkZlYXR1cmUgQ1NWIG5vdCBmb3VuZDoge2ZlYXR1cmVzX3BhdGh9IikNCg0KICAgIGJ1bmRsZSA9IGxvYWRfbW9kZWxfYnVuZGxlKG1vZGVsX3BhdGgpDQogICAgZmVhdHVyZV90YWJsZSA9IHBkLnJlYWRfY3N2KGZlYXR1cmVzX3BhdGgpDQogICAgcHJlZGljdGlvbnMgPSBwcmVkaWN0X2Zyb21fYnVuZGxlKGZlYXR1cmVfdGFibGUsIGJ1bmRsZSkNCg0KICAgIGlkX2NvbHVtbnMgPSBpZF9jb2x1bW5zIG9yIFtdDQogICAgbWlzc2luZ19pZF9jb2x1bW5zID0gW2NvbHVtbiBmb3IgY29sdW1uIGluIGlkX2NvbHVtbnMgaWYgY29sdW1uIG5vdCBpbiBmZWF0dXJlX3RhYmxlLmNvbHVtbnNdDQogICAgaWYgbWlzc2luZ19pZF9jb2x1bW5zOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTWlzc2luZyByZXF1ZXN0ZWQgaWQgY29sdW1uczoge21pc3NpbmdfaWRfY29sdW1uc30iKQ0KDQogICAgaWYgaWRfY29sdW1uczoNCiAgICAgICAgcHJlZGljdGlvbnMgPSBwZC5jb25jYXQoW2ZlYXR1cmVfdGFibGUubG9jWzosIGlkX2NvbHVtbnNdLCBwcmVkaWN0aW9uc10sIGF4aXM9MSkNCg0KICAgIG91dHB1dF9wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgcHJlZGljdGlvbnMudG9fY3N2KG91dHB1dF9wYXRoLCBpbmRleD1GYWxzZSkNCiAgICByZXR1cm4gb3V0cHV0X3BhdGgNCg0KDQpkZWYgbWFpbigpIC0+IGludDoNCiAgICBhcmdzID0gcGFyc2VfYXJncygpDQogICAgb3V0cHV0X3BhdGggPSBydW5fcHJlZGljdGlvbigNCiAgICAgICAgbW9kZWxfcGF0aD1hcmdzLm1vZGVsX3BhdGgsDQogICAgICAgIGZlYXR1cmVzX3BhdGg9YXJncy5mZWF0dXJlc19wYXRoLA0KICAgICAgICBvdXRwdXRfcGF0aD1hcmdzLm91dHB1dF9wYXRoLA0KICAgICAgICBpZF9jb2x1bW5zPWFyZ3MuaWRfY29sdW1ucywNCiAgICApDQogICAgcHJpbnQoZiJQcmVkaWN0aW9ucyB3cml0dGVuIHRvOiB7b3V0cHV0X3BhdGh9IikNCiAgICByZXR1cm4gMA0KDQoNCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6DQogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpDQo=', 'scripts/run_pipeline.py': 'IiIiUnVuIHRoZSBTRUNPTSBtYW51ZmFjdHVyaW5nIGFuYWx5dGljcyBwaXBlbGluZSBlbmQgdG8gZW5kLiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBhcmdwYXJzZQ0KZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzDQppbXBvcnQgc3lzDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KUFJPSkVDVF9ST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0NCmlmIHN0cihQUk9KRUNUX1JPT1QpIG5vdCBpbiBzeXMucGF0aDoNCiAgICBzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBST0pFQ1RfUk9PVCkpDQpmcm9tIHR5cGluZyBpbXBvcnQgQ2FsbGFibGUNCg0KZnJvbSBzY3JpcHRzLmFzc2VtYmxlX2ZlYXR1cmVfdGFibGUgaW1wb3J0IGFzc2VtYmxlX2Zyb21fcGF0aHMNCmZyb20gc2NyaXB0cy5nZW5lcmF0ZV9tb25pdG9yaW5nX3JlcG9ydCBpbXBvcnQgZ2VuZXJhdGVfbW9uaXRvcmluZ19yZXBvcnRzDQpmcm9tIHNjcmlwdHMucHJlZGljdF93aXRoX21vZGVsIGltcG9ydCBydW5fcHJlZGljdGlvbg0KZnJvbSBzY3JpcHRzLnJ1bl9xdWFsaXR5X3JlcG9ydCBpbXBvcnQgcnVuX3F1YWxpdHlfcmVwb3J0DQoNCg0KQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkNCmNsYXNzIFBpcGVsaW5lUGF0aHM6DQogICAgIiIiSW5wdXQgYW5kIG91dHB1dCBwYXRocyBmb3IgYW4gZW5kLXRvLWVuZCBwaXBlbGluZSBydW4uIiIiDQoNCiAgICByYXdfZGF0YV9kaXI6IFBhdGgNCiAgICBzZW5zb3JfcGF0aDogUGF0aA0KICAgIHdhZmVyX3BhdGg6IFBhdGggfCBOb25lDQogICAgZXF1aXBtZW50X3BhdGg6IFBhdGggfCBOb25lDQogICAgbW9kZWxfcGF0aDogUGF0aCB8IE5vbmUNCiAgICByZXBvcnRzX2RpcjogUGF0aA0KICAgIGZlYXR1cmVzX3BhdGg6IFBhdGgNCiAgICBwcmVkaWN0aW9uc19wYXRoOiBQYXRoDQogICAgbW9uaXRvcmluZ19kaXI6IFBhdGgNCg0KDQpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQ0KY2xhc3MgUGlwZWxpbmVSZXN1bHQ6DQogICAgIiIiT3V0cHV0IGFydGlmYWN0IHBhdGhzIGZyb20gYSBwaXBlbGluZSBydW4uIiIiDQoNCiAgICBxdWFsaXR5X3JlcG9ydF9kaXI6IFBhdGgNCiAgICBmZWF0dXJlX3RhYmxlX3BhdGg6IFBhdGgNCiAgICBqb2luX3JlcG9ydF9wYXRoOiBQYXRoDQogICAgZmVhdHVyZV9taXNzaW5nbmVzc19wYXRoOiBQYXRoDQogICAgcHJlZGljdGlvbnNfcGF0aDogUGF0aCB8IE5vbmUNCiAgICBtb25pdG9yaW5nX3BhdGhzOiBkaWN0W3N0ciwgUGF0aF0NCg0KDQpkZWYgcGFyc2Vfb3B0aW9uYWxfcGF0aCh2YWx1ZTogc3RyIHwgTm9uZSkgLT4gUGF0aCB8IE5vbmU6DQogICAgIiIiUGFyc2Ugb3B0aW9uYWwgcGF0aCBhcmd1bWVudHMuIiIiDQogICAgaWYgdmFsdWUgaXMgTm9uZSBvciBub3Qgc3RyKHZhbHVlKS5zdHJpcCgpOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIHJldHVybiBQYXRoKHZhbHVlKQ0KDQoNCmRlZiBwYXJzZV9jb2x1bW5zKHZhbHVlOiBzdHIgfCBOb25lKSAtPiBsaXN0W3N0cl06DQogICAgIiIiUGFyc2Ugb3B0aW9uYWwgY29tbWEtc2VwYXJhdGVkIGNvbHVtbnMuIiIiDQogICAgaWYgdmFsdWUgaXMgTm9uZSBvciBub3QgdmFsdWUuc3RyaXAoKToNCiAgICAgICAgcmV0dXJuIFtdDQogICAgcmV0dXJuIFtpdGVtLnN0cmlwKCkgZm9yIGl0ZW0gaW4gdmFsdWUuc3BsaXQoIiwiKSBpZiBpdGVtLnN0cmlwKCldDQoNCg0KZGVmIHBhcnNlX2FyZ3MoKSAtPiBhcmdwYXJzZS5OYW1lc3BhY2U6DQogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IlJ1biB0aGUgbWFudWZhY3R1cmluZyBhbmFseXRpY3MgcGlwZWxpbmUuIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJhdy1kYXRhLWRpciIsIHR5cGU9UGF0aCwgZGVmYXVsdD1QYXRoKCJkYXRhL3JhdyIpKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2Vuc29yLXBhdGgiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS13YWZlci1wYXRoIiwgdHlwZT1wYXJzZV9vcHRpb25hbF9wYXRoLCBkZWZhdWx0PU5vbmUpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1lcXVpcG1lbnQtcGF0aCIsIHR5cGU9cGFyc2Vfb3B0aW9uYWxfcGF0aCwgZGVmYXVsdD1Ob25lKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbW9kZWwtcGF0aCIsIHR5cGU9cGFyc2Vfb3B0aW9uYWxfcGF0aCwgZGVmYXVsdD1Ob25lKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcmVwb3J0cy1kaXIiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9UGF0aCgib3V0cHV0cy9yZXBvcnRzIikpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1mZWF0dXJlcy1wYXRoIiwgdHlwZT1QYXRoLCBkZWZhdWx0PVBhdGgoIm91dHB1dHMvZmVhdHVyZXMvbW9kZWxpbmdfdGFibGUuY3N2IikpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wcmVkaWN0aW9ucy1wYXRoIiwgdHlwZT1QYXRoLCBkZWZhdWx0PVBhdGgoIm91dHB1dHMvcHJlZGljdGlvbnMvcHJlZGljdGlvbnMuY3N2IikpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1tb25pdG9yaW5nLWRpciIsIHR5cGU9UGF0aCwgZGVmYXVsdD1QYXRoKCJvdXRwdXRzL3JlcG9ydHMvbW9uaXRvcmluZyIpKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZG93bmxvYWQtc2Vjb20iLCBhY3Rpb249InN0b3JlX3RydWUiKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2tpcC1xdWFsaXR5LXJlcG9ydCIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1za2lwLXByZWRpY3Rpb24iLCBhY3Rpb249InN0b3JlX3RydWUiKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0taWQtY29sdW1ucyIsIHR5cGU9cGFyc2VfY29sdW1ucywgZGVmYXVsdD1bXSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1vbml0b3JpbmctZ3JvdXAtY29sdW1ucyIsIHR5cGU9cGFyc2VfY29sdW1ucywgZGVmYXVsdD1bXSkNCiAgICByZXR1cm4gcGFyc2VyLnBhcnNlX2FyZ3MoKQ0KDQoNCmRlZiBydW5fcGlwZWxpbmUoDQogICAgcGF0aHM6IFBpcGVsaW5lUGF0aHMsDQogICAgZG93bmxvYWRfc2Vjb206IGJvb2wgPSBGYWxzZSwNCiAgICBza2lwX3F1YWxpdHlfcmVwb3J0OiBib29sID0gRmFsc2UsDQogICAgc2tpcF9wcmVkaWN0aW9uOiBib29sID0gRmFsc2UsDQogICAgaWRfY29sdW1uczogbGlzdFtzdHJdIHwgTm9uZSA9IE5vbmUsDQogICAgbW9uaXRvcmluZ19ncm91cF9jb2x1bW5zOiBsaXN0W3N0cl0gfCBOb25lID0gTm9uZSwNCiAgICBxdWFsaXR5X3N0ZXA6IENhbGxhYmxlID0gcnVuX3F1YWxpdHlfcmVwb3J0LA0KICAgIGFzc2VtYmxlX3N0ZXA6IENhbGxhYmxlID0gYXNzZW1ibGVfZnJvbV9wYXRocywNCiAgICBwcmVkaWN0aW9uX3N0ZXA6IENhbGxhYmxlID0gcnVuX3ByZWRpY3Rpb24sDQogICAgbW9uaXRvcmluZ19zdGVwOiBDYWxsYWJsZSA9IGdlbmVyYXRlX21vbml0b3JpbmdfcmVwb3J0cywNCikgLT4gUGlwZWxpbmVSZXN1bHQ6DQogICAgIiIiUnVuIHF1YWxpdHksIGZlYXR1cmUgYXNzZW1ibHksIHByZWRpY3Rpb24sIGFuZCBtb25pdG9yaW5nIHN0ZXBzLiIiIg0KICAgIHF1YWxpdHlfcmVwb3J0X2RpciA9IHBhdGhzLnJlcG9ydHNfZGlyIC8gInF1YWxpdHkiDQogICAgaWYgbm90IHNraXBfcXVhbGl0eV9yZXBvcnQ6DQogICAgICAgIHF1YWxpdHlfc3RlcCgNCiAgICAgICAgICAgIHJhd19kYXRhX2Rpcj1wYXRocy5yYXdfZGF0YV9kaXIsDQogICAgICAgICAgICBvdXRwdXRfZGlyPXF1YWxpdHlfcmVwb3J0X2RpciwNCiAgICAgICAgICAgIGRvd25sb2FkPWRvd25sb2FkX3NlY29tLA0KICAgICAgICApDQoNCiAgICBmZWF0dXJlX3RhYmxlX3BhdGgsIGpvaW5fcmVwb3J0X3BhdGgsIGZlYXR1cmVfbWlzc2luZ25lc3NfcGF0aCA9IGFzc2VtYmxlX3N0ZXAoDQogICAgICAgIHNlbnNvcl9wYXRoPXBhdGhzLnNlbnNvcl9wYXRoLA0KICAgICAgICB3YWZlcl9wYXRoPXBhdGhzLndhZmVyX3BhdGgsDQogICAgICAgIGVxdWlwbWVudF9wYXRoPXBhdGhzLmVxdWlwbWVudF9wYXRoLA0KICAgICAgICBvdXRwdXRfcGF0aD1wYXRocy5mZWF0dXJlc19wYXRoLA0KICAgICAgICByZXBvcnRfZGlyPXBhdGhzLnJlcG9ydHNfZGlyLA0KICAgICkNCg0KICAgIHByZWRpY3Rpb25zX3BhdGggPSBOb25lDQogICAgbW9uaXRvcmluZ19wYXRoczogZGljdFtzdHIsIFBhdGhdID0ge30NCiAgICBzaG91bGRfcHJlZGljdCA9IG5vdCBza2lwX3ByZWRpY3Rpb24gYW5kIHBhdGhzLm1vZGVsX3BhdGggaXMgbm90IE5vbmUNCiAgICBpZiBzaG91bGRfcHJlZGljdDoNCiAgICAgICAgcHJlZGljdGlvbnNfcGF0aCA9IHByZWRpY3Rpb25fc3RlcCgNCiAgICAgICAgICAgIG1vZGVsX3BhdGg9cGF0aHMubW9kZWxfcGF0aCwNCiAgICAgICAgICAgIGZlYXR1cmVzX3BhdGg9ZmVhdHVyZV90YWJsZV9wYXRoLA0KICAgICAgICAgICAgb3V0cHV0X3BhdGg9cGF0aHMucHJlZGljdGlvbnNfcGF0aCwNCiAgICAgICAgICAgIGlkX2NvbHVtbnM9aWRfY29sdW1ucyBvciBbXSwNCiAgICAgICAgKQ0KICAgICAgICBtb25pdG9yaW5nX3BhdGhzID0gbW9uaXRvcmluZ19zdGVwKA0KICAgICAgICAgICAgcHJlZGljdGlvbnNfcGF0aD1wcmVkaWN0aW9uc19wYXRoLA0KICAgICAgICAgICAgb3V0cHV0X2Rpcj1wYXRocy5tb25pdG9yaW5nX2RpciwNCiAgICAgICAgICAgIGdyb3VwX2NvbHVtbnM9bW9uaXRvcmluZ19ncm91cF9jb2x1bW5zIG9yIFtdLA0KICAgICAgICApDQoNCiAgICByZXR1cm4gUGlwZWxpbmVSZXN1bHQoDQogICAgICAgIHF1YWxpdHlfcmVwb3J0X2Rpcj1xdWFsaXR5X3JlcG9ydF9kaXIsDQogICAgICAgIGZlYXR1cmVfdGFibGVfcGF0aD1mZWF0dXJlX3RhYmxlX3BhdGgsDQogICAgICAgIGpvaW5fcmVwb3J0X3BhdGg9am9pbl9yZXBvcnRfcGF0aCwNCiAgICAgICAgZmVhdHVyZV9taXNzaW5nbmVzc19wYXRoPWZlYXR1cmVfbWlzc2luZ25lc3NfcGF0aCwNCiAgICAgICAgcHJlZGljdGlvbnNfcGF0aD1wcmVkaWN0aW9uc19wYXRoLA0KICAgICAgICBtb25pdG9yaW5nX3BhdGhzPW1vbml0b3JpbmdfcGF0aHMsDQogICAgKQ0KDQoNCmRlZiBtYWluKCkgLT4gaW50Og0KICAgIGFyZ3MgPSBwYXJzZV9hcmdzKCkNCiAgICByZXN1bHQgPSBydW5fcGlwZWxpbmUoDQogICAgICAgIHBhdGhzPVBpcGVsaW5lUGF0aHMoDQogICAgICAgICAgICByYXdfZGF0YV9kaXI9YXJncy5yYXdfZGF0YV9kaXIsDQogICAgICAgICAgICBzZW5zb3JfcGF0aD1hcmdzLnNlbnNvcl9wYXRoLA0KICAgICAgICAgICAgd2FmZXJfcGF0aD1hcmdzLndhZmVyX3BhdGgsDQogICAgICAgICAgICBlcXVpcG1lbnRfcGF0aD1hcmdzLmVxdWlwbWVudF9wYXRoLA0KICAgICAgICAgICAgbW9kZWxfcGF0aD1hcmdzLm1vZGVsX3BhdGgsDQogICAgICAgICAgICByZXBvcnRzX2Rpcj1hcmdzLnJlcG9ydHNfZGlyLA0KICAgICAgICAgICAgZmVhdHVyZXNfcGF0aD1hcmdzLmZlYXR1cmVzX3BhdGgsDQogICAgICAgICAgICBwcmVkaWN0aW9uc19wYXRoPWFyZ3MucHJlZGljdGlvbnNfcGF0aCwNCiAgICAgICAgICAgIG1vbml0b3JpbmdfZGlyPWFyZ3MubW9uaXRvcmluZ19kaXIsDQogICAgICAgICksDQogICAgICAgIGRvd25sb2FkX3NlY29tPWFyZ3MuZG93bmxvYWRfc2Vjb20sDQogICAgICAgIHNraXBfcXVhbGl0eV9yZXBvcnQ9YXJncy5za2lwX3F1YWxpdHlfcmVwb3J0LA0KICAgICAgICBza2lwX3ByZWRpY3Rpb249YXJncy5za2lwX3ByZWRpY3Rpb24sDQogICAgICAgIGlkX2NvbHVtbnM9YXJncy5pZF9jb2x1bW5zLA0KICAgICAgICBtb25pdG9yaW5nX2dyb3VwX2NvbHVtbnM9YXJncy5tb25pdG9yaW5nX2dyb3VwX2NvbHVtbnMsDQogICAgKQ0KICAgIHByaW50KGYiRmVhdHVyZSB0YWJsZToge3Jlc3VsdC5mZWF0dXJlX3RhYmxlX3BhdGh9IikNCiAgICBwcmludChmIkpvaW4gcmVwb3J0OiB7cmVzdWx0LmpvaW5fcmVwb3J0X3BhdGh9IikNCiAgICBwcmludChmIkZlYXR1cmUgbWlzc2luZ25lc3MgcmVwb3J0OiB7cmVzdWx0LmZlYXR1cmVfbWlzc2luZ25lc3NfcGF0aH0iKQ0KICAgIGlmIHJlc3VsdC5wcmVkaWN0aW9uc19wYXRoOg0KICAgICAgICBwcmludChmIlByZWRpY3Rpb25zOiB7cmVzdWx0LnByZWRpY3Rpb25zX3BhdGh9IikNCiAgICBmb3IgbmFtZSwgcGF0aCBpbiByZXN1bHQubW9uaXRvcmluZ19wYXRocy5pdGVtcygpOg0KICAgICAgICBwcmludChmIk1vbml0b3Jpbmcge25hbWV9OiB7cGF0aH0iKQ0KICAgIHJldHVybiAwDQoNCg0KaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoNCiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkNCg==', 'scripts/run_quality_report.py': 'IiIiR2VuZXJhdGUgU0VDT00gZGF0YSBxdWFsaXR5IHJlcG9ydHMgZnJvbSBsb2NhbCBvciBkb3dubG9hZGVkIHJhdyBmaWxlcy4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgYXJncGFyc2UNCmltcG9ydCBzeXMNCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KDQpQUk9KRUNUX1JPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXQ0KaWYgc3RyKFBST0pFQ1RfUk9PVCkgbm90IGluIHN5cy5wYXRoOg0KICAgIHN5cy5wYXRoLmluc2VydCgwLCBzdHIoUFJPSkVDVF9ST09UKSkNCg0KaW1wb3J0IHBhbmRhcyBhcyBwZA0KDQpmcm9tIHNyYy5xdWFsaXR5X3JlcG9ydHMgaW1wb3J0IHF1YWxpdHlfcmVwb3J0X2J1bmRsZQ0KZnJvbSBzcmMuc2Vjb21fZGF0YSBpbXBvcnQgZGVmYXVsdF9zZWNvbV9wYXRocywgZG93bmxvYWRfc2Vjb21fZGF0YXNldCwgbG9hZF9zZWNvbV9kYXRhDQoNCg0KZGVmIHBhcnNlX2FyZ3MoKSAtPiBhcmdwYXJzZS5OYW1lc3BhY2U6DQogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IkdlbmVyYXRlIFNFQ09NIGRhdGEgcXVhbGl0eSByZXBvcnRzLiIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgNCiAgICAgICAgIi0tcmF3LWRhdGEtZGlyIiwNCiAgICAgICAgdHlwZT1QYXRoLA0KICAgICAgICBkZWZhdWx0PVBhdGgoImRhdGEvcmF3IiksDQogICAgICAgIGhlbHA9IkRpcmVjdG9yeSBjb250YWluaW5nIHNlY29tLmRhdGEgYW5kIHNlY29tX2xhYmVscy5kYXRhLiIsDQogICAgKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoDQogICAgICAgICItLW91dHB1dC1kaXIiLA0KICAgICAgICB0eXBlPVBhdGgsDQogICAgICAgIGRlZmF1bHQ9UGF0aCgib3V0cHV0cy9yZXBvcnRzIiksDQogICAgICAgIGhlbHA9IkRpcmVjdG9yeSB3aGVyZSByZXBvcnQgQ1NWIGZpbGVzIHdpbGwgYmUgd3JpdHRlbi4iLA0KICAgICkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KA0KICAgICAgICAiLS1kb3dubG9hZCIsDQogICAgICAgIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsDQogICAgICAgIGhlbHA9IkRvd25sb2FkIFVDSSBTRUNPTSByYXcgZmlsZXMgYmVmb3JlIGxvYWRpbmcuIiwNCiAgICApDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgNCiAgICAgICAgIi0tbWlzc2luZy10aHJlc2hvbGQiLA0KICAgICAgICB0eXBlPWZsb2F0LA0KICAgICAgICBkZWZhdWx0PTAuNSwNCiAgICAgICAgaGVscD0iTWlzc2luZyByYXRpbyB0aHJlc2hvbGQgZm9yIGhpZ2gtbWlzc2luZyBmZWF0dXJlIHJlcG9ydGluZy4iLA0KICAgICkNCiAgICByZXR1cm4gcGFyc2VyLnBhcnNlX2FyZ3MoKQ0KDQoNCmRlZiB3cml0ZV9yZXBvcnRfYnVuZGxlKHJlcG9ydF9idW5kbGU6IGRpY3QsIG91dHB1dF9kaXI6IFBhdGgpIC0+IE5vbmU6DQogICAgIiIiV3JpdGUgcXVhbGl0eSByZXBvcnQgYnVuZGxlIG9iamVjdHMgdG8gQ1NWIGZpbGVzLiIiIg0KICAgIG91dHB1dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KDQogICAgb3ZlcnZpZXcgPSByZXBvcnRfYnVuZGxlWyJvdmVydmlldyJdLnRvX2ZyYW1lKCkucmVzZXRfaW5kZXgoKQ0KICAgIG92ZXJ2aWV3LmNvbHVtbnMgPSBbIm1ldHJpYyIsICJ2YWx1ZSJdDQogICAgb3ZlcnZpZXcudG9fY3N2KG91dHB1dF9kaXIgLyAib3ZlcnZpZXcuY3N2IiwgaW5kZXg9RmFsc2UpDQoNCiAgICByZXBvcnRfYnVuZGxlWyJjbGFzc19kaXN0cmlidXRpb24iXS50b19jc3Yob3V0cHV0X2RpciAvICJjbGFzc19kaXN0cmlidXRpb24uY3N2IiwgaW5kZXg9RmFsc2UpDQogICAgcmVwb3J0X2J1bmRsZVsibWlzc2luZ25lc3MiXS50b19jc3Yob3V0cHV0X2RpciAvICJtaXNzaW5nbmVzcy5jc3YiLCBpbmRleF9sYWJlbD0iZmVhdHVyZSIpDQoNCiAgICBwZC5TZXJpZXMocmVwb3J0X2J1bmRsZVsiaGlnaF9taXNzaW5nX2ZlYXR1cmVzIl0sIG5hbWU9ImZlYXR1cmUiKS50b19jc3YoDQogICAgICAgIG91dHB1dF9kaXIgLyAiaGlnaF9taXNzaW5nX2ZlYXR1cmVzLmNzdiIsDQogICAgICAgIGluZGV4PUZhbHNlLA0KICAgICkNCiAgICBwZC5TZXJpZXMocmVwb3J0X2J1bmRsZVsiY29uc3RhbnRfZmVhdHVyZXMiXSwgbmFtZT0iZmVhdHVyZSIpLnRvX2NzdigNCiAgICAgICAgb3V0cHV0X2RpciAvICJjb25zdGFudF9mZWF0dXJlcy5jc3YiLA0KICAgICAgICBpbmRleD1GYWxzZSwNCiAgICApDQoNCg0KZGVmIHJ1bl9xdWFsaXR5X3JlcG9ydCgNCiAgICByYXdfZGF0YV9kaXI6IFBhdGgsDQogICAgb3V0cHV0X2RpcjogUGF0aCwNCiAgICBkb3dubG9hZDogYm9vbCA9IEZhbHNlLA0KICAgIG1pc3NpbmdfdGhyZXNob2xkOiBmbG9hdCA9IDAuNSwNCikgLT4gUGF0aDoNCiAgICAiIiJMb2FkIFNFQ09NIGRhdGEgYW5kIHdyaXRlIHF1YWxpdHkgcmVwb3J0cyB0byBhbiBvdXRwdXQgZGlyZWN0b3J5LiIiIg0KICAgIGlmIGRvd25sb2FkOg0KICAgICAgICBmZWF0dXJlX3BhdGgsIGxhYmVsX3BhdGggPSBkb3dubG9hZF9zZWNvbV9kYXRhc2V0KHJhd19kYXRhX2RpcikNCiAgICBlbHNlOg0KICAgICAgICBmZWF0dXJlX3BhdGgsIGxhYmVsX3BhdGggPSBkZWZhdWx0X3NlY29tX3BhdGhzKHJhd19kYXRhX2RpcikNCg0KICAgIGRhdGFzZXQgPSBsb2FkX3NlY29tX2RhdGEoZmVhdHVyZV9wYXRoLCBsYWJlbF9wYXRoKQ0KICAgIHJlcG9ydF9idW5kbGUgPSBxdWFsaXR5X3JlcG9ydF9idW5kbGUoZGF0YXNldCwgbWlzc2luZ190aHJlc2hvbGQ9bWlzc2luZ190aHJlc2hvbGQpDQogICAgd3JpdGVfcmVwb3J0X2J1bmRsZShyZXBvcnRfYnVuZGxlLCBvdXRwdXRfZGlyKQ0KICAgIHJldHVybiBvdXRwdXRfZGlyDQoNCg0KZGVmIG1haW4oKSAtPiBpbnQ6DQogICAgYXJncyA9IHBhcnNlX2FyZ3MoKQ0KICAgIG91dHB1dF9kaXIgPSBydW5fcXVhbGl0eV9yZXBvcnQoDQogICAgICAgIHJhd19kYXRhX2Rpcj1hcmdzLnJhd19kYXRhX2RpciwNCiAgICAgICAgb3V0cHV0X2Rpcj1hcmdzLm91dHB1dF9kaXIsDQogICAgICAgIGRvd25sb2FkPWFyZ3MuZG93bmxvYWQsDQogICAgICAgIG1pc3NpbmdfdGhyZXNob2xkPWFyZ3MubWlzc2luZ190aHJlc2hvbGQsDQogICAgKQ0KICAgIHByaW50KGYiUXVhbGl0eSByZXBvcnRzIHdyaXR0ZW4gdG86IHtvdXRwdXRfZGlyfSIpDQogICAgcmV0dXJuIDANCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQ0K', 'scripts/train_secom_model.py': 'IiIiVHJhaW4gYW5kIHNhdmUgYSBTRUNPTSBQYXNzL0ZhaWwgY2xhc3NpZmljYXRpb24gbW9kZWwuIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IGFyZ3BhcnNlDQppbXBvcnQgc3lzDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KUFJPSkVDVF9ST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0NCmlmIHN0cihQUk9KRUNUX1JPT1QpIG5vdCBpbiBzeXMucGF0aDoNCiAgICBzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBST0pFQ1RfUk9PVCkpDQpmcm9tIHR5cGluZyBpbXBvcnQgQW55DQoNCmltcG9ydCBwYW5kYXMgYXMgcGQNCg0KZnJvbSBzcmMubW9kZWxfcmVnaXN0cnkgaW1wb3J0IE1vZGVsQnVuZGxlLCBzYXZlX21vZGVsX2J1bmRsZQ0KZnJvbSBzcmMuc2Vjb21fZGF0YSBpbXBvcnQgZGVmYXVsdF9zZWNvbV9wYXRocywgZG93bmxvYWRfc2Vjb21fZGF0YXNldCwgbG9hZF9zZWNvbV9kYXRhDQpmcm9tIHNyYy5zZWNvbV9tb2RlbGluZyBpbXBvcnQgKA0KICAgIEhpZ2hNaXNzaW5nRmVhdHVyZURyb3BwZXIsDQogICAgZXZhbHVhdGVfY2xhc3NpZmllciwNCiAgICBnZXRfcG9zaXRpdmVfcHJvYmEsDQogICAgbWFrZV9saW5lYXJfcGlwZWxpbmUsDQogICAgbWFrZV90cmVlX3BpcGVsaW5lLA0KICAgIHByZWRpY3Rfd2l0aF90aHJlc2hvbGQsDQopDQpmcm9tIHNyYy5zZWNvbV90cmFpbmluZyBpbXBvcnQgKA0KICAgIERFRkFVTFRfTU9ERUxfUkFOS0lORywNCiAgICBERUZBVUxUX1RIUkVTSE9MRF9SQU5LSU5HLA0KICAgIGJ1aWxkX3RocmVzaG9sZF9tZXRyaWNzLA0KICAgIHBhcnNlX3RocmVzaG9sZHMsDQogICAgcmFua19yZXN1bHRzLA0KICAgIHNlbGVjdF90b3BfcmVzdWx0LA0KKQ0KDQoNCmRlZiBwYXJzZV9vcHRpb25hbF9wYXRoKHZhbHVlOiBzdHIgfCBOb25lKSAtPiBQYXRoIHwgTm9uZToNCiAgICAiIiJQYXJzZSBvcHRpb25hbCBwYXRoIGFyZ3VtZW50cy4iIiINCiAgICBpZiB2YWx1ZSBpcyBOb25lIG9yIG5vdCBzdHIodmFsdWUpLnN0cmlwKCk6DQogICAgICAgIHJldHVybiBOb25lDQogICAgcmV0dXJuIFBhdGgodmFsdWUpDQoNCg0KZGVmIHBhcnNlX2FyZ3MoKSAtPiBhcmdwYXJzZS5OYW1lc3BhY2U6DQogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IlRyYWluIGEgU0VDT00gUGFzcy9GYWlsIG1vZGVsLiIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1yYXctZGF0YS1kaXIiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9UGF0aCgiZGF0YS9yYXciKSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWZlYXR1cmUtcGF0aCIsIHR5cGU9cGFyc2Vfb3B0aW9uYWxfcGF0aCwgZGVmYXVsdD1Ob25lKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbGFiZWwtcGF0aCIsIHR5cGU9cGFyc2Vfb3B0aW9uYWxfcGF0aCwgZGVmYXVsdD1Ob25lKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZG93bmxvYWQtc2Vjb20iLCBhY3Rpb249InN0b3JlX3RydWUiKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbW9kZWwtb3V0cHV0IiwgdHlwZT1QYXRoLCBkZWZhdWx0PVBhdGgoIm91dHB1dHMvbW9kZWxzL3NlY29tX2ZpbmFsX3BpcGVsaW5lLmpvYmxpYiIpKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbWV0cmljcy1vdXRwdXQiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9UGF0aCgib3V0cHV0cy9yZXBvcnRzL21vZGVsX21ldHJpY3MuY3N2IikpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10aHJlc2hvbGQtb3V0cHV0IiwgdHlwZT1QYXRoLCBkZWZhdWx0PVBhdGgoIm91dHB1dHMvcmVwb3J0cy90aHJlc2hvbGRfbWV0cmljcy5jc3YiKSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXByZWRpY3Rpb25zLW91dHB1dCIsIHR5cGU9UGF0aCwgZGVmYXVsdD1QYXRoKCJvdXRwdXRzL3ByZWRpY3Rpb25zL3Rlc3RfcHJlZGljdGlvbnMuY3N2IikpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10ZXN0LXNpemUiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuMikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXZhbGlkYXRpb24tc2l6ZSIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4yNSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1pc3NpbmctdGhyZXNob2xkIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjUpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1sb3ctdmFyaWFuY2UtdGhyZXNob2xkIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0xZS04KQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcmFuZG9tLXN0YXRlIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NDIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10aHJlc2hvbGRzIiwgdHlwZT1wYXJzZV90aHJlc2hvbGRzLCBkZWZhdWx0PXBhcnNlX3RocmVzaG9sZHMoTm9uZSkpDQogICAgcmV0dXJuIHBhcnNlci5wYXJzZV9hcmdzKCkNCg0KDQpkZWYgcmVzb2x2ZV9zZWNvbV9wYXRocygNCiAgICByYXdfZGF0YV9kaXI6IFBhdGgsDQogICAgZmVhdHVyZV9wYXRoOiBQYXRoIHwgTm9uZSA9IE5vbmUsDQogICAgbGFiZWxfcGF0aDogUGF0aCB8IE5vbmUgPSBOb25lLA0KICAgIGRvd25sb2FkOiBib29sID0gRmFsc2UsDQopIC0+IHR1cGxlW1BhdGgsIFBhdGhdOg0KICAgICIiIlJlc29sdmUgU0VDT00gZmVhdHVyZSBhbmQgbGFiZWwgcGF0aHMsIG9wdGlvbmFsbHkgZG93bmxvYWRpbmcgdGhlbS4iIiINCiAgICBpZiBkb3dubG9hZDoNCiAgICAgICAgZG93bmxvYWRlZF9mZWF0dXJlX3BhdGgsIGRvd25sb2FkZWRfbGFiZWxfcGF0aCA9IGRvd25sb2FkX3NlY29tX2RhdGFzZXQocmF3X2RhdGFfZGlyKQ0KICAgICAgICBmZWF0dXJlX3BhdGggPSBmZWF0dXJlX3BhdGggb3IgZG93bmxvYWRlZF9mZWF0dXJlX3BhdGgNCiAgICAgICAgbGFiZWxfcGF0aCA9IGxhYmVsX3BhdGggb3IgZG93bmxvYWRlZF9sYWJlbF9wYXRoDQogICAgZWxzZToNCiAgICAgICAgZGVmYXVsdF9mZWF0dXJlX3BhdGgsIGRlZmF1bHRfbGFiZWxfcGF0aCA9IGRlZmF1bHRfc2Vjb21fcGF0aHMocmF3X2RhdGFfZGlyKQ0KICAgICAgICBmZWF0dXJlX3BhdGggPSBmZWF0dXJlX3BhdGggb3IgZGVmYXVsdF9mZWF0dXJlX3BhdGgNCiAgICAgICAgbGFiZWxfcGF0aCA9IGxhYmVsX3BhdGggb3IgZGVmYXVsdF9sYWJlbF9wYXRoDQoNCiAgICBtaXNzaW5nID0gW3BhdGggZm9yIHBhdGggaW4gW2ZlYXR1cmVfcGF0aCwgbGFiZWxfcGF0aF0gaWYgcGF0aCBpcyBOb25lIG9yIG5vdCBQYXRoKHBhdGgpLmV4aXN0cygpXQ0KICAgIGlmIG1pc3Npbmc6DQogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKA0KICAgICAgICAgICAgIlNFQ09NIGZpbGVzIGFyZSBtaXNzaW5nLiBQYXNzIC0tZG93bmxvYWQtc2Vjb20gb3IgcHJvdmlkZSAtLWZlYXR1cmUtcGF0aCBhbmQgLS1sYWJlbC1wYXRoLiINCiAgICAgICAgKQ0KICAgIHJldHVybiBQYXRoKGZlYXR1cmVfcGF0aCksIFBhdGgobGFiZWxfcGF0aCkNCg0KDQpkZWYgbWFrZV9jYW5kaWRhdGVfbW9kZWxzKA0KICAgIG1pc3NpbmdfdGhyZXNob2xkOiBmbG9hdCA9IDAuNSwNCiAgICBsb3dfdmFyaWFuY2VfdGhyZXNob2xkOiBmbG9hdCA9IDFlLTgsDQogICAgcmFuZG9tX3N0YXRlOiBpbnQgPSA0MiwNCikgLT4gZGljdFtzdHIsIHR1cGxlW3N0ciwgQW55XV06DQogICAgIiIiQ3JlYXRlIGNhbmRpZGF0ZSBtb2RlbHMgZm9yIGJhc2VsaW5lLCBsaW5lYXIsIGFuZCB0cmVlLWJhc2VkIGNvbXBhcmlzb24uIiIiDQogICAgZnJvbSBza2xlYXJuLmR1bW15IGltcG9ydCBEdW1teUNsYXNzaWZpZXINCiAgICBmcm9tIHNrbGVhcm4uZmVhdHVyZV9zZWxlY3Rpb24gaW1wb3J0IFZhcmlhbmNlVGhyZXNob2xkDQogICAgZnJvbSBza2xlYXJuLmltcHV0ZSBpbXBvcnQgU2ltcGxlSW1wdXRlcg0KICAgIGZyb20gc2tsZWFybi5waXBlbGluZSBpbXBvcnQgUGlwZWxpbmUNCg0KICAgIGR1bW15X3BpcGVsaW5lID0gUGlwZWxpbmUoDQogICAgICAgIHN0ZXBzPVsNCiAgICAgICAgICAgICgiZHJvcF9oaWdoX21pc3NpbmciLCBIaWdoTWlzc2luZ0ZlYXR1cmVEcm9wcGVyKHRocmVzaG9sZD1taXNzaW5nX3RocmVzaG9sZCkpLA0KICAgICAgICAgICAgKCJpbXB1dGVyIiwgU2ltcGxlSW1wdXRlcihzdHJhdGVneT0ibWVkaWFuIikpLA0KICAgICAgICAgICAgKCJ2YXJpYW5jZSIsIFZhcmlhbmNlVGhyZXNob2xkKHRocmVzaG9sZD1sb3dfdmFyaWFuY2VfdGhyZXNob2xkKSksDQogICAgICAgICAgICAoIm1vZGVsIiwgRHVtbXlDbGFzc2lmaWVyKHN0cmF0ZWd5PSJtb3N0X2ZyZXF1ZW50IiwgcmFuZG9tX3N0YXRlPXJhbmRvbV9zdGF0ZSkpLA0KICAgICAgICBdDQogICAgKQ0KICAgIHJldHVybiB7DQogICAgICAgICJEdW1teSBNb3N0IEZyZXF1ZW50IjogKCJiYXNlbGluZSIsIGR1bW15X3BpcGVsaW5lKSwNCiAgICAgICAgIkxvZ2lzdGljIFJlZ3Jlc3Npb24iOiAoDQogICAgICAgICAgICAibm9uZSIsDQogICAgICAgICAgICBtYWtlX2xpbmVhcl9waXBlbGluZSgNCiAgICAgICAgICAgICAgICBtaXNzaW5nX3RocmVzaG9sZD1taXNzaW5nX3RocmVzaG9sZCwNCiAgICAgICAgICAgICAgICBsb3dfdmFyaWFuY2VfdGhyZXNob2xkPWxvd192YXJpYW5jZV90aHJlc2hvbGQsDQogICAgICAgICAgICAgICAgY2xhc3Nfd2VpZ2h0PU5vbmUsDQogICAgICAgICAgICAgICAgcmFuZG9tX3N0YXRlPXJhbmRvbV9zdGF0ZSwNCiAgICAgICAgICAgICksDQogICAgICAgICksDQogICAgICAgICJMb2dpc3RpYyBSZWdyZXNzaW9uIEJhbGFuY2VkIjogKA0KICAgICAgICAgICAgImNsYXNzX3dlaWdodD1iYWxhbmNlZCIsDQogICAgICAgICAgICBtYWtlX2xpbmVhcl9waXBlbGluZSgNCiAgICAgICAgICAgICAgICBtaXNzaW5nX3RocmVzaG9sZD1taXNzaW5nX3RocmVzaG9sZCwNCiAgICAgICAgICAgICAgICBsb3dfdmFyaWFuY2VfdGhyZXNob2xkPWxvd192YXJpYW5jZV90aHJlc2hvbGQsDQogICAgICAgICAgICAgICAgY2xhc3Nfd2VpZ2h0PSJiYWxhbmNlZCIsDQogICAgICAgICAgICAgICAgcmFuZG9tX3N0YXRlPXJhbmRvbV9zdGF0ZSwNCiAgICAgICAgICAgICksDQogICAgICAgICksDQogICAgICAgICJSYW5kb20gRm9yZXN0IEJhbGFuY2VkIjogKA0KICAgICAgICAgICAgImNsYXNzX3dlaWdodD1iYWxhbmNlZCIsDQogICAgICAgICAgICBtYWtlX3RyZWVfcGlwZWxpbmUoDQogICAgICAgICAgICAgICAgbWlzc2luZ190aHJlc2hvbGQ9bWlzc2luZ190aHJlc2hvbGQsDQogICAgICAgICAgICAgICAgbG93X3ZhcmlhbmNlX3RocmVzaG9sZD1sb3dfdmFyaWFuY2VfdGhyZXNob2xkLA0KICAgICAgICAgICAgICAgIGNsYXNzX3dlaWdodD0iYmFsYW5jZWQiLA0KICAgICAgICAgICAgICAgIHJhbmRvbV9zdGF0ZT1yYW5kb21fc3RhdGUsDQogICAgICAgICAgICApLA0KICAgICAgICApLA0KICAgIH0NCg0KDQpkZWYgd3JpdGVfdGVzdF9wcmVkaWN0aW9ucygNCiAgICBvdXRwdXRfcGF0aDogUGF0aCwNCiAgICBtb2RlbDogQW55LA0KICAgIFhfdGVzdDogcGQuRGF0YUZyYW1lLA0KICAgIHlfdGVzdDogcGQuU2VyaWVzLA0KICAgIHRpbWVzdGFtcHM6IHBkLlNlcmllcywNCiAgICB0aHJlc2hvbGQ6IGZsb2F0LA0KKSAtPiBQYXRoOg0KICAgICIiIldyaXRlIGhlbGQtb3V0IHRlc3QgcHJlZGljdGlvbnMgd2l0aCBsYWJlbHMgYW5kIHRpbWVzdGFtcHMuIiIiDQogICAgcHJlZGljdGlvbl9mcmFtZSA9IHByZWRpY3Rfd2l0aF90aHJlc2hvbGQobW9kZWwsIFhfdGVzdCwgdGhyZXNob2xkPXRocmVzaG9sZCkudG9fZnJhbWUoaW5kZXg9WF90ZXN0LmluZGV4KQ0KICAgIG91dHB1dCA9IHBkLkRhdGFGcmFtZSgNCiAgICAgICAgew0KICAgICAgICAgICAgInNhbXBsZV9pbmRleCI6IFhfdGVzdC5pbmRleCwNCiAgICAgICAgICAgICJ0aW1lc3RhbXAiOiB0aW1lc3RhbXBzLmxvY1tYX3Rlc3QuaW5kZXhdLmFzdHlwZShzdHIpLnRvX251bXB5KCksDQogICAgICAgICAgICAiYWN0dWFsX2xhYmVsIjogeV90ZXN0LnRvX251bXB5KCksDQogICAgICAgICAgICAiYWN0dWFsX25hbWUiOiB5X3Rlc3QucmVwbGFjZSh7MDogIlBhc3MiLCAxOiAiRmFpbCJ9KS50b19udW1weSgpLA0KICAgICAgICB9LA0KICAgICAgICBpbmRleD1YX3Rlc3QuaW5kZXgsDQogICAgKQ0KICAgIG91dHB1dCA9IHBkLmNvbmNhdChbb3V0cHV0LCBwcmVkaWN0aW9uX2ZyYW1lXSwgYXhpcz0xKQ0KICAgIG91dHB1dF9wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgb3V0cHV0LnRvX2NzdihvdXRwdXRfcGF0aCwgaW5kZXg9RmFsc2UpDQogICAgcmV0dXJuIG91dHB1dF9wYXRoDQoNCg0KZGVmIHRyYWluX3NlY29tX21vZGVsKA0KICAgIHJhd19kYXRhX2RpcjogUGF0aCA9IFBhdGgoImRhdGEvcmF3IiksDQogICAgZmVhdHVyZV9wYXRoOiBQYXRoIHwgTm9uZSA9IE5vbmUsDQogICAgbGFiZWxfcGF0aDogUGF0aCB8IE5vbmUgPSBOb25lLA0KICAgIGRvd25sb2FkX3NlY29tOiBib29sID0gRmFsc2UsDQogICAgbW9kZWxfb3V0cHV0OiBQYXRoID0gUGF0aCgib3V0cHV0cy9tb2RlbHMvc2Vjb21fZmluYWxfcGlwZWxpbmUuam9ibGliIiksDQogICAgbWV0cmljc19vdXRwdXQ6IFBhdGggPSBQYXRoKCJvdXRwdXRzL3JlcG9ydHMvbW9kZWxfbWV0cmljcy5jc3YiKSwNCiAgICB0aHJlc2hvbGRfb3V0cHV0OiBQYXRoID0gUGF0aCgib3V0cHV0cy9yZXBvcnRzL3RocmVzaG9sZF9tZXRyaWNzLmNzdiIpLA0KICAgIHByZWRpY3Rpb25zX291dHB1dDogUGF0aCA9IFBhdGgoIm91dHB1dHMvcHJlZGljdGlvbnMvdGVzdF9wcmVkaWN0aW9ucy5jc3YiKSwNCiAgICB0ZXN0X3NpemU6IGZsb2F0ID0gMC4yLA0KICAgIHZhbGlkYXRpb25fc2l6ZTogZmxvYXQgPSAwLjI1LA0KICAgIG1pc3NpbmdfdGhyZXNob2xkOiBmbG9hdCA9IDAuNSwNCiAgICBsb3dfdmFyaWFuY2VfdGhyZXNob2xkOiBmbG9hdCA9IDFlLTgsDQogICAgcmFuZG9tX3N0YXRlOiBpbnQgPSA0MiwNCiAgICB0aHJlc2hvbGRzOiBsaXN0W2Zsb2F0XSB8IE5vbmUgPSBOb25lLA0KKSAtPiBkaWN0W3N0ciwgUGF0aF06DQogICAgIiIiVHJhaW4gY2FuZGlkYXRlIG1vZGVscywgdHVuZSB0aHJlc2hvbGQsIGFuZCBzYXZlIGZpbmFsIGFydGlmYWN0cy4iIiINCiAgICBmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCB0cmFpbl90ZXN0X3NwbGl0DQoNCiAgICBmZWF0dXJlX3BhdGgsIGxhYmVsX3BhdGggPSByZXNvbHZlX3NlY29tX3BhdGhzKA0KICAgICAgICByYXdfZGF0YV9kaXI9cmF3X2RhdGFfZGlyLA0KICAgICAgICBmZWF0dXJlX3BhdGg9ZmVhdHVyZV9wYXRoLA0KICAgICAgICBsYWJlbF9wYXRoPWxhYmVsX3BhdGgsDQogICAgICAgIGRvd25sb2FkPWRvd25sb2FkX3NlY29tLA0KICAgICkNCiAgICBkYXRhc2V0ID0gbG9hZF9zZWNvbV9kYXRhKGZlYXR1cmVfcGF0aCwgbGFiZWxfcGF0aCkNCiAgICB0aHJlc2hvbGRzID0gdGhyZXNob2xkcyBvciBwYXJzZV90aHJlc2hvbGRzKE5vbmUpDQoNCiAgICBYX3RyYWluX3ZhbGlkLCBYX3Rlc3QsIHlfdHJhaW5fdmFsaWQsIHlfdGVzdCA9IHRyYWluX3Rlc3Rfc3BsaXQoDQogICAgICAgIGRhdGFzZXQuZmVhdHVyZXMsDQogICAgICAgIGRhdGFzZXQubGFiZWxzLA0KICAgICAgICB0ZXN0X3NpemU9dGVzdF9zaXplLA0KICAgICAgICBzdHJhdGlmeT1kYXRhc2V0LmxhYmVscywNCiAgICAgICAgcmFuZG9tX3N0YXRlPXJhbmRvbV9zdGF0ZSwNCiAgICApDQogICAgWF90cmFpbiwgWF92YWxpZCwgeV90cmFpbiwgeV92YWxpZCA9IHRyYWluX3Rlc3Rfc3BsaXQoDQogICAgICAgIFhfdHJhaW5fdmFsaWQsDQogICAgICAgIHlfdHJhaW5fdmFsaWQsDQogICAgICAgIHRlc3Rfc2l6ZT12YWxpZGF0aW9uX3NpemUsDQogICAgICAgIHN0cmF0aWZ5PXlfdHJhaW5fdmFsaWQsDQogICAgICAgIHJhbmRvbV9zdGF0ZT1yYW5kb21fc3RhdGUsDQogICAgKQ0KDQogICAgdmFsaWRhdGlvbl9yb3dzID0gW10NCiAgICBjYW5kaWRhdGVzID0gbWFrZV9jYW5kaWRhdGVfbW9kZWxzKA0KICAgICAgICBtaXNzaW5nX3RocmVzaG9sZD1taXNzaW5nX3RocmVzaG9sZCwNCiAgICAgICAgbG93X3ZhcmlhbmNlX3RocmVzaG9sZD1sb3dfdmFyaWFuY2VfdGhyZXNob2xkLA0KICAgICAgICByYW5kb21fc3RhdGU9cmFuZG9tX3N0YXRlLA0KICAgICkNCiAgICBmb3IgbW9kZWxfbmFtZSwgKGltYmFsYW5jZV9tZXRob2QsIG1vZGVsKSBpbiBjYW5kaWRhdGVzLml0ZW1zKCk6DQogICAgICAgIG1vZGVsLmZpdChYX3RyYWluLCB5X3RyYWluKQ0KICAgICAgICB2YWxpZGF0aW9uX3Jvd3MuYXBwZW5kKA0KICAgICAgICAgICAgew0KICAgICAgICAgICAgICAgICJzcGxpdCI6ICJ2YWxpZGF0aW9uIiwNCiAgICAgICAgICAgICAgICAqKmV2YWx1YXRlX2NsYXNzaWZpZXIoDQogICAgICAgICAgICAgICAgICAgIG1vZGVsX25hbWUsDQogICAgICAgICAgICAgICAgICAgIGltYmFsYW5jZV9tZXRob2QsDQogICAgICAgICAgICAgICAgICAgIG1vZGVsLA0KICAgICAgICAgICAgICAgICAgICBYX3ZhbGlkLA0KICAgICAgICAgICAgICAgICAgICB5X3ZhbGlkLA0KICAgICAgICAgICAgICAgICAgICB0aHJlc2hvbGQ9MC41LA0KICAgICAgICAgICAgICAgICksDQogICAgICAgICAgICB9DQogICAgICAgICkNCg0KICAgIHZhbGlkYXRpb25fbWV0cmljcyA9IHJhbmtfcmVzdWx0cyhwZC5EYXRhRnJhbWUodmFsaWRhdGlvbl9yb3dzKSwgcmFua2luZ19jb2x1bW5zPURFRkFVTFRfTU9ERUxfUkFOS0lORykNCiAgICBzZWxlY3RlZF9tb2RlbF9uYW1lID0gc3RyKHNlbGVjdF90b3BfcmVzdWx0KHZhbGlkYXRpb25fbWV0cmljcywgREVGQVVMVF9NT0RFTF9SQU5LSU5HKS5yb3dbIm1vZGVsIl0pDQoNCiAgICBzZWxlY3RlZF9jYW5kaWRhdGVzID0gbWFrZV9jYW5kaWRhdGVfbW9kZWxzKA0KICAgICAgICBtaXNzaW5nX3RocmVzaG9sZD1taXNzaW5nX3RocmVzaG9sZCwNCiAgICAgICAgbG93X3ZhcmlhbmNlX3RocmVzaG9sZD1sb3dfdmFyaWFuY2VfdGhyZXNob2xkLA0KICAgICAgICByYW5kb21fc3RhdGU9cmFuZG9tX3N0YXRlLA0KICAgICkNCiAgICBzZWxlY3RlZF9pbWJhbGFuY2VfbWV0aG9kLCB0aHJlc2hvbGRfbW9kZWwgPSBzZWxlY3RlZF9jYW5kaWRhdGVzW3NlbGVjdGVkX21vZGVsX25hbWVdDQogICAgdGhyZXNob2xkX21vZGVsLmZpdChYX3RyYWluLCB5X3RyYWluKQ0KICAgIHZhbGlkYXRpb25fcHJvYmFiaWxpdHkgPSBnZXRfcG9zaXRpdmVfcHJvYmEodGhyZXNob2xkX21vZGVsLCBYX3ZhbGlkKQ0KICAgIHRocmVzaG9sZF9tZXRyaWNzID0gcmFua19yZXN1bHRzKA0KICAgICAgICBidWlsZF90aHJlc2hvbGRfbWV0cmljcyh5X3ZhbGlkLCB2YWxpZGF0aW9uX3Byb2JhYmlsaXR5LCB0aHJlc2hvbGRzKSwNCiAgICAgICAgcmFua2luZ19jb2x1bW5zPURFRkFVTFRfVEhSRVNIT0xEX1JBTktJTkcsDQogICAgKQ0KICAgIHNlbGVjdGVkX3RocmVzaG9sZCA9IGZsb2F0KA0KICAgICAgICBzZWxlY3RfdG9wX3Jlc3VsdCh0aHJlc2hvbGRfbWV0cmljcywgREVGQVVMVF9USFJFU0hPTERfUkFOS0lORykucm93WyJ0aHJlc2hvbGQiXQ0KICAgICkNCg0KICAgIGZpbmFsX2NhbmRpZGF0ZXMgPSBtYWtlX2NhbmRpZGF0ZV9tb2RlbHMoDQogICAgICAgIG1pc3NpbmdfdGhyZXNob2xkPW1pc3NpbmdfdGhyZXNob2xkLA0KICAgICAgICBsb3dfdmFyaWFuY2VfdGhyZXNob2xkPWxvd192YXJpYW5jZV90aHJlc2hvbGQsDQogICAgICAgIHJhbmRvbV9zdGF0ZT1yYW5kb21fc3RhdGUsDQogICAgKQ0KICAgIF8sIGZpbmFsX21vZGVsID0gZmluYWxfY2FuZGlkYXRlc1tzZWxlY3RlZF9tb2RlbF9uYW1lXQ0KICAgIGZpbmFsX21vZGVsLmZpdChYX3RyYWluX3ZhbGlkLCB5X3RyYWluX3ZhbGlkKQ0KDQogICAgdGVzdF9tZXRyaWNzID0gcGQuRGF0YUZyYW1lKA0KICAgICAgICBbDQogICAgICAgICAgICB7DQogICAgICAgICAgICAgICAgInNwbGl0IjogInRlc3QiLA0KICAgICAgICAgICAgICAgICoqZXZhbHVhdGVfY2xhc3NpZmllcigNCiAgICAgICAgICAgICAgICAgICAgc2VsZWN0ZWRfbW9kZWxfbmFtZSwNCiAgICAgICAgICAgICAgICAgICAgc2VsZWN0ZWRfaW1iYWxhbmNlX21ldGhvZCwNCiAgICAgICAgICAgICAgICAgICAgZmluYWxfbW9kZWwsDQogICAgICAgICAgICAgICAgICAgIFhfdGVzdCwNCiAgICAgICAgICAgICAgICAgICAgeV90ZXN0LA0KICAgICAgICAgICAgICAgICAgICB0aHJlc2hvbGQ9c2VsZWN0ZWRfdGhyZXNob2xkLA0KICAgICAgICAgICAgICAgICksDQogICAgICAgICAgICB9DQogICAgICAgIF0NCiAgICApDQogICAgbWV0cmljcyA9IHBkLmNvbmNhdChbdmFsaWRhdGlvbl9tZXRyaWNzLCB0ZXN0X21ldHJpY3NdLCBpZ25vcmVfaW5kZXg9VHJ1ZSkNCg0KICAgIGJ1bmRsZSA9IE1vZGVsQnVuZGxlKA0KICAgICAgICBtb2RlbD1maW5hbF9tb2RlbCwNCiAgICAgICAgdGhyZXNob2xkPXNlbGVjdGVkX3RocmVzaG9sZCwNCiAgICAgICAgZmVhdHVyZV9jb2x1bW5zPWRhdGFzZXQuZmVhdHVyZXMuY29sdW1ucy50b2xpc3QoKSwNCiAgICAgICAgdGFyZ2V0X21hcHBpbmc9ezA6ICJQYXNzIiwgMTogIkZhaWwifSwNCiAgICAgICAgbW9kZWxfbmFtZT1zZWxlY3RlZF9tb2RlbF9uYW1lLA0KICAgICAgICBtZXRyaWNzPXRlc3RfbWV0cmljcy5pbG9jWzBdLnRvX2RpY3QoKSwNCiAgICApDQoNCiAgICBtZXRyaWNzX291dHB1dC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KICAgIHRocmVzaG9sZF9vdXRwdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICBtZXRyaWNzLnRvX2NzdihtZXRyaWNzX291dHB1dCwgaW5kZXg9RmFsc2UpDQogICAgdGhyZXNob2xkX21ldHJpY3MudG9fY3N2KHRocmVzaG9sZF9vdXRwdXQsIGluZGV4PUZhbHNlKQ0KICAgIHNhdmVfbW9kZWxfYnVuZGxlKGJ1bmRsZSwgbW9kZWxfb3V0cHV0KQ0KICAgIHdyaXRlX3Rlc3RfcHJlZGljdGlvbnMoDQogICAgICAgIG91dHB1dF9wYXRoPXByZWRpY3Rpb25zX291dHB1dCwNCiAgICAgICAgbW9kZWw9ZmluYWxfbW9kZWwsDQogICAgICAgIFhfdGVzdD1YX3Rlc3QsDQogICAgICAgIHlfdGVzdD15X3Rlc3QsDQogICAgICAgIHRpbWVzdGFtcHM9ZGF0YXNldC50aW1lc3RhbXBzLA0KICAgICAgICB0aHJlc2hvbGQ9c2VsZWN0ZWRfdGhyZXNob2xkLA0KICAgICkNCg0KICAgIHJldHVybiB7DQogICAgICAgICJtb2RlbCI6IG1vZGVsX291dHB1dCwNCiAgICAgICAgIm1ldHJpY3MiOiBtZXRyaWNzX291dHB1dCwNCiAgICAgICAgInRocmVzaG9sZHMiOiB0aHJlc2hvbGRfb3V0cHV0LA0KICAgICAgICAidGVzdF9wcmVkaWN0aW9ucyI6IHByZWRpY3Rpb25zX291dHB1dCwNCiAgICB9DQoNCg0KZGVmIG1haW4oKSAtPiBpbnQ6DQogICAgYXJncyA9IHBhcnNlX2FyZ3MoKQ0KICAgIG91dHB1dHMgPSB0cmFpbl9zZWNvbV9tb2RlbCgNCiAgICAgICAgcmF3X2RhdGFfZGlyPWFyZ3MucmF3X2RhdGFfZGlyLA0KICAgICAgICBmZWF0dXJlX3BhdGg9YXJncy5mZWF0dXJlX3BhdGgsDQogICAgICAgIGxhYmVsX3BhdGg9YXJncy5sYWJlbF9wYXRoLA0KICAgICAgICBkb3dubG9hZF9zZWNvbT1hcmdzLmRvd25sb2FkX3NlY29tLA0KICAgICAgICBtb2RlbF9vdXRwdXQ9YXJncy5tb2RlbF9vdXRwdXQsDQogICAgICAgIG1ldHJpY3Nfb3V0cHV0PWFyZ3MubWV0cmljc19vdXRwdXQsDQogICAgICAgIHRocmVzaG9sZF9vdXRwdXQ9YXJncy50aHJlc2hvbGRfb3V0cHV0LA0KICAgICAgICBwcmVkaWN0aW9uc19vdXRwdXQ9YXJncy5wcmVkaWN0aW9uc19vdXRwdXQsDQogICAgICAgIHRlc3Rfc2l6ZT1hcmdzLnRlc3Rfc2l6ZSwNCiAgICAgICAgdmFsaWRhdGlvbl9zaXplPWFyZ3MudmFsaWRhdGlvbl9zaXplLA0KICAgICAgICBtaXNzaW5nX3RocmVzaG9sZD1hcmdzLm1pc3NpbmdfdGhyZXNob2xkLA0KICAgICAgICBsb3dfdmFyaWFuY2VfdGhyZXNob2xkPWFyZ3MubG93X3ZhcmlhbmNlX3RocmVzaG9sZCwNCiAgICAgICAgcmFuZG9tX3N0YXRlPWFyZ3MucmFuZG9tX3N0YXRlLA0KICAgICAgICB0aHJlc2hvbGRzPWFyZ3MudGhyZXNob2xkcywNCiAgICApDQogICAgZm9yIG5hbWUsIHBhdGggaW4gb3V0cHV0cy5pdGVtcygpOg0KICAgICAgICBwcmludChmIntuYW1lfToge3BhdGh9IikNCiAgICByZXR1cm4gMA0KDQoNCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6DQogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpDQo=', 'tests/test_assemble_feature_table.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB1bml0dGVzdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gc2NyaXB0cy5hc3NlbWJsZV9mZWF0dXJlX3RhYmxlIGltcG9ydCBhc3NlbWJsZV9mcm9tX3BhdGhzLCBwYXJzZV9rZXlfbGlzdAoKCmNsYXNzIEFzc2VtYmxlRmVhdHVyZVRhYmxlQ2xpVGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHRlc3RfcGFyc2Vfa2V5X2xpc3Qoc2VsZikgLT4gTm9uZToKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHBhcnNlX2tleV9saXN0KCJ3YWZlcl9pZCwgbG90X2lkIiksIFsid2FmZXJfaWQiLCAibG90X2lkIl0pCgogICAgZGVmIHRlc3RfYXNzZW1ibGVfZnJvbV9wYXRoc193cml0ZXNfb3V0cHV0cyhzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KCkgYXMgdG1wZGlyOgogICAgICAgICAgICB0bXBfcGF0aCA9IFBhdGgodG1wZGlyKQogICAgICAgICAgICBzZW5zb3JfcGF0aCA9IHRtcF9wYXRoIC8gInNlbnNvci5jc3YiCiAgICAgICAgICAgIHdhZmVyX3BhdGggPSB0bXBfcGF0aCAvICJ3YWZlci5jc3YiCiAgICAgICAgICAgIGVxdWlwbWVudF9wYXRoID0gdG1wX3BhdGggLyAiZXF1aXBtZW50LmNzdiIKICAgICAgICAgICAgb3V0cHV0X3BhdGggPSB0bXBfcGF0aCAvICJvdXQiIC8gIm1vZGVsaW5nX3RhYmxlLmNzdiIKICAgICAgICAgICAgcmVwb3J0X2RpciA9IHRtcF9wYXRoIC8gInJlcG9ydHMiCgogICAgICAgICAgICBwZC5EYXRhRnJhbWUoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgInNhbXBsZV9pZCI6IFsiUzEiLCAiUzIiLCAiUzMiXSwKICAgICAgICAgICAgICAgICAgICAid2FmZXJfaWQiOiBbIlcxIiwgIlcyIiwgIlczIl0sCiAgICAgICAgICAgICAgICAgICAgImVxdWlwbWVudF9pZCI6IFsiRVExIiwgIkVRMSIsICJFUTIiXSwKICAgICAgICAgICAgICAgICAgICAiZmVhdHVyZV8wMDAiOiBbMC4xLCAwLjIsIDAuM10sCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkudG9fY3N2KHNlbnNvcl9wYXRoLCBpbmRleD1GYWxzZSkKICAgICAgICAgICAgcGQuRGF0YUZyYW1lKAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICJ3YWZlcl9pZCI6IFsiVzEiLCAiVzIiXSwKICAgICAgICAgICAgICAgICAgICAiZGVmZWN0X2NvdW50IjogWzQsIDFdLAogICAgICAgICAgICAgICAgfQogICAgICAgICAgICApLnRvX2Nzdih3YWZlcl9wYXRoLCBpbmRleD1GYWxzZSkKICAgICAgICAgICAgcGQuRGF0YUZyYW1lKAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICJlcXVpcG1lbnRfaWQiOiBbIkVRMSIsICJFUTIiXSwKICAgICAgICAgICAgICAgICAgICAiZXZlbnRfY291bnQiOiBbNSwgMl0sCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkudG9fY3N2KGVxdWlwbWVudF9wYXRoLCBpbmRleD1GYWxzZSkKCiAgICAgICAgICAgIG1vZGVsaW5nX3BhdGgsIGpvaW5fcmVwb3J0X3BhdGgsIG1pc3NpbmduZXNzX3BhdGggPSBhc3NlbWJsZV9mcm9tX3BhdGhzKAogICAgICAgICAgICAgICAgc2Vuc29yX3BhdGg9c2Vuc29yX3BhdGgsCiAgICAgICAgICAgICAgICB3YWZlcl9wYXRoPXdhZmVyX3BhdGgsCiAgICAgICAgICAgICAgICBlcXVpcG1lbnRfcGF0aD1lcXVpcG1lbnRfcGF0aCwKICAgICAgICAgICAgICAgIHNlbnNvcl93YWZlcl9rZXlzPVsid2FmZXJfaWQiXSwKICAgICAgICAgICAgICAgIHNlbnNvcl9lcXVpcG1lbnRfa2V5cz1bImVxdWlwbWVudF9pZCJdLAogICAgICAgICAgICAgICAgb3V0cHV0X3BhdGg9b3V0cHV0X3BhdGgsCiAgICAgICAgICAgICAgICByZXBvcnRfZGlyPXJlcG9ydF9kaXIsCiAgICAgICAgICAgICkKCiAgICAgICAgICAgIG1vZGVsaW5nX3RhYmxlID0gcGQucmVhZF9jc3YobW9kZWxpbmdfcGF0aCkKICAgICAgICAgICAgam9pbl9yZXBvcnQgPSBwZC5yZWFkX2Nzdihqb2luX3JlcG9ydF9wYXRoKQogICAgICAgICAgICBtaXNzaW5nbmVzcyA9IHBkLnJlYWRfY3N2KG1pc3NpbmduZXNzX3BhdGgpCgogICAgICAgICAgICBzZWxmLmFzc2VydFRydWUobW9kZWxpbmdfcGF0aC5leGlzdHMoKSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRUcnVlKGpvaW5fcmVwb3J0X3BhdGguZXhpc3RzKCkpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShtaXNzaW5nbmVzc19wYXRoLmV4aXN0cygpKQogICAgICAgICAgICBzZWxmLmFzc2VydEluKCJ3YWZlcl9kZWZlY3RfY291bnQiLCBtb2RlbGluZ190YWJsZS5jb2x1bW5zKQogICAgICAgICAgICBzZWxmLmFzc2VydEluKCJlcXVpcG1lbnRfZXZlbnRfY291bnQiLCBtb2RlbGluZ190YWJsZS5jb2x1bW5zKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGpvaW5fcmVwb3J0WyJ1bm1hdGNoZWRfcm93cyJdLnRvbGlzdCgpLCBbMSwgMF0pCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0SW4oIndhZmVyX2RlZmVjdF9jb3VudCIsIG1pc3NpbmduZXNzWyJmZWF0dXJlIl0udG9saXN0KCkpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHVuaXR0ZXN0Lm1haW4oKQo=', 'tests/test_build_auxiliary_features.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB1bml0dGVzdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gc2NyaXB0cy5idWlsZF9hdXhpbGlhcnlfZmVhdHVyZXMgaW1wb3J0IGJ1aWxkX2F1eGlsaWFyeV9mZWF0dXJlcwoKCmNsYXNzIEJ1aWxkQXV4aWxpYXJ5RmVhdHVyZXNDbGlUZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBkZWYgdGVzdF9idWlsZF9hdXhpbGlhcnlfZmVhdHVyZXNfd3JpdGVzX3dhZmVyX2FuZF9lcXVpcG1lbnRfb3V0cHV0cyhzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KCkgYXMgdG1wZGlyOgogICAgICAgICAgICB0bXBfcGF0aCA9IFBhdGgodG1wZGlyKQogICAgICAgICAgICB3YWZlcl9pbnB1dCA9IHRtcF9wYXRoIC8gIndhZmVyX2luc3BlY3Rpb24uY3N2IgogICAgICAgICAgICBlcXVpcG1lbnRfaW5wdXQgPSB0bXBfcGF0aCAvICJlcXVpcG1lbnRfZXZlbnRzLmNzdiIKICAgICAgICAgICAgd2FmZXJfb3V0cHV0ID0gdG1wX3BhdGggLyAiZmVhdHVyZXMiIC8gIndhZmVyX2ZlYXR1cmVzLmNzdiIKICAgICAgICAgICAgZXF1aXBtZW50X291dHB1dCA9IHRtcF9wYXRoIC8gImZlYXR1cmVzIiAvICJlcXVpcG1lbnRfZmVhdHVyZXMuY3N2IgoKICAgICAgICAgICAgcGQuRGF0YUZyYW1lKAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICJ3YWZlcl9pZCI6IFsiVzEiLCAiVzEiLCAiVzIiLCAiVzIiXSwKICAgICAgICAgICAgICAgICAgICAieCI6IFswLCAxLCA5LCAtOV0sCiAgICAgICAgICAgICAgICAgICAgInkiOiBbMCwgMSwgOSwgLTldLAogICAgICAgICAgICAgICAgICAgICJkZWZlY3RfdHlwZSI6IFsiZG90IiwgInNjcmF0Y2giLCAiZWRnZSIsICJlZGdlIl0sCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkudG9fY3N2KHdhZmVyX2lucHV0LCBpbmRleD1GYWxzZSkKICAgICAgICAgICAgcGQuRGF0YUZyYW1lKAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICJlcXVpcG1lbnRfaWQiOiBbIkVRMSIsICJFUTEiLCAiRVEyIl0sCiAgICAgICAgICAgICAgICAgICAgInRpbWVzdGFtcCI6IFsiMjAyNi0wMS0wMSAwMDowMDowMCIsICIyMDI2LTAxLTAxIDAyOjAwOjAwIiwgIjIwMjYtMDEtMDIgMDA6MDA6MDAiXSwKICAgICAgICAgICAgICAgICAgICAiZXZlbnRfdHlwZSI6IFsiYWxhcm0iLCAid2FybmluZyIsICJub3JtYWwiXSwKICAgICAgICAgICAgICAgICAgICAiZmFpbHVyZV9sYWJlbCI6IFswLCAxLCAwXSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgKS50b19jc3YoZXF1aXBtZW50X2lucHV0LCBpbmRleD1GYWxzZSkKCiAgICAgICAgICAgIG91dHB1dHMgPSBidWlsZF9hdXhpbGlhcnlfZmVhdHVyZXMoCiAgICAgICAgICAgICAgICB3YWZlcl9pbnB1dD13YWZlcl9pbnB1dCwKICAgICAgICAgICAgICAgIGVxdWlwbWVudF9pbnB1dD1lcXVpcG1lbnRfaW5wdXQsCiAgICAgICAgICAgICAgICB3YWZlcl9vdXRwdXQ9d2FmZXJfb3V0cHV0LAogICAgICAgICAgICAgICAgZXF1aXBtZW50X291dHB1dD1lcXVpcG1lbnRfb3V0cHV0LAogICAgICAgICAgICAgICAgYWRkX3dhZmVyX3BhdHRlcm5fbGFiZWw9VHJ1ZSwKICAgICAgICAgICAgKQoKICAgICAgICAgICAgd2FmZXJfZmVhdHVyZXMgPSBwZC5yZWFkX2NzdihvdXRwdXRzWyJ3YWZlciJdKQogICAgICAgICAgICBlcXVpcG1lbnRfZmVhdHVyZXMgPSBwZC5yZWFkX2NzdihvdXRwdXRzWyJlcXVpcG1lbnQiXSkKCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZSh3YWZlcl9vdXRwdXQuZXhpc3RzKCkpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShlcXVpcG1lbnRfb3V0cHV0LmV4aXN0cygpKQogICAgICAgICAgICBzZWxmLmFzc2VydEluKCJwYXR0ZXJuX2xhYmVsIiwgd2FmZXJfZmVhdHVyZXMuY29sdW1ucykKICAgICAgICAgICAgc2VsZi5hc3NlcnRJbigiZXZlbnRfY291bnQiLCBlcXVpcG1lbnRfZmVhdHVyZXMuY29sdW1ucykKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChlcXVpcG1lbnRfZmVhdHVyZXNbImV2ZW50X2NvdW50Il0uc3VtKCksIDMpCgogICAgZGVmIHRlc3RfYnVpbGRfYXV4aWxpYXJ5X2ZlYXR1cmVzX3JlcXVpcmVzX2F0X2xlYXN0X29uZV9pbnB1dChzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KCkgYXMgdG1wZGlyOgogICAgICAgICAgICB0bXBfcGF0aCA9IFBhdGgodG1wZGlyKQoKICAgICAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgICAgIGJ1aWxkX2F1eGlsaWFyeV9mZWF0dXJlcygKICAgICAgICAgICAgICAgICAgICB3YWZlcl9pbnB1dD1Ob25lLAogICAgICAgICAgICAgICAgICAgIGVxdWlwbWVudF9pbnB1dD1Ob25lLAogICAgICAgICAgICAgICAgICAgIHdhZmVyX291dHB1dD10bXBfcGF0aCAvICJ3YWZlci5jc3YiLAogICAgICAgICAgICAgICAgICAgIGVxdWlwbWVudF9vdXRwdXQ9dG1wX3BhdGggLyAiZXF1aXBtZW50LmNzdiIsCiAgICAgICAgICAgICAgICApCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHVuaXR0ZXN0Lm1haW4oKQo=', 'tests/test_data_contracts.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHVuaXR0ZXN0CgppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIHNyYy5kYXRhX2NvbnRyYWN0cyBpbXBvcnQgKAogICAgcmVxdWlyZV9jb2x1bW5zLAogICAgcmVxdWlyZV9wcm9iYWJpbGl0eV9jb2x1bW4sCiAgICB2YWxpZGF0ZV9lcXVpcG1lbnRfZXZlbnRzX2NvbnRyYWN0LAogICAgdmFsaWRhdGVfbW9kZWxpbmdfdGFibGVfY29udHJhY3QsCiAgICB2YWxpZGF0ZV9wcmVkaWN0aW9uc19jb250cmFjdCwKICAgIHZhbGlkYXRlX3dhZmVyX2luc3BlY3Rpb25fY29udHJhY3QsCikKCgpjbGFzcyBEYXRhQ29udHJhY3RUZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBkZWYgdGVzdF9yZXF1aXJlX2NvbHVtbnNfcmVwb3J0c19taXNzaW5nX2NvbHVtbnMoc2VsZikgLT4gTm9uZToKICAgICAgICByZXN1bHQgPSByZXF1aXJlX2NvbHVtbnMocGQuRGF0YUZyYW1lKHsiYSI6IFsxXX0pLCBbImEiLCAiYiJdLCAidGFibGUiKQoKICAgICAgICBzZWxmLmFzc2VydEZhbHNlKHJlc3VsdC5vaykKICAgICAgICBzZWxmLmFzc2VydEluKCJtaXNzaW5nIGNvbHVtbnMiLCByZXN1bHQuZXJyb3JzWzBdKQoKICAgIGRlZiB0ZXN0X3JhaXNlX2lmX2ZhaWxlZF9yYWlzZXNfdmFsdWVfZXJyb3Ioc2VsZikgLT4gTm9uZToKICAgICAgICByZXN1bHQgPSByZXF1aXJlX2NvbHVtbnMocGQuRGF0YUZyYW1lKHsiYSI6IFsxXX0pLCBbIm1pc3NpbmciXSwgInRhYmxlIikKCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgcmVzdWx0LnJhaXNlX2lmX2ZhaWxlZCgpCgogICAgZGVmIHRlc3RfdmFsaWRhdGVfd2FmZXJfaW5zcGVjdGlvbl9jb250cmFjdF9hY2NlcHRzX3ZhbGlkX3RhYmxlKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgZGYgPSBwZC5EYXRhRnJhbWUoeyJ3YWZlcl9pZCI6IFsiVzEiXSwgIngiOiBbMS4wXSwgInkiOiBbMi4wXX0pCgogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZSh2YWxpZGF0ZV93YWZlcl9pbnNwZWN0aW9uX2NvbnRyYWN0KGRmKS5vaykKCiAgICBkZWYgdGVzdF92YWxpZGF0ZV93YWZlcl9pbnNwZWN0aW9uX2NvbnRyYWN0X3JlamVjdHNfbm9uX251bWVyaWNfY29vcmRpbmF0ZXMoc2VsZikgLT4gTm9uZToKICAgICAgICBkZiA9IHBkLkRhdGFGcmFtZSh7IndhZmVyX2lkIjogWyJXMSJdLCAieCI6IFsiYmFkIl0sICJ5IjogWzIuMF19KQoKICAgICAgICByZXN1bHQgPSB2YWxpZGF0ZV93YWZlcl9pbnNwZWN0aW9uX2NvbnRyYWN0KGRmKQoKICAgICAgICBzZWxmLmFzc2VydEZhbHNlKHJlc3VsdC5vaykKICAgICAgICBzZWxmLmFzc2VydEluKCJub24tbnVtZXJpYyBjb2x1bW46IHgiLCByZXN1bHQuZXJyb3JzKQoKICAgIGRlZiB0ZXN0X3ZhbGlkYXRlX2VxdWlwbWVudF9ldmVudHNfY29udHJhY3Qoc2VsZikgLT4gTm9uZToKICAgICAgICBkZiA9IHBkLkRhdGFGcmFtZSgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgImVxdWlwbWVudF9pZCI6IFsiRVExIl0sCiAgICAgICAgICAgICAgICAidGltZXN0YW1wIjogWyIyMDI2LTAxLTAxIl0sCiAgICAgICAgICAgICAgICAiZXZlbnRfdHlwZSI6IFsiYWxhcm0iXSwKICAgICAgICAgICAgfQogICAgICAgICkKCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKHZhbGlkYXRlX2VxdWlwbWVudF9ldmVudHNfY29udHJhY3QoZGYpLm9rKQoKICAgIGRlZiB0ZXN0X3ZhbGlkYXRlX3ByZWRpY3Rpb25zX2NvbnRyYWN0X3JlamVjdHNfYmFkX3Byb2JhYmlsaXR5X2FuZF9sYWJlbChzZWxmKSAtPiBOb25lOgogICAgICAgIGRmID0gcGQuRGF0YUZyYW1lKHsicHJlZGljdGlvbiI6IFsiUGFzcyIsICJNYXliZSJdLCAiZmFpbF9wcm9iYWJpbGl0eSI6IFswLjEsIDEuMl19KQoKICAgICAgICByZXN1bHQgPSB2YWxpZGF0ZV9wcmVkaWN0aW9uc19jb250cmFjdChkZikKCiAgICAgICAgc2VsZi5hc3NlcnRGYWxzZShyZXN1bHQub2spCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKGFueSgiaW52YWxpZCB2YWx1ZXMiIGluIGVycm9yIGZvciBlcnJvciBpbiByZXN1bHQuZXJyb3JzKSkKICAgICAgICBzZWxmLmFzc2VydFRydWUoYW55KCJwcm9iYWJpbGl0eSBvdXQgb2YgcmFuZ2UiIGluIGVycm9yIGZvciBlcnJvciBpbiByZXN1bHQuZXJyb3JzKSkKCiAgICBkZWYgdGVzdF9yZXF1aXJlX3Byb2JhYmlsaXR5X2NvbHVtbl9hY2NlcHRzX3ZhbGlkX3Byb2JhYmlsaXR5KHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcmVzdWx0ID0gcmVxdWlyZV9wcm9iYWJpbGl0eV9jb2x1bW4ocGQuRGF0YUZyYW1lKHsicCI6IFswLjAsIDAuNSwgMS4wXX0pLCAicCIsICJ0YWJsZSIpCgogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShyZXN1bHQub2spCgogICAgZGVmIHRlc3RfdmFsaWRhdGVfbW9kZWxpbmdfdGFibGVfY29udHJhY3Qoc2VsZikgLT4gTm9uZToKICAgICAgICBkZiA9IHBkLkRhdGFGcmFtZSh7ImZlYXR1cmVfYSI6IFsxXSwgImZlYXR1cmVfYiI6IFsyXX0pCgogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZSh2YWxpZGF0ZV9tb2RlbGluZ190YWJsZV9jb250cmFjdChkZiwgWyJmZWF0dXJlX2EiLCAiZmVhdHVyZV9iIl0pLm9rKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2UodmFsaWRhdGVfbW9kZWxpbmdfdGFibGVfY29udHJhY3QoZGYsIFsiZmVhdHVyZV9jIl0pLm9rKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICB1bml0dGVzdC5tYWluKCkK', 'tests/test_equipment_features.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHVuaXR0ZXN0CgppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIHNyYy5lcXVpcG1lbnRfZmVhdHVyZXMgaW1wb3J0ICgKICAgIGFkZF90aW1lX3NpbmNlX3ByZXZpb3VzX2V2ZW50LAogICAgZXF1aXBtZW50X2V2ZW50X2ZlYXR1cmVzLAogICAgdmFsaWRhdGVfZXF1aXBtZW50X2NvbHVtbnMsCikKCgpjbGFzcyBFcXVpcG1lbnRGZWF0dXJlVGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHRlc3RfZXF1aXBtZW50X2V2ZW50X2ZlYXR1cmVzX2FnZ3JlZ2F0ZXNfY291bnRzX2FuZF9mYWlsdXJlKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgZXZlbnRzX2RmID0gcGQuRGF0YUZyYW1lKAogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAiZXF1aXBtZW50X2lkIjogWyJFUTEiLCAiRVExIiwgIkVRMSIsICJFUTIiXSwKICAgICAgICAgICAgICAgICJ0aW1lc3RhbXAiOiBbCiAgICAgICAgICAgICAgICAgICAgIjIwMjYtMDEtMDEgMDA6MDA6MDAiLAogICAgICAgICAgICAgICAgICAgICIyMDI2LTAxLTAxIDAyOjAwOjAwIiwKICAgICAgICAgICAgICAgICAgICAiMjAyNi0wMS0wMSAwMzowMDowMCIsCiAgICAgICAgICAgICAgICAgICAgIjIwMjYtMDEtMDIgMDA6MDA6MDAiLAogICAgICAgICAgICAgICAgXSwKICAgICAgICAgICAgICAgICJldmVudF90eXBlIjogWyJhbGFybSIsICJ3YXJuaW5nIiwgImFsYXJtIiwgIm5vcm1hbCJdLAogICAgICAgICAgICAgICAgImZhaWx1cmVfbGFiZWwiOiBbMCwgMCwgMSwgMF0sCiAgICAgICAgICAgIH0KICAgICAgICApCgogICAgICAgIGZlYXR1cmVzID0gZXF1aXBtZW50X2V2ZW50X2ZlYXR1cmVzKGV2ZW50c19kZikKICAgICAgICBlcTEgPSBmZWF0dXJlcy5sb2NbZmVhdHVyZXNbImVxdWlwbWVudF9pZCJdID09ICJFUTEiXS5pbG9jWzBdCgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZmVhdHVyZXMuc2hhcGVbMF0sIDIpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChlcTFbImV2ZW50X2NvdW50Il0sIDMpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChlcTFbImZhaWx1cmVfbGFiZWwiXSwgMSkKICAgICAgICBzZWxmLmFzc2VydEluKCJldmVudF90eXBlX2NvdW50X2FsYXJtIiwgZmVhdHVyZXMuY29sdW1ucykKICAgICAgICBzZWxmLmFzc2VydEdyZWF0ZXIoZXExWyJldmVudF9yYXRlX3Blcl9ob3VyIl0sIDApCgogICAgZGVmIHRlc3RfYWRkX3RpbWVfc2luY2VfcHJldmlvdXNfZXZlbnRfY29tcHV0ZXNfcGVyX2VxdWlwbWVudF9lbGFwc2VkX2hvdXJzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgZXZlbnRzX2RmID0gcGQuRGF0YUZyYW1lKAogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAiZXF1aXBtZW50X2lkIjogWyJFUTEiLCAiRVExIiwgIkVRMiJdLAogICAgICAgICAgICAgICAgInRpbWVzdGFtcCI6IFsKICAgICAgICAgICAgICAgICAgICAiMjAyNi0wMS0wMSAwMDowMDowMCIsCiAgICAgICAgICAgICAgICAgICAgIjIwMjYtMDEtMDEgMDI6MzA6MDAiLAogICAgICAgICAgICAgICAgICAgICIyMDI2LTAxLTAxIDA0OjAwOjAwIiwKICAgICAgICAgICAgICAgIF0sCiAgICAgICAgICAgICAgICAiZXZlbnRfdHlwZSI6IFsiYWxhcm0iLCAid2FybmluZyIsICJub3JtYWwiXSwKICAgICAgICAgICAgfQogICAgICAgICkKCiAgICAgICAgZW5yaWNoZWQgPSBhZGRfdGltZV9zaW5jZV9wcmV2aW91c19ldmVudChldmVudHNfZGYpCgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZW5yaWNoZWQubG9jWzAsICJob3Vyc19zaW5jZV9wcmV2aW91c19ldmVudCJdLCAwKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZW5yaWNoZWQubG9jWzEsICJob3Vyc19zaW5jZV9wcmV2aW91c19ldmVudCJdLCAyLjUpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChlbnJpY2hlZC5sb2NbMiwgImhvdXJzX3NpbmNlX3ByZXZpb3VzX2V2ZW50Il0sIDApCgogICAgZGVmIHRlc3RfdmFsaWRhdGVfZXF1aXBtZW50X2NvbHVtbnNfcmVqZWN0c19taXNzaW5nX2NvbHVtbnMoc2VsZikgLT4gTm9uZToKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICB2YWxpZGF0ZV9lcXVpcG1lbnRfY29sdW1ucyhwZC5EYXRhRnJhbWUoeyJlcXVpcG1lbnRfaWQiOiBbIkVRMSJdLCAidGltZXN0YW1wIjogWyIyMDI2LTAxLTAxIl19KSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgdW5pdHRlc3QubWFpbigpCg==', 'tests/test_feature_store.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHVuaXR0ZXN0CgppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIHNyYy5mZWF0dXJlX3N0b3JlIGltcG9ydCAoCiAgICBhc3NlbWJsZV9mZWF0dXJlX3RhYmxlLAogICAgZmVhdHVyZV9taXNzaW5nbmVzc19yZXBvcnQsCiAgICBsZWZ0X2pvaW5fZmVhdHVyZXMsCiAgICB2YWxpZGF0ZV9rZXlfY29sdW1ucywKKQoKCmNsYXNzIEZlYXR1cmVTdG9yZVRlc3RzKHVuaXR0ZXN0LlRlc3RDYXNlKToKICAgIGRlZiB0ZXN0X2Fzc2VtYmxlX2ZlYXR1cmVfdGFibGVfam9pbnNfd2FmZXJfYW5kX2VxdWlwbWVudF9mZWF0dXJlcyhzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbnNvcl9kZiA9IHBkLkRhdGFGcmFtZSgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgInNhbXBsZV9pZCI6IFsiUzEiLCAiUzIiLCAiUzMiXSwKICAgICAgICAgICAgICAgICJ3YWZlcl9pZCI6IFsiVzEiLCAiVzIiLCAiVzMiXSwKICAgICAgICAgICAgICAgICJlcXVpcG1lbnRfaWQiOiBbIkVRMSIsICJFUTEiLCAiRVEyIl0sCiAgICAgICAgICAgICAgICAiZmVhdHVyZV8wMDAiOiBbMC4xLCAwLjIsIDAuM10sCiAgICAgICAgICAgIH0KICAgICAgICApCiAgICAgICAgd2FmZXJfZGYgPSBwZC5EYXRhRnJhbWUoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJ3YWZlcl9pZCI6IFsiVzEiLCAiVzIiXSwKICAgICAgICAgICAgICAgICJkZWZlY3RfY291bnQiOiBbNSwgMl0sCiAgICAgICAgICAgICAgICAiem9uZV9yYXRpb19lZGdlIjogWzAuOCwgMC4xXSwKICAgICAgICAgICAgfQogICAgICAgICkKICAgICAgICBlcXVpcG1lbnRfZGYgPSBwZC5EYXRhRnJhbWUoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJlcXVpcG1lbnRfaWQiOiBbIkVRMSIsICJFUTIiXSwKICAgICAgICAgICAgICAgICJldmVudF9jb3VudCI6IFsxMCwgM10sCiAgICAgICAgICAgICAgICAiZmFpbHVyZV9sYWJlbCI6IFsxLCAwXSwKICAgICAgICAgICAgfQogICAgICAgICkKCiAgICAgICAgZmVhdHVyZV90YWJsZSwgam9pbl9yZXBvcnQgPSBhc3NlbWJsZV9mZWF0dXJlX3RhYmxlKAogICAgICAgICAgICBzZW5zb3JfZGYsCiAgICAgICAgICAgIHdhZmVyX2ZlYXR1cmVzPXdhZmVyX2RmLAogICAgICAgICAgICBlcXVpcG1lbnRfZmVhdHVyZXM9ZXF1aXBtZW50X2RmLAogICAgICAgICkKCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChmZWF0dXJlX3RhYmxlLnNoYXBlWzBdLCAzKQogICAgICAgIHNlbGYuYXNzZXJ0SW4oIndhZmVyX2RlZmVjdF9jb3VudCIsIGZlYXR1cmVfdGFibGUuY29sdW1ucykKICAgICAgICBzZWxmLmFzc2VydEluKCJlcXVpcG1lbnRfZXZlbnRfY291bnQiLCBmZWF0dXJlX3RhYmxlLmNvbHVtbnMpCiAgICAgICAgc2VsZi5hc3NlcnRUcnVlKHBkLmlzbmEoZmVhdHVyZV90YWJsZS5sb2NbMiwgIndhZmVyX2RlZmVjdF9jb3VudCJdKSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGpvaW5fcmVwb3J0WyJ1bm1hdGNoZWRfcm93cyJdLnRvbGlzdCgpLCBbMSwgMF0pCgogICAgZGVmIHRlc3RfbGVmdF9qb2luX2ZlYXR1cmVzX3ByZWZpeGVzX25vbl9rZXlfY29sdW1ucyhzZWxmKSAtPiBOb25lOgogICAgICAgIGJhc2UgPSBwZC5EYXRhRnJhbWUoeyJpZCI6IFsxLCAyXSwgImJhc2VfdmFsdWUiOiBbMTAsIDIwXX0pCiAgICAgICAgZmVhdHVyZSA9IHBkLkRhdGFGcmFtZSh7ImlkIjogWzFdLCAic2NvcmUiOiBbMC43XX0pCgogICAgICAgIGpvaW5lZCwgcmVwb3J0ID0gbGVmdF9qb2luX2ZlYXR1cmVzKAogICAgICAgICAgICBiYXNlLAogICAgICAgICAgICBmZWF0dXJlLAogICAgICAgICAgICBrZXlzPVsiaWQiXSwKICAgICAgICAgICAgdGFibGVfbmFtZT0ic2NvcmVfdGFibGUiLAogICAgICAgICAgICBmZWF0dXJlX3ByZWZpeD0ic2NvcmVfIiwKICAgICAgICApCgogICAgICAgIHNlbGYuYXNzZXJ0SW4oInNjb3JlX3Njb3JlIiwgam9pbmVkLmNvbHVtbnMpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXBvcnQudW5tYXRjaGVkX3Jvd3MsIDEpCgogICAgZGVmIHRlc3RfdmFsaWRhdGVfa2V5X2NvbHVtbnNfcmVqZWN0c19taXNzaW5nX2tleShzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgIHZhbGlkYXRlX2tleV9jb2x1bW5zKHBkLkRhdGFGcmFtZSh7ImlkIjogWzFdfSksIFsibWlzc2luZyJdLCAidGFibGUiKQoKICAgIGRlZiB0ZXN0X2ZlYXR1cmVfbWlzc2luZ25lc3NfcmVwb3J0X3NvcnRzX2J5X21pc3NpbmdfcmF0aW8oc2VsZikgLT4gTm9uZToKICAgICAgICBmZWF0dXJlX3RhYmxlID0gcGQuRGF0YUZyYW1lKAogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAiZnVsbCI6IFsxLCAyLCAzXSwKICAgICAgICAgICAgICAgICJzb21lX21pc3NpbmciOiBbMSwgTm9uZSwgTm9uZV0sCiAgICAgICAgICAgICAgICAiYWxsX21pc3NpbmciOiBbTm9uZSwgTm9uZSwgTm9uZV0sCiAgICAgICAgICAgIH0KICAgICAgICApCgogICAgICAgIHJlcG9ydCA9IGZlYXR1cmVfbWlzc2luZ25lc3NfcmVwb3J0KGZlYXR1cmVfdGFibGUpCgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVwb3J0LmluZGV4WzBdLCAiYWxsX21pc3NpbmciKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVwb3J0LmxvY1sic29tZV9taXNzaW5nIiwgIm1pc3NpbmdfY291bnQiXSwgMikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgdW5pdHRlc3QubWFpbigpCg==', 'tests/test_generate_monitoring_report.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB1bml0dGVzdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gc2NyaXB0cy5nZW5lcmF0ZV9tb25pdG9yaW5nX3JlcG9ydCBpbXBvcnQgZ2VuZXJhdGVfbW9uaXRvcmluZ19yZXBvcnRzLCBwYXJzZV9ncm91cF9jb2x1bW5zCgoKY2xhc3MgR2VuZXJhdGVNb25pdG9yaW5nUmVwb3J0Q2xpVGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHRlc3RfcGFyc2VfZ3JvdXBfY29sdW1ucyhzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocGFyc2VfZ3JvdXBfY29sdW1ucygid2FmZXJfaWQsZXF1aXBtZW50X2lkIiksIFsid2FmZXJfaWQiLCAiZXF1aXBtZW50X2lkIl0pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChwYXJzZV9ncm91cF9jb2x1bW5zKCIiKSwgW10pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChwYXJzZV9ncm91cF9jb2x1bW5zKE5vbmUpLCBbXSkKCiAgICBkZWYgdGVzdF9nZW5lcmF0ZV9tb25pdG9yaW5nX3JlcG9ydHNfd3JpdGVzX2Nzdl9maWxlcyhzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KCkgYXMgdG1wZGlyOgogICAgICAgICAgICB0bXBfcGF0aCA9IFBhdGgodG1wZGlyKQogICAgICAgICAgICBwcmVkaWN0aW9uc19wYXRoID0gdG1wX3BhdGggLyAicHJlZGljdGlvbnMuY3N2IgogICAgICAgICAgICBvdXRwdXRfZGlyID0gdG1wX3BhdGggLyAibW9uaXRvcmluZyIKCiAgICAgICAgICAgIHBkLkRhdGFGcmFtZSgKICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAic2FtcGxlX2lkIjogWyJTMSIsICJTMiIsICJTMyJdLAogICAgICAgICAgICAgICAgICAgICJ3YWZlcl9pZCI6IFsiVzEiLCAiVzEiLCAiVzIiXSwKICAgICAgICAgICAgICAgICAgICAicHJlZGljdGlvbiI6IFsiUGFzcyIsICJGYWlsIiwgIkZhaWwiXSwKICAgICAgICAgICAgICAgICAgICAiZmFpbF9wcm9iYWJpbGl0eSI6IFswLjIsIDAuNywgMC45XSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgKS50b19jc3YocHJlZGljdGlvbnNfcGF0aCwgaW5kZXg9RmFsc2UpCgogICAgICAgICAgICBwYXRocyA9IGdlbmVyYXRlX21vbml0b3JpbmdfcmVwb3J0cygKICAgICAgICAgICAgICAgIHByZWRpY3Rpb25zX3BhdGg9cHJlZGljdGlvbnNfcGF0aCwKICAgICAgICAgICAgICAgIG91dHB1dF9kaXI9b3V0cHV0X2RpciwKICAgICAgICAgICAgICAgIGdyb3VwX2NvbHVtbnM9WyJ3YWZlcl9pZCJdLAogICAgICAgICAgICAgICAgaGlnaF9yaXNrX3RocmVzaG9sZD0wLjUsCiAgICAgICAgICAgICAgICBhbGVydF9yYXRpb190aHJlc2hvbGQ9MC41LAogICAgICAgICAgICAgICAgdG9wX249MiwKICAgICAgICAgICAgKQoKICAgICAgICAgICAgc2VsZi5hc3NlcnRUcnVlKHBhdGhzWyJvdmVyYWxsIl0uZXhpc3RzKCkpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShwYXRoc1sidG9wX3Jpc2siXS5leGlzdHMoKSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRUcnVlKHBhdGhzWyJncm91cCJdLmV4aXN0cygpKQoKICAgICAgICAgICAgdG9wX3Jpc2sgPSBwZC5yZWFkX2NzdihwYXRoc1sidG9wX3Jpc2siXSkKICAgICAgICAgICAgZ3JvdXAgPSBwZC5yZWFkX2NzdihwYXRoc1siZ3JvdXAiXSkKCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwodG9wX3Jpc2tbInNhbXBsZV9pZCJdLnRvbGlzdCgpLCBbIlMzIiwgIlMyIl0pCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0SW4oImFsZXJ0X2ZsYWciLCBncm91cC5jb2x1bW5zKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICB1bml0dGVzdC5tYWluKCkK', 'tests/test_model_registry.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgaW1wb3J0bGliLnV0aWwNCmltcG9ydCB0ZW1wZmlsZQ0KaW1wb3J0IHVuaXR0ZXN0DQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCkpPQkxJQl9BVkFJTEFCTEUgPSBpbXBvcnRsaWIudXRpbC5maW5kX3NwZWMoImpvYmxpYiIpIGlzIG5vdCBOb25lDQoNCmZyb20gc3JjLm1vZGVsX3JlZ2lzdHJ5IGltcG9ydCAoDQogICAgTW9kZWxCdW5kbGUsDQogICAgbG9hZF9tb2RlbF9idW5kbGUsDQogICAgcHJlZGljdF9mcm9tX2J1bmRsZSwNCiAgICBwcmVwYXJlX2ZlYXR1cmVzX2Zvcl9idW5kbGUsDQogICAgc2F2ZV9tb2RlbF9idW5kbGUsDQopDQoNCg0KY2xhc3MgRmFrZUJ1bmRsZU1vZGVsOg0KICAgIGRlZiBwcmVkaWN0X3Byb2JhKHNlbGYsIFgpOg0KICAgICAgICBmZWF0dXJlX3N1bSA9IFguc3VtKGF4aXM9MSkudG9fbnVtcHkoZHR5cGU9ZmxvYXQpDQogICAgICAgIHByb2JhID0gbnAuY2xpcChmZWF0dXJlX3N1bSAvIDEwLjAsIDAuMCwgMS4wKQ0KICAgICAgICByZXR1cm4gbnAuY29sdW1uX3N0YWNrKFsxLjAgLSBwcm9iYSwgcHJvYmFdKQ0KDQoNCmNsYXNzIE1vZGVsUmVnaXN0cnlUZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6DQogICAgZGVmIG1ha2VfYnVuZGxlKHNlbGYpIC0+IE1vZGVsQnVuZGxlOg0KICAgICAgICByZXR1cm4gTW9kZWxCdW5kbGUoDQogICAgICAgICAgICBtb2RlbD1GYWtlQnVuZGxlTW9kZWwoKSwNCiAgICAgICAgICAgIHRocmVzaG9sZD0wLjUsDQogICAgICAgICAgICBmZWF0dXJlX2NvbHVtbnM9WyJmZWF0dXJlX2EiLCAiZmVhdHVyZV9iIl0sDQogICAgICAgICAgICB0YXJnZXRfbWFwcGluZz17MDogIlBhc3MiLCAxOiAiRmFpbCJ9LA0KICAgICAgICAgICAgbW9kZWxfbmFtZT0iZmFrZV9tb2RlbCIsDQogICAgICAgICAgICBtZXRyaWNzPXsiZmFpbF9yZWNhbGwiOiAwLjh9LA0KICAgICAgICApDQoNCiAgICBAdW5pdHRlc3Quc2tpcFVubGVzcyhKT0JMSUJfQVZBSUxBQkxFLCAiam9ibGliIGlzIG5vdCBpbnN0YWxsZWQiKQ0KICAgIGRlZiB0ZXN0X3NhdmVfYW5kX2xvYWRfbW9kZWxfYnVuZGxlKHNlbGYpIC0+IE5vbmU6DQogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KCkgYXMgdG1wZGlyOg0KICAgICAgICAgICAgcGF0aCA9IFBhdGgodG1wZGlyKSAvICJtb2RlbC5qb2JsaWIiDQogICAgICAgICAgICBidW5kbGUgPSBzZWxmLm1ha2VfYnVuZGxlKCkNCg0KICAgICAgICAgICAgc2F2ZWRfcGF0aCA9IHNhdmVfbW9kZWxfYnVuZGxlKGJ1bmRsZSwgcGF0aCkNCiAgICAgICAgICAgIGxvYWRlZCA9IGxvYWRfbW9kZWxfYnVuZGxlKHNhdmVkX3BhdGgpDQoNCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2F2ZWRfcGF0aCwgcGF0aCkNCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobG9hZGVkLnRocmVzaG9sZCwgMC41KQ0KICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsb2FkZWQuZmVhdHVyZV9jb2x1bW5zLCBbImZlYXR1cmVfYSIsICJmZWF0dXJlX2IiXSkNCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobG9hZGVkLm1ldHJpY3NbImZhaWxfcmVjYWxsIl0sIDAuOCkNCg0KICAgIGRlZiB0ZXN0X3ByZXBhcmVfZmVhdHVyZXNfZm9yX2J1bmRsZV9vcmRlcnNfY29sdW1ucyhzZWxmKSAtPiBOb25lOg0KICAgICAgICBidW5kbGUgPSBzZWxmLm1ha2VfYnVuZGxlKCkNCiAgICAgICAgZmVhdHVyZV90YWJsZSA9IHBkLkRhdGFGcmFtZSgNCiAgICAgICAgICAgIHsNCiAgICAgICAgICAgICAgICAic2FtcGxlX2lkIjogWyJTMSIsICJTMiJdLA0KICAgICAgICAgICAgICAgICJmZWF0dXJlX2IiOiBbMi4wLCA0LjBdLA0KICAgICAgICAgICAgICAgICJmZWF0dXJlX2EiOiBbMy4wLCAxLjBdLA0KICAgICAgICAgICAgfQ0KICAgICAgICApDQoNCiAgICAgICAgcHJlcGFyZWQgPSBwcmVwYXJlX2ZlYXR1cmVzX2Zvcl9idW5kbGUoZmVhdHVyZV90YWJsZSwgYnVuZGxlKQ0KDQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocHJlcGFyZWQuY29sdW1ucy50b2xpc3QoKSwgWyJmZWF0dXJlX2EiLCAiZmVhdHVyZV9iIl0pDQoNCiAgICBkZWYgdGVzdF9wcmVwYXJlX2ZlYXR1cmVzX2Zvcl9idW5kbGVfcmVqZWN0c19taXNzaW5nX2NvbHVtbnMoc2VsZikgLT4gTm9uZToNCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToNCiAgICAgICAgICAgIHByZXBhcmVfZmVhdHVyZXNfZm9yX2J1bmRsZShwZC5EYXRhRnJhbWUoeyJmZWF0dXJlX2EiOiBbMS4wXX0pLCBzZWxmLm1ha2VfYnVuZGxlKCkpDQoNCiAgICBkZWYgdGVzdF9wcmVkaWN0X2Zyb21fYnVuZGxlX3VzZXNfdGhyZXNob2xkKHNlbGYpIC0+IE5vbmU6DQogICAgICAgIGJ1bmRsZSA9IHNlbGYubWFrZV9idW5kbGUoKQ0KICAgICAgICBmZWF0dXJlX3RhYmxlID0gcGQuRGF0YUZyYW1lKA0KICAgICAgICAgICAgew0KICAgICAgICAgICAgICAgICJmZWF0dXJlX2EiOiBbMS4wLCA0LjBdLA0KICAgICAgICAgICAgICAgICJmZWF0dXJlX2IiOiBbMS4wLCA0LjBdLA0KICAgICAgICAgICAgfSwNCiAgICAgICAgICAgIGluZGV4PVsibG93IiwgImhpZ2giXSwNCiAgICAgICAgKQ0KDQogICAgICAgIHByZWRpY3Rpb25zID0gcHJlZGljdF9mcm9tX2J1bmRsZShmZWF0dXJlX3RhYmxlLCBidW5kbGUpDQoNCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChwcmVkaWN0aW9uc1sicHJlZGljdGlvbiJdLnRvbGlzdCgpLCBbIlBhc3MiLCAiRmFpbCJdKQ0KICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHByZWRpY3Rpb25zLmluZGV4LnRvbGlzdCgpLCBbImxvdyIsICJoaWdoIl0pDQoNCg0KaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoNCiAgICB1bml0dGVzdC5tYWluKCkNCg==', 'tests/test_monitoring.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHVuaXR0ZXN0CgppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIHNyYy5tb25pdG9yaW5nIGltcG9ydCAoCiAgICBncm91cF9yaXNrX3N1bW1hcnksCiAgICBvdmVyYWxsX3Jpc2tfc3VtbWFyeSwKICAgIHRvcF9yaXNrX3ByZWRpY3Rpb25zLAogICAgdmFsaWRhdGVfcHJlZGljdGlvbl9jb2x1bW5zLAopCgoKY2xhc3MgTW9uaXRvcmluZ1Rlc3RzKHVuaXR0ZXN0LlRlc3RDYXNlKToKICAgIGRlZiBtYWtlX3ByZWRpY3Rpb25zKHNlbGYpIC0+IHBkLkRhdGFGcmFtZToKICAgICAgICByZXR1cm4gcGQuRGF0YUZyYW1lKAogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAic2FtcGxlX2lkIjogWyJTMSIsICJTMiIsICJTMyIsICJTNCJdLAogICAgICAgICAgICAgICAgIndhZmVyX2lkIjogWyJXMSIsICJXMSIsICJXMiIsICJXMiJdLAogICAgICAgICAgICAgICAgImVxdWlwbWVudF9pZCI6IFsiRVExIiwgIkVRMSIsICJFUTIiLCAiRVEyIl0sCiAgICAgICAgICAgICAgICAicHJlZGljdGlvbiI6IFsiUGFzcyIsICJGYWlsIiwgIlBhc3MiLCAiRmFpbCJdLAogICAgICAgICAgICAgICAgImZhaWxfcHJvYmFiaWxpdHkiOiBbMC4xLCAwLjksIDAuMywgMC44XSwKICAgICAgICAgICAgfQogICAgICAgICkKCiAgICBkZWYgdGVzdF9vdmVyYWxsX3Jpc2tfc3VtbWFyeShzZWxmKSAtPiBOb25lOgogICAgICAgIHN1bW1hcnkgPSBvdmVyYWxsX3Jpc2tfc3VtbWFyeShzZWxmLm1ha2VfcHJlZGljdGlvbnMoKSwgaGlnaF9yaXNrX3RocmVzaG9sZD0wLjUpCgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc3VtbWFyeVsibl9wcmVkaWN0aW9ucyJdLCA0KQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc3VtbWFyeVsicHJlZGljdGVkX2ZhaWxfY291bnQiXSwgMikKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHN1bW1hcnlbImhpZ2hfcmlza19jb3VudCJdLCAyKQoKICAgIGRlZiB0ZXN0X2dyb3VwX3Jpc2tfc3VtbWFyeV9zZXRzX2FsZXJ0cyhzZWxmKSAtPiBOb25lOgogICAgICAgIHN1bW1hcnkgPSBncm91cF9yaXNrX3N1bW1hcnkoCiAgICAgICAgICAgIHNlbGYubWFrZV9wcmVkaWN0aW9ucygpLAogICAgICAgICAgICBncm91cF9jb2xzPVsid2FmZXJfaWQiXSwKICAgICAgICAgICAgaGlnaF9yaXNrX3RocmVzaG9sZD0wLjUsCiAgICAgICAgICAgIGFsZXJ0X3JhdGlvX3RocmVzaG9sZD0wLjQsCiAgICAgICAgKQoKICAgICAgICBzZWxmLmFzc2VydEluKCJhbGVydF9mbGFnIiwgc3VtbWFyeS5jb2x1bW5zKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShzdW1tYXJ5WyJhbGVydF9mbGFnIl0uYWxsKCkpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzdW1tYXJ5WyJuX3ByZWRpY3Rpb25zIl0udG9saXN0KCksIFsyLCAyXSkKCiAgICBkZWYgdGVzdF90b3Bfcmlza19wcmVkaWN0aW9uc19zb3J0c19kZXNjZW5kaW5nKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgdG9wID0gdG9wX3Jpc2tfcHJlZGljdGlvbnMoc2VsZi5tYWtlX3ByZWRpY3Rpb25zKCksIHRvcF9uPTIpCgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwodG9wWyJzYW1wbGVfaWQiXS50b2xpc3QoKSwgWyJTMiIsICJTNCJdKQoKICAgIGRlZiB0ZXN0X3ZhbGlkYXRlX3ByZWRpY3Rpb25fY29sdW1uc19yZWplY3RzX21pc3NpbmdfY29sdW1ucyhzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgIHZhbGlkYXRlX3ByZWRpY3Rpb25fY29sdW1ucyhwZC5EYXRhRnJhbWUoeyJwcmVkaWN0aW9uIjogWyJQYXNzIl19KSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgdW5pdHRlc3QubWFpbigpCg==', 'tests/test_predict_with_model.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGltcG9ydGxpYi51dGlsCmltcG9ydCB0ZW1wZmlsZQppbXBvcnQgdW5pdHRlc3QKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSBzY3JpcHRzLnByZWRpY3Rfd2l0aF9tb2RlbCBpbXBvcnQgcGFyc2VfaWRfY29sdW1ucywgcnVuX3ByZWRpY3Rpb24KZnJvbSBzcmMubW9kZWxfcmVnaXN0cnkgaW1wb3J0IE1vZGVsQnVuZGxlLCBzYXZlX21vZGVsX2J1bmRsZQoKCkpPQkxJQl9BVkFJTEFCTEUgPSBpbXBvcnRsaWIudXRpbC5maW5kX3NwZWMoImpvYmxpYiIpIGlzIG5vdCBOb25lCgoKY2xhc3MgRmFrZVByZWRpY3Rpb25Nb2RlbDoKICAgIGRlZiBwcmVkaWN0X3Byb2JhKHNlbGYsIFgpOgogICAgICAgIHByb2JhID0gbnAuY2xpcChYLnN1bShheGlzPTEpLnRvX251bXB5KGR0eXBlPWZsb2F0KSAvIDEwLjAsIDAuMCwgMS4wKQogICAgICAgIHJldHVybiBucC5jb2x1bW5fc3RhY2soWzEuMCAtIHByb2JhLCBwcm9iYV0pCgoKY2xhc3MgUHJlZGljdFdpdGhNb2RlbENsaVRlc3RzKHVuaXR0ZXN0LlRlc3RDYXNlKToKICAgIGRlZiB0ZXN0X3BhcnNlX2lkX2NvbHVtbnMoc2VsZikgLT4gTm9uZToKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHBhcnNlX2lkX2NvbHVtbnMoInNhbXBsZV9pZCwgd2FmZXJfaWQiKSwgWyJzYW1wbGVfaWQiLCAid2FmZXJfaWQiXSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHBhcnNlX2lkX2NvbHVtbnMoIiIpLCBbXSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHBhcnNlX2lkX2NvbHVtbnMoTm9uZSksIFtdKQoKICAgIEB1bml0dGVzdC5za2lwVW5sZXNzKEpPQkxJQl9BVkFJTEFCTEUsICJqb2JsaWIgaXMgbm90IGluc3RhbGxlZCIpCiAgICBkZWYgdGVzdF9ydW5fcHJlZGljdGlvbl93cml0ZXNfcHJlZGljdGlvbl9jc3Yoc2VsZikgLT4gTm9uZToKICAgICAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpIGFzIHRtcGRpcjoKICAgICAgICAgICAgdG1wX3BhdGggPSBQYXRoKHRtcGRpcikKICAgICAgICAgICAgbW9kZWxfcGF0aCA9IHRtcF9wYXRoIC8gIm1vZGVsLmpvYmxpYiIKICAgICAgICAgICAgZmVhdHVyZXNfcGF0aCA9IHRtcF9wYXRoIC8gImZlYXR1cmVzLmNzdiIKICAgICAgICAgICAgb3V0cHV0X3BhdGggPSB0bXBfcGF0aCAvICJwcmVkaWN0aW9ucyIgLyAicHJlZGljdGlvbnMuY3N2IgoKICAgICAgICAgICAgYnVuZGxlID0gTW9kZWxCdW5kbGUoCiAgICAgICAgICAgICAgICBtb2RlbD1GYWtlUHJlZGljdGlvbk1vZGVsKCksCiAgICAgICAgICAgICAgICB0aHJlc2hvbGQ9MC41LAogICAgICAgICAgICAgICAgZmVhdHVyZV9jb2x1bW5zPVsiZmVhdHVyZV9hIiwgImZlYXR1cmVfYiJdLAogICAgICAgICAgICAgICAgdGFyZ2V0X21hcHBpbmc9ezA6ICJQYXNzIiwgMTogIkZhaWwifSwKICAgICAgICAgICAgICAgIG1vZGVsX25hbWU9ImZha2UiLAogICAgICAgICAgICAgICAgbWV0cmljcz17ImZhaWxfcmVjYWxsIjogMS4wfSwKICAgICAgICAgICAgKQogICAgICAgICAgICBzYXZlX21vZGVsX2J1bmRsZShidW5kbGUsIG1vZGVsX3BhdGgpCgogICAgICAgICAgICBwZC5EYXRhRnJhbWUoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgInNhbXBsZV9pZCI6IFsiUzEiLCAiUzIiXSwKICAgICAgICAgICAgICAgICAgICAiZmVhdHVyZV9iIjogWzEuMCwgNC4wXSwKICAgICAgICAgICAgICAgICAgICAiZmVhdHVyZV9hIjogWzEuMCwgNC4wXSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgKS50b19jc3YoZmVhdHVyZXNfcGF0aCwgaW5kZXg9RmFsc2UpCgogICAgICAgICAgICByZXN1bHRfcGF0aCA9IHJ1bl9wcmVkaWN0aW9uKAogICAgICAgICAgICAgICAgbW9kZWxfcGF0aD1tb2RlbF9wYXRoLAogICAgICAgICAgICAgICAgZmVhdHVyZXNfcGF0aD1mZWF0dXJlc19wYXRoLAogICAgICAgICAgICAgICAgb3V0cHV0X3BhdGg9b3V0cHV0X3BhdGgsCiAgICAgICAgICAgICAgICBpZF9jb2x1bW5zPVsic2FtcGxlX2lkIl0sCiAgICAgICAgICAgICkKCiAgICAgICAgICAgIHByZWRpY3Rpb25zID0gcGQucmVhZF9jc3YocmVzdWx0X3BhdGgpCgogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHByZWRpY3Rpb25zWyJzYW1wbGVfaWQiXS50b2xpc3QoKSwgWyJTMSIsICJTMiJdKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHByZWRpY3Rpb25zWyJwcmVkaWN0aW9uIl0udG9saXN0KCksIFsiUGFzcyIsICJGYWlsIl0pCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0SW4oImZhaWxfcHJvYmFiaWxpdHkiLCBwcmVkaWN0aW9ucy5jb2x1bW5zKQoKICAgIEB1bml0dGVzdC5za2lwVW5sZXNzKEpPQkxJQl9BVkFJTEFCTEUsICJqb2JsaWIgaXMgbm90IGluc3RhbGxlZCIpCiAgICBkZWYgdGVzdF9ydW5fcHJlZGljdGlvbl9yZWplY3RzX21pc3NpbmdfaWRfY29sdW1ucyhzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KCkgYXMgdG1wZGlyOgogICAgICAgICAgICB0bXBfcGF0aCA9IFBhdGgodG1wZGlyKQogICAgICAgICAgICBtb2RlbF9wYXRoID0gdG1wX3BhdGggLyAibW9kZWwuam9ibGliIgogICAgICAgICAgICBmZWF0dXJlc19wYXRoID0gdG1wX3BhdGggLyAiZmVhdHVyZXMuY3N2IgoKICAgICAgICAgICAgc2F2ZV9tb2RlbF9idW5kbGUoCiAgICAgICAgICAgICAgICBNb2RlbEJ1bmRsZSgKICAgICAgICAgICAgICAgICAgICBtb2RlbD1GYWtlUHJlZGljdGlvbk1vZGVsKCksCiAgICAgICAgICAgICAgICAgICAgdGhyZXNob2xkPTAuNSwKICAgICAgICAgICAgICAgICAgICBmZWF0dXJlX2NvbHVtbnM9WyJmZWF0dXJlX2EiXSwKICAgICAgICAgICAgICAgICAgICB0YXJnZXRfbWFwcGluZz17MDogIlBhc3MiLCAxOiAiRmFpbCJ9LAogICAgICAgICAgICAgICAgICAgIG1vZGVsX25hbWU9ImZha2UiLAogICAgICAgICAgICAgICAgKSwKICAgICAgICAgICAgICAgIG1vZGVsX3BhdGgsCiAgICAgICAgICAgICkKICAgICAgICAgICAgcGQuRGF0YUZyYW1lKHsiZmVhdHVyZV9hIjogWzEuMF19KS50b19jc3YoZmVhdHVyZXNfcGF0aCwgaW5kZXg9RmFsc2UpCgogICAgICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICAgICAgcnVuX3ByZWRpY3Rpb24oCiAgICAgICAgICAgICAgICAgICAgbW9kZWxfcGF0aD1tb2RlbF9wYXRoLAogICAgICAgICAgICAgICAgICAgIGZlYXR1cmVzX3BhdGg9ZmVhdHVyZXNfcGF0aCwKICAgICAgICAgICAgICAgICAgICBvdXRwdXRfcGF0aD10bXBfcGF0aCAvICJwcmVkaWN0aW9ucy5jc3YiLAogICAgICAgICAgICAgICAgICAgIGlkX2NvbHVtbnM9WyJtaXNzaW5nX2lkIl0sCiAgICAgICAgICAgICAgICApCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHVuaXR0ZXN0Lm1haW4oKQo=', 'tests/test_quality_reports.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHVuaXR0ZXN0CgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSBzcmMucXVhbGl0eV9yZXBvcnRzIGltcG9ydCAoCiAgICBjbGFzc19kaXN0cmlidXRpb24sCiAgICBjb25zdGFudF9mZWF0dXJlcywKICAgIGRhdGFzZXRfb3ZlcnZpZXcsCiAgICBoaWdoX21pc3NpbmdfZmVhdHVyZXMsCiAgICBtaXNzaW5nbmVzc19yZXBvcnQsCiAgICBxdWFsaXR5X3JlcG9ydF9idW5kbGUsCikKZnJvbSBzcmMuc2Vjb21fZGF0YSBpbXBvcnQgU2Vjb21EYXRhc2V0CgoKZGVmIG1ha2VfZGF0YXNldCgpIC0+IFNlY29tRGF0YXNldDoKICAgIGZlYXR1cmVzID0gcGQuRGF0YUZyYW1lKAogICAgICAgIHsKICAgICAgICAgICAgImZlYXR1cmVfMDAwIjogWzEuMCwgMi4wLCAzLjAsIDQuMF0sCiAgICAgICAgICAgICJmZWF0dXJlXzAwMSI6IFtucC5uYW4sIG5wLm5hbiwgMS4wLCBucC5uYW5dLAogICAgICAgICAgICAiZmVhdHVyZV8wMDIiOiBbNy4wLCA3LjAsIDcuMCwgNy4wXSwKICAgICAgICAgICAgImZlYXR1cmVfMDAzIjogWzEuMCwgbnAuaW5mLCAyLjAsIDMuMF0sCiAgICAgICAgfQogICAgKQogICAgbGFiZWxzID0gcGQuU2VyaWVzKFswLCAwLCAxLCAxXSwgbmFtZT0idGFyZ2V0IikKICAgIHRpbWVzdGFtcHMgPSBwZC50b19kYXRldGltZSgKICAgICAgICBwZC5TZXJpZXMoCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICAgICIyMDI2LTAxLTAxIDAwOjAwOjAwIiwKICAgICAgICAgICAgICAgICIyMDI2LTAxLTAyIDAwOjAwOjAwIiwKICAgICAgICAgICAgICAgICIyMDI2LTAxLTAzIDAwOjAwOjAwIiwKICAgICAgICAgICAgICAgIE5vbmUsCiAgICAgICAgICAgIF0KICAgICAgICApCiAgICApLnJlbmFtZSgidGltZXN0YW1wIikKICAgIHJhd19sYWJlbHMgPSBwZC5EYXRhRnJhbWUoezA6IFstMSwgLTEsIDEsIDFdfSkKICAgIHJldHVybiBTZWNvbURhdGFzZXQoZmVhdHVyZXM9ZmVhdHVyZXMsIGxhYmVscz1sYWJlbHMsIHRpbWVzdGFtcHM9dGltZXN0YW1wcywgcmF3X2xhYmVscz1yYXdfbGFiZWxzKQoKCmNsYXNzIFF1YWxpdHlSZXBvcnRUZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBkZWYgdGVzdF9kYXRhc2V0X292ZXJ2aWV3X3JlcG9ydHNfY29yZV9jb3VudHMoc2VsZikgLT4gTm9uZToKICAgICAgICBvdmVydmlldyA9IGRhdGFzZXRfb3ZlcnZpZXcobWFrZV9kYXRhc2V0KCkpCgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwob3ZlcnZpZXdbIm5fc2FtcGxlcyJdLCA0KQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwob3ZlcnZpZXdbIm5fZmVhdHVyZXMiXSwgNCkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKG92ZXJ2aWV3WyJwYXNzX2NvdW50Il0sIDIpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChvdmVydmlld1siZmFpbF9jb3VudCJdLCAyKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwob3ZlcnZpZXdbInRpbWVzdGFtcF9taXNzaW5nIl0sIDEpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChvdmVydmlld1sibWlzc2luZ192YWx1ZXMiXSwgMykKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKG92ZXJ2aWV3WyJpbmZpbml0ZV92YWx1ZXMiXSwgMSkKCiAgICBkZWYgdGVzdF9jbGFzc19kaXN0cmlidXRpb25fcmVwb3J0c19yYXRpb3Moc2VsZikgLT4gTm9uZToKICAgICAgICBkaXN0cmlidXRpb24gPSBjbGFzc19kaXN0cmlidXRpb24obWFrZV9kYXRhc2V0KCkubGFiZWxzKQoKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHNldChkaXN0cmlidXRpb25bImNsYXNzIl0pLCB7IlBhc3MiLCAiRmFpbCJ9KQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZSgoZGlzdHJpYnV0aW9uWyJyYXRpbyJdID09IDAuNSkuYWxsKCkpCgogICAgZGVmIHRlc3RfbWlzc2luZ25lc3NfYW5kX2hpZ2hfbWlzc2luZ19mZWF0dXJlcyhzZWxmKSAtPiBOb25lOgogICAgICAgIGZlYXR1cmVzID0gbWFrZV9kYXRhc2V0KCkuZmVhdHVyZXMKICAgICAgICByZXBvcnQgPSBtaXNzaW5nbmVzc19yZXBvcnQoZmVhdHVyZXMpCgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVwb3J0LmluZGV4WzBdLCAiZmVhdHVyZV8wMDEiKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoaGlnaF9taXNzaW5nX2ZlYXR1cmVzKGZlYXR1cmVzLCB0aHJlc2hvbGQ9MC41KSwgWyJmZWF0dXJlXzAwMSJdKQoKICAgIGRlZiB0ZXN0X2NvbnN0YW50X2ZlYXR1cmVzX2NvdW50c19uYW5fYXNfdmFsdWUoc2VsZikgLT4gTm9uZToKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGNvbnN0YW50X2ZlYXR1cmVzKG1ha2VfZGF0YXNldCgpLmZlYXR1cmVzKSwgWyJmZWF0dXJlXzAwMiJdKQoKICAgIGRlZiB0ZXN0X3F1YWxpdHlfcmVwb3J0X2J1bmRsZV9jb250YWluc19zdGFuZGFyZF9yZXBvcnRzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgYnVuZGxlID0gcXVhbGl0eV9yZXBvcnRfYnVuZGxlKG1ha2VfZGF0YXNldCgpLCBtaXNzaW5nX3RocmVzaG9sZD0wLjUpCgogICAgICAgIHNlbGYuYXNzZXJ0SW4oIm92ZXJ2aWV3IiwgYnVuZGxlKQogICAgICAgIHNlbGYuYXNzZXJ0SW4oImNsYXNzX2Rpc3RyaWJ1dGlvbiIsIGJ1bmRsZSkKICAgICAgICBzZWxmLmFzc2VydEluKCJtaXNzaW5nbmVzcyIsIGJ1bmRsZSkKICAgICAgICBzZWxmLmFzc2VydEluKCJoaWdoX21pc3NpbmdfZmVhdHVyZXMiLCBidW5kbGUpCiAgICAgICAgc2VsZi5hc3NlcnRJbigiY29uc3RhbnRfZmVhdHVyZXMiLCBidW5kbGUpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHVuaXR0ZXN0Lm1haW4oKQo=', 'tests/test_reporting.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB1bml0dGVzdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gc3JjLnJlcG9ydGluZyBpbXBvcnQgYnVpbGRfc3VtbWFyeV9yZXBvcnQsIGRhdGFmcmFtZV90b19tYXJrZG93bgoKCmNsYXNzIFJlcG9ydGluZ1Rlc3RzKHVuaXR0ZXN0LlRlc3RDYXNlKToKICAgIGRlZiB0ZXN0X2RhdGFmcmFtZV90b19tYXJrZG93bihzZWxmKSAtPiBOb25lOgogICAgICAgIGRmID0gcGQuRGF0YUZyYW1lKHsibWV0cmljIjogWyJuX3NhbXBsZXMiXSwgInZhbHVlIjogWzNdfSkKCiAgICAgICAgbWFya2Rvd24gPSBkYXRhZnJhbWVfdG9fbWFya2Rvd24oZGYpCgogICAgICAgIHNlbGYuYXNzZXJ0SW4oInwgbWV0cmljIHwgdmFsdWUgfCIsIG1hcmtkb3duKQogICAgICAgIHNlbGYuYXNzZXJ0SW4oInwgbl9zYW1wbGVzIHwgMyB8IiwgbWFya2Rvd24pCgogICAgZGVmIHRlc3RfYnVpbGRfc3VtbWFyeV9yZXBvcnRfdXNlc19hdmFpbGFibGVfY3N2X2ZpbGVzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB0bXBkaXI6CiAgICAgICAgICAgIHRtcF9wYXRoID0gUGF0aCh0bXBkaXIpCiAgICAgICAgICAgIHJlcG9ydHNfZGlyID0gdG1wX3BhdGggLyAicmVwb3J0cyIKICAgICAgICAgICAgcXVhbGl0eV9kaXIgPSByZXBvcnRzX2RpciAvICJxdWFsaXR5IgogICAgICAgICAgICBtb25pdG9yaW5nX2RpciA9IHJlcG9ydHNfZGlyIC8gIm1vbml0b3JpbmciCiAgICAgICAgICAgIG91dHB1dF9wYXRoID0gcmVwb3J0c19kaXIgLyAic3VtbWFyeV9yZXBvcnQubWQiCiAgICAgICAgICAgIHF1YWxpdHlfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgICAgICAgICAgbW9uaXRvcmluZ19kaXIubWtkaXIocGFyZW50cz1UcnVlKQoKICAgICAgICAgICAgcGQuRGF0YUZyYW1lKHsibWV0cmljIjogWyJuX3NhbXBsZXMiXSwgInZhbHVlIjogWzNdfSkudG9fY3N2KAogICAgICAgICAgICAgICAgcXVhbGl0eV9kaXIgLyAib3ZlcnZpZXcuY3N2IiwKICAgICAgICAgICAgICAgIGluZGV4PUZhbHNlLAogICAgICAgICAgICApCiAgICAgICAgICAgIHBkLkRhdGFGcmFtZSh7ImNsYXNzIjogWyJQYXNzIiwgIkZhaWwiXSwgImNvdW50IjogWzIsIDFdLCAicmF0aW8iOiBbMC42NywgMC4zM119KS50b19jc3YoCiAgICAgICAgICAgICAgICBxdWFsaXR5X2RpciAvICJjbGFzc19kaXN0cmlidXRpb24uY3N2IiwKICAgICAgICAgICAgICAgIGluZGV4PUZhbHNlLAogICAgICAgICAgICApCiAgICAgICAgICAgIHBkLkRhdGFGcmFtZSh7Im1ldHJpYyI6IFsiaGlnaF9yaXNrX2NvdW50Il0sICJ2YWx1ZSI6IFsxXX0pLnRvX2NzdigKICAgICAgICAgICAgICAgIG1vbml0b3JpbmdfZGlyIC8gIm92ZXJhbGxfcmlza19zdW1tYXJ5LmNzdiIsCiAgICAgICAgICAgICAgICBpbmRleD1GYWxzZSwKICAgICAgICAgICAgKQoKICAgICAgICAgICAgcmVwb3J0ID0gYnVpbGRfc3VtbWFyeV9yZXBvcnQoCiAgICAgICAgICAgICAgICByZXBvcnRzX2Rpcj1yZXBvcnRzX2RpciwKICAgICAgICAgICAgICAgIG1vbml0b3JpbmdfZGlyPW1vbml0b3JpbmdfZGlyLAogICAgICAgICAgICAgICAgb3V0cHV0X3BhdGg9b3V0cHV0X3BhdGgsCiAgICAgICAgICAgICkKCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShvdXRwdXRfcGF0aC5leGlzdHMoKSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRJbigiIyBTRUNPTSBNYW51ZmFjdHVyaW5nIEFuYWx5dGljcyBTdW1tYXJ5IiwgcmVwb3J0KQogICAgICAgICAgICBzZWxmLmFzc2VydEluKCIjIyBEYXRhIE92ZXJ2aWV3IiwgcmVwb3J0KQogICAgICAgICAgICBzZWxmLmFzc2VydEluKCIjIyBPdmVyYWxsIFJpc2sgU3VtbWFyeSIsIHJlcG9ydCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgdW5pdHRlc3QubWFpbigpCg==', 'tests/test_run_pipeline.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB1bml0dGVzdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmZyb20gc2NyaXB0cy5ydW5fcGlwZWxpbmUgaW1wb3J0IFBpcGVsaW5lUGF0aHMsIHJ1bl9waXBlbGluZQoKCmNsYXNzIFJ1blBpcGVsaW5lVGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIG1ha2VfcGF0aHMoc2VsZiwgdG1wX3BhdGg6IFBhdGgpIC0+IFBpcGVsaW5lUGF0aHM6CiAgICAgICAgcmV0dXJuIFBpcGVsaW5lUGF0aHMoCiAgICAgICAgICAgIHJhd19kYXRhX2Rpcj10bXBfcGF0aCAvICJyYXciLAogICAgICAgICAgICBzZW5zb3JfcGF0aD10bXBfcGF0aCAvICJzZW5zb3IuY3N2IiwKICAgICAgICAgICAgd2FmZXJfcGF0aD10bXBfcGF0aCAvICJ3YWZlci5jc3YiLAogICAgICAgICAgICBlcXVpcG1lbnRfcGF0aD10bXBfcGF0aCAvICJlcXVpcG1lbnQuY3N2IiwKICAgICAgICAgICAgbW9kZWxfcGF0aD10bXBfcGF0aCAvICJtb2RlbC5qb2JsaWIiLAogICAgICAgICAgICByZXBvcnRzX2Rpcj10bXBfcGF0aCAvICJyZXBvcnRzIiwKICAgICAgICAgICAgZmVhdHVyZXNfcGF0aD10bXBfcGF0aCAvICJmZWF0dXJlcyIgLyAibW9kZWxpbmcuY3N2IiwKICAgICAgICAgICAgcHJlZGljdGlvbnNfcGF0aD10bXBfcGF0aCAvICJwcmVkaWN0aW9ucyIgLyAicHJlZGljdGlvbnMuY3N2IiwKICAgICAgICAgICAgbW9uaXRvcmluZ19kaXI9dG1wX3BhdGggLyAibW9uaXRvcmluZyIsCiAgICAgICAgKQoKICAgIGRlZiB0ZXN0X3J1bl9waXBlbGluZV9jYWxsc19zdGVwc19hbmRfcmV0dXJuc19wYXRocyhzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KCkgYXMgdG1wZGlyOgogICAgICAgICAgICB0bXBfcGF0aCA9IFBhdGgodG1wZGlyKQogICAgICAgICAgICBwYXRocyA9IHNlbGYubWFrZV9wYXRocyh0bXBfcGF0aCkKICAgICAgICAgICAgY2FsbHMgPSBbXQoKICAgICAgICAgICAgZGVmIHF1YWxpdHlfc3RlcCgqKmt3YXJncyk6CiAgICAgICAgICAgICAgICBjYWxscy5hcHBlbmQoKCJxdWFsaXR5Iiwga3dhcmdzKSkKICAgICAgICAgICAgICAgIHJldHVybiBrd2FyZ3NbIm91dHB1dF9kaXIiXQoKICAgICAgICAgICAgZGVmIGFzc2VtYmxlX3N0ZXAoKiprd2FyZ3MpOgogICAgICAgICAgICAgICAgY2FsbHMuYXBwZW5kKCgiYXNzZW1ibGUiLCBrd2FyZ3MpKQogICAgICAgICAgICAgICAgcmV0dXJuICgKICAgICAgICAgICAgICAgICAgICBrd2FyZ3NbIm91dHB1dF9wYXRoIl0sCiAgICAgICAgICAgICAgICAgICAga3dhcmdzWyJyZXBvcnRfZGlyIl0gLyAiZmVhdHVyZV9qb2luX3JlcG9ydC5jc3YiLAogICAgICAgICAgICAgICAgICAgIGt3YXJnc1sicmVwb3J0X2RpciJdIC8gImZlYXR1cmVfbWlzc2luZ25lc3NfcmVwb3J0LmNzdiIsCiAgICAgICAgICAgICAgICApCgogICAgICAgICAgICBkZWYgcHJlZGljdGlvbl9zdGVwKCoqa3dhcmdzKToKICAgICAgICAgICAgICAgIGNhbGxzLmFwcGVuZCgoInByZWRpY3Rpb24iLCBrd2FyZ3MpKQogICAgICAgICAgICAgICAgcmV0dXJuIGt3YXJnc1sib3V0cHV0X3BhdGgiXQoKICAgICAgICAgICAgZGVmIG1vbml0b3Jpbmdfc3RlcCgqKmt3YXJncyk6CiAgICAgICAgICAgICAgICBjYWxscy5hcHBlbmQoKCJtb25pdG9yaW5nIiwga3dhcmdzKSkKICAgICAgICAgICAgICAgIHJldHVybiB7Im92ZXJhbGwiOiBrd2FyZ3NbIm91dHB1dF9kaXIiXSAvICJvdmVyYWxsX3Jpc2tfc3VtbWFyeS5jc3YifQoKICAgICAgICAgICAgcmVzdWx0ID0gcnVuX3BpcGVsaW5lKAogICAgICAgICAgICAgICAgcGF0aHM9cGF0aHMsCiAgICAgICAgICAgICAgICBpZF9jb2x1bW5zPVsic2FtcGxlX2lkIl0sCiAgICAgICAgICAgICAgICBtb25pdG9yaW5nX2dyb3VwX2NvbHVtbnM9WyJ3YWZlcl9pZCJdLAogICAgICAgICAgICAgICAgcXVhbGl0eV9zdGVwPXF1YWxpdHlfc3RlcCwKICAgICAgICAgICAgICAgIGFzc2VtYmxlX3N0ZXA9YXNzZW1ibGVfc3RlcCwKICAgICAgICAgICAgICAgIHByZWRpY3Rpb25fc3RlcD1wcmVkaWN0aW9uX3N0ZXAsCiAgICAgICAgICAgICAgICBtb25pdG9yaW5nX3N0ZXA9bW9uaXRvcmluZ19zdGVwLAogICAgICAgICAgICApCgogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKFtuYW1lIGZvciBuYW1lLCBfIGluIGNhbGxzXSwgWyJxdWFsaXR5IiwgImFzc2VtYmxlIiwgInByZWRpY3Rpb24iLCAibW9uaXRvcmluZyJdKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJlc3VsdC5mZWF0dXJlX3RhYmxlX3BhdGgsIHBhdGhzLmZlYXR1cmVzX3BhdGgpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVzdWx0LnByZWRpY3Rpb25zX3BhdGgsIHBhdGhzLnByZWRpY3Rpb25zX3BhdGgpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0SW4oIm92ZXJhbGwiLCByZXN1bHQubW9uaXRvcmluZ19wYXRocykKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChjYWxsc1syXVsxXVsiaWRfY29sdW1ucyJdLCBbInNhbXBsZV9pZCJdKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGNhbGxzWzNdWzFdWyJncm91cF9jb2x1bW5zIl0sIFsid2FmZXJfaWQiXSkKCiAgICBkZWYgdGVzdF9ydW5fcGlwZWxpbmVfY2FuX3NraXBfcXVhbGl0eV9hbmRfcHJlZGljdGlvbihzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KCkgYXMgdG1wZGlyOgogICAgICAgICAgICB0bXBfcGF0aCA9IFBhdGgodG1wZGlyKQogICAgICAgICAgICBwYXRocyA9IHNlbGYubWFrZV9wYXRocyh0bXBfcGF0aCkKICAgICAgICAgICAgY2FsbHMgPSBbXQoKICAgICAgICAgICAgZGVmIGFzc2VtYmxlX3N0ZXAoKiprd2FyZ3MpOgogICAgICAgICAgICAgICAgY2FsbHMuYXBwZW5kKCgiYXNzZW1ibGUiLCBrd2FyZ3MpKQogICAgICAgICAgICAgICAgcmV0dXJuICgKICAgICAgICAgICAgICAgICAgICBrd2FyZ3NbIm91dHB1dF9wYXRoIl0sCiAgICAgICAgICAgICAgICAgICAga3dhcmdzWyJyZXBvcnRfZGlyIl0gLyAiZmVhdHVyZV9qb2luX3JlcG9ydC5jc3YiLAogICAgICAgICAgICAgICAgICAgIGt3YXJnc1sicmVwb3J0X2RpciJdIC8gImZlYXR1cmVfbWlzc2luZ25lc3NfcmVwb3J0LmNzdiIsCiAgICAgICAgICAgICAgICApCgogICAgICAgICAgICByZXN1bHQgPSBydW5fcGlwZWxpbmUoCiAgICAgICAgICAgICAgICBwYXRocz1wYXRocywKICAgICAgICAgICAgICAgIHNraXBfcXVhbGl0eV9yZXBvcnQ9VHJ1ZSwKICAgICAgICAgICAgICAgIHNraXBfcHJlZGljdGlvbj1UcnVlLAogICAgICAgICAgICAgICAgYXNzZW1ibGVfc3RlcD1hc3NlbWJsZV9zdGVwLAogICAgICAgICAgICApCgogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKFtuYW1lIGZvciBuYW1lLCBfIGluIGNhbGxzXSwgWyJhc3NlbWJsZSJdKQogICAgICAgICAgICBzZWxmLmFzc2VydElzTm9uZShyZXN1bHQucHJlZGljdGlvbnNfcGF0aCkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHQubW9uaXRvcmluZ19wYXRocywge30pCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHVuaXR0ZXN0Lm1haW4oKQo=', 'tests/test_run_quality_report.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB1bml0dGVzdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gc2NyaXB0cy5ydW5fcXVhbGl0eV9yZXBvcnQgaW1wb3J0IHJ1bl9xdWFsaXR5X3JlcG9ydAoKCmNsYXNzIFJ1blF1YWxpdHlSZXBvcnRUZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBkZWYgdGVzdF9ydW5fcXVhbGl0eV9yZXBvcnRfd3JpdGVzX2V4cGVjdGVkX2Nzdl9maWxlcyhzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KCkgYXMgdG1wZGlyOgogICAgICAgICAgICB0bXBfcGF0aCA9IFBhdGgodG1wZGlyKQogICAgICAgICAgICByYXdfZGlyID0gdG1wX3BhdGggLyAicmF3IgogICAgICAgICAgICBvdXRwdXRfZGlyID0gdG1wX3BhdGggLyAicmVwb3J0cyIKICAgICAgICAgICAgcmF3X2Rpci5ta2RpcigpCgogICAgICAgICAgICAocmF3X2RpciAvICJzZWNvbS5kYXRhIikud3JpdGVfdGV4dCgKICAgICAgICAgICAgICAgICIxLjAgMi4wIE5hTlxuIgogICAgICAgICAgICAgICAgIjMuMCBOYU4gNS4wXG4iCiAgICAgICAgICAgICAgICAiNi4wIDcuMCA4LjBcbiIsCiAgICAgICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICAgICApCiAgICAgICAgICAgIChyYXdfZGlyIC8gInNlY29tX2xhYmVscy5kYXRhIikud3JpdGVfdGV4dCgKICAgICAgICAgICAgICAgICctMSAiMTkvMDcvMjAwOCAxMTo1NTowMCJcbicKICAgICAgICAgICAgICAgICcxICIyMC8wNy8yMDA4IDEyOjEwOjAwIlxuJwogICAgICAgICAgICAgICAgJy0xICIyMS8wNy8yMDA4IDEzOjE1OjAwIlxuJywKICAgICAgICAgICAgICAgIGVuY29kaW5nPSJ1dGYtOCIsCiAgICAgICAgICAgICkKCiAgICAgICAgICAgIHJlc3VsdF9kaXIgPSBydW5fcXVhbGl0eV9yZXBvcnQoCiAgICAgICAgICAgICAgICByYXdfZGF0YV9kaXI9cmF3X2RpciwKICAgICAgICAgICAgICAgIG91dHB1dF9kaXI9b3V0cHV0X2RpciwKICAgICAgICAgICAgICAgIGRvd25sb2FkPUZhbHNlLAogICAgICAgICAgICAgICAgbWlzc2luZ190aHJlc2hvbGQ9MC4zNCwKICAgICAgICAgICAgKQoKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHRfZGlyLCBvdXRwdXRfZGlyKQogICAgICAgICAgICBleHBlY3RlZF9maWxlcyA9IHsKICAgICAgICAgICAgICAgICJvdmVydmlldy5jc3YiLAogICAgICAgICAgICAgICAgImNsYXNzX2Rpc3RyaWJ1dGlvbi5jc3YiLAogICAgICAgICAgICAgICAgIm1pc3NpbmduZXNzLmNzdiIsCiAgICAgICAgICAgICAgICAiaGlnaF9taXNzaW5nX2ZlYXR1cmVzLmNzdiIsCiAgICAgICAgICAgICAgICAiY29uc3RhbnRfZmVhdHVyZXMuY3N2IiwKICAgICAgICAgICAgfQogICAgICAgICAgICBzZWxmLmFzc2VydFRydWUoZXhwZWN0ZWRfZmlsZXMuaXNzdWJzZXQoe3BhdGgubmFtZSBmb3IgcGF0aCBpbiBvdXRwdXRfZGlyLml0ZXJkaXIoKX0pKQoKICAgICAgICAgICAgb3ZlcnZpZXcgPSBwZC5yZWFkX2NzdihvdXRwdXRfZGlyIC8gIm92ZXJ2aWV3LmNzdiIpCiAgICAgICAgICAgIG1pc3NpbmduZXNzID0gcGQucmVhZF9jc3Yob3V0cHV0X2RpciAvICJtaXNzaW5nbmVzcy5jc3YiKQoKICAgICAgICAgICAgc2VsZi5hc3NlcnRJbigibl9zYW1wbGVzIiwgb3ZlcnZpZXdbIm1ldHJpYyJdLnRvbGlzdCgpKQogICAgICAgICAgICBzZWxmLmFzc2VydEluKCJmZWF0dXJlXzAwMiIsIG1pc3NpbmduZXNzWyJmZWF0dXJlIl0udG9saXN0KCkpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHVuaXR0ZXN0Lm1haW4oKQo=', 'tests/test_secom_data.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB1bml0dGVzdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gc3JjLnNlY29tX2RhdGEgaW1wb3J0ICgKICAgIGZpcnN0X2V4aXN0aW5nX3BhdGgsCiAgICBsb2FkX3NlY29tX2RhdGEsCiAgICByZWFkX29wdGlvbmFsX3RhYmxlLAopCgoKY2xhc3MgU2Vjb21EYXRhTG9hZGVyVGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHRlc3RfbG9hZF9zZWNvbV9kYXRhX21hcHNfbGFiZWxzX2FuZF90aW1lc3RhbXBzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB0bXBkaXI6CiAgICAgICAgICAgIHRtcF9wYXRoID0gUGF0aCh0bXBkaXIpCiAgICAgICAgICAgIGZlYXR1cmVfcGF0aCA9IHRtcF9wYXRoIC8gInNlY29tLmRhdGEiCiAgICAgICAgICAgIGxhYmVsX3BhdGggPSB0bXBfcGF0aCAvICJzZWNvbV9sYWJlbHMuZGF0YSIKCiAgICAgICAgICAgIGZlYXR1cmVfcGF0aC53cml0ZV90ZXh0KAogICAgICAgICAgICAgICAgIjEuMCAyLjAgTmFOXG4iCiAgICAgICAgICAgICAgICAiNC4wIE5hTiA2LjBcbiIsCiAgICAgICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICAgICApCiAgICAgICAgICAgIGxhYmVsX3BhdGgud3JpdGVfdGV4dCgKICAgICAgICAgICAgICAgICctMSAiMTkvMDcvMjAwOCAxMTo1NTowMCJcbicKICAgICAgICAgICAgICAgICcxICIyMC8wNy8yMDA4IDEyOjEwOjAwIlxuJywKICAgICAgICAgICAgICAgIGVuY29kaW5nPSJ1dGYtOCIsCiAgICAgICAgICAgICkKCiAgICAgICAgICAgIGRhdGFzZXQgPSBsb2FkX3NlY29tX2RhdGEoZmVhdHVyZV9wYXRoLCBsYWJlbF9wYXRoKQoKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChkYXRhc2V0LmZlYXR1cmVzLnNoYXBlLCAoMiwgMykpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZGF0YXNldC5mZWF0dXJlcy5jb2x1bW5zLnRvbGlzdCgpLCBbImZlYXR1cmVfMDAwIiwgImZlYXR1cmVfMDAxIiwgImZlYXR1cmVfMDAyIl0pCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZGF0YXNldC5sYWJlbHMudG9saXN0KCksIFswLCAxXSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChkYXRhc2V0LnRpbWVzdGFtcHMuaXNuYSgpLnN1bSgpLCAwKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGRhdGFzZXQudGltZXN0YW1wcy5kdC5kYXkudG9saXN0KCksIFsxOSwgMjBdKQoKICAgIGRlZiB0ZXN0X2xvYWRfc2Vjb21fZGF0YV9yZWplY3RzX3Jvd19taXNtYXRjaChzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KCkgYXMgdG1wZGlyOgogICAgICAgICAgICB0bXBfcGF0aCA9IFBhdGgodG1wZGlyKQogICAgICAgICAgICBmZWF0dXJlX3BhdGggPSB0bXBfcGF0aCAvICJzZWNvbS5kYXRhIgogICAgICAgICAgICBsYWJlbF9wYXRoID0gdG1wX3BhdGggLyAic2Vjb21fbGFiZWxzLmRhdGEiCgogICAgICAgICAgICBmZWF0dXJlX3BhdGgud3JpdGVfdGV4dCgiMS4wIDIuMFxuMy4wIDQuMFxuIiwgZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICAgICAgbGFiZWxfcGF0aC53cml0ZV90ZXh0KCctMSAiMTkvMDcvMjAwOCAxMTo1NTowMCJcbicsIGVuY29kaW5nPSJ1dGYtOCIpCgogICAgICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICAgICAgbG9hZF9zZWNvbV9kYXRhKGZlYXR1cmVfcGF0aCwgbGFiZWxfcGF0aCkKCiAgICBkZWYgdGVzdF9yZWFkX29wdGlvbmFsX3RhYmxlX3N1cHBvcnRzX2Fic2VudF9jc3ZfYW5kX3RzdihzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KCkgYXMgdG1wZGlyOgogICAgICAgICAgICB0bXBfcGF0aCA9IFBhdGgodG1wZGlyKQogICAgICAgICAgICBjc3ZfcGF0aCA9IHRtcF9wYXRoIC8gIndhZmVyX2luc3BlY3Rpb24uY3N2IgogICAgICAgICAgICB0c3ZfcGF0aCA9IHRtcF9wYXRoIC8gImVxdWlwbWVudF9ldmVudHMudHN2IgoKICAgICAgICAgICAgc2VsZi5hc3NlcnRJc05vbmUocmVhZF9vcHRpb25hbF90YWJsZSh0bXBfcGF0aCAvICJtaXNzaW5nLmNzdiIpKQoKICAgICAgICAgICAgY3N2X3BhdGgud3JpdGVfdGV4dCgid2FmZXJfaWQseCx5LGRlZmVjdF90eXBlXG5XMSwxLDIsY2VudGVyXG4iLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgICAgICB0c3ZfcGF0aC53cml0ZV90ZXh0KCJlcXVpcG1lbnRfaWRcdHRpbWVzdGFtcFx0ZXZlbnRfdHlwZVxuRVExXHQyMDI2LTAxLTAxXHRhbGFybVxuIiwgZW5jb2Rpbmc9InV0Zi04IikKCiAgICAgICAgICAgIGNzdl9kZiA9IHJlYWRfb3B0aW9uYWxfdGFibGUoY3N2X3BhdGgpCiAgICAgICAgICAgIHRzdl9kZiA9IHJlYWRfb3B0aW9uYWxfdGFibGUodHN2X3BhdGgpCgogICAgICAgICAgICBzZWxmLmFzc2VydElzSW5zdGFuY2UoY3N2X2RmLCBwZC5EYXRhRnJhbWUpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0SXNJbnN0YW5jZSh0c3ZfZGYsIHBkLkRhdGFGcmFtZSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChjc3ZfZGYubG9jWzAsICJkZWZlY3RfdHlwZSJdLCAiY2VudGVyIikKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbCh0c3ZfZGYubG9jWzAsICJlcXVpcG1lbnRfaWQiXSwgIkVRMSIpCgogICAgZGVmIHRlc3RfZmlyc3RfZXhpc3RpbmdfcGF0aF9yZXR1cm5zX2ZpcnN0X21hdGNoKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB0bXBkaXI6CiAgICAgICAgICAgIHRtcF9wYXRoID0gUGF0aCh0bXBkaXIpCiAgICAgICAgICAgIGZpcnN0ID0gdG1wX3BhdGggLyAibWlzc2luZy5jc3YiCiAgICAgICAgICAgIHNlY29uZCA9IHRtcF9wYXRoIC8gInByZXNlbnQuY3N2IgogICAgICAgICAgICBzZWNvbmQud3JpdGVfdGV4dCgiYVxuMVxuIiwgZW5jb2Rpbmc9InV0Zi04IikKCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZmlyc3RfZXhpc3RpbmdfcGF0aChbZmlyc3QsIHNlY29uZF0pLCBzZWNvbmQpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHVuaXR0ZXN0Lm1haW4oKQo=', 'tests/test_secom_modeling.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHVuaXR0ZXN0CgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSBzcmMuc2Vjb21fbW9kZWxpbmcgaW1wb3J0ICgKICAgIEhpZ2hNaXNzaW5nRmVhdHVyZURyb3BwZXIsCiAgICBnZXRfcG9zaXRpdmVfcHJvYmEsCiAgICBwcmVkaWN0X3dpdGhfdGhyZXNob2xkLAopCgoKY2xhc3MgRmFrZVByb2JhTW9kZWw6CiAgICBkZWYgcHJlZGljdF9wcm9iYShzZWxmLCBYKToKICAgICAgICByZXR1cm4gbnAuYXJyYXkoCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICAgIFswLjgsIDAuMl0sCiAgICAgICAgICAgICAgICBbMC40LCAwLjZdLAogICAgICAgICAgICAgICAgWzAuMSwgMC45XSwKICAgICAgICAgICAgXQogICAgICAgIClbOiBsZW4oWCldCgoKY2xhc3MgRmFrZURlY2lzaW9uTW9kZWw6CiAgICBkZWYgX19pbml0X18oc2VsZiwgc2NvcmVzKToKICAgICAgICBzZWxmLnNjb3JlcyA9IG5wLmFzYXJyYXkoc2NvcmVzLCBkdHlwZT1mbG9hdCkKCiAgICBkZWYgZGVjaXNpb25fZnVuY3Rpb24oc2VsZiwgWCk6CiAgICAgICAgcmV0dXJuIHNlbGYuc2NvcmVzWzogbGVuKFgpXQoKCmNsYXNzIFNlY29tTW9kZWxpbmdUZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBkZWYgdGVzdF9oaWdoX21pc3NpbmdfZmVhdHVyZV9kcm9wcGVyX2Ryb3BzX2NvbHVtbnNfYXRfdGhyZXNob2xkKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgWCA9IHBkLkRhdGFGcmFtZSgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgImtlZXBfZnVsbCI6IFsxLjAsIDIuMCwgMy4wLCA0LjBdLAogICAgICAgICAgICAgICAgImtlZXBfYmVsb3dfdGhyZXNob2xkIjogWzEuMCwgbnAubmFuLCAzLjAsIDQuMF0sCiAgICAgICAgICAgICAgICAiZHJvcF9hdF90aHJlc2hvbGQiOiBbMS4wLCBucC5uYW4sIDMuMCwgbnAubmFuXSwKICAgICAgICAgICAgICAgICJkcm9wX2Fib3ZlX3RocmVzaG9sZCI6IFtucC5uYW4sIG5wLm5hbiwgMy4wLCBucC5uYW5dLAogICAgICAgICAgICB9CiAgICAgICAgKQoKICAgICAgICBkcm9wcGVyID0gSGlnaE1pc3NpbmdGZWF0dXJlRHJvcHBlcih0aHJlc2hvbGQ9MC41KQogICAgICAgIHRyYW5zZm9ybWVkID0gZHJvcHBlci5maXRfdHJhbnNmb3JtKFgpCgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwodHJhbnNmb3JtZWQuY29sdW1ucy50b2xpc3QoKSwgWyJrZWVwX2Z1bGwiLCAia2VlcF9iZWxvd190aHJlc2hvbGQiXSkKCiAgICBkZWYgdGVzdF9oaWdoX21pc3NpbmdfZmVhdHVyZV9kcm9wcGVyX3JlcXVpcmVzX2ZpdChzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoUnVudGltZUVycm9yKToKICAgICAgICAgICAgSGlnaE1pc3NpbmdGZWF0dXJlRHJvcHBlcigpLnRyYW5zZm9ybShwZC5EYXRhRnJhbWUoeyJhIjogWzFdfSkpCgogICAgZGVmIHRlc3RfZ2V0X3Bvc2l0aXZlX3Byb2JhX3VzZXNfcHJlZGljdF9wcm9iYShzZWxmKSAtPiBOb25lOgogICAgICAgIFggPSBwZC5EYXRhRnJhbWUoeyJmZWF0dXJlIjogWzEsIDIsIDNdfSkKCiAgICAgICAgcHJvYmEgPSBnZXRfcG9zaXRpdmVfcHJvYmEoRmFrZVByb2JhTW9kZWwoKSwgWCkKCiAgICAgICAgbnAudGVzdGluZy5hc3NlcnRfYWxsY2xvc2UocHJvYmEsIG5wLmFycmF5KFswLjIsIDAuNiwgMC45XSkpCgogICAgZGVmIHRlc3RfZ2V0X3Bvc2l0aXZlX3Byb2JhX3NjYWxlc19kZWNpc2lvbl9mdW5jdGlvbihzZWxmKSAtPiBOb25lOgogICAgICAgIFggPSBwZC5EYXRhRnJhbWUoeyJmZWF0dXJlIjogWzEsIDIsIDNdfSkKCiAgICAgICAgcHJvYmEgPSBnZXRfcG9zaXRpdmVfcHJvYmEoRmFrZURlY2lzaW9uTW9kZWwoWzEwLCAyMCwgMzBdKSwgWCkKCiAgICAgICAgbnAudGVzdGluZy5hc3NlcnRfYWxsY2xvc2UocHJvYmEsIG5wLmFycmF5KFswLjAsIDAuNSwgMS4wXSkpCgogICAgZGVmIHRlc3RfZ2V0X3Bvc2l0aXZlX3Byb2JhX2hhbmRsZXNfY29uc3RhbnRfZGVjaXNpb25fc2NvcmVzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgWCA9IHBkLkRhdGFGcmFtZSh7ImZlYXR1cmUiOiBbMSwgMiwgM119KQoKICAgICAgICBwcm9iYSA9IGdldF9wb3NpdGl2ZV9wcm9iYShGYWtlRGVjaXNpb25Nb2RlbChbNywgNywgN10pLCBYKQoKICAgICAgICBucC50ZXN0aW5nLmFzc2VydF9hbGxjbG9zZShwcm9iYSwgbnAuYXJyYXkoWzAuNSwgMC41LCAwLjVdKSkKCiAgICBkZWYgdGVzdF9wcmVkaWN0X3dpdGhfdGhyZXNob2xkX3JldHVybnNfZnJhbWVfcmVhZHlfcmVzdWx0KHNlbGYpIC0+IE5vbmU6CiAgICAgICAgWCA9IHBkLkRhdGFGcmFtZSh7ImZlYXR1cmUiOiBbMSwgMiwgM119LCBpbmRleD1bImEiLCAiYiIsICJjIl0pCgogICAgICAgIHJlc3VsdCA9IHByZWRpY3Rfd2l0aF90aHJlc2hvbGQoRmFrZVByb2JhTW9kZWwoKSwgWCwgdGhyZXNob2xkPTAuNikudG9fZnJhbWUoaW5kZXg9WC5pbmRleCkKCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHRbInByZWRpY3Rpb24iXS50b2xpc3QoKSwgWyJQYXNzIiwgIkZhaWwiLCAiRmFpbCJdKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVzdWx0LmluZGV4LnRvbGlzdCgpLCBbImEiLCAiYiIsICJjIl0pCiAgICAgICAgbnAudGVzdGluZy5hc3NlcnRfYWxsY2xvc2UocmVzdWx0WyJkZWNpc2lvbl90aHJlc2hvbGQiXS50b19udW1weSgpLCBucC5hcnJheShbMC42LCAwLjYsIDAuNl0pKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICB1bml0dGVzdC5tYWluKCkK', 'tests/test_secom_training.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHVuaXR0ZXN0CgppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIHNyYy5zZWNvbV90cmFpbmluZyBpbXBvcnQgcGFyc2VfdGhyZXNob2xkcywgcmFua19yZXN1bHRzLCBzZWxlY3RfdG9wX3Jlc3VsdAoKCmNsYXNzIFNlY29tVHJhaW5pbmdUZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBkZWYgdGVzdF9wYXJzZV90aHJlc2hvbGRzX2FjY2VwdHNfY3N2X2FuZF9pdGVyYWJsZShzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocGFyc2VfdGhyZXNob2xkcygiMC4yLCAwLjUsMC44IiksIFswLjIsIDAuNSwgMC44XSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHBhcnNlX3RocmVzaG9sZHMoWzAuMSwgMC45XSksIFswLjEsIDAuOV0pCgogICAgZGVmIHRlc3RfcGFyc2VfdGhyZXNob2xkc19yZWplY3RzX2ludmFsaWRfdmFsdWVzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgcGFyc2VfdGhyZXNob2xkcygiMC4xLDEuMiIpCgogICAgZGVmIHRlc3RfcmFua19yZXN1bHRzX3NvcnRzX2J5X3ByaW9yaXR5X21ldHJpY3Moc2VsZikgLT4gTm9uZToKICAgICAgICByZXN1bHRzID0gcGQuRGF0YUZyYW1lKAogICAgICAgICAgICBbCiAgICAgICAgICAgICAgICB7Im1vZGVsIjogIkEiLCAiZmFpbF9yZWNhbGwiOiAwLjgsICJmYWlsX2YxIjogMC4yLCAicHJfYXVjIjogMC40fSwKICAgICAgICAgICAgICAgIHsibW9kZWwiOiAiQiIsICJmYWlsX3JlY2FsbCI6IDAuOCwgImZhaWxfZjEiOiAwLjUsICJwcl9hdWMiOiAwLjN9LAogICAgICAgICAgICAgICAgeyJtb2RlbCI6ICJDIiwgImZhaWxfcmVjYWxsIjogMC42LCAiZmFpbF9mMSI6IDAuOSwgInByX2F1YyI6IDAuOX0sCiAgICAgICAgICAgIF0KICAgICAgICApCgogICAgICAgIHJhbmtlZCA9IHJhbmtfcmVzdWx0cyhyZXN1bHRzKQoKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJhbmtlZFsibW9kZWwiXS50b2xpc3QoKSwgWyJCIiwgIkEiLCAiQyJdKQoKICAgIGRlZiB0ZXN0X3NlbGVjdF90b3BfcmVzdWx0X3JldHVybnNfYmVzdF9yb3coc2VsZikgLT4gTm9uZToKICAgICAgICByZXN1bHRzID0gcGQuRGF0YUZyYW1lKAogICAgICAgICAgICBbCiAgICAgICAgICAgICAgICB7Im1vZGVsIjogImJhc2VsaW5lIiwgImZhaWxfcmVjYWxsIjogMC4wLCAiZmFpbF9mMSI6IDAuMCwgInByX2F1YyI6IDAuMX0sCiAgICAgICAgICAgICAgICB7Im1vZGVsIjogImJhbGFuY2VkIiwgImZhaWxfcmVjYWxsIjogMC43LCAiZmFpbF9mMSI6IDAuNCwgInByX2F1YyI6IDAuM30sCiAgICAgICAgICAgIF0KICAgICAgICApCgogICAgICAgIHNlbGVjdGVkID0gc2VsZWN0X3RvcF9yZXN1bHQocmVzdWx0cykKCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzZWxlY3RlZC5uYW1lLCAiYmFsYW5jZWQiKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2VsZWN0ZWQucmFua2luZ19jb2x1bW5zLCAoImZhaWxfcmVjYWxsIiwgImZhaWxfZjEiLCAicHJfYXVjIikpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHVuaXR0ZXN0Lm1haW4oKQo=', 'tests/test_train_secom_model.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB1bml0dGVzdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIHNjcmlwdHMudHJhaW5fc2Vjb21fbW9kZWwgaW1wb3J0IHJlc29sdmVfc2Vjb21fcGF0aHMsIHdyaXRlX3Rlc3RfcHJlZGljdGlvbnMKCgpjbGFzcyBGYWtlTW9kZWw6CiAgICBkZWYgcHJlZGljdF9wcm9iYShzZWxmLCBYKToKICAgICAgICBwcm9iYWJpbGl0aWVzID0gbnAuYXJyYXkoWzAuMiwgMC44XSlbOiBsZW4oWCldCiAgICAgICAgcmV0dXJuIG5wLmNvbHVtbl9zdGFjayhbMS4wIC0gcHJvYmFiaWxpdGllcywgcHJvYmFiaWxpdGllc10pCgoKY2xhc3MgVHJhaW5TZWNvbU1vZGVsVGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHRlc3RfcmVzb2x2ZV9zZWNvbV9wYXRoc191c2VzX2RlZmF1bHRzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB0bXBkaXI6CiAgICAgICAgICAgIHJhd19kYXRhX2RpciA9IFBhdGgodG1wZGlyKQogICAgICAgICAgICBmZWF0dXJlX3BhdGggPSByYXdfZGF0YV9kaXIgLyAic2Vjb20uZGF0YSIKICAgICAgICAgICAgbGFiZWxfcGF0aCA9IHJhd19kYXRhX2RpciAvICJzZWNvbV9sYWJlbHMuZGF0YSIKICAgICAgICAgICAgZmVhdHVyZV9wYXRoLndyaXRlX3RleHQoIjEgMlxuIiwgZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICAgICAgbGFiZWxfcGF0aC53cml0ZV90ZXh0KCItMSBub3dcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCgogICAgICAgICAgICByZXNvbHZlZCA9IHJlc29sdmVfc2Vjb21fcGF0aHMocmF3X2RhdGFfZGlyPXJhd19kYXRhX2RpcikKCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocmVzb2x2ZWQsIChmZWF0dXJlX3BhdGgsIGxhYmVsX3BhdGgpKQoKICAgIGRlZiB0ZXN0X3Jlc29sdmVfc2Vjb21fcGF0aHNfcmVqZWN0c19taXNzaW5nX2ZpbGVzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB0bXBkaXI6CiAgICAgICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoRmlsZU5vdEZvdW5kRXJyb3IpOgogICAgICAgICAgICAgICAgcmVzb2x2ZV9zZWNvbV9wYXRocyhyYXdfZGF0YV9kaXI9UGF0aCh0bXBkaXIpKQoKICAgIGRlZiB0ZXN0X3dyaXRlX3Rlc3RfcHJlZGljdGlvbnNfaW5jbHVkZXNfbGFiZWxzX2FuZF90aHJlc2hvbGQoc2VsZikgLT4gTm9uZToKICAgICAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpIGFzIHRtcGRpcjoKICAgICAgICAgICAgb3V0cHV0X3BhdGggPSBQYXRoKHRtcGRpcikgLyAicHJlZGljdGlvbnMuY3N2IgogICAgICAgICAgICBYX3Rlc3QgPSBwZC5EYXRhRnJhbWUoeyJmZWF0dXJlXzAwMCI6IFsxLjAsIDIuMF19LCBpbmRleD1bMTAsIDExXSkKICAgICAgICAgICAgeV90ZXN0ID0gcGQuU2VyaWVzKFswLCAxXSwgaW5kZXg9WF90ZXN0LmluZGV4KQogICAgICAgICAgICB0aW1lc3RhbXBzID0gcGQuU2VyaWVzKHBkLnRvX2RhdGV0aW1lKFsiMjAyNi0wMS0wMSIsICIyMDI2LTAxLTAyIl0pLCBpbmRleD1YX3Rlc3QuaW5kZXgpCgogICAgICAgICAgICB3cml0dGVuX3BhdGggPSB3cml0ZV90ZXN0X3ByZWRpY3Rpb25zKAogICAgICAgICAgICAgICAgb3V0cHV0X3BhdGg9b3V0cHV0X3BhdGgsCiAgICAgICAgICAgICAgICBtb2RlbD1GYWtlTW9kZWwoKSwKICAgICAgICAgICAgICAgIFhfdGVzdD1YX3Rlc3QsCiAgICAgICAgICAgICAgICB5X3Rlc3Q9eV90ZXN0LAogICAgICAgICAgICAgICAgdGltZXN0YW1wcz10aW1lc3RhbXBzLAogICAgICAgICAgICAgICAgdGhyZXNob2xkPTAuNSwKICAgICAgICAgICAgKQoKICAgICAgICAgICAgcHJlZGljdGlvbnMgPSBwZC5yZWFkX2Nzdih3cml0dGVuX3BhdGgpCgogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHdyaXR0ZW5fcGF0aCwgb3V0cHV0X3BhdGgpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocHJlZGljdGlvbnNbInByZWRpY3Rpb24iXS50b2xpc3QoKSwgWyJQYXNzIiwgIkZhaWwiXSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChwcmVkaWN0aW9uc1siYWN0dWFsX25hbWUiXS50b2xpc3QoKSwgWyJQYXNzIiwgIkZhaWwiXSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChwcmVkaWN0aW9uc1siZGVjaXNpb25fdGhyZXNob2xkIl0udG9saXN0KCksIFswLjUsIDAuNV0pCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHVuaXR0ZXN0Lm1haW4oKQo=', 'tests/test_wafer_features.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHVuaXR0ZXN0CgppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIHNyYy53YWZlcl9mZWF0dXJlcyBpbXBvcnQgKAogICAgaGV1cmlzdGljX3dhZmVyX3BhdHRlcm5fbGFiZWwsCiAgICB2YWxpZGF0ZV93YWZlcl9jb2x1bW5zLAogICAgd2FmZXJfZGVmZWN0X2ZlYXR1cmVzLAopCgoKY2xhc3MgV2FmZXJGZWF0dXJlVGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHRlc3Rfd2FmZXJfZGVmZWN0X2ZlYXR1cmVzX2V4dHJhY3RzX3NwYXRpYWxfcmF0aW9zKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgd2FmZXJfZGYgPSBwZC5EYXRhRnJhbWUoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJ3YWZlcl9pZCI6IFsiVzEiLCAiVzEiLCAiVzEiLCAiVzIiLCAiVzIiLCAiVzIiLCAiVzIiXSwKICAgICAgICAgICAgICAgICJ4IjogWzAsIDEsIC0xLCA5LCAxMCwgLTEwLCAtOV0sCiAgICAgICAgICAgICAgICAieSI6IFswLCAxLCAtMSwgOSwgMTAsIC0xMCwgLTldLAogICAgICAgICAgICAgICAgImRlZmVjdF90eXBlIjogWyJkb3QiLCAiZG90IiwgInNjcmF0Y2giLCAiZWRnZSIsICJlZGdlIiwgImVkZ2UiLCAiZG90Il0sCiAgICAgICAgICAgIH0KICAgICAgICApCgogICAgICAgIGZlYXR1cmVzID0gd2FmZXJfZGVmZWN0X2ZlYXR1cmVzKHdhZmVyX2RmKQoKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGZlYXR1cmVzLnNoYXBlWzBdLCAyKQogICAgICAgIHNlbGYuYXNzZXJ0SW4oInpvbmVfcmF0aW9fY2VudGVyIiwgZmVhdHVyZXMuY29sdW1ucykKICAgICAgICBzZWxmLmFzc2VydEluKCJ6b25lX3JhdGlvX2VkZ2UiLCBmZWF0dXJlcy5jb2x1bW5zKQogICAgICAgIHNlbGYuYXNzZXJ0SW4oImRlZmVjdF90eXBlX3JhdGlvX2RvdCIsIGZlYXR1cmVzLmNvbHVtbnMpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChmZWF0dXJlcy5sb2NbZmVhdHVyZXNbIndhZmVyX2lkIl0gPT0gIlcxIiwgImRlZmVjdF9jb3VudCJdLmlsb2NbMF0sIDMpCgogICAgZGVmIHRlc3RfaGV1cmlzdGljX3BhdHRlcm5fbGFiZWxfdXNlc19lbmdpbmVlcmVkX2ZlYXR1cmVzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgZmVhdHVyZXMgPSBwZC5EYXRhRnJhbWUoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJ6b25lX3JhdGlvX2NlbnRlciI6IFswLjcsIDAuMSwgMC4yLCAwLjNdLAogICAgICAgICAgICAgICAgInpvbmVfcmF0aW9fZWRnZSI6IFswLjEsIDAuOCwgMC4yLCAwLjJdLAogICAgICAgICAgICAgICAgInF1YWRyYW50X2ltYmFsYW5jZSI6IFswLjEsIDAuMSwgMC43LCAwLjFdLAogICAgICAgICAgICB9CiAgICAgICAgKQoKICAgICAgICBsYWJlbHMgPSBoZXVyaXN0aWNfd2FmZXJfcGF0dGVybl9sYWJlbChmZWF0dXJlcykKCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsYWJlbHMudG9saXN0KCksIFsiY2VudGVyIiwgImVkZ2UiLCAibG9jYWxpemVkIiwgIm1peGVkIl0pCgogICAgZGVmIHRlc3RfdmFsaWRhdGVfd2FmZXJfY29sdW1uc19yZWplY3RzX21pc3NpbmdfY29sdW1ucyhzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgIHZhbGlkYXRlX3dhZmVyX2NvbHVtbnMocGQuRGF0YUZyYW1lKHsid2FmZXJfaWQiOiBbIlcxIl0sICJ4IjogWzFdfSkpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHVuaXR0ZXN0Lm1haW4oKQo='}

project_root = Path.cwd()
for relative_path, encoded in PROJECT_FILES_B64.items():
    target = project_root / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(base64.b64decode(encoded))

print(f"Created or updated {len(PROJECT_FILES_B64)} project files in Colab runtime.")
print(f"Project root: {project_root}")


### 33.1 Verify Synced Project

Run this cell after syncing. If Colab reports missing packages, run `%pip install -r requirements.txt` first and retry.


In [ ]:
import subprocess
import sys

commands = [
    [sys.executable, "validate_notebook.py"],
    [sys.executable, "-m", "unittest", "discover", "-s", "tests"],
]

for command in commands:
    print("$", " ".join(command))
    subprocess.run(command, check=True)
